# Week 1: pricing in the developed market

I am Manuel Rodríguez, an actuarial student and individual participant in the IAA AI Hackathon 2026. In this notebook I explain and reconstruct my Week 1 risk-premium model, developed with AI assistance.

I estimate the expected ultimate claim cost for each 2025 policy period in motor, buildings and contents. The target is net of excess and recoveries and excludes declined claims. It is a period risk premium, not a commercial premium. Claims valued at the end of 2024 do not reveal the full ultimate cost, so the development assumptions and their sensitivity matter.


## Notebook guide

| Purpose | What I reconstruct |
| --- | --- |
| Selected recipe | S41, my Week 1 submission |
| Required inputs | 11 developed-market Parquet files from Part 1 |
| Final exports | `submission_week1_S41.csv` |
| External model or repository download | None; source and frozen configuration are embedded |

For a first reading, I follow the [method](#approach-and-limitations), [execution plan](#inspect-the-dependency-graph) and [output checks](#apply-the-s41-level-factors-and-check-the-output). The [embedded source package](#embedded-source-package) preserves the individual programs for inspection. Its source cells are marked as hidden in JupyterLab's presentation metadata; I can expand them. GitHub may display them in full.

For execution, I attach the data, check [dependencies and storage](#dependencies-and-workspace) and run cells in order. The later [recipe execution](#materialize-and-run-the-frozen-recipe) trains models and can take hours. Reading the notebook does not require starting that execution.

My earlier resumed reconstruction recovered the historical S41 CSV with 160 intermediate differences, including three unexplained feature-table fingerprints. A new execution of the preceding frozen edition is recorded separately in the [execution index](../registro/ejecucion_notebooks_actual.json); I do not claim its final S41 check before it finishes. This English edition uses neutral runtime names and revised comments/docstrings. Its 123 embedded sources across both notebooks passed normalized executable-structure checks; the data and path checks are documented in the [source review](../registro/documentacion_codigo_actual.json). Those checks are separate from another full Run All of this edition. Hosted Kaggle execution and independent professional review remain unverified.


## Scope and reproducibility

I rebuild the historical S30 models and apply the S41 public-probe level factors from the official original Parquet files. All source code, parameters and aggregate verification hashes used below are embedded in this notebook; no team files, models or predictions are required. The complete adapted notebook has not yet been verified end to end on Kaggle.

The original F0 construction used restricted row-level decision ledgers for two audit-only capital flags. Both are excluded from every model. I reconstruct CAP01 directly from the original data and its frozen rule. For the other flag, I use a rule inferred after inspecting the historical matrix: a normal period in the same policy and, for buildings, the same address. The rule exactly matched the historical flag on this data; it is a retrospective reconstruction, not the recovered decision process or proof that the rule generalizes. I verify the complete F0 content hashes before training and check the final CSV separately.

Attach the Part 1 competition files and run in order. I keep intermediate data, models and logs in a fresh temporary directory outside `/kaggle/working`; only the final CSV goes into `/kaggle/working`. The temporary directory does not persist with the notebook output. Input files remain read-only, and the row-level final CSV must remain private under the competition rules.


## Attach the competition data

I use the eleven developed-market Parquet files from Part 1. I resolve the folders in this order: explicit market variables (`IAA_W1_DATA`), an optional `IAA_DATA_ROOT`, the local workspace's `RUTAS_PROYECTO.json`, then attached data under `/kaggle/input`.

In my local project, I can open this notebook from the workspace without entering machine-specific paths. `IAA_PROJECT_ROOT` identifies that workspace when the kernel starts elsewhere. For another local checkout I set the market variables to the actual Parquet directories, or point `IAA_DATA_ROOT` to their common parent. In Kaggle I attach the corresponding competition; folder discovery stops if multiple candidates are possible.

Before training, I check every required file against the frozen sizes and SHA-256 fingerprints of my archived official downloads, then check the template's identifiers and order against the test files. Week 2 validates both markets. An incorrect path, missing file, mixed market or different fingerprint stops execution. These hashes identify the downloaded bytes; they are not represented as hashes published by Kaggle.

Only after these checks pass do I create the run folder. Its `input_manifest.json` records the resolved paths, input fingerprints and template counts. I keep inputs read-only and prevent output directories inside them.


## Optional dependency installation

Kaggle usually includes these libraries. If the environment does not match the recorded reference, set the switch to `True` and run this cell with Internet enabled. Restart the kernel after installation, set the switch back to `False`, and run from the top. Keep the same Kaggle session so the installed packages remain available. The reference versions may not support every Kaggle Python image.


In [ ]:
INSTALL_REFERENCE_PACKAGES = False
if INSTALL_REFERENCE_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==2.5.3", "pandas==3.0.5", "scipy==1.18.1", "scikit-learn==1.9.1", "lightgbm==4.7.0", "catboost==1.2.10", "pyarrow==26.0.0"])
    raise RuntimeError("Packages installed. Restart the kernel, turn installation off, and run all cells again.")


In [ ]:
import os, sys, hashlib, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

for package in ("numpy", "pandas", "pyarrow"):
    print(f"{package}: {importlib.metadata.version(package)}")
print("Python:", sys.version)

DEVELOPED_DIR = os.environ.get("IAA_W1_DATA", "")  # e.g. /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-1/public


In [ ]:
"""Resolve official notebook inputs and validate them without training models."""
from pathlib import Path
import hashlib
import json
import os

LINES = ('motor', 'buildings', 'contents')
BASE_FILES = ['addresses.parquet', 'claims.parquet', 'customers.parquet',
              'weather.parquet', 'sample_submission.parquet'] + [
    f'{prefix}_{line}.parquet' for prefix in ('policies', 'test') for line in LINES]
INPUT_CONTRACT = {}
NOTEBOOK_WEEK = None
_RESOLVED_INPUTS = {}
_VERIFIED_INPUTS = {}
_VERIFIED_TEMPLATES = {}
_VARIABLES = {'public': 'IAA_W1_DATA', 'developed': 'IAA_W2_DEVELOPED',
              'emerging': 'IAA_W2_EMERGING'}
_ROUTE_KEYS = {'public': 'week_1_data', 'developed': 'week_2_developed_reference',
               'emerging': 'week_2_emerging_data'}


def _directory(value):
    return Path(value).expanduser().resolve()


def _missing(folder, required):
    return [name for name in required if not (folder / name).is_file()]


def _discover(root, required, market):
    """Search only an explicitly chosen data tree or the Kaggle input mount."""
    candidates = set()
    if not root.is_dir():
        return []
    for current, directories, filenames in os.walk(root, followlinks=False):
        folder = Path(current)
        if len(folder.relative_to(root).parts) >= 6:
            directories[:] = []
        else:
            directories[:] = [d for d in directories if not d.startswith('.')]
        if 'sample_submission.parquet' not in filenames or _missing(folder, required):
            continue
        # The emerging folder also contains the eleven base filenames.
        if market != 'emerging' and (folder / 'reference_policies.parquet').is_file():
            continue
        candidates.add(folder.resolve())
    return sorted(candidates, key=str)


def _project_folder(market):
    explicit = os.environ.get('IAA_PROJECT_ROOT', '')
    if explicit:
        configs = [_directory(explicit) / 'RUTAS_PROYECTO.json']
        if not configs[0].is_file():
            raise RuntimeError(f'IAA_PROJECT_ROOT has no RUTAS_PROYECTO.json: {configs[0]}')
    else:
        start = Path.cwd().resolve()
        configs = [p / 'RUTAS_PROYECTO.json' for p in (start, *start.parents)
                   if (p / 'RUTAS_PROYECTO.json').is_file()][:1]
    if not configs:
        return None
    config = configs[0]
    routes = json.loads(config.read_text(encoding='utf-8-sig'))
    value = routes.get(_ROUTE_KEYS[market])
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f'{config.name} lacks the data route {_ROUTE_KEYS[market]}')
    return (config.parent / value).resolve()


def find_market(market, required, override):
    """Resolve one market, never silently select between multiple candidates."""
    if market not in INPUT_CONTRACT or market not in _VARIABLES:
        raise RuntimeError(f'No frozen input contract for {market!r}')
    origin = _VARIABLES[market]
    if override:
        candidates = [_directory(override)]
    elif os.environ.get('IAA_DATA_ROOT'):
        origin = 'IAA_DATA_ROOT'
        candidates = _discover(_directory(os.environ['IAA_DATA_ROOT']), required, market)
    else:
        project = _project_folder(market)
        origin = 'RUTAS_PROYECTO.json' if project is not None else '/kaggle/input'
        candidates = [project] if project is not None else _discover(Path('/kaggle/input'), required, market)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one {market} directory via {origin}; found '
                           f'{[str(p) for p in candidates]}. Set {_VARIABLES[market]} '
                           'to the directory containing the required Parquet files.')
    folder = candidates[0]
    missing = _missing(folder, required)
    if missing:
        raise RuntimeError(f'Missing {market} files in {folder}: {", ".join(missing)}. '
                           f'Check {_VARIABLES[market]}. No fallback was selected.')
    if market != 'emerging' and (folder / 'reference_policies.parquet').is_file():
        raise RuntimeError(f'{folder} contains emerging-market reference policies; '
                           f'check {_VARIABLES[market]} instead of mixing markets.')
    _RESOLVED_INPUTS[str(folder)] = {'market': market, 'origin': origin}
    _VERIFIED_INPUTS.pop(market, None)
    _VERIFIED_TEMPLATES.pop(market, None)
    return folder


def inventory(folder, filenames):
    import pandas as pd
    import pyarrow.parquet as pq
    folder = _directory(folder)
    binding = _RESOLVED_INPUTS.get(str(folder))
    if binding is None:
        raise RuntimeError('Resolve this input with find_market before inventory.')
    market = binding['market']
    _VERIFIED_INPUTS.pop(market, None)
    _VERIFIED_TEMPLATES.pop(market, None)
    expected = INPUT_CONTRACT[market]
    if set(filenames) != set(expected):
        raise RuntimeError(f'File list does not match the frozen {market} contract')
    rows = []
    for name in filenames:
        path = folder / name
        before = path.stat()
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(chunk)
        actual = digest.hexdigest()
        if before.st_size != expected[name]['bytes'] or actual != expected[name]['sha256']:
            raise RuntimeError(f'Official-download fingerprint mismatch: {market}/{name}. '
                               'Check the input version; training has not started.')
        parquet = pq.ParquetFile(path)
        after = path.stat()
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise RuntimeError(f'Input changed while being checked: {path}')
        rows.append({'file': name, 'rows': parquet.metadata.num_rows,
                     'columns': len(parquet.schema_arrow), 'bytes': after.st_size,
                     'sha256': actual, 'matches_download': True,
                     'mtime_ns': after.st_mtime_ns})
    _VERIFIED_INPUTS[market] = {'directory': str(folder), 'origin': binding['origin'], 'files': rows}
    return pd.DataFrame([{k: v for k, v in row.items() if k != 'mtime_ns'} for row in rows])


def check_template(folder):
    import pandas as pd
    folder = _directory(folder)
    market = _RESOLVED_INPUTS[str(folder)]['market']
    _VERIFIED_TEMPLATES.pop(market, None)
    if market not in _VERIFIED_INPUTS:
        raise RuntimeError('Verify input fingerprints before checking the template.')
    sample = pd.read_parquet(folder / 'sample_submission.parquet')
    test = pd.concat([pd.read_parquet(folder / f'test_{line}.parquet', columns=['row_id'])
                      for line in LINES], ignore_index=True)
    if ('row_id' not in sample or not sample.row_id.notna().all()
            or not test.row_id.notna().all() or not test.row_id.is_unique
            or not sample.row_id.is_unique or not sample.row_id.equals(test.row_id)):
        raise RuntimeError(f'{market}: template and test identifiers/order do not agree')
    _VERIFIED_TEMPLATES[market] = len(sample)
    print(f'{market}: template matches test identifiers in order ({len(sample):,} rows)')
    return sample


def _safe_output_root(root):
    root = _directory(root)
    for binding in _RESOLVED_INPUTS:
        source = Path(binding)
        if root == source or source in root.parents:
            raise RuntimeError(f'Output directory is inside read-only input data: {root}')
    return root


def prepare_workspace(week):
    """Create a run only after all input contracts and templates have passed."""
    import tempfile
    import uuid
    from datetime import datetime, timezone
    if week not in (1, 2) or week != NOTEBOOK_WEEK:
        raise RuntimeError('Workspace week does not match the notebook input contract')
    required = {'public'} if week == 1 else {'developed', 'emerging'}
    if not required.issubset(_VERIFIED_INPUTS) or not required.issubset(_VERIFIED_TEMPLATES):
        raise RuntimeError('Run the input inventory and template cells before creating the workspace.')
    for market in required:
        for row in _VERIFIED_INPUTS[market]['files']:
            path = Path(_VERIFIED_INPUTS[market]['directory']) / row['file']
            stat = path.stat()
            if (stat.st_size, stat.st_mtime_ns) != (row['bytes'], row['mtime_ns']):
                raise RuntimeError(f'Input changed after validation: {market}/{row["file"]}')
    work_root = _safe_output_root(os.environ.get('IAA_NOTEBOOK_WORK') or tempfile.gettempdir())
    export_value = os.environ.get('IAA_NOTEBOOK_EXPORT', '')
    export_root = _safe_output_root(export_value) if export_value else (
        Path('/kaggle/working') if Path('/kaggle/working').is_dir() else None)
    run = work_root / ('iaa_week' + str(week) + '_' + uuid.uuid4().hex[:12])
    run.mkdir(parents=True, exist_ok=False)
    export = (export_root if export_root is not None else run) / (run.name + '_exports')
    export.mkdir(parents=True, exist_ok=False)
    receipt = {'utc': datetime.now(timezone.utc).isoformat(), 'week': week,
               'inputs': {m: _VERIFIED_INPUTS[m] for m in sorted(required)},
               'template_rows': {m: _VERIFIED_TEMPLATES[m] for m in sorted(required)},
               'training_started': False}
    (run / 'input_manifest.json').write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
    print('New run:', run)
    print('Final outputs:', export)
    return run, export



NOTEBOOK_WEEK = 1
INPUT_CONTRACT = {'public': {'addresses.parquet': {'bytes': 8209304, 'sha256': '2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1'}, 'claims.parquet': {'bytes': 6393494, 'sha256': '323adf2e0f6c5975a026479d725d4cd04f51fce3aa069f897a99f664c6e392c0'}, 'customers.parquet': {'bytes': 9840123, 'sha256': '065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb'}, 'policies_buildings.parquet': {'bytes': 28411001, 'sha256': 'eb84372c23c16ed0619d89c8eb1396ab6916e5ec46a5907ce42c3dfce4a91cbe'}, 'policies_contents.parquet': {'bytes': 28203870, 'sha256': '8c067cc80baed583100df999e1c3d5e389295702f7bac21d5331bbd639b83fcd'}, 'policies_motor.parquet': {'bytes': 44218528, 'sha256': '4d2d7ae03971d5d6df3a71bbaf0ebc168250a6807b27d1561d38b638434dc95e'}, 'sample_submission.parquet': {'bytes': 3271617, 'sha256': '9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95'}, 'test_buildings.parquet': {'bytes': 7917305, 'sha256': '7b2b96507de29241c70617c17df66146ed30b51f826ce5219bbda5da536d4957'}, 'test_contents.parquet': {'bytes': 7795903, 'sha256': 'c2212f62f3b7a21431c0073277549c3db5e63857565a8b88855e79a65fa1b8b6'}, 'test_motor.parquet': {'bytes': 12035693, 'sha256': '8d28fb9fd091caf2a044913a013592226a4c0917d5c6fcc9c5e8cdcec2ab4fc5'}, 'weather.parquet': {'bytes': 41863008, 'sha256': 'b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49'}}}


In [ ]:
developed = find_market("public", BASE_FILES, DEVELOPED_DIR)
print("Developed data:", developed)
display(inventory(developed, BASE_FILES))
sample = check_template(developed)


## Approach and limitations

I assign claims using policy, line and occurrence date, then build observed net costs and reporting-development adjustments. S30 combines total-cost Tweedie models with peril frequency times severity; its level uses historical temporal cuts. S41 applies three frozen credibility-weighted factors obtained from public leaderboard probes. Since the public leaderboard informed those factors, it is not an independent test of their level.

Case reserves and unreported claims remain uncertain. The historical temporal cuts informed model decisions; feature availability at quote time still needs confirmation. I directed the modeling, data-rule reconstruction and verification work with AI assistance; the project records preserve each contribution. This is self-reproduction, not independent actuarial review.

The embedded source retains the historical programs and their stage boundaries. Each source is written into a fresh working directory at runtime; the notebook does not fetch a repository or execute a team file.


## From claims to a risk premium

Each prediction is the expected ultimate cost for one policy period, in the competition's fictional currency. It is already a period amount. The target is not the insurer's written premium and it should not be multiplied by exposure a second time when interpreting the output.

I first join each claim to a period using its line, policy identifier and occurrence date. A policy can have more than one period, so an identifier-only join can duplicate or misassign cost. I exclude declined claims and use incurred cost less recoveries for accepted, uniquely assigned claims. The valuation is a snapshot at the end of 2024: open claims may develop further, and some claims have not yet been reported.

I estimate reporting completeness by peril from a count development model. For training, the recipe reduces a period's effective exposure according to that completeness. It does not also inflate the observed label for the same unreported claims. This offset is a modeling assumption, not proof that case reserves are sufficient or that the development pattern will remain stable.

## Validation and information timing

The 2023 cut trains on earlier years and separates customers for calibration and selection. The 2024 cut trains on 2021–2023 and evaluates the next year. The 2025 fit uses the permitted 2021–2024 history. These historical cuts helped choose the model, so I do not describe them as untouched final tests.

The feature contract distinguishes policy and property attributes, prior claims history, prior-year weather, conditional customer attributes and existing pricing information. It excludes the current-period service-contact and renewal outcome fields and the audit flags from predictors. History features are cut at the start of the policy year. Whether every retained field is actually available at a real quote remains an open question.

## Models, aggregation and calibration

I combine total-cost Tweedie models with peril-specific frequency multiplied by severity. In motor the final weights are 40% total cost and 60% frequency–severity; the same split applies to buildings. In contents the split is 30% and 70%. Component predictions are normalized to a line-level reference before blending. The recipe applies the historical shape and exposure adjustments and then calibrates the 2025 level from the two historical cuts.

For Tweedie power 1.5, a constant multiplier that minimizes deviance over a fixed set of positive predictions is `sum(y / sqrt(mu)) / sum(sqrt(mu))`. S41 adds one fixed multiplier per line. Those factors were derived from six public leaderboard probes and a credibility rule specified before their scores were observed. They use information from the public part of 2025, so I do not treat that score as a blind evaluation. The exact factors, rather than their rounded presentation values, are embedded in the final cell.

## How to read the run

The source cells below preserve the original program structure, including the data producer, actuarial bases, feature contract, model stages and blend. The execution cell materializes them inside a fresh run folder and saves logs and a receipt for each node. If a stage fails, inspect its log before rerunning. Code 3 reports a matching final S30 CSV with intermediate content differences; it is not a clean match for every artifact. A different final CSV hash stops the run. The final check also enforces the template's identifiers, their order and positive finite predictions.


## Interpreting the metric

For nonnegative observed cost `y` and positive prediction `mu`, Tweedie deviance at power 1.5 is `4 * (y / sqrt(mu) + sqrt(mu) - 2 * sqrt(y))`. The local D² is one minus the total model deviance divided by the deviance of the constant mean prediction. Higher is better; a negative value means worse than that constant on the evaluated rows. The observed target is still a valuation proxy for ultimate cost. I do not claim this local calculation proves every detail of Kaggle's unpublished evaluator.

A score measures one aspect of the exercise. I also examine portfolio slices, large claims, reserve assumptions and whether the input fields exist at quote time. None of those checks turns an adaptive development cut into an independent professional review.


In [ ]:
from sklearn.metrics import mean_tweedie_deviance, d2_tweedie_score
# A small synthetic calculation, including a zero-cost observation.
y_demo = np.array([0.0, 0.25, 2.0, 100.0])
mu_demo = np.array([0.1, 0.3, 1.5, 25.0])
d_demo = 4 * (y_demo / np.sqrt(mu_demo) + np.sqrt(mu_demo) - 2 * np.sqrt(y_demo))
null_demo = 4 * (y_demo / np.sqrt(y_demo.mean()) + np.sqrt(y_demo.mean()) - 2 * np.sqrt(y_demo))
assert np.isclose(d_demo.mean(), mean_tweedie_deviance(y_demo, mu_demo, power=1.5), rtol=1e-13)
assert np.isclose(1 - d_demo.sum() / null_demo.sum(), d2_tweedie_score(y_demo, mu_demo, power=1.5), rtol=1e-13)
print("The formula agrees with the local scikit-learn calculation.")


## Dependencies and workspace

The historical reference used Python 3.13.16, NumPy 2.5.3, pandas 3.0.5, SciPy 1.18.1, scikit-learn 1.9.1, LightGBM 4.7.0, CatBoost 1.2.10 and PyArrow 26.0.0. Check the installed versions below; if needed install compatible versions separately and restart the kernel. Installation may require Internet.

The historical 36-node reconstruction, which also included the S22 backup, recorded about 10.6 hours on two cores. This notebook runs the 30-node S30 dependency graph; its time on hosted Kaggle has not been measured. [Kaggle's notebook documentation](https://www.kaggle.com/docs/notebooks), consulted on 28 October 2026, describes a 12-hour CPU session and 20 GB of saved working storage. Execution speed and disk use vary, so the available margin is not guaranteed. A hosted end-to-end run is still needed to establish whether this recipe finishes there.


In [ ]:
import json, shutil, subprocess, time, uuid
from datetime import datetime, timezone
from importlib import metadata
REFERENCE = {"numpy": "2.5.3", "pandas": "3.0.5", "scipy": "1.18.1", "scikit-learn": "1.9.1", "lightgbm": "4.7.0", "catboost": "1.2.10", "pyarrow": "26.0.0"}
installed = {name: metadata.version(name) for name in REFERENCE}
print("Installed:", installed, "Reference:", REFERENCE)
if installed != REFERENCE: print("Environment differs from the reference; verify output identity.")
RUN, EXPORT = prepare_workspace(1)
PACKAGE = RUN / "embedded_code"
PACKAGE.mkdir()
SOURCES = {}


## Embedded source package

I keep the source files below as text assignments. Running these cells registers the code in memory; the later execution section writes it into the new workspace and starts the recipe. I retain the original program boundaries, authorship and frozen configuration. No past row-level predictions or fitted models are embedded.


### Embedded source: `reproducir.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['reproducir.py'] = '"""Reconstruct the frozen S30 blend and its S41 delivery calibration.\n\nThe runner checks required artifacts against the frozen recipe. An exact final CSV\nmatch is a separate check from equality of intermediate artifacts.\n\nCommands:\n  plan          Print dependency order and commands without running models.\n  mezcla        Rebuild final blends from verified, previously saved components.\n  componentes   Rebuild features, selections, production models and blends from F0.\n  verificar     Check existing outputs without training or changing files.\n  datos         Rebuild prepared tables, actuarial bases and F0 from 11 original files.\n  todo          Run datos, then componentes on the newly generated data.\n\nOptions:\n  --semana1=<directory>     Week 1 workspace containing data and historical runs.\n  --trabajo=<directory>     New private workspace for outputs, logs and journal.\n  --objetivo=S30,S22        Requested blend targets; both are used by default.\n  --artefactos=<directory>  Additional component directories for mezcla; repeatable.\n  --solo=<node>[,...]       Run only named nodes whose dependencies already exist.\n  --originales=<directory>  Original Parquet directory for datos and todo.\n  --decisiones=<directory>  Frozen audit-flag decision lists, where required.\n  --continuar-si-difiere    Record intermediate content differences and continue.\n  --relanzar-parciales      Preserve incomplete outputs as .parcial_<UTC> and retry.\n\nMissing, unreadable or unpinned required artifacts always stop execution. The\ncontinue option never bypasses the exact SHA-256 check on a final CSV. Completed\nnodes can be reused only after their required fingerprints have been checked.\nEvery executed node records its command, duration, exit code and output checks in\nregistro.jsonl and writes its standard output to registros/<node>.log.\n\nExit codes:\n  0  All requested artifact checks and final CSV fingerprints match.\n  1  A required input, stage, artifact or final fingerprint failed.\n  2  Invalid command or arguments.\n  3  Final CSVs match, but recorded intermediate content differences remain.\n\nThe full model graph can take several hours on two CPU cores. Model sources use\nthe runtime paths supplied by the runner; original data remain read-only.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nVERSION = "1.2"\nVERSION_FECHA = "2026-10-19"\n\nREPO = Path(__file__).resolve().parent\nsys.path.insert(0, str(REPO / "herramientas"))\nfrom huella import huellas  # noqa: E402\n\nRECETA = json.loads((REPO / "receta" / "receta.json").read_text(encoding="utf-8"))\nNODOS = {n["id"]: n for n in RECETA["nodos"]}\nORDEN = [n["id"] for n in RECETA["nodos"]]\nCSV_FINALES = {v["csv"] for v in RECETA["objetivos"].values()}\n\n\nclass NoConforme(SystemExit):\n    """Stop because a required input or artifact violates the recipe contract (exit code 1)."""\n\n    def __init__(self, mensaje: str):\n        super().__init__(f"NO CONFORME: {mensaje}")\n\n\ndef opciones(argv):\n    o = {}\n    for a in argv:\n        if a.startswith("--"):\n            k, _, v = a[2:].partition("=")\n            o[k] = v if v else True\n    return o\n\n\ndef cierre(objetivos, solo=None):\n    """Return the target dependency closure in the frozen recipe order."""\n    if solo:\n        desconocidos = sorted(set(solo) - set(ORDEN))\n        if desconocidos:\n            raise SystemExit(f"--solo nombra nodos que no están en la receta: {desconocidos}")\n        return [i for i in ORDEN if i in solo]\n    raices = [RECETA["objetivos"][o]["nodo"] for o in objetivos]\n    necesarios, pila = set(), list(raices)\n    while pila:\n        i = pila.pop()\n        if i in necesarios:\n            continue\n        necesarios.add(i)\n        pila += NODOS[i]["requiere"]\n    return [i for i in ORDEN if i in necesarios]\n\n\ndef orden_de(nodo):\n    args = [str(REPO / a) if a.startswith("receta/") else a for a in nodo["args"]]\n    # Write both final CSVs with LF line endings, matching the reference serialization.\n    # Keep the required SHA unchanged; do not normalize a file after verifying it.\n    if nodo["script"] in {"src/modelos/mezcla3.py", "src/linea_b/mezcla3_b.py"}:\n        return [sys.executable, "-W", "ignore", str(REPO / "herramientas" / "ejecutar_csv_lf.py"), str(REPO / nodo["script"])] + args\n    return [sys.executable, "-W", "ignore", str(REPO / nodo["script"])] + args\n\n\ndef entorno(semana1: Path, work: Path):\n    env = dict(os.environ)\n    env.update({"HAKATON_SEMANA1": str(semana1), "MODELOS_WORK": str(work), "MODELOS_BASE": str(REPO / "src" / "modelos"),\n                "PYTHONHASHSEED": "0"})\n    return env\n\n\ndef esperadas():\n    """Load expected fingerprints; execution requires this recipe contract."""\n    p = REPO / "receta" / "huellas_esperadas.json"\n    if not p.is_file():\n        raise NoConforme(f"falta {p.relative_to(REPO)}; sin huellas esperadas no se puede verificar nada")\n    esp = json.loads(p.read_text(encoding="utf-8"))\n    for clave in ("entradas", "salidas"):\n        if not isinstance(esp.get(clave), dict) or not esp[clave]:\n            raise NoConforme(f"receta/huellas_esperadas.json no tiene la sección «{clave}» o está vacía")\n    return esp\n\n\ndef comparar(rel, work: Path, esp: dict, seccion: str = "salidas", clave: str | None = None):\n    """Compare work/<rel> with esp[section][key]. Return diagnostics instead of raising."""\n    p = work / rel\n    e = esp.get(seccion, {}).get(clave or rel)\n    r = {"ruta": rel, "existe": p.is_file(), "huella_esperada": isinstance(e, dict) and bool(e.get("contenido"))}\n    if not p.is_file():\n        return r\n    try:\n        h = huellas(p)\n    except Exception as exc:  # A required file may be truncated, corrupt or unreadable.\n        r.update({"legible": False, "error": f"{type(exc).__name__}: {exc}"})\n        return r\n    r.update({"legible": True, "sha256": h["sha256"], "contenido": h["contenido"]})\n    if r["huella_esperada"]:\n        r.update({"igual_bytes": h["sha256"] == e.get("sha256"), "igual_contenido": h["contenido"] == e["contenido"]})\n    return r\n\n\ndef no_conformidades(filas, tolerar_diferencias=False):\n    """Separate blocking failures from allowed intermediate differences. Final CSVs also require exact bytes."""\n    bloqueantes, diferencias = [], []\n    for f in filas:\n        if not f.get("existe"):\n            bloqueantes.append((f["ruta"], "falta"))\n        elif not f.get("legible", True):\n            bloqueantes.append((f["ruta"], f"no se puede leer: {f.get(\'error\')}"))\n        elif not f.get("huella_esperada"):\n            bloqueantes.append((f["ruta"], "sin huella esperada en la receta"))\n        elif f.get("igual_contenido") is not True or (f["ruta"] in CSV_FINALES and f.get("igual_bytes") is not True):\n            motivo = "contenido distinto" if f.get("igual_contenido") is not True else "CSV final con SHA-256 distinto"\n            (diferencias if tolerar_diferencias else bloqueantes).append((f["ruta"], motivo))\n    return bloqueantes, diferencias\n\n\ndef exigir_artefactos(filas, contexto: str, tolerar_diferencias=False):\n    """Stop on blocking artifact failures and return any explicitly tolerated differences."""\n    if not filas:\n        raise NoConforme(f"{contexto}: no hay artefactos requeridos que verificar")\n    bloqueantes, diferencias = no_conformidades(filas, tolerar_diferencias)\n    if bloqueantes:\n        raise NoConforme(f"{contexto}: artefactos requeridos no conformes: {resumir(bloqueantes)}")\n    return diferencias\n\n\ndef resumir(lista, maximo=8):\n    """Summarize long lists; the JSON output and journal retain every item."""\n    return str(lista) if len(lista) <= maximo else f"{lista[:maximo]} y {len(lista) - maximo} más (detalle en la salida JSON y en registro.jsonl)"\n\n\ndef exigir_datos(informe):\n    """Require every mandatory data-stage check to be present and successful at the entry point."""\n    contrato = esperadas().get("datos")\n    if not contrato or not isinstance(informe, dict):\n        raise NoConforme("datos: falta el contrato esperado de datos o el informe no es válido")\n    fallos = []\n    for nombre in contrato["capa"]:\n        if informe.get("capa", {}).get(nombre) is not True:\n            fallos.append("capa/" + nombre)\n    for grupo in ("bases", "f0"):\n        for nombre in contrato[grupo]:\n            item = informe.get(grupo, {}).get(nombre)\n            if not isinstance(item, dict) or item.get("igual_contenido") is not True:\n                fallos.append(grupo + "/" + nombre)\n    if informe.get("f0", {}).get("contrato_bloques_iguales") is not True:\n        fallos.append("f0/contrato_bloques_iguales")\n    if fallos:\n        raise NoConforme(f"datos: controles requeridos no conformes: {fallos}")\n\n\ndef bloques_contrato(p: Path) -> str:\n    """Fingerprint the F0 feature-block assignments consumed by comun.contract."""\n    import hashlib\n    cols = json.loads(p.read_text(encoding="utf-8"))["columnas"]\n    return hashlib.sha256(json.dumps({l: {c: v["bloque"] for c, v in cc.items()} for l, cc in cols.items()}, sort_keys=True).encode()).hexdigest()\n\n\ndef comprobar_entradas(semana1: Path, esp: dict):\n    malas = []\n    bloques_esperados = esp.get("datos", {}).get("f0_contrato_bloques")\n    for rel, e in esp["entradas"].items():\n        p = semana1 / rel\n        if not p.is_file():\n            malas.append((rel, "no existe"))\n            continue\n        try:\n            if rel.endswith("contrato_disponibilidad.json"):\n                # Arrow column types can vary with the pandas version.\n                if not bloques_esperados:\n                    malas.append((rel, "la receta no tiene la huella esperada de los bloques (datos/f0_contrato_bloques)"))\n                elif bloques_contrato(p) != bloques_esperados:\n                    malas.append((rel, "bloques distintos"))\n            elif not e.get("contenido"):\n                malas.append((rel, "sin huella de contenido esperada"))\n            elif huellas(p)["contenido"] != e["contenido"]:\n                malas.append((rel, "contenido distinto"))\n        except Exception as exc:\n            malas.append((rel, f"no se puede leer: {type(exc).__name__}: {exc}"))\n    return malas\n\n\ndef registrar(trabajo: Path, fila: dict):\n    with open(trabajo / "registro.jsonl", "a", encoding="utf-8") as f:\n        f.write(json.dumps(fila, ensure_ascii=False) + "\\n")\n\n\ndef ahora():\n    return datetime.now(timezone.utc).isoformat(timespec="seconds")\n\n\ndef apartar_parciales(nodo: dict, work: Path, trabajo: Path):\n    """Preserve incomplete node outputs with a .parcial_<UTC> suffix; delete nothing."""\n    marca = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")\n    apartadas = []\n    for s in nodo["salidas"]:\n        p = work / s\n        if p.exists() or p.is_symlink():\n            destino = p.with_name(p.name + f".parcial_{marca}")\n            p.rename(destino)\n            apartadas.append({"de": s, "a": str(destino.relative_to(work))})\n    registrar(trabajo, {"nodo": nodo["id"], "estado": "parciales_apartadas", "apartadas": apartadas, "utc": ahora()})\n    print(f"[{nodo[\'id\']}] salidas parciales apartadas: {[a[\'a\'] for a in apartadas]}", flush=True)\n\n\ndef ejecutar(ids, semana1: Path, trabajo: Path, continuar=False, relanzar_parciales=False):\n    """Run nodes in dependency order and return the recorded intermediate differences."""\n    work = trabajo / "work"\n    (trabajo / "registros").mkdir(parents=True, exist_ok=True)\n    work.mkdir(parents=True, exist_ok=True)\n    esp = esperadas()\n    env = entorno(semana1, work)\n    toleradas = []\n    for i in ids:\n        n = NODOS[i]\n        presentes = [s for s in n["salidas"] if (work / s).is_file()]\n        if len(presentes) == len(n["salidas"]):\n            comp = [comparar(s, work, esp) for s in n["verificar"]]\n            dif = exigir_artefactos(comp, f"reanudar {i}", continuar)\n            estado = "reanudado_con_diferencias" if dif else "reanudado_verificado"\n            registrar(trabajo, {"nodo": i, "estado": estado, "verificacion": comp, "utc": ahora()})\n            print(f"[{i}] ya hecho, huellas requeridas verificadas" + (f"; difieren: {[d[0] for d in dif]}" if dif else ""), flush=True)\n            toleradas += [(i, *d) for d in dif]\n            continue\n        if presentes or any((work / s).exists() or (work / s).is_symlink() for s in n["salidas"]):\n            if not relanzar_parciales:\n                raise NoConforme(f"{i}: salidas parciales en work/ ({presentes or \'rutas presentes sin archivo\'}); "\n                                 "revisar la carpeta de trabajo o relanzar con --relanzar-parciales")\n            apartar_parciales(n, work, trabajo)\n        cmd = orden_de(n)\n        print(f"[{i}] {\' \'.join(cmd[3:])}", flush=True)\n        t0 = time.time()\n        with open(trabajo / "registros" / f"{i}.log", "a", encoding="utf-8") as log:\n            log.write(f"# {ahora()} {\' \'.join(cmd)}\\n")\n            log.flush()\n            rc = subprocess.run(cmd, cwd=str(trabajo), env=env, stdout=log, stderr=subprocess.STDOUT).returncode  # Use a private cwd because CatBoost writes catboost_info/ there.\n        seg = round(time.time() - t0, 1)\n        comp = [comparar(s, work, esp) for s in n["verificar"]]\n        bloq, dif = no_conformidades(comp, continuar)\n        estado = "error" if rc else ("no_conforme" if bloq else ("difiere" if dif else "igual"))\n        fila = {"nodo": i, "orden": cmd[2:], "rc": rc, "segundos": seg, "utc_fin": ahora(), "estado": estado, "verificacion": comp}\n        registrar(trabajo, fila)\n        print(f"[{i}] {estado} en {seg:.0f} s" + (f"; no conformes: {bloq}" if bloq else "") + (f"; difieren: {[d[0] for d in dif]}" if dif else ""), flush=True)\n        if rc:\n            raise SystemExit(f"El nodo {i} terminó con código {rc}; ver registros/{i}.log")\n        exigir_artefactos(comp, f"salida {i}", continuar)\n        toleradas += [(i, *d) for d in dif]\n    return toleradas\n\n\ndef enlazar_archivo(src: Path, dst: Path):\n    """Use a private copy on Windows, then apply the same checks as for a symbolic link."""\n    if os.name == "nt":\n        shutil.copy2(src, dst)\n    else:\n        dst.symlink_to(src)\n\n\ndef preparar_mezcla(ids, semana1: Path, trabajo: Path, extra: list[Path], esp: dict):\n    """Stage saved component and exposure files in work/, verify them, and return missing or nonconforming inputs."""\n    work = trabajo / "work"\n    work.mkdir(parents=True, exist_ok=True)\n    origenes = [semana1 / "Modelos_base", semana1 / "Modelos_line_b"] + extra\n    sys.path.insert(0, str(REPO / "herramientas"))\n    from inventario import localizar\n    faltan, comp = [], []\n    directos = sorted({r for i in ids if NODOS[i]["etapa"] == "mezcla" for r in NODOS[i]["requiere"]})\n    for i in directos:\n        n = NODOS[i]\n        for rel in n["verificar"]:\n            if not rel.endswith(".parquet"):\n                continue  # Blends read only v24.parquet, prediccion_2025.parquet and sel_<name>.parquet.\n            dst = work / rel\n            if not dst.exists():\n                src = localizar(rel, origenes)\n                if src is None:\n                    faltan.append(rel)\n                    continue\n                dst.parent.mkdir(parents=True, exist_ok=True)\n                enlazar_archivo(src, dst)\n            comp.append(comparar(rel, work, esp))\n    for line in ("motor", "buildings", "contents"):\n        rel = f"feat_{line}.parquet"\n        dst = work / rel\n        if not dst.exists():\n            src = next((o / "insumos" / f"exposicion_{line}.parquet" for o in origenes if (o / "insumos" / f"exposicion_{line}.parquet").is_file()), None)\n            src = src or next((o / rel for o in origenes if (o / rel).is_file()), None)\n            if src is None:\n                faltan.append(f"{rel} (o insumos/exposicion_{line}.parquet)")\n                continue\n            enlazar_archivo(src, dst)\n        # Exposure may come from the full feature matrix checked under salidas,\n        # or from the frozen insumos/exposicion_<line>.parquet extract checked under mezcla.\n        # Accept exactly one of the two contract-approved inputs.\n        # Resolving a Windows copy does not retain its original source filename.\n        completo = comparar(rel, work, esp)\n        extracto = comparar(rel, work, esp, "mezcla", clave=f"insumos/exposicion_{line}.parquet")\n        if completo.get("igual_contenido") is True:\n            completo["contrato"] = "salidas"\n            comp.append(completo)\n        else:\n            extracto["contrato"] = "mezcla"\n            comp.append(extracto)\n    bloq, _ = no_conformidades(comp, tolerar_diferencias=False)\n    registrar(trabajo, {"modo": "mezcla", "estado": "componentes_enlazados", "faltan": faltan, "no_conformes": bloq, "verificacion": comp, "utc": ahora()})\n    return faltan, bloq\n\n\ndef main():\n    if len(sys.argv) < 2 or sys.argv[1] not in ("plan", "mezcla", "componentes", "verificar", "datos", "todo"):\n        print(__doc__)\n        raise SystemExit(2)\n    modo, o = sys.argv[1], opciones(sys.argv[2:])\n    print(f"reproducir.py versión {VERSION} ({VERSION_FECHA}), modo {modo}", flush=True)\n    if o.get("parar-si-difiere"):\n        print("--parar-si-difiere ya es el comportamiento por defecto desde la versión 1.1; se ignora", flush=True)\n    continuar = bool(o.get("continuar-si-difiere"))\n    objetivos = [x for x in str(o.get("objetivo", "S30,S22")).split(",") if x]\n    desconocidos = [x for x in objetivos if x not in RECETA["objetivos"]]\n    if desconocidos:\n        raise SystemExit(f"Objetivos desconocidos: {desconocidos}; la receta define {sorted(RECETA[\'objetivos\'])}")\n    semana1 = Path(o.get("semana1") or os.environ.get("IAA_SEMANA1", "")).expanduser().resolve()\n    trabajo = Path(o.get("trabajo", "trabajo")).expanduser().resolve()\n    solo = set(str(o["solo"]).split(",")) if o.get("solo") else None\n    ids = cierre(objetivos, solo)\n    if modo == "plan":\n        for i in ids:\n            n = NODOS[i]\n            print(f"{i:30s} [{n[\'etapa\']}, {n[\'linea\']}, ~{n[\'minutos_ref\']} min] {\' \'.join(orden_de(n)[3:])}")\n        print(f"{len(ids)} nodos; tiempo de referencia ~{sum(NODOS[i][\'minutos_ref\'] or 0 for i in ids) / 60:.1f} h con dos núcleos")\n        return\n    if modo == "verificar":\n        esp = esperadas()\n        work = trabajo / "work"\n        filas = [comparar(rel, work, esp) for i in ids for rel in NODOS[i]["verificar"]]\n        bloq, _ = no_conformidades(filas, tolerar_diferencias=False)\n        resumen = {"version": VERSION, "comparados": len(filas), "iguales_contenido": sum(f.get("igual_contenido") is True for f in filas),\n                   "iguales_bytes": sum(f.get("igual_bytes") is True for f in filas),\n                   "difieren": [f["ruta"] for f in filas if f.get("existe") and f.get("legible", True) and f.get("igual_contenido") is False],\n                   "faltan": [f["ruta"] for f in filas if not f.get("existe")],\n                   "ilegibles": [f["ruta"] for f in filas if f.get("existe") and not f.get("legible", True)],\n                   "sin_huella_esperada": [f["ruta"] for f in filas if f.get("existe") and not f.get("huella_esperada")],\n                   "conforme": not bloq}\n        print(json.dumps(resumen, ensure_ascii=False, indent=1))\n        exigir_artefactos(filas, "verificar")\n        return\n    if not semana1.is_dir():\n        raise SystemExit("Falta --semana1=<carpeta Hackaton/Semana 1> (o la variable IAA_SEMANA1)")\n    trabajo.mkdir(parents=True, exist_ok=True)\n    registrar(trabajo, {"modo": modo, "estado": "inicio", "version": VERSION, "version_fecha": VERSION_FECHA, "orden": sys.argv[1:], "utc": ahora()})\n    if modo in ("datos", "todo"):\n        from etapas_datos import preparar_datos\n        originales = Path(o.get("originales") or (semana1 / "Datos")).expanduser()\n        decisiones = Path(o.get("decisiones") or semana1).expanduser()\n        semana1, informe = preparar_datos(originales, decisiones, trabajo)\n        exigir_datos(informe)\n        registrar(trabajo, {"modo": "datos", "semana1": str(semana1), "informe": informe, "conforme": True, "utc": ahora()})\n        print(json.dumps(informe, ensure_ascii=False, indent=1))\n        if modo == "datos":\n            print("datos: conforme", flush=True)\n            return\n        ids = cierre(objetivos, solo)\n    esp = esperadas()\n    if modo == "mezcla":\n        extra = [Path(x) for x in str(o.get("artefactos", "")).split(",") if x]\n        faltan, bloq = preparar_mezcla(ids, semana1, trabajo, extra, esp)\n        if faltan:\n            raise NoConforme(f"no encuentro estas salidas guardadas: {faltan}")\n        if bloq:\n            raise NoConforme(f"componentes guardados que no coinciden con la receta: {bloq}")\n        ids = [i for i in ids if NODOS[i]["etapa"] == "mezcla"]\n    else:\n        malas = comprobar_entradas(semana1, esp)\n        if malas:\n            raise NoConforme(f"las entradas no coinciden con las de la receta: {malas}")\n    toleradas = ejecutar(ids, semana1, trabajo, continuar=continuar, relanzar_parciales=bool(o.get("relanzar-parciales")))\n    work = trabajo / "work"\n    final, no_construidos = {}, []\n    for ob in objetivos:\n        t = RECETA["objetivos"][ob]\n        if t["nodo"] not in ids:\n            no_construidos.append(ob)\n            continue\n        p = work / t["csv"]\n        sha = huellas(p)["sha256"] if p.is_file() else None\n        final[ob] = {"csv": str(p), "sha256": sha, "esperado": t["sha256"], "coincide": sha == t["sha256"]}\n    csv_ok = all(v["coincide"] for v in final.values())\n    estado = "igual" if csv_ok and not toleradas else ("csv_iguales_con_diferencias_intermedias" if csv_ok else "csv_distinto")\n    resultado = {"modo": modo, "version": VERSION, "objetivos": final, "objetivos_no_construidos": no_construidos,\n                 "diferencias_toleradas": [{"nodo": n, "ruta": r, "motivo": m} for n, r, m in toleradas], "estado": estado, "utc": ahora()}\n    registrar(trabajo, resultado)\n    print(json.dumps(resultado, ensure_ascii=False, indent=1))\n    if not csv_ok:\n        raise SystemExit("NO CONFORME: algún CSV final no coincide con el entregado")\n    if toleradas:\n        print(f"Reconstrucción con {len(toleradas)} diferencias de contenido toleradas (--continuar-si-difiere): no acreditada", flush=True)\n        raise SystemExit(3)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `herramientas/huella.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['herramientas/huella.py'] = '"""Fingerprint reconstructed artifacts against their frozen references.\n\nsha256 hashes file bytes and is mandatory for final CSVs. contenido hashes values\nindependently of the writer: Parquet columns are decoded and normalized for string\nand timestamp representations; NumPy arrays include dtype, shape and bytes; NPZ\nmembers are hashed individually because ZIP timestamps vary. CSV line endings\nand JSON key ordering are normalized only for this content comparison.\n\nUsage: python huella.py <file> [<file> ...]\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef sha256_bytes(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef _array_digest(h, arr: np.ndarray) -> None:\n    arr = np.ascontiguousarray(arr)\n    h.update(f"{arr.dtype.str}|{arr.shape}|".encode())\n    if arr.dtype.kind == "O":\n        h.update("\\x1f".join("\\x00" if v is None else str(v) for v in arr.ravel()).encode("utf-8"))\n    else:\n        h.update(arr.tobytes())\n\n\ndef _parquet_digest(path: Path) -> str:\n    import pyarrow as pa\n    import pyarrow.compute as pc\n    import pyarrow.parquet as pq\n\n    t = pq.read_table(path)\n    h = hashlib.sha256()\n    h.update(f"filas={t.num_rows}|".encode())\n    for name in t.column_names:\n        if name.startswith("__index_level_"):\n            continue\n        col = t.column(name).combine_chunks()\n        if pa.types.is_dictionary(col.type):\n            col = col.dictionary_decode()\n        typ = col.type\n        es_vista = getattr(pa.types, "is_string_view", lambda _t: False)\n        if pa.types.is_large_string(typ) or es_vista(typ):\n            col = col.cast(pa.string()); typ = col.type\n        if pa.types.is_timestamp(typ):\n            col = pc.cast(col.cast(pa.timestamp("ns", tz=typ.tz)), pa.int64()); tag = "ts_ns"\n        elif pa.types.is_date(typ):\n            col = pc.cast(col.cast(pa.date32()), pa.int32()); tag = "date32"\n        elif pa.types.is_string(typ):\n            tag = "str"\n        elif pa.types.is_boolean(typ):\n            tag = "bool"\n        elif pa.types.is_integer(typ):\n            tag = "int"\n        elif pa.types.is_floating(typ):\n            tag = "float"\n        else:\n            tag = str(typ)\n        h.update(f"{name}|{tag}|nulos={col.null_count}|".encode())\n        if tag == "str":\n            vals = col.to_pylist()\n            h.update("\\x1f".join("\\x00" if v is None else v for v in vals).encode("utf-8"))\n        elif tag == "bool":\n            h.update(np.asarray(col.fill_null(False).to_numpy(zero_copy_only=False), dtype=np.uint8).tobytes())\n            if col.null_count:\n                h.update(np.asarray(col.is_null().to_numpy(zero_copy_only=False), dtype=np.uint8).tobytes())\n        elif tag in ("int", "ts_ns", "date32"):\n            v = col.fill_null(0).to_numpy(zero_copy_only=False).astype(np.int64)\n            h.update(v.tobytes())\n            if col.null_count:\n                h.update(np.asarray(col.is_null().to_numpy(zero_copy_only=False), dtype=np.uint8).tobytes())\n        elif tag == "float":\n            v = col.to_numpy(zero_copy_only=False).astype(np.float64)\n            h.update(v.tobytes())\n        else:\n            h.update(json.dumps(col.to_pylist(), default=str).encode("utf-8"))\n    return h.hexdigest()\n\n\ndef huella_contenido(path: Path) -> str:\n    path = Path(path)\n    suf = path.suffix.lower()\n    if suf == ".parquet":\n        return _parquet_digest(path)\n    h = hashlib.sha256()\n    if suf == ".npy":\n        _array_digest(h, np.load(path, allow_pickle=False))\n    elif suf == ".npz":\n        with np.load(path, allow_pickle=False) as z:\n            for k in sorted(z.files):\n                h.update(f"[{k}]".encode())\n                _array_digest(h, z[k])\n    elif suf == ".csv":\n        h.update(path.read_bytes().replace(b"\\r\\n", b"\\n"))\n    elif suf == ".json":\n        h.update(json.dumps(json.loads(path.read_text(encoding="utf-8-sig")), sort_keys=True, ensure_ascii=False).encode("utf-8"))\n    else:\n        h.update(path.read_bytes())\n    return h.hexdigest()\n\n\ndef huellas(path: Path) -> dict:\n    path = Path(path)\n    return {"bytes": path.stat().st_size, "sha256": sha256_bytes(path), "contenido": huella_contenido(path)}\n\n\nif __name__ == "__main__":\n    for a in sys.argv[1:]:\n        print(json.dumps({"archivo": a, **huellas(Path(a))}, ensure_ascii=False))\n'


### Embedded source: `herramientas/etapas_datos.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['herramientas/etapas_datos.py'] = '"""Rebuild prepared data, actuarial bases and F0 from 11 original Parquet files.\n\nCreate a minimal project tree under <work>/proyecto, including RUTAS_PROYECTO.json,\nthe prepared-data catalog, verification records and the expected Week 1 runtime\ndirectories. Run the N62 preparation, bases_actuariales.py and construir_f0.py\nin that isolated tree. Link the newly generated runs under the names consumed\nby comun.py and verify each step against receta/huellas_esperadas.json.\n\nThe notebook adaptation reconstructs audit-only flags without copying restricted\ndecision ledgers. Return the generated Week 1 directory for the model stages.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nREPO = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(REPO / "herramientas"))\nfrom huella import huellas  # noqa: E402\n\nPREAUDIT = "preaudit_20261016T211816Z_817285a5"\nF0_NOMBRE = "f0_20261017T141732Z_65df7fca"\nBASES_NOMBRE = "bases_20261017T120251Z_45443ff0"\n\n\ndef sha(p: Path) -> str:\n    h = hashlib.sha256()\n    with open(p, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef correr(cmd, cwd, log: Path):\n    with open(log, "a", encoding="utf-8") as f:\n        f.write("# " + " ".join(map(str, cmd)) + "\\n")\n        f.flush()\n        rc = subprocess.run([str(c) for c in cmd], cwd=str(cwd), stdout=f, stderr=subprocess.STDOUT).returncode\n    if rc:\n        raise SystemExit(f"Falló {\' \'.join(map(str, cmd[1:3]))}; ver {log}")\n\n\ndef copiar_script(src: Path, dst_dir: Path) -> Path:\n    dst_dir.mkdir(parents=True, exist_ok=True)\n    dst = dst_dir / src.name\n    if not dst.exists():\n        shutil.copy2(src, dst)\n    if sha(dst) != sha(src):\n        raise SystemExit(f"La copia de {src.name} en la raíz mínima no coincide con la del repositorio")\n    return dst\n\n\ndef preparar_datos(originales: Path, decisiones: Path, trabajo: Path) -> tuple[Path, dict]:\n    esp = json.loads((REPO / "receta" / "huellas_esperadas.json").read_text(encoding="utf-8"))["datos"]\n    root = trabajo / "proyecto"\n    s1 = root / "Hackaton" / "Semana 1"\n    logs = trabajo / "registros"\n    logs.mkdir(parents=True, exist_ok=True)\n    s1.mkdir(parents=True, exist_ok=True)\n    (root / "RUTAS_PROYECTO.json").write_text("{}\\n", encoding="utf-8")\n    informe = {}\n\n    # Stage 00: frozen originals and the N62 prepared-data producer.\n    malos = [n for n, h in esp["originales"].items() if sha(originales / (n if n.endswith(".parquet") else f"{n}.parquet")) != h]\n    if malos:\n        raise SystemExit(f"Los originales no tienen el SHA-256 congelado: {malos}")\n    capa = s1 / "Limpieza" / "runs" / PREAUDIT\n    if not (capa / "COMPLETE.json").exists():\n        correr([sys.executable, REPO / "src/datos/n62/rebuild_base.py", "--raw", originales,\n                "--rules", REPO / "src/datos/n62/rules.json", "--out", capa], trabajo, logs / "datos_capa.log")\n    rel = f"Hackaton/Semana 1/Limpieza/runs/{PREAUDIT}"\n    catalogo = {f.stem: {"workspace_relative_path": f"{rel}/data/{f.name}", "sha256": sha(f)} for f in sorted((capa / "data").glob("*.parquet"))}\n    informe["capa"] = {n: catalogo.get(n, {}).get("sha256") == h for n, h in esp["capa"].items()}\n    if not all(informe["capa"].values()):\n        raise SystemExit(f"La capa reconstruida no coincide con preaudit: {informe[\'capa\']}")\n    (capa / "dataset_catalog.json").write_text(json.dumps(catalogo, indent=1), encoding="utf-8")\n    (capa / "verification.json").write_text(json.dumps({"status": "capa reconstruida por N62 desde los once originales; SHA-256 iguales a preaudit",\n                                                        "passed": len(catalogo), "failed": 0}, indent=1), encoding="utf-8")\n    (s1 / "Limpieza" / "current_dataset.json").write_text(json.dumps({"workspace_relative_run": rel, "status": "rebuilt_N62_from_originals",\n                                                                       "catalog_workspace_relative": f"{rel}/dataset_catalog.json", "approved_for_modeling": False}, indent=1), encoding="utf-8")\n    datos = s1 / "Datos"\n    if not datos.exists():\n        shutil.copytree(capa / "data", datos) if sys.platform == "win32" else datos.symlink_to(capa / "data", target_is_directory=True)\n\n    # Restricted audit ledgers are deliberately not copied into this run.\n\n    # Stage 01: actuarial bases.\n    bdir = s1 / "Bases_actuariales"\n    script = copiar_script(REPO / "src/datos/bases_actuariales.py", bdir)\n    corridas = sorted(p for p in (bdir / "runs").glob("bases_*") if not p.is_symlink() and (p / "manifest.json").exists()) if (bdir / "runs").exists() else []\n    if not corridas:\n        correr([sys.executable, script], bdir, logs / "datos_bases.log")\n        corridas = sorted(p for p in (bdir / "runs").glob("bases_*") if not p.is_symlink() and (p / "manifest.json").exists())\n    bases = corridas[-1]\n    (bdir / "current_run.json").write_text(json.dumps({"workspace_relative_run": f"Hackaton/Semana 1/Bases_actuariales/runs/{bases.name}"}, indent=1), encoding="utf-8")\n    alias = bdir / "runs" / BASES_NOMBRE\n    if not alias.exists():\n        shutil.copytree(bases, alias) if sys.platform == "win32" else alias.symlink_to(bases.name, target_is_directory=True)\n    informe["bases"] = {}\n    for relp, e in esp["bases"].items():\n        h = huellas(bases / relp)\n        informe["bases"][relp] = {"igual_bytes": h["sha256"] == e["sha256"], "igual_contenido": h["contenido"] == e["contenido"]}\n\n    if not all(v["igual_contenido"] for v in informe["bases"].values()):\n        raise SystemExit(f"Las bases requeridas difieren en contenido: {informe[\'bases\']}")\n\n    # Stage 02: partitioned F0 matrix.\n    fdir = s1 / "Matriz_F0"\n    script = copiar_script(REPO / "src/datos/construir_f0.py", fdir)\n    corridas = sorted(p for p in (fdir / "runs").glob("f0_*") if not p.is_symlink() and (p / "manifest.json").exists()) if (fdir / "runs").exists() else []\n    if not corridas:\n        log = logs / "datos_f0.log"\n        correr([sys.executable, script, "--ramo", "motor"], fdir, log)\n        run_id = next(l.split(":", 1)[1].strip() for l in log.read_text(encoding="utf-8").splitlines() if l.startswith("Corrida creada:"))\n        for parte in ("buildings", "contents"):\n            correr([sys.executable, script, "--run", run_id, "--ramo", parte], fdir, log)\n        correr([sys.executable, script, "--run", run_id, "--finalizar"], fdir, log)\n        corridas = [fdir / "runs" / run_id]\n    f0 = corridas[-1]\n    alias = fdir / "runs" / F0_NOMBRE\n    if not alias.exists():\n        shutil.copytree(f0, alias) if sys.platform == "win32" else alias.symlink_to(f0.name, target_is_directory=True)\n    informe["f0"] = {}\n    for relp, e in esp["f0"].items():\n        h = huellas(f0 / relp)\n        informe["f0"][relp] = {"igual_bytes": h["sha256"] == e["sha256"], "igual_contenido": h["contenido"] == e["contenido"]}\n    cols = json.loads((f0 / "contrato_disponibilidad.json").read_text(encoding="utf-8"))["columnas"]\n    bloques = hashlib.sha256(json.dumps({l: {c: v["bloque"] for c, v in cc.items()} for l, cc in cols.items()}, sort_keys=True).encode()).hexdigest()\n    informe["f0"]["contrato_bloques_iguales"] = bloques == esp["f0_contrato_bloques"]\n    if not all(v["igual_contenido"] for k, v in informe["f0"].items() if k != "contrato_bloques_iguales"):\n        raise SystemExit(f"Las tablas F0 requeridas difieren en contenido: {informe[\'f0\']}")\n    if not informe["f0"]["contrato_bloques_iguales"]:\n        raise SystemExit("Los bloques del contrato F0 difieren")\n    return s1, informe\n'


### Embedded source: `herramientas/ejecutar_csv_lf.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['herramientas/ejecutar_csv_lf.py'] = '"""Run final writers with process-local LF defaults in pandas.\n\nThe reference CSV uses LF; pandas otherwise defaults to os.linesep. This child\nprocess changes the writer default without rewriting an already generated CSV.\n"""\nimport os\nimport runpy\nimport sys\nfrom pathlib import Path\n\nREPO = Path(__file__).resolve().parents[1]\npermitidos = {REPO / "src/modelos/mezcla3.py", REPO / "src/linea_b/mezcla3_b.py"}\nscript = Path(sys.argv[1]).resolve()\nif script not in permitidos:\n    raise SystemExit("El escritor no pertenece a los dos puntos de salida finales de la receta")\nos.linesep = "\\n"\nsys.argv = [str(script)] + sys.argv[2:]\nsys.path.insert(0, str(script.parent))\nrunpy.run_path(str(script), run_name="__main__")\n'


### Embedded source: `herramientas/inventario.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['herramientas/inventario.py'] = '"""Inventory frozen recipe artifacts from their original run directories.\n\nFor every recipe output, try <source>/<path>, <source>/runs/<path> and the recorded\nselection-table locations. Inspect Week 1 inputs in the same way. Record file\nsize, byte SHA-256, content fingerprint and provenance in\nreceta/huellas_esperadas.json, and report missing files.\n\nUsage: python herramientas/inventario.py --semana1=<directory> --origen=<directory>\nRepeat --origen to search additional source directories.\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nAQUI = Path(__file__).resolve().parent\nsys.path.insert(0, str(AQUI))\nfrom huella import huellas  # noqa: E402\n\nREPO = AQUI.parent\n\n\ndef localizar(rel: str, origenes: list[Path]) -> Path | None:\n    for o in origenes:\n        cands = [o / rel, o / "runs" / rel]\n        name = Path(rel).name\n        if "/" not in rel and name.startswith("sel_"):\n            corrida = name[4:-8]\n            cands += [o / "runs" / corrida / name, o / "selecciones_v23" / name, o / corrida / name]\n        for c in cands:\n            if c.is_file():\n                return c\n    # Last selection-table fallback: a nested run such as runs/S08h_hogar_F3s2/sel_hS2.parquet.\n    if "/" not in rel and Path(rel).name.startswith("sel_"):\n        for o in origenes:\n            hall = sorted((o / "runs").glob(f"*/{Path(rel).name}")) if (o / "runs").is_dir() else []\n            if hall:\n                return hall[0]\n    return None\n\n\ndef main():\n    opts = [a for a in sys.argv[1:] if a.startswith("--")]\n    semana1 = Path(next(a.split("=", 1)[1] for a in opts if a.startswith("--semana1=")))\n    origenes = [Path(a.split("=", 1)[1]) for a in opts if a.startswith("--origen=")]\n    receta = json.loads((REPO / "receta" / "receta.json").read_text(encoding="utf-8"))\n    out = {"creado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "origenes": [str(o) for o in origenes],\n           "semana1": str(semana1), "entradas": {}, "salidas": {}, "faltan": []}\n    for rel in receta["entradas"]["archivos"]:\n        p = semana1 / rel\n        if p.is_file():\n            out["entradas"][rel] = huellas(p) | {"origen": str(p)}\n        else:\n            out["faltan"].append(rel)\n    for n in receta["nodos"]:\n        for rel in n["salidas"]:\n            p = localizar(rel, origenes)\n            if p is None:\n                out["faltan"].append(rel)\n                continue\n            out["salidas"][rel] = huellas(p) | {"origen": str(p), "nodo": n["id"]}\n            print(n["id"], rel, out["salidas"][rel]["sha256"][:12], flush=True)\n    (REPO / "receta" / "huellas_esperadas.json").write_text(json.dumps(out, indent=1, ensure_ascii=False) + "\\n", encoding="utf-8")\n    print("salidas:", len(out["salidas"]), "entradas:", len(out["entradas"]), "faltan:", out["faltan"])\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/anexar_entrega.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/anexar_entrega.py'] = '"""Append existing validation evidence to a candidate delivery record.\n\nRead paired-bootstrap, simulated-public, reserve-sensitivity, robustness,\nhalf-year and level-drift records from the workspace. Add optional line means,\nrow-level prediction provenance and the preregistered decision criterion when\navailable. Preserve CRLF in the delivery JSON; do not recalculate model results.\n\nUsage: python anexar_entrega.py <key> <candidate> <previous> <sha256> <weight_selection> [<shape>]\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport sys\nfrom pathlib import Path\n\nfrom comun import WORK\n\nO = Path("/mnt/user-data/outputs/Envios")\n\n\ndef c(v, k=5):\n    return f"{v:+.{k}f}".replace(".", ",")\n\n\ndef main():\n    key, cand, prev, sha, seleccion = sys.argv[1:6]\n    forma = sys.argv[6] if len(sys.argv) > 6 else "motor:1.09:0.95,buildings:1.08:1.14,contents:1.07:1"\n    P = O / f"{key.upper()}_model_delivery.json"\n    j = json.loads(P.read_text(encoding="utf-8"))\n    b = json.loads((WORK / f"bootstrap_reference_{key}.json").read_text())\n    ps = json.loads((WORK / f"publico_simulado_{key}.json").read_text())\n    inf = json.loads((WORK / f"influencia_reservas_{key}.json").read_text())\n    rob = json.loads((WORK / f"robustez_{key}_frente_{prev.lower()}.json").read_text())["candidatos"][cand]\n    sem = json.loads((WORK / f"semestres_{key}.json").read_text())\n    der = json.loads((WORK / f"deriva_nivel_{key}.json").read_text())\n\n    def ic_txt(d):\n        t = f"{c(d[\'dif\'], 6)}; IC95 cliente [{c(d[\'ic95_cliente\'][0], 6)}; {c(d[\'ic95_cliente\'][1], 6)}]"\n        if "ic95_direccion" in d:\n            t += f", dirección [{c(d[\'ic95_direccion\'][0], 6)}; {c(d[\'ic95_direccion\'][1], 6)}]"\n        return t\n\n    def fmt(d):\n        return (f"{c(d[\'dif\'])}; IC95 cliente [{c(d[\'ic95_cliente\'][0])}; {c(d[\'ic95_cliente\'][1])}], "\n                f"dirección [{c(d[\'ic95_direccion\'][0])}; {c(d[\'ic95_direccion\'][1])}]")\n    j["validacion"]["intervalos_2024_frente_a_A_G_y_H"] = {\n        "valores": {k: {"frente_a_AG": fmt(b[k]["CL_menos_AG"]), "frente_a_H": fmt(b[k]["CL_menos_H"])} for k in ("motor", "buildings", "contents", "agregado")},\n        "metodo": "Modelos_base/bootstrap_reference.py: 1.000 réplicas Poisson por grupo, denominador fijo; mismas filas y T0",\n        "nota": "A_G y H ajustan su meta dentro de 2024, así que la comparación local los favorece"}\n    j["validacion"][f"frente_a_{prev}"] = {f: {k: f"{c(v[\'dif\'])} (P>0 {str(v[\'P_mejora\']).replace(\'.\', \',\')})" for k, v in rob[f].items()} for f in ("V23", "V24")}\n    j["validacion"][f"frente_a_{prev}"]["sin_el_0_1_por_ciento_superior_V23_V24"] = {k: f"{c(rob[\'V23\'][k][\'dif_sin_top01\'])} / {c(rob[\'V24\'][k][\'dif_sin_top01\'])}" for k in rob["V24"]}\n    tab = {}\n    cn = inf["candidato"]; other = inf["comparadores"][0]\n    for a in ("0.75", "1.0", "1.25"):\n        tab[a] = {}\n        for scope in ("motor", "buildings", "contents", "agregado"):\n            row = inf["escenarios"][a][scope]\n            tab[a][scope] = {f"d2_{cn}_{other}_H_A_G": [row["d2"][k] for k in (cn, other, "H", "A_G")],\n                             f"{cn}_menos_H": ic_txt(row[f"{cn}_menos_H"]),\n                             f"{cn}_menos_{other}": ic_txt(row[f"{cn}_menos_{other}"]),\n                             f"{cn}_menos_A_G": ic_txt(row[f"{cn}_menos_A_G"]),\n                             "diez_clientes_mas_favorables_frente_a_H": c(row[f"{cn}_menos_H"]["diez_clientes_mas_favorables"], 6),\n                             "frente_a_H_sin_esos_diez": c(row[f"{cn}_menos_H"]["dif_sin_esos_diez"], 6)}\n            if a == "1.25" and "cambio_numerador_por_peligro_a_mas25" in row[f"{cn}_menos_H"]:\n                tab[a][scope]["cambio_numerador_frente_a_H_por_peligro"] = row[f"{cn}_menos_H"]["cambio_numerador_por_peligro_a_mas25"]\n    j["validacion"]["reservas_de_caso_2024"] = {"tabla": tab, "metodo": "Modelos_base/influencia_reservas.py: etiqueta T0 + (a - 1) x reserva de caso de los abiertos aceptados (a = 0,75; 1; 1,25), predicciones sin cambio; IC95 por cliente; los diez clientes más favorables son un diagnóstico de influencia, no un criterio de descarte"}\n    st = {}\n    for a, res in sem["resultados"].items():\n        st[a] = {k: {"filas": v["filas"], "peso_exposicion": v["peso_exposicion"], "madurez": v["madurez"],\n                     f"{cn}_menos_H": ic_txt(v[f"{cn}_menos_H"]), f"{cn}_menos_{other}": ic_txt(v[f"{cn}_menos_{other}"]),\n                     f"{cn}_menos_A_G": ic_txt(v[f"{cn}_menos_A_G"])} for k, v in res.items()}\n    j["validacion"]["semestres_2024"] = {"tabla": st, "metodo": "Modelos_base/semestres.py: semestre de inicio del periodo (H1 enero-junio, H2 julio-diciembre); madurez = exposición efectiva / nominal; IC95 por cliente y, si se calculó, por dirección"}\n    if (WORK / f"medias_por_ramo_{key}.json").exists():\n        med = json.loads((WORK / f"medias_por_ramo_{key}.json").read_text())\n        j["validacion"]["medias_por_ramo"] = {"tabla": med["medias"], "definiciones": med["definiciones"],\n                                              "metodo": f"Modelos_base/exportar_predicciones.py (experimentos/medias_por_ramo_{key}.json)"}\n    pq = WORK / f"predicciones_v23_v24_{key}.parquet"\n    if pq.exists():\n        h = hashlib.sha256(pq.read_bytes()).hexdigest()\n        j["predicciones_por_fila_v23_v24"] = {"archivo": f"Hackaton/Semana 1/Modelos_base/experimentos/predicciones_v23_v24_{key}.parquet", "sha256": h,\n                                              "columnas": ["row_id", "line", "corte", "y_T0", "reserva_caso", "prediction"],\n                                              "nota": "las mismas predicciones que miden d2_sel (V23, sin factor) y d2_v24_f23 (V24, con el factor de V23); reserva_caso permite rehacer T0 + (a - 1) x reserva"}\n    if (WORK / f"criterio_{key}.json").exists():\n        j["criterio_fijado_antes_de_resultados"] = json.loads((WORK / f"criterio_{key}.json").read_text(encoding="utf-8"))\n    p30 = ps["V24"]["30%"]; p30v23 = ps["V23"]["30%"]\n    k_sp = f"{cand}-S09b"; k_prev = f"{prev}-{cand}"\n    f1 = lambda x: f"{100 * x:.1f}".replace(".", ",")\n    j["publico_simulado"] = (f"En 1.000 subconjuntos aleatorios del 30 % de los clientes, {cand} supera a S09b en el {f1(p30[k_sp][\'P_mayor_que_0\'])} % de los casos en V24 y el {f1(p30v23[k_sp][\'P_mayor_que_0\'])} % en V23, "\n                             f"y a {prev} en el {f1(1 - p30[k_prev][\'P_mayor_que_0\'])} % (V24) y el {f1(1 - p30v23[k_prev][\'P_mayor_que_0\'])} % (V23) (Modelos_base/experimentos/publico_simulado_{key}.json)")\n    j["validacion"]["seleccion_de_pesos"] = seleccion\n    j["validacion"]["regla_de_nivel"] = {"metodo": "mezcla3.py --nivel=s17: fuera de la referencia de nivel los componentes cuyo total 2025/V24 se aparta más de un 1,5 % del ancla del ramo",\n                                         "deriva_por_componente": {line: {r: v["deriva"] for r, v in comps.items()} for line, comps in der.items()}}\n    j["reproduccion"] = (f"En el equipo: python3 reproducir_mezcla.py experimentos/spec_{key}_propio.json {forma} s17 {cand}_main {sha} "\n                         "(desde Modelos_base; repite la mezcla a partir de las predicciones guardadas de cada componente)")\n    txt = json.dumps(j, indent=1, ensure_ascii=False).replace("\\n", "\\r\\n") + "\\r\\n"\n    P.write_text(txt, encoding="utf-8", newline="")\n    print(j["publico_simulado"])\n    print(json.dumps(j["validacion"][f"frente_a_{prev}"], ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/bootstrap_reference.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/bootstrap_reference.py'] = '"""Compute paired 2024 uncertainty intervals against the A_G and H references.\n\nUse 1,000 Poisson(1) group-bootstrap replicates by customer or address, with a\nfixed denominator, for line-level and aggregate differences in Tweedie D2.\nReference predictions are read without modification. Their within-2024 customer\ncross-validation does not make the local meta-model result an out-of-time test.\n\nUsage: python bootstrap_reference.py <output.json> <spec>@<shape>\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK, deviance\nfrom mezcla3 import labels\nfrom robustez import blend, parse\n\nC = "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelado_reference/nocturna_20261017_N01"\nB = 1000\n\n\ndef ci(dif, groups, null, rng):\n    codes, inv = np.unique(groups, return_inverse=True)\n    d = np.bincount(inv, weights=dif)\n    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null\n    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)], round(float((bs > 0).mean()), 3)\n\n\ndef main():\n    out, arg = sys.argv[1], sys.argv[2]\n    _, spec, forma = parse("CL=" + arg)\n    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam\n    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate\n    rng = np.random.default_rng(20261018)\n    rep, allv = {}, {"y": [], "CL": [], "AG": [], "H": [], "cu": [], "ad": []}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "address_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        _, _, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)\n        y = y24.values\n        P = {"CL": b24, "AG": ag.reindex(y24.index).values, "H": hh.reindex(y24.index).values}\n        cu = fe.customer_id.reindex(y24.index).astype(str).values; ad = fe.address_id.reindex(y24.index).astype(str).values\n        null = deviance(y, np.full_like(y, y.mean())).sum()\n        r = {}\n        for ref in ("AG", "H"):\n            dif = deviance(y, P[ref]) - deviance(y, P["CL"])\n            r[f"CL_menos_{ref}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ci(dif, cu, null, rng)[0], "ic95_direccion": ci(dif, ad, null, rng)[0]}\n        rep[line] = r\n        for k, v in (("y", y), ("cu", cu), ("ad", ad)):\n            allv[k].append(v)\n        for k in ("CL", "AG", "H"):\n            allv[k].append(P[k])\n        print(line, r, flush=True)\n    y = np.concatenate(allv["y"]); cu = np.concatenate(allv["cu"]); ad = np.concatenate(allv["ad"])\n    null = deviance(y, np.full_like(y, y.mean())).sum()\n    g = {}\n    for ref in ("AG", "H"):\n        dif = deviance(y, np.concatenate(allv[ref])) - deviance(y, np.concatenate(allv["CL"]))\n        g[f"CL_menos_{ref}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ci(dif, cu, null, rng)[0], "ic95_direccion": ci(dif, ad, null, rng)[0]}\n    rep["agregado"] = g\n    print("agregado", g, flush=True)\n    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/calibracion_deciles.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/calibracion_deciles.py'] = '"""Inspect observed-to-expected ratios by predicted-rate decile in V23 and V24.\n\nUse the evaluated blend, including its V23-derived V24 level factor. Form deciles\nfrom prediction/effective exposure and also report ratios excluding the largest\n0.1% of observed losses, to show the influence of the household tail.\n\nUsage: python calibracion_deciles.py <spec>@<shape>@<level> <output.json>\n"""\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK\nfrom mezcla3 import labels\nfrom robustez import blend, parse\n\n\ndef main():\n    _, spec, forma = parse("C=" + sys.argv[1])\n    rep = {}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        _, bsel, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)\n        rep[line] = {}\n        for fold, y, mu, e in (("V23", ysel.values, bsel, expos[0]), ("V24", y24.values, b24, expos[1])):\n            rate = mu / e\n            q = np.quantile(rate, np.linspace(0, 1, 11)); dec = np.clip(np.searchsorted(q, rate, side="right") - 1, 0, 9)\n            keep = y <= np.quantile(y, 0.999)\n            ae = [round(float(y[dec == k].sum() / mu[dec == k].sum()), 3) for k in range(10)]\n            ae_sin = [round(float(y[(dec == k) & keep].sum() / mu[(dec == k) & keep].sum()), 3) for k in range(10)]\n            rep[line][fold] = {"real_sobre_esperado_por_decil": ae, "sin_0_1_por_ciento_superior": ae_sin, "total": round(float(y.sum() / mu.sum()), 4)}\n            print(line, fold, "total", rep[line][fold]["total"], "deciles", ae, flush=True)\n            print("   sin top 0,1 %", ae_sin, flush=True)\n    (WORK / sys.argv[2]).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/calibrar_siguiente_anio.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/calibrar_siguiente_anio.py'] = '"""Calibrate each line to the year after training with Tweedie power 1.5.\n\nV23 uses training years 2021-2022 and the 2023 selection partition; V24 uses\ntraining years 2021-2023 and the 2024 observations. For each cut calculate\nf = sum(y/sqrt(mu)) / sum(sqrt(mu)); apply the mean of both factors to 2025.\nThe predictions already include effective-exposure development, so the factor\ndoes not apply IBNR a second time. The historical transfer check applies the\nV23 factor to V24; these cuts informed model decisions.\n\nUsage: python calibrar_siguiente_anio.py <base_run> <output> <line>:<V23_selection>[,...]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import BASE, LINES, WORK, d2, digest\n\n\ndef ftw(y, mu):\n    return float((y / np.sqrt(mu)).sum() / np.sqrt(mu).sum())\n\n\ndef main():\n    base_run, out_name = sys.argv[1], sys.argv[2]\n    sel_map = dict(kv.split(":") for kv in sys.argv[3].split(","))\n    base = WORK / base_run\n    pred = pd.read_parquet(base / "prediccion_2025.parquet")\n    v24 = pd.read_parquet(base / "v24.parquet")\n    out = WORK / out_name\n    out.mkdir(parents=True, exist_ok=True)\n    rep = {"base_run": base_run, "lines": {}}\n    v24c = v24.copy()\n    for line in LINES:\n        s = pd.read_parquet(WORK / f"sel_{sel_map[line]}.parquet")\n        s = s[s.line == line]\n        b = v24[v24.line == line]\n        f23, f24 = ftw(s.y.values, s.mu.values), ftw(b.y.values, b.mu.values)\n        f = (f23 + f24) / 2\n        rep["lines"][line] = {"f_v23": round(f23, 4), "f_v24": round(f24, 4), "f_aplicado": round(f, 4),\n                              "d2_v24_sin": round(d2(b.y, b.mu), 5), "d2_v24_con_f_v23": round(d2(b.y, b.mu * f23), 5), "sel_v23": sel_map[line]}\n        v24c.loc[v24c.line == line, "mu"] = b.mu.values * f23\n        pred.loc[pred.line == line, "prediction"] = pred.loc[pred.line == line, "prediction"].values * f\n    rep["d2_v24_global_sin"] = round(d2(v24.y, v24.mu), 5)\n    rep["d2_v24_global_con_f_v23"] = round(d2(v24c.y, v24c.mu), 5)\n    sample = pd.read_parquet(BASE / "Datos" / "sample_submission.parquet", columns=["row_id"])\n    sub = sample.merge(pred[["row_id", "prediction"]], on="row_id", how="left", validate="one_to_one")\n    assert len(sub) == 626639 and np.isfinite(sub.prediction).all() and (sub.prediction > 0).all()\n    csv = out / f"{out_name}.csv"\n    sub.to_csv(csv, index=False, float_format="%.6f")\n    back = pd.read_csv(csv, dtype={"row_id": str})\n    assert (back.row_id.values == sample.row_id.values).all() and len(back) == 626639\n    rep["file"] = {"path": str(csv), "sha256": digest(csv), "rows": 626639, "total_CU": round(float(back.prediction.sum()), 2),\n                   "by_line_CU": {l: round(float(pred[pred.line == l].prediction.sum()), 2) for l in LINES}}\n    pred.to_parquet(out / "prediccion_2025.parquet", index=False)\n    (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n    print(json.dumps(rep, indent=1))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/cat_experimento.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/cat_experimento.py'] = '"""Fit CatBoost Tweedie under the same exposure contract as LightGBM in V23.\n\nUse effective exposure, the motor cap with a uniform surcharge and motor F3\nfeatures. Stop on the calibration half and compare on the selection half.\nSave sel_<name>.parquet for the frozen blending workflow.\n\nUsage: python cat_experimento.py <name> [<lines>] [--depth=6] [--lr=0.05] [--f3s2]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostRegressor, Pool\n\nfrom comun import LINES, WORK, d2\nfrom experimentos import features, load\n\nCAPS = {"motor": 15000.0}\n# The seven approved household customer features are enabled only with --f3s2 (D80).\nF3S2 = ["prior_claims_5y", "credit_band", "missed_payments_12m", "household_size", "years_at_address", "marital_status", "licence_points"]\n\n\ndef frame(df, feats):\n    X = df[feats].copy()\n    cats = []\n    for c in X.columns:\n        s = X[c]\n        if pd.api.types.is_bool_dtype(s.dtype):\n            X[c] = s.astype("float32")\n        elif pd.api.types.is_numeric_dtype(s.dtype):\n            X[c] = s.astype("float64")\n        else:\n            if pd.api.types.infer_dtype(s, skipna=True) == "boolean":\n                X[c] = s.map({True: 1.0, False: 0.0}).astype("float32")\n            else:\n                X[c] = s.astype(str).fillna("nan")\n                cats.append(c)\n    return X, cats\n\n\ndef main():\n    name = sys.argv[1]\n    lines = sys.argv[2].split(",") if len(sys.argv) > 2 and not sys.argv[2].startswith("--") else LINES\n    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[2:] if a.startswith("--"))\n    depth, lr = int(opts.get("depth", 6)), float(opts.get("lr", 0.05))\n    power = float(opts.get("p", 1.5))\n    if "cap" in opts:\n        CAPS["motor"] = float(opts["cap"])\n    rep, frames = {}, []\n    for line in lines:\n        t = time.time()\n        df = load(line)\n        feats = features(line, df, "F01234" if line == "motor" else "F0124")\n        if line != "motor" and "f3s2" in opts:\n            feats += [c for c in F3S2 if c in df.columns and c not in feats]\n        X, cats = frame(df, feats)\n        tr = df.year.isin([2021, 2022]).values\n        cal = ((df.year == 2023) & (df.cal_half == 0)).values\n        sel = ((df.year == 2023) & (df.cal_half == 1)).values\n        y = df["T0"].fillna(0).values\n        off = df["e_ef_coste"].values\n        cap = CAPS.get(line)\n        yt = np.minimum(y, cap) if cap else y\n        loading = y[tr].sum() / yt[tr].sum() if cap else 1.0\n        base = float(np.log(yt[tr].sum() / off[tr].sum()))\n        ptr = Pool(X[tr], yt[tr], cat_features=cats, baseline=np.log(off[tr]) + base)\n        pcal = Pool(X[cal], yt[cal], cat_features=cats, baseline=np.log(off[cal]) + base)\n        m = CatBoostRegressor(loss_function=f"Tweedie:variance_power={power}", iterations=4000, learning_rate=lr, depth=depth, l2_leaf_reg=10,\n                              od_type="Iter", od_wait=200, thread_count=2, random_seed=20261017, verbose=0, boost_from_average=False,\n                              one_hot_max_size=8, use_best_model=True)\n        m.fit(ptr, eval_set=pcal)\n        raw = m.predict(Pool(X[sel], cat_features=cats), prediction_type="RawFormulaVal")\n        mu = np.exp(raw + np.log(off[sel]) + base) * loading\n        rep[line] = {"d2_sel": round(d2(y[sel], mu), 5), "best": int(m.get_best_iteration()), "balance_sel": round(float(mu.sum() / y[sel].sum()), 4), "secs": round(time.time() - t)}\n        print(name, line, rep[line], flush=True)\n        frames.append(pd.DataFrame({"row_id": df.row_id[sel].astype(str).values, "line": line, "y": y[sel], "mu": mu}))\n    out = pd.concat(frames, ignore_index=True)\n    out.to_parquet(WORK / f"sel_{name}.parquet", index=False)\n    if len(lines) == 3:\n        rep["global"] = round(d2(out.y, out.mu), 5)\n        print("== ", name, "global", rep["global"], flush=True)\n    (WORK / f"eval_{name}.json").write_text(json.dumps(rep, indent=1))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/cat_frecsev.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/cat_frecsev.py'] = '"""Fit peril-specific CatBoost Poisson frequency times training mean severity.\n\nmu = sum_p lambda_p(x) * s_p. Frequency uses the log effective-exposure baseline;\ns_p is uncapped training peril cost divided by its claim count. Use motor F3\nfeatures and, with --f3s2, the seven approved household customer features.\nV23 stops on calibration and saves selection predictions. --prod adds V24 with\n1.15 times the selected iterations and 2025 with 1.3 times, using nominal exposure\nand the requested seeds. Save each completed peril as v23_<p>.npz, v24_<p>.npy or\nprod_<p>.npy so the same command can resume completed work after validation.\nNumeric features use float32 for memory; the historical motor V23/V24 fit used\nfloat64. --desde and --semilla_inicial reuse only the selected V23 iteration\ncounts when adding new seeds for V24 and production.\n\nUsage: python cat_frecsev.py <name> <line> [--prod] [--solo_prod] [--semillas=2]\n       [--depth=6] [--lr=0.05] [--f3s2] [--desde=<run>] [--semilla_inicial=0]\n"""\nfrom __future__ import annotations\n\nimport gc\nimport json\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostRegressor, Pool\n\nfrom cat_experimento import frame\nfrom comun import PERILS, WORK, d2\nfrom experimentos import features, load\n\nF3S2 = ["prior_claims_5y", "credit_band", "missed_payments_12m", "household_size", "years_at_address", "marital_status", "licence_points"]\n\n\ndef main():\n    name, line = sys.argv[1], sys.argv[2]\n    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[3:] if a.startswith("--"))\n    seeds, depth, lr = int(opts.get("semillas", 2)), int(opts.get("depth", 6)), float(opts.get("lr", 0.05))\n    s0 = int(opts.get("semilla_inicial", 0))  # Offset the seed sequence when adding seeds to an earlier run.\n    t0 = time.time()\n    df = load(line)\n    feats = features(line, df, "F01234" if line == "motor" else "F0124")\n    if line != "motor" and "f3s2" in opts:\n        feats += [c for c in F3S2 if c in df.columns and c not in feats]\n    X, cats = frame(df, feats)\n    for c in X.columns:\n        if c in cats:\n            X[c] = X[c].astype("category")\n        elif X[c].dtype == "float64":\n            X[c] = X[c].astype("float32")\n    y = df["T0"].fillna(0).values\n    yr = df.year.values\n    expo = df["exposure"].values\n    row_ids = df.row_id.astype(str).values\n    cal_half = df.cal_half.values\n    per = {p: (df[f"n_{p}"].fillna(0).values.astype(float), df[f"y_{p}"].fillna(0).values, df[f"e_ef_{p}"].values) for p in PERILS[line]}\n    del df\n    gc.collect()\n    tr = yr <= 2022; cal = (yr == 2023) & (cal_half == 0); sel = (yr == 2023) & (cal_half == 1)\n    out = WORK / name\n    out.mkdir(parents=True, exist_ok=True)\n    if "solo_prod" in opts:\n        # Production only: use saved V23 iterations and preserve each completed peril.\n        rep = json.loads((out / "resumen.json").read_text())\n        te, trm = yr == 2025, yr <= 2024\n        # Keep compact training/test copies instead of the full feature frame.\n        Xtr, Xte = X.loc[trm].copy(), X.loc[te].copy()\n        del X\n        for Z in (Xtr, Xte):\n            for c in Z.columns:\n                if Z[c].dtype == "float64":\n                    Z[c] = Z[c].astype("float32")\n        gc.collect()\n        mu = np.zeros(te.sum())\n        for p, (n, yp, off) in per.items():\n            fp = out / f"prod_{p}.npy"\n            if fp.exists():\n                mu += np.load(fp)\n                print(name, "2025", p, "reanudado", flush=True)\n                continue\n            sev = yp[trm].sum() / max(n[trm].sum(), 1.0)\n            base = float(np.log(n[trm].sum() / off[trm].sum()))\n            ptr = Pool(Xtr, n[trm], cat_features=cats, baseline=np.log(off[trm]) + base)\n            pev = Pool(Xte, cat_features=cats)\n            part = np.zeros(te.sum())\n            for s in range(seeds):\n                m = CatBoostRegressor(loss_function="Poisson", iterations=int(round(max(rep["perils"][p]["best"], 20) * 1.3)), learning_rate=lr, depth=depth,\n                                      l2_leaf_reg=10, thread_count=2, random_seed=20261017 + 101 * (s0 + s), verbose=0, boost_from_average=False, one_hot_max_size=8)\n                m.fit(ptr)\n                part += np.exp(m.predict(pev, prediction_type="RawFormulaVal") + np.log(expo[te]) + base) * sev / seeds\n            np.save(fp, part)\n            del ptr, pev, m\n            gc.collect()\n            mu += part\n            print(name, "2025", p, round(time.time() - t0), flush=True)\n        rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)\n        pd.DataFrame({"row_id": row_ids[te], "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)\n        (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n        print(name, "2025 total", rep["pred_2025_total_CU"], flush=True)\n        return\n    rep = {"line": line, "depth": depth, "lr": lr, "seeds": seeds, "f3s2": "f3s2" in opts, "perils": {}}\n    mu_sel = np.zeros(sel.sum())\n    best = {}\n    if "desde" in opts:\n        # Additional seeds reuse the earlier V23 iteration counts for V24 and 2025.\n        src = json.loads((WORK / opts["desde"] / "resumen.json").read_text())\n        best = {p: int(src["perils"][p]["best"]) for p in per}\n        rep.update({"desde": opts["desde"], "semilla_inicial": s0, "perils": {p: {"best": best[p]} for p in per}})\n        (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n    for p, (n, yp, off) in ([] if "desde" in opts else per.items()):\n        sev = yp[tr].sum() / max(n[tr].sum(), 1.0)\n        base = float(np.log(n[tr].sum() / off[tr].sum()))\n        fv = out / f"v23_{p}.npz"\n        if fv.exists():\n            z = np.load(fv)\n            best[p], lam = int(z["best"]), z["lam"]\n            print(name, "V23", p, "reanudado", flush=True)\n        else:\n            m = CatBoostRegressor(loss_function="Poisson", iterations=3000, learning_rate=lr, depth=depth, l2_leaf_reg=10, od_type="Iter", od_wait=150,\n                                  thread_count=2, random_seed=20261017, verbose=0, boost_from_average=False, one_hot_max_size=8, use_best_model=True)\n            m.fit(Pool(X[tr], n[tr], cat_features=cats, baseline=np.log(off[tr]) + base),\n                  eval_set=Pool(X[cal], n[cal], cat_features=cats, baseline=np.log(off[cal]) + base))\n            best[p] = max(int(m.get_best_iteration()) + 1, 20)\n            lam = np.exp(m.predict(Pool(X[sel], cat_features=cats), prediction_type="RawFormulaVal") + np.log(off[sel]) + base)\n            np.savez(fv, best=best[p], lam=lam)\n            del m\n            gc.collect()\n        mu_sel += lam * sev\n        rep["perils"][p] = {"best": best[p], "sev": round(float(sev), 2), "d2_coste_peligro_sel": round(d2(yp[sel], lam * sev), 5)}\n        print(name, p, rep["perils"][p], round(time.time() - t0), flush=True)\n    if "desde" not in opts:\n        rep["d2_sel"] = round(d2(y[sel], mu_sel), 5)\n        print(name, "V23 selección", rep["d2_sel"], flush=True)\n        pd.DataFrame({"row_id": row_ids[sel], "line": line, "y": y[sel], "mu": mu_sel}).to_parquet(WORK / f"sel_{name}.parquet", index=False)\n        (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n    if "prod" not in opts:\n        return\n    for tag, trm, evm, mult in (("v24", yr <= 2023, yr == 2024, 1.15), ("2025", yr <= 2024, yr == 2025, 1.3)):\n        mu = np.zeros(evm.sum())\n        for p, (n, yp, off) in per.items():\n            fp = out / (f"v24_{p}.npy" if tag == "v24" else f"prod_{p}.npy")\n            if fp.exists():\n                mu += np.load(fp)\n                print(name, tag, p, "reanudado", flush=True)\n                continue\n            sev = yp[trm].sum() / max(n[trm].sum(), 1.0)\n            base = float(np.log(n[trm].sum() / off[trm].sum()))\n            ptr = Pool(X[trm], n[trm], cat_features=cats, baseline=np.log(off[trm]) + base)\n            pev = Pool(X[evm], cat_features=cats)\n            offe = off[evm] if tag == "v24" else expo[evm]\n            part = np.zeros(evm.sum())\n            for s in range(seeds):\n                m = CatBoostRegressor(loss_function="Poisson", iterations=int(round(best[p] * mult)), learning_rate=lr, depth=depth, l2_leaf_reg=10,\n                                      thread_count=2, random_seed=20261017 + 101 * (s0 + s), verbose=0, boost_from_average=False, one_hot_max_size=8)\n                m.fit(ptr)\n                part += np.exp(m.predict(pev, prediction_type="RawFormulaVal") + np.log(offe) + base) * sev / seeds\n                del m\n            np.save(fp, part)\n            mu += part\n            del ptr, pev\n            gc.collect()\n            print(name, tag, p, round(time.time() - t0), flush=True)\n        if tag == "v24":\n            rep["d2_v24"] = round(d2(y[evm], mu), 5)\n            pd.DataFrame({"row_id": row_ids[evm], "line": line, "y": y[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)\n            print(name, "V24", rep["d2_v24"], flush=True)\n        else:\n            rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)\n            pd.DataFrame({"row_id": row_ids[evm], "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)\n        (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n    rep["secs"] = round(time.time() - t0)\n    (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n    print(name, json.dumps(rep), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/cat_produccion.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/cat_produccion.py'] = '"""Fit production CatBoost Tweedie models for one insurance line.\n\nV24 trains on 2021-2023 and predicts 2024; production trains on 2021-2024 for 2025.\nKeep the effective-exposure offset, motor cap/uniform surcharge and motor F3\ncontract. Use the V23-selected iteration count times 1.15 for V24 and 1.3 for\nproduction. Write v24.parquet, prediccion_2025.parquet and resumen.json.\n\nUsage: python cat_produccion.py <name> <line> <V23_iterations> [--semillas=2]\n       [--semilla_inicial=0] [--depth=6] [--lr=0.05] [--drop=a,b]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostRegressor, Pool\n\nfrom cat_experimento import CAPS, frame\nfrom comun import WORK, d2\nfrom experimentos import features, load\n\n\ndef main():\n    name, line, it = sys.argv[1], sys.argv[2], int(sys.argv[3])\n    opts = dict(a[2:].split("=", 1) for a in sys.argv[4:] if a.startswith("--"))\n    seeds, depth, lr = int(opts.get("semillas", 2)), int(opts.get("depth", 6)), float(opts.get("lr", 0.05))\n    s0 = int(opts.get("semilla_inicial", 0))  # Offset the seed sequence to avoid repeating previously fitted seeds.\n    drop = set(opts["drop"].split(",")) if opts.get("drop") else set()\n    t = time.time()\n    df = load(line)\n    feats = [f for f in features(line, df, "F01234" if line == "motor" else "F0124") if f not in drop]\n    X, cats = frame(df, feats)\n    for c in cats:  # Compact categorical arrays preserve the information with lower memory use.\n        X[c] = X[c].astype("category")\n    y = df["T0"].fillna(0).values\n    off = df["e_ef_coste"].values\n    expo = df["exposure"].values\n    years = df["year"].values\n    row_ids = df["row_id"].astype(str).values\n    del df\n    import gc\n    gc.collect()\n    cap = CAPS.get(line)\n    out = WORK / name\n    out.mkdir(parents=True, exist_ok=True)\n    res = {"line": line, "iter_v23": it, "seeds": seeds, "depth": depth, "lr": lr, "drop": sorted(drop)}\n    tasks = (("v24", years <= 2023, years == 2024, off, 1.15), ("2025", years <= 2024, years == 2025, expo, 1.3))\n    if opts.get("solo2025"):\n        tasks = tasks[1:]\n    for tag, trm, evm, offe, mult in tasks:\n        yt = np.minimum(y, cap) if cap else y\n        loading = y[trm].sum() / yt[trm].sum() if cap else 1.0\n        base = float(np.log(yt[trm].sum() / off[trm].sum()))\n        ptr = Pool(X[trm], yt[trm], cat_features=cats, baseline=np.log(off[trm]) + base)\n        pev = Pool(X[evm], cat_features=cats)\n        mu = np.zeros(evm.sum())\n        for s in range(s0, s0 + seeds):\n            m = CatBoostRegressor(loss_function="Tweedie:variance_power=1.5", iterations=int(round(it * mult)), learning_rate=lr, depth=depth,\n                                  l2_leaf_reg=10, thread_count=2, random_seed=20261017 + 101 * s, verbose=0, boost_from_average=False, one_hot_max_size=8)\n            m.fit(ptr)\n            mu += np.exp(m.predict(pev, prediction_type="RawFormulaVal") + np.log(offe[evm]) + base) * loading / seeds\n        if tag == "v24":\n            res["d2_v24"] = round(d2(y[evm], mu), 5)\n            res["balance_v24"] = round(float(mu.sum() / y[evm].sum()), 4)\n            pd.DataFrame({"row_id": row_ids[evm], "line": line, "y": y[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)\n        else:\n            res["pred_2025_total_CU"] = round(float(mu.sum()), 2)\n            pd.DataFrame({"row_id": row_ids[evm], "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)\n        print(tag, res, flush=True)\n    res["secs"] = round(time.time() - t)\n    prev = json.loads((out / "resumen.json").read_text()) if (out / "resumen.json").exists() else {}\n    (out / "resumen.json").write_text(json.dumps(prev | res, indent=1))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/combinar_semillas.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/combinar_semillas.py'] = '"""Average production runs of the same model using different random seeds.\n\nWeight each run by its seed count. Keep the single early-stopped V23 selection\nfit unchanged. Combine v24.parquet and prediccion_2025.parquet.\nUsage: python combinar_semillas.py <output_name> <run1> <run2> [...]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport pandas as pd\n\nfrom comun import WORK, d2\n\nname, runs = sys.argv[1], sys.argv[2:]\nout = WORK / name\nout.mkdir(parents=True, exist_ok=True)\nreps = [json.loads((WORK / r / "resumen.json").read_text()) for r in runs]\nw = [r.get("seeds", 1) for r in reps]\nrep = {"corridas": runs, "semillas": w}\nfor f, col in (("v24.parquet", "mu"), ("prediccion_2025.parquet", "prediction")):\n    frames = [pd.read_parquet(WORK / r / f).set_index("row_id") for r in runs]\n    base = frames[0].copy()\n    for fr in frames[1:]:\n        assert (fr.index == base.index).all()\n    base[col] = sum(wi * fr[col] for wi, fr in zip(w, frames)) / sum(w)\n    base.reset_index().to_parquet(out / f, index=False)\n    if f == "v24.parquet":\n        rep["d2_v24"] = round(d2(base.y.values, base.mu.values), 5)\n        rep["d2_v24_por_corrida"] = [round(d2(fr.y.values, fr.mu.values), 5) for fr in frames]\n    else:\n        rep["pred_2025_total_CU"] = round(float(base.prediction.sum()), 2)\n(out / "resumen.json").write_text(json.dumps(rep, indent=1))\nprint(json.dumps(rep), flush=True)\n'


### Embedded source: `src/modelos/compare_reference.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/compare_reference.py'] = '"""Compare a blend with A_G and H on identical 2024 rows and fixed mixtures.\n\nA_G is a GAM meta-model and H a nested geometric meta-model from the N01 archive.\nTheir customer cross-validation occurs within 2024, so their local D2 is optimistic\nrelative to an out-of-time test. Reconstruct the candidate as in mezcla3, applying\nthe V23 factor to V24. Report both its original level and its line-specific 2024\nTweedie-optimal level to distinguish level from shape. Read references unchanged.\nUsage: python compare_reference.py <output.json> <spec> <shape>\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK, d2\nfrom mezcla3 import ftw, labels\nfrom robustez import blend, parse\n\nC = "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelado_reference/nocturna_20261017_N01"\n\n\ndef main():\n    out_path, spec_arg = sys.argv[1], sys.argv[2]\n    name, spec, forma = parse("CL=" + spec_arg)\n    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "T0", "meta_gam", "S09b"]).set_index("row_id")\n    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate", "anchor"]).set_index("row_id")\n    rep = {}\n    allv = {k: [] for k in ("y", "CL", "AG", "H", "S09b")}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        _, _, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)\n        idx = y24.index\n        assert ag.index.isin(idx).sum() >= len(idx) * 0.999\n        y = y24.values\n        assert np.allclose(ag.T0.reindex(idx).values, y, atol=0.01)\n        P = {"CL": b24, "AG": ag.meta_gam.reindex(idx).values, "H": hh.candidate.reindex(idx).values, "S09b": ag.S09b.reindex(idx).values}\n        assert all(np.isfinite(v).all() for v in P.values())\n        r = {k: {"d2": round(d2(y, v), 5), "d2_renivelado": round(d2(y, v * ftw(y, v)), 5), "nivel_sobre_real": round(float(v.sum() / y.sum()), 4)} for k, v in P.items()}\n        for other in ("AG", "H"):\n            for w in (0.3, 0.5, 0.7):\n                for kind in ("arit", "geom"):\n                    m = w * P["CL"] + (1 - w) * P[other] if kind == "arit" else np.exp(w * np.log(P["CL"]) + (1 - w) * np.log(P[other]))\n                    r[f"{kind}_{w}CL_{other}"] = {"d2": round(d2(y, m), 5), "d2_renivelado": round(d2(y, m * ftw(y, m)), 5)}\n        rep[line] = r\n        for k in allv:\n            allv[k].append(y if k == "y" else P[k])\n        print(line, {k: (v["d2"], v["d2_renivelado"]) for k, v in r.items()}, flush=True)\n    y = np.concatenate(allv["y"]); P = {k: np.concatenate(v) for k, v in allv.items() if k != "y"}\n    g = {k: round(d2(y, v), 5) for k, v in P.items()}\n    for other in ("AG", "H"):\n        for w in (0.3, 0.5, 0.7):\n            g[f"arit_{w}CL_{other}"] = round(d2(y, w * P["CL"] + (1 - w) * P[other]), 5)\n            g[f"geom_{w}CL_{other}"] = round(d2(y, np.exp(w * np.log(P["CL"]) + (1 - w) * np.log(P[other]))), 5)\n    rep["global"] = g\n    print("global", g, flush=True)\n    (WORK / out_path).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/comun.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/comun.py'] = '"""Shared model paths, Tweedie metric, feature loading and block assignments.\n\nAdditional features use information before 1 January of the policy-period year,\nwith the same cutoff in training and test. Inputs are fingerprint-verified F0\nmatrices, claim assignments and original weather tables.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nBASE = Path(os.environ.get("HAKATON_SEMANA1", "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1"))\nF0RUN = "f0_20261017T141732Z_65df7fca"\nF0DIR = BASE / "Matriz_F0" / "runs" / F0RUN\nBASESRUN = "bases_20261017T120251Z_45443ff0"\nBASESDIR = BASE / "Bases_actuariales" / "runs" / BASESRUN\nWORK = Path(os.environ.get("MODELOS_WORK", "/home/participant/work"))\nLINES = ["motor", "buildings", "contents"]\nPERILS = {"motor": ["own_damage", "third_party_prop", "bodily_injury", "theft", "glass_hail"],\n          "buildings": ["storm", "flood", "escape_of_water", "subsidence", "fire", "accidental_damage"],\n          "contents": ["theft", "escape_of_water", "accidental_damage", "flood"]}\nWEATHER_VARS = ["hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index", "wildfire_index", "rainfall_mm", "mean_temperature_c"]\nPOWER = 1.5\nNEVER = {"row_id", "policy_id", "customer_id", "address_id", "period_start", "period_end", "is_test", "year"}\n\n\ndef digest(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef deviance(y, mu, p=POWER):\n    y = np.asarray(y, float)\n    mu = np.asarray(mu, float)\n    return 2.0 * (np.power(y, 2 - p) / ((1 - p) * (2 - p)) - y * np.power(mu, 1 - p) / (1 - p) + np.power(mu, 2 - p) / (2 - p))\n\n\ndef d2(y, mu, p=POWER) -> float:\n    y = np.asarray(y, float)\n    return float(1.0 - deviance(y, mu, p).sum() / deviance(y, np.full_like(y, y.mean()), p).sum())\n\n\ndef contract() -> dict:\n    return json.loads((F0DIR / "contrato_disponibilidad.json").read_text(encoding="utf-8"))["columnas"]\n\n\ndef build_extra(line: str) -> pd.DataFrame:\n    """Load a line\'s F0 plus F1, F2 and F4 features using its 1 January cutoff."""\n    df = pd.read_parquet(F0DIR / "tables" / f"f0_{line}.parquet")\n    a = pd.read_parquet(BASESDIR / "tables" / "asignacion_por_siniestro.parquet")\n    a["report_year"] = a.report_date.dt.year.astype(int)\n\n    # F1 by policy and peril: notifications reported before 1 January of the period year.\n    ap = a[a.line == line]\n    parts = []\n    for y in range(2021, 2026):\n        prev = ap[ap.report_year < y]\n        if len(prev):\n            w = prev.groupby(["policy_id", "peril"]).size().unstack(fill_value=0)\n            w.columns = [f"f1x_notif_{c}" for c in w.columns]\n            w = w.reset_index()\n            w["year"] = y\n            parts.append(w)\n    hist = pd.concat(parts, ignore_index=True)\n    df = df.merge(hist, on=["policy_id", "year"], how="left")\n    for p in PERILS[line]:\n        col = f"f1x_notif_{p}"\n        if col not in df.columns:\n            df[col] = 0\n        df[col] = df[col].fillna(0).astype("int32")\n\n    # Previously observed policy exposure and prior notification rate.\n    tr = df.loc[~df.is_test, ["policy_id", "year", "exposure"]]\n    pe = tr.groupby(["policy_id", "year"])["exposure"].sum().unstack(fill_value=0.0)\n    cum = pe.cumsum(axis=1).shift(1, axis=1).fillna(0.0)\n    cum[2025] = pe.sum(axis=1)\n    cum = cum.stack().rename("f1x_exposicion_previa").reset_index().rename(columns={"level_1": "year"})\n    cum["year"] = cum["year"].astype(int)\n    df = df.merge(cum, on=["policy_id", "year"], how="left")\n    df["f1x_exposicion_previa"] = df["f1x_exposicion_previa"].fillna(0.0)\n    df["f1x_tasa_notif_previa"] = np.where(df.f1x_exposicion_previa > 0, df.f1_notificaciones_previas / df.f1x_exposicion_previa.clip(lower=0.25), np.nan)\n\n    # Household address history includes earlier buildings and contents notifications.\n    if line in ("buildings", "contents"):\n        rows = pd.concat([pd.read_parquet(F0DIR / "tables" / f"f0_{l}.parquet", columns=["row_id", "address_id"]).assign(src=l) for l in ("buildings", "contents")], ignore_index=True)\n        ah = a[a.line.isin(["buildings", "contents"]) & a.row_id.notna()].merge(rows[["row_id", "address_id"]], on="row_id", how="inner")\n        other = "contents" if line == "buildings" else "buildings"\n        parts = []\n        for y in range(2021, 2026):\n            prev = ah[ah.report_year < y]\n            g = prev.groupby("address_id").agg(addr_notif_hogar=("claim_id", "size")).reset_index()\n            o = prev[prev.line == other].groupby("address_id").agg(addr_notif_otro_ramo=("claim_id", "size")).reset_index()\n            fl = prev[prev.peril.isin(["flood", "subsidence", "storm", "escape_of_water"])].groupby(["address_id", "peril"]).size().unstack(fill_value=0)\n            fl.columns = [f"addr_notif_{c}" for c in fl.columns]\n            g = g.merge(o, on="address_id", how="outer").merge(fl.reset_index(), on="address_id", how="outer")\n            g["year"] = y\n            parts.append(g)\n        ad = pd.concat(parts, ignore_index=True)\n        df = df.merge(ad, on=["address_id", "year"], how="left")\n        for c in [c for c in df.columns if c.startswith("addr_notif_")]:\n            df[c] = df[c].fillna(0).astype("int32")\n\n    # F2: address-level weather averages using years before the policy period.\n    w = pd.read_parquet(BASE / "Datos" / "weather.parquet").sort_values(["address_id", "year"])\n    g = w.groupby("address_id")\n    n_prev = g.cumcount()\n    for v in WEATHER_VARS:\n        s = g[v].cumsum() - w[v]\n        w[f"{v}_mprev"] = np.where(n_prev > 0, s / n_prev.replace(0, np.nan), np.nan)\n    w = w[["address_id", "year"] + [f"{v}_mprev" for v in WEATHER_VARS]]\n    df = df.merge(w, on=["address_id", "year"], how="left")\n\n    # F4: annualized written premium relative to its line/year mean, removing tariff-level trend.\n    rate = df.written_premium / df.exposure\n    ymean = df.groupby("year")["written_premium"].sum() / df.groupby("year")["exposure"].sum()\n    df["prem_rel"] = rate / df.year.map(ymean)\n    df["log_prem_rel"] = np.log(df.prem_rel.clip(lower=1e-6))\n    return df\n\n\ndef blocks_for(line: str, df: pd.DataFrame) -> dict:\n    """Assign candidate columns to feature blocks under the D68 availability contract."""\n    c = contract()[line]\n    out = {}\n    for col in df.columns:\n        if col in NEVER or col.startswith("e_ef_"):\n            continue\n        if col in c:\n            b = c[col]["bloque"]\n        elif col.startswith("f1x_") or col.startswith("addr_notif"):\n            b = "F1"\n        elif col.endswith("_mprev"):\n            b = "F2"\n        elif col in ("prem_rel", "log_prem_rel"):\n            b = "F4"\n        else:\n            b = "revisar"\n        if b in ("F0", "F1", "F2", "F3", "F4"):\n            out[col] = b\n    return out\n\n\ndef to_model_frame(df: pd.DataFrame, feats: list[str]) -> pd.DataFrame:\n    """Convert all rows of a line together so LightGBM shares categorical levels across cuts."""\n    X = df[feats].copy()\n    for col in X.columns:\n        s = X[col]\n        if isinstance(s.dtype, pd.CategoricalDtype):\n            continue\n        if pd.api.types.is_bool_dtype(s.dtype):\n            X[col] = s.astype("float32")\n        elif pd.api.types.is_numeric_dtype(s.dtype):\n            if str(s.dtype).startswith(("Int", "Float")):\n                X[col] = s.astype("float64")\n        else:\n            if pd.api.types.infer_dtype(s, skipna=True) == "boolean":\n                X[col] = s.map({True: 1.0, False: 0.0}).astype("float32")\n            else:\n                X[col] = s.astype("category")\n    return X\n'


### Embedded source: `src/modelos/construir_variables.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/construir_variables.py'] = '"""Build and save each line\'s augmented features as WORK/feat_<line>.parquet."""\nimport sys, time\nfrom comun import LINES, WORK, build_extra\nWORK.mkdir(parents=True, exist_ok=True)\nfor line in (sys.argv[1:] or LINES):\n    t = time.time()\n    df = build_extra(line)\n    df.to_parquet(WORK / f"feat_{line}.parquet", index=False)\n    extra = [c for c in df.columns if c.startswith(("f1x_", "addr_notif")) or c.endswith("_mprev") or c in ("prem_rel", "log_prem_rel")]\n    print(line, df.shape, f"{time.time()-t:.0f}s", extra, flush=True)\n'


### Embedded source: `src/modelos/deriva_nivel.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/deriva_nivel.py'] = '"""Inspect unlabeled component level drift: 2025/V24 totals against each line\'s anchor.\n\nThe s17 rule excludes components drifting more than 1.5% from the anchor from\nthe blend\'s level reference. Anchors are CAT_motor_4s, S12h_edif_md1000 and\nS08h_hogar_F3s2 for motor, buildings and contents respectively.\nUsage: python deriva_nivel.py <spec_json> [<output.json>]\n"""\nimport json\nimport sys\n\nfrom comun import WORK\nfrom mezcla3 import component, sin_referencia\n\nANCLA = {"motor": ("CAT_motor_4s", "cat1a"), "buildings": ("S12h_edif_md1000", "hS3"), "contents": ("S08h_hogar_F3s2", "hS2")}\n\n\ndef tot(run, sel, line):\n    s, v, p = component(run, sel, line)\n    return float(v.sum()), float(p.sum())\n\n\ndef main():\n    spec = json.load(open(sys.argv[1]))\n    excl = sin_referencia({"nivel": "s17"})\n    rep = {}\n    for line, comps in spec.items():\n        a24, a25 = tot(*ANCLA[line], line)\n        rep[line] = {}\n        for run, sel, w in comps:\n            v24, p25 = tot(run, sel, line)\n            der = (p25 / v24) / (a25 / a24) - 1.0\n            rep[line][run] = {"peso": w, "deriva": round(der, 4), "fuera_de_la_referencia": run in excl, "regla_coherente": (abs(der) > 0.015) == (run in excl) or run.startswith("MLP")}\n            print(line, run, rep[line][run], flush=True)\n    if len(sys.argv) > 2:\n        (WORK / sys.argv[2]).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/diagnostico_cuerpo_cola.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/diagnostico_cuerpo_cola.py'] = '"""Diagnose candidate ordering in the body of the 2024 loss distribution.\n\nFor archived A_G, H, S09b, S21 and S22 predictions, report D2 on all rows and\nafter excluding observed losses above p99.9, p99.5 and p99. Relevel the optional\ncomparison within each subset to distinguish shape from level. Filtering by the\nobserved label is a diagnostic, not a valid prospective scoring or selection rule.\nRead all archived predictions unchanged.\nUsage: python diagnostico_cuerpo_cola.py <output.json>\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import WORK, d2\n\nC = "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelado_reference/nocturna_20261017_N01"\nPUBLICO = {"A_G": 0.12908, "S09b": 0.12889, "H": 0.12886, "S21": 0.12874}  # Historical comparison and submission ledger reference: COD-20261018-049.\n\n\ndef ftw(y, mu):\n    return float((y / np.sqrt(mu)).sum() / np.sqrt(mu).sum())\n\n\ndef main():\n    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "line", "T0", "S09b", "meta_gam"]).set_index("row_id")\n    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id")\n    df = ag.join(hh).rename(columns={"meta_gam": "A_G", "candidate": "H"})\n    for k in ("s21", "s22"):\n        p = pd.read_parquet(WORK / f"predicciones_v23_v24_{k}.parquet")\n        df = df.join(p[p.corte == "V24"].set_index("row_id")[["prediction"]].rename(columns={"prediction": k.upper()}))\n    cands = ["A_G", "S09b", "H", "S21", "S22"]\n    assert df[cands].notna().all().all()\n    y = df.T0.values\n    rep = {"publico": PUBLICO, "orden_publico": sorted(PUBLICO, key=PUBLICO.get, reverse=True), "bruto": {}, "renivelado": {},\n           "media_predicha_V24": {k: round(float(df[k].mean()), 2) for k in cands}, "media_T0_V24": round(float(y.mean()), 2)}\n    for q in (None, 0.999, 0.995, 0.99):\n        m = np.ones(len(y), bool) if q is None else (y <= np.quantile(y, q))\n        lab = "completo" if q is None else f"sin_p{q * 100:g}"\n        bruto, ren = {}, {}\n        for k in cands:\n            mu = df[k].values[m]; yy = y[m]; ln = df.line.values[m]\n            bruto[k] = round(d2(yy, mu), 5)\n            mu2 = mu.copy()\n            for line in ("motor", "buildings", "contents"):\n                l = ln == line\n                mu2[l] = mu[l] * ftw(yy[l], mu[l])\n            ren[k] = round(d2(yy, mu2), 5)\n        rep["bruto"][lab] = {"d2": bruto, "orden": sorted(bruto, key=bruto.get, reverse=True)}\n        rep["renivelado"][lab] = {"d2": ren, "orden": sorted(ren, key=ren.get, reverse=True)}\n        print(lab, "bruto", rep["bruto"][lab]["orden"], "renivelado", rep["renivelado"][lab]["orden"], ren, flush=True)\n    (WORK / sys.argv[1]).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/diagnostico_forma_nivel.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/diagnostico_forma_nivel.py'] = '"""Measure the effect of shape calibration on 2025 level without 2025 labels.\n\nApply mu\' = e * (e/e_bar)^(b-1) * g * (r/g)^a. V23/V24 labels allow the later\nTweedie factor to absorb changes in their totals; 2025 uses the mean of those\nfactors. Different exposure definitions or rate dispersion can therefore shift\n2025 relative to an unshaped blend. Report shaped/unshaped total ratios and\nfactors by line and cut, together with the resulting 2025 total.\nUsage: python diagnostico_forma_nivel.py <spec_json> <shape> <level|-> [<output.json>]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK, d2\nfrom mezcla3 import component, ftw, labels, shape_rate, sin_referencia\n\n\ndef main():\n    spec = json.load(open(sys.argv[1]))\n    forma = {kv.split(":")[0]: (float(kv.split(":")[1]), float(kv.split(":")[2])) for kv in sys.argv[2].split(",") if kv}\n    nivel = sys.argv[3]\n    excl = sin_referencia({"nivel": None if nivel == "-" else nivel})\n    rep = {}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        ex = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        exn = (fe.exposure.reindex(ysel.index).values, fe.exposure.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        raw = []\n        for run, sel_name, w in spec[line]:\n            s, v, p = component(run, sel_name, line)\n            raw.append((run, w, s.reindex(ysel.index).values, v.reindex(y24.index).values, p.reindex(idx25).values))\n        own = [r for r in raw if r[0] not in excl]\n        wo = sum(r[1] for r in own)\n        ref = [sum(r[1] * r[j].sum() for r in own) / wo for j in (2, 3, 4)]\n        ws = np.array([r[1] for r in raw], float); ws /= ws.sum()\n        blends = [sum(w * r[j] * ref[j - 2] / r[j].sum() for w, r in zip(ws, raw)) for j in (2, 3, 4)]\n        a, b = forma.get(line, (1.0, 1.0))\n        shaped = [shape_rate(m, e, a, b) for m, e in zip(blends, ex)]\n        L = [float(s.sum() / m.sum()) for s, m in zip(shaped, blends)]\n        f_sin = [ftw(ysel.values, blends[0]), ftw(y24.values, blends[1])]\n        f_con = [ftw(ysel.values, shaped[0]), ftw(y24.values, shaped[1])]\n        tot25_sin = float(blends[2].sum() * np.mean(f_sin)); tot25_con = float(shaped[2].sum() * np.mean(f_con))\n        # Exposure dispersion: mean of (e/e_bar)^(b-1) and exposure quantiles in each cut.\n        disp = {}\n        for k, e in zip(("V23", "V24", "2025"), ex):\n            eb = np.exp(np.log(e).mean())\n            disp[k] = {"media_(e/eb)^(b-1)": round(float(np.mean((e / eb) ** (b - 1.0))), 4), "e_p10_p50_p90": [round(float(q), 3) for q in np.quantile(e, [0.1, 0.5, 0.9])],\n                       "media_e": round(float(e.mean()), 4)}\n        rep[line] = {"forma": [a, b], "L_V23_V24_2025": [round(x, 4) for x in L], "f_sin_forma": [round(x, 4) for x in f_sin], "f_con_forma": [round(x, 4) for x in f_con],\n                     "total_2025_sin_forma": round(tot25_sin, 1), "total_2025_con_forma": round(tot25_con, 1), "cociente_con_sin": round(tot25_con / tot25_sin, 4),\n                     "d2_sin": [round(d2(ysel.values, blends[0]), 5), round(d2(y24.values, blends[1] * f_sin[0]), 5)],\n                     "d2_con": [round(d2(ysel.values, shaped[0]), 5), round(d2(y24.values, shaped[1] * f_con[0]), 5)], "exposicion": disp}\n        print(line, json.dumps(rep[line], ensure_ascii=False), flush=True)\n    if len(sys.argv) > 4:\n        (WORK / sys.argv[4]).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/escenarios_reserva.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/escenarios_reserva.py'] = '"""Evaluate case-reserve sensitivity with fixed model predictions.\n\nFor accepted open claims, T1(a) = T0 + (a-1)*R, where T0 is net incurred cost and\nR the row\'s case reserve. Use a=0.75 and 1.25 as evaluation scenarios; reserves\nare never predictors and the T0-derived next-year factors remain unchanged.\nReport V23-selection and V24 D2 plus paired customer-bootstrap differences using\n1,000 Poisson(1) replicates and a fixed denominator. Compare A_G/H only in V24.\nUsage: python escenarios_reserva.py <output.json> <ref>=<spec>[@<shape>[@<level>]] <cand>=<spec> [...]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import BASE, BASESDIR, LINES, WORK, d2, deviance\nfrom mezcla3 import labels\nfrom robustez import blend, parse\n\nC = "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelado_reference/nocturna_20261017_N01"\nCLAIMS = BASE / "Limpieza" / "runs" / "preaudit_20261016T211816Z_817285a5" / "data" / "claims.parquet"\nA = (0.75, 1.0, 1.25)\nB = 1000\n\n\ndef reserva_por_fila():\n    c = pd.read_parquet(CLAIMS, columns=["claim_id", "status", "paid_to_date", "incurred_to_date", "audit_pre00"])\n    a = pd.read_parquet(BASESDIR / "tables" / "asignacion_por_siniestro.parquet", columns=["claim_id", "category", "row_id"])\n    m = a.merge(c, on="claim_id", how="left", validate="one_to_one")\n    ok = (m.status == "open") & (m.category == "unica") & (~m.audit_pre00.fillna(False).astype(bool)) & m.row_id.notna()\n    r = (m.incurred_to_date - m.paid_to_date)[ok].groupby(m.row_id[ok].astype(str)).sum()\n    return r\n\n\ndef boot(dif, groups, null, rng):\n    codes, inv = np.unique(groups, return_inverse=True)\n    d = np.bincount(inv, weights=dif)\n    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null\n    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)], round(float((bs > 0).mean()), 3)\n\n\ndef main():\n    out = sys.argv[1]\n    cands = [parse(x) for x in sys.argv[2:]]\n    ref = cands[0][0]\n    R = reserva_por_fila()\n    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam\n    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate\n    rng = np.random.default_rng(20261018)\n    store = {f: {"y": [], "r": [], "cu": [], "P": {}} for f in ("V23", "V24")}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        for f, ys in (("V23", ysel), ("V24", y24)):\n            store[f]["y"].append(ys.values); store[f]["r"].append(R.reindex(ys.index).fillna(0.0).values)\n            store[f]["cu"].append(fe.customer_id.reindex(ys.index).astype(str).values)\n        for name, spec, forma in cands:\n            _, bsel, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)\n            store["V23"]["P"].setdefault(name, []).append(bsel); store["V24"]["P"].setdefault(name, []).append(b24)\n        store["V24"]["P"].setdefault("A_G", []).append(ag.reindex(y24.index).values)\n        store["V24"]["P"].setdefault("H", []).append(hh.reindex(y24.index).values)\n        store["V23"].setdefault("line", []).append(np.full(len(ysel), line)); store["V24"].setdefault("line", []).append(np.full(len(y24), line))\n    rep = {"referencia": ref, "reserva_abiertos_CU": {f: round(float(np.concatenate(store[f]["r"]).sum()), 0) for f in store}, "escenarios": {}}\n    for a in A:\n        ra = {}\n        for f in ("V23", "V24"):\n            y0 = np.concatenate(store[f]["y"]); r = np.concatenate(store[f]["r"]); cu = np.concatenate(store[f]["cu"]); ln = np.concatenate(store[f]["line"])\n            y = y0 + (a - 1.0) * r\n            P = {k: np.concatenate(v) for k, v in store[f]["P"].items()}\n            res = {}\n            for scope in LINES + ["agregado"]:\n                msk = np.ones(len(y), bool) if scope == "agregado" else (ln == scope)\n                ys, cus = y[msk], cu[msk]\n                null = deviance(ys, np.full_like(ys, ys.mean())).sum()\n                row = {"d2": {k: round(d2(ys, p[msk]), 5) for k, p in P.items()}}\n                for k in P:\n                    if k == ref:\n                        continue\n                    dif = deviance(ys, P[ref][msk]) - deviance(ys, P[k][msk])\n                    ic, pm = boot(dif, cus, null, rng)\n                    row[f"{k}_menos_{ref}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ic, "P_mejora": pm}\n                res[scope] = row\n            ra[f] = res\n            print(f"a={a} {f} agregado", {k: v for k, v in res["agregado"]["d2"].items()}, flush=True)\n            for scope in LINES + ["agregado"]:\n                print("   ", scope, {k: (v["dif"], v["ic95_cliente"]) for k, v in res[scope].items() if k != "d2"}, flush=True)\n        rep["escenarios"][str(a)] = ra\n    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/evaluar.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/evaluar.py'] = '"""Evaluate a line configuration on the V23 calibration/selection split.\n\ntotal fits line T0 using e_ef_coste; componentes fits each peril\'s y_<peril>\nusing e_ef_<peril> and sums predictions. Configurations may specify a line cap\nor peril-specific caps, and per-line drop/add column lists. Save selection\npredictions for subsequent frozen blends.\nUsage: python evaluar.py <name> <feature_set> <config_json> <mode> [<lines>]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, PERILS, WORK, d2, to_model_frame\nfrom experimentos import features, fit, load, predict\n\n\ndef main():\n    name, set_name, cfg_path, mode = sys.argv[1:5]\n    cfg = json.loads(open(cfg_path).read())\n    rep, ys, mus, frames = {}, [], [], []\n    lines = sys.argv[5].split(",") if len(sys.argv) > 5 else LINES\n    for line in lines:\n        t = time.time()\n        df = load(line)\n        feats = [f for f in features(line, df, set_name) if f not in set(cfg[line].get("drop", []))]\n        feats += [f for f in cfg[line].get("add", []) if f in df.columns and f not in feats]\n        X = to_model_frame(df, feats)\n        tr = df.year.isin([2021, 2022]).values\n        cal = ((df.year == 2023) & (df.cal_half == 0)).values\n        sel = ((df.year == 2023) & (df.cal_half == 1)).values\n        y = df["T0"].fillna(0).values\n        if mode == "total":\n            off = df["e_ef_coste"].values\n            b, base = fit(X[tr], y[tr], off[tr], X[cal], y[cal], off[cal], rounds=6000, early=200, params=dict(cfg[line]["params"]))\n            best = b.best_iteration or b.current_iteration()\n            mu = predict(b, base, X[sel], off[sel], best)\n            info = {"best": best}\n        else:\n            mu = np.zeros(sel.sum())\n            info = {}\n            for p in PERILS[line]:\n                yp = df[f"y_{p}"].fillna(0).values\n                op = df[f"e_ef_{p}"].values\n                params = dict(cfg[line]["params"])\n                params.pop("cap", None)\n                c = cfg[line].get("caps", {}).get(p)\n                if c:\n                    params["cap"] = c\n                b, base = fit(X[tr], yp[tr], op[tr], X[cal], yp[cal], op[cal], rounds=6000, early=200, params=params)\n                best = b.best_iteration or b.current_iteration()\n                mp = predict(b, base, X[sel], op[sel], best)\n                info[p] = {"best": best, "d2": round(d2(yp[sel], mp), 5)}\n                mu += mp\n        rep[line] = {"d2_sel": round(d2(y[sel], mu), 5), "balance_sel": round(float(mu.sum() / y[sel].sum()), 4), "n_feats": len(feats), **info, "secs": round(time.time() - t)}\n        print(name, line, rep[line], flush=True)\n        ys.append(y[sel]); mus.append(mu)\n        frames.append(pd.DataFrame({"row_id": df.row_id[sel].astype(str).values, "line": line, "y": y[sel], "mu": mu}))\n    rep["global"] = round(d2(np.concatenate(ys), np.concatenate(mus)), 5)\n    print("== ", name, "global sel", rep["global"], flush=True)\n    pd.concat(frames, ignore_index=True).to_parquet(WORK / f"sel_{name}.parquet", index=False)\n    (WORK / f"eval_{name}.json").write_text(json.dumps(rep, indent=1, default=float))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/experimentos.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/experimentos.py'] = '"""Run line-specific LightGBM Tweedie fits with effective-exposure offsets.\n\nV23 trains on 2021-2022, stops on 2023 cal_half=0 and compares on cal_half=1.\nThe registered V24 evaluation trains on 2021-2023 using V23-fixed rounds.\nCalculate power-1.5 D2 on amounts, per line and across concatenated lines.\nUsage: python experimentos.py <output_name> <feature_set>[,...] [--v24]\nFeature sets: M00, F0, F012, F0124, F01234 (without marketing_segment), F01234m.\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\n\nfrom comun import F0DIR, LINES, WORK, blocks_for, d2, to_model_frame\n\nPARAMS = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.05, num_leaves=31, min_data_in_leaf=500,\n              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, max_bin=255, num_threads=2,\n              verbose=-1, boost_from_average=False, cat_smooth=50, min_data_per_group=200, max_cat_to_onehot=8, seed=20261017)\nSETS = {"F0": ["F0"], "F012": ["F0", "F1", "F2"], "F0124": ["F0", "F1", "F2", "F4"], "F01234": ["F0", "F1", "F2", "F3", "F4"], "F01234m": ["F0", "F1", "F2", "F3", "F4"]}\nALWAYS_OUT = {"marketing_segment"}\n\n\ndef load(line):\n    df = pd.read_parquet(WORK / f"feat_{line}.parquet")\n    part = pd.read_parquet(F0DIR / "tables" / "particiones.parquet", columns=["row_id", "cal_half"])\n    df = df.merge(part, on="row_id", how="left", validate="one_to_one")\n    return df\n\n\ndef features(line, df, set_name):\n    blocks = blocks_for(line, df)\n    keep = SETS[set_name]\n    feats = [c for c, b in blocks.items() if b in keep]\n    if set_name != "F01234m":\n        feats = [c for c in feats if c not in ALWAYS_OUT]\n    return feats\n\n\ndef fit(X, y, off, Xv=None, yv=None, offv=None, rounds=4000, early=150, params=None):\n    """Fit LightGBM Tweedie with a log-exposure offset; cap training rows if specified, return excess as a uniform loading, and evaluate full labels."""\n    params = dict(PARAMS, **(params or {}))\n    cap = params.pop("cap", None)\n    loading = 1.0\n    if cap:\n        yc = np.minimum(y, cap)\n        loading = float(y.sum() / yc.sum())\n        y = yc\n        if yv is not None:\n            yv = np.minimum(yv, cap)\n    base = float(np.log(y.sum() / off.sum()))\n    dtr = lgb.Dataset(X, label=y, init_score=np.log(off) + base, free_raw_data=False)\n    valid = []\n    if Xv is not None:\n        valid = [lgb.Dataset(Xv, label=yv, init_score=np.log(offv) + base, reference=dtr)]\n    cb = [lgb.early_stopping(early, verbose=False)] if (Xv is not None and early) else []\n    booster = lgb.train(params, dtr, num_boost_round=rounds, valid_sets=valid, callbacks=cb)\n    booster.loading = loading\n    return booster, base\n\n\ndef predict(booster, base, X, off, iters=None):\n    raw = booster.predict(X, raw_score=True, num_iteration=iters)\n    return np.exp(raw + base + np.log(off)) * getattr(booster, "loading", 1.0)\n\n\ndef main():\n    out_name = sys.argv[1]\n    sets = sys.argv[2].split(",")\n    do_v24 = "--v24" in sys.argv\n    extra = {}\n    for a in sys.argv[3:]:\n        if a.startswith("--param="):\n            k, v = a[len("--param="):].split(":", 1)\n            extra[k] = float(v) if v.replace(".", "", 1).replace("-", "", 1).isdigit() else v\n    results = {"params": dict(PARAMS, **extra), "sets": {}}\n    for set_name in sets:\n        sel_y, sel_mu, v24_y, v24_mu = [], [], [], []\n        per_line = {}\n        for line in LINES:\n            t = time.time()\n            df = load(line)\n            tr = df.year.isin([2021, 2022]).values\n            cal = ((df.year == 2023) & (df.cal_half == 0)).values\n            sel = ((df.year == 2023) & (df.cal_half == 1)).values\n            y = df["T0"].fillna(0).values\n            off = df["e_ef_coste"].values\n            if set_name == "M00":\n                rate = y[tr].sum() / off[tr].sum()\n                mu_sel = rate * off[sel]\n                best = 0\n                info = {"rate": rate}\n                if do_v24:\n                    tr24 = (df.year <= 2023).values\n                    ev24 = (df.year == 2024).values\n                    mu24 = y[tr24].sum() / off[tr24].sum() * off[ev24]\n            else:\n                feats = features(line, df, set_name)\n                X = to_model_frame(df, feats)\n                booster, base = fit(X[tr], y[tr], off[tr], X[cal], y[cal], off[cal], params=extra)\n                best = booster.best_iteration or booster.current_iteration()\n                mu_sel = predict(booster, base, X[sel], off[sel], best)\n                imp = pd.Series(booster.feature_importance("gain"), index=feats).sort_values(ascending=False)\n                info = {"best_iter": best, "n_feats": len(feats), "top_gain": {k: round(float(v / imp.sum()), 4) for k, v in imp.head(15).items()}}\n                if do_v24:\n                    tr24 = (df.year <= 2023).values\n                    ev24 = (df.year == 2024).values\n                    b24, base24 = fit(X[tr24], y[tr24], off[tr24], rounds=int(round(best * 1.1)) or 100, early=None, params=extra)\n                    mu24 = predict(b24, base24, X[ev24], off[ev24])\n            per_line[line] = {"d2_sel": round(d2(y[sel], mu_sel), 5), "balance_sel": round(float(mu_sel.sum() / y[sel].sum()), 4), **info, "secs": round(time.time() - t)}\n            sel_y.append(y[sel]); sel_mu.append(mu_sel)\n            if do_v24:\n                per_line[line]["d2_v24"] = round(d2(y[ev24], mu24), 5)\n                per_line[line]["balance_v24"] = round(float(mu24.sum() / y[ev24].sum()), 4)\n                v24_y.append(y[ev24]); v24_mu.append(mu24)\n            print(set_name, line, per_line[line], flush=True)\n        res = {"d2_sel_global": round(d2(np.concatenate(sel_y), np.concatenate(sel_mu)), 5), "lines": per_line}\n        if do_v24:\n            res["d2_v24_global"] = round(d2(np.concatenate(v24_y), np.concatenate(v24_mu)), 5)\n        results["sets"][set_name] = res\n        print("== ", set_name, "global sel", res["d2_sel_global"], "v24", res.get("d2_v24_global"), flush=True)\n        (WORK / f"exp_{out_name}.json").write_text(json.dumps(results, indent=1, default=float))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/exportar_predicciones.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/exportar_predicciones.py'] = '"""Export evaluated V23-selection and V24 blend predictions with audit labels.\n\nV23 uses 2021-2022 training, 2023 cal_half=1 and no next-year factor. V24 uses\n2021-2023 training and the V23 level factor. Preserve the blend\'s shape/level\nrule. Include T0 and accepted-open-claim reserves so a reviewer can reconstruct\nT0+(a-1)*R scenarios at a=0.75, 1 and 1.25 without changing predictions.\nWrite private row-level Parquet and aggregate line-mean JSON outputs.\nUsage: python exportar_predicciones.py <key> <candidate>=<spec>@<shape>@<level>\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK, d2\nfrom escenarios_reserva import reserva_por_fila\nfrom mezcla3 import labels\nfrom robustez import blend, parse\n\n\ndef main():\n    key = sys.argv[1]\n    cn, spec, fm = parse(sys.argv[2])\n    R = reserva_por_fila()\n    filas, medias = [], {}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        res, bsel, b24, b25 = blend(spec, fm, line, ysel, y24, idx25, expos)\n        medias[line] = {}\n        for corte, ys, pr in (("V23", ysel, bsel), ("V24", y24, b24)):\n            r = R.reindex(ys.index.astype(str)).fillna(0.0).values\n            filas.append(pd.DataFrame({"row_id": ys.index.astype(str), "line": line, "corte": corte, "y_T0": ys.values,\n                                       "reserva_caso": r, "prediction": pr}))\n            m = {"filas": int(len(ys)), "media_predicha": round(float(pr.mean()), 4)}\n            for a in (0.75, 1.0, 1.25):\n                y = ys.values + (a - 1.0) * r\n                m[f"a={a}"] = {"media_observada": round(float(y.mean()), 4), "observada_entre_predicha": round(float(y.sum() / pr.sum()), 4),\n                               "d2": round(d2(y, pr), 6)}\n            medias[line][corte] = m\n        medias[line]["2025"] = {"filas": int(len(idx25)), "media_predicha": round(float(b25.mean()), 4)}\n        print(line, json.dumps(medias[line], ensure_ascii=False), flush=True)\n    df = pd.concat(filas, ignore_index=True)\n    df.to_parquet(WORK / f"predicciones_v23_v24_{key}.parquet", index=False)\n    agg = {}\n    for corte in ("V23", "V24"):\n        d = df[df.corte == corte]\n        agg[corte] = {"filas": int(len(d)), "media_predicha": round(float(d.prediction.mean()), 4)}\n        for a in (0.75, 1.0, 1.25):\n            y = d.y_T0.values + (a - 1.0) * d.reserva_caso.values\n            agg[corte][f"a={a}"] = {"media_observada": round(float(y.mean()), 4), "observada_entre_predicha": round(float(y.sum() / d.prediction.sum()), 4),\n                                    "d2": round(d2(y, d.prediction.values), 6)}\n    medias["agregado"] = agg\n    rep = {"candidato": cn, "medias": medias,\n           "definiciones": {"V23": "selección: entrenamiento 2021-22, evaluación 2023 con cal_half = 1; mezcla con forma y regla de nivel, sin factor (d2_sel)",\n                            "V24": "entrenamiento 2021-23, evaluación 2024; mezcla con forma y regla de nivel por el factor ftw de V23 (d2_v24_f23)",\n                            "a": "etiqueta T0 + (a - 1) x reserva de caso de los siniestros abiertos aceptados asignados a la fila"}}\n    (WORK / f"medias_por_ramo_{key}.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n    print(json.dumps(agg, ensure_ascii=False), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/frecsev2.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/frecsev2.py'] = '"""Fit peril frequency with LightGBM Poisson and positive severity with gamma boosting.\n\nFrequency uses peril-specific effective exposure. Fit weighted positive row\nseverity y_p/n_p when at least MIN_SEV positive claims are available; otherwise\nuse constant mean severity. Multiply by the positive-cost claim proportion to\ninclude zero-cost claims. Use the approved household customer features and motor\nF01234. V23 stops on calibration and compares on selection; --prod adds V24\nand 2025 with 1.15 and 1.3 times the selected rounds and the requested seeds.\n--aditivo uses two-leaf trees at learning rate 0.05 for additive models without\ninteractions. --desde and --semilla_inicial add frequency seeds using earlier\nV23 rounds; the gamma severity fit is deterministic. --partes also saves each\nperil\'s frequency times training mean severity for severidad_cfs.py.\nUsage: python frecsev2.py <name> <line> [--prod] [--semillas=2] [--aditivo]\n       [--desde=<run>] [--semilla_inicial=0] [--partes]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import PERILS, WORK, d2, to_model_frame\nfrom experimentos import features, fit, load, predict\n\nMIN_SEV = 1500\nF3S2 = ["prior_claims_5y", "credit_band", "missed_payments_12m", "household_size", "years_at_address", "marital_status", "licence_points"]\nSEV_PARAMS = {"objective": "gamma", "learning_rate": 0.03, "num_leaves": 5, "min_data_in_leaf": 300, "feature_fraction": 0.7, "lambda_l2": 20.0,\n              "cat_smooth": 100, "min_data_per_group": 300, "max_cat_threshold": 16}\n\n\ndef main():\n    name, line = sys.argv[1], sys.argv[2]\n    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[3:] if a.startswith("--"))\n    seeds = int(opts.get("semillas", 2))\n    s0 = int(opts.get("semilla_inicial", 0))  # Add new seeds for averaging with an existing run.\n    t0 = time.time()\n    cfg = json.load(open(WORK / "cfg_total.json"))[line]["params"]\n    df = load(line)\n    feats = features(line, df, "F01234" if line == "motor" else "F0124")\n    if line != "motor":\n        feats += [c for c in F3S2 if c in df.columns and c not in feats]\n    X = to_model_frame(df, feats)\n    y = df["T0"].fillna(0).values\n    yr = df.year.values\n    expo = df["exposure"].values\n    tr = yr <= 2022; cal = (yr == 2023) & (df.cal_half.values == 0); sel = (yr == 2023) & (df.cal_half.values == 1)\n    pf = dict(cfg); pf.pop("cap", None); pf["objective"] = "poisson"; pf.pop("tweedie_variance_power", None)\n    sev_params = dict(SEV_PARAMS)\n    if "aditivo" in opts:\n        pf.update({"num_leaves": 2, "learning_rate": 0.05, "min_data_in_leaf": 2000})\n        sev_params.update({"num_leaves": 2, "learning_rate": 0.05})\n    rep = {"line": line, "aditivo": "aditivo" in opts, "perils": {}}\n    out = WORK / name\n    out.mkdir(parents=True, exist_ok=True)\n    best = {}\n\n    def sev_fit(p, n, yp, trm, valm=None, rounds=None):\n        pos_tr = trm & (yp > 0) & (n > 0)\n        share = float((yp[trm] > 0).sum() / max((n[trm] > 0).sum(), 1))  # Claim-bearing rows with positive observed cost.\n        mean_pos = float(yp[pos_tr].sum() / n[pos_tr].sum())\n        if pos_tr.sum() < MIN_SEV:\n            return None, mean_pos, share, 0\n        s_obs = yp / np.maximum(n, 1)\n        params = dict(sev_params, num_threads=2, verbose=-1, seed=20261017)\n        import lightgbm as lgb\n        dtr = lgb.Dataset(X[pos_tr], label=s_obs[pos_tr], weight=n[pos_tr], init_score=np.full(pos_tr.sum(), np.log(mean_pos)))\n        if valm is not None:\n            pos_v = valm & (yp > 0) & (n > 0)\n            dv = lgb.Dataset(X[pos_v], label=s_obs[pos_v], weight=n[pos_v], init_score=np.full(pos_v.sum(), np.log(mean_pos)), reference=dtr)\n            b = lgb.train(params, dtr, num_boost_round=3000, valid_sets=[dv], callbacks=[lgb.early_stopping(150, verbose=False)])\n            it = max(b.best_iteration, 10)\n        else:\n            b = lgb.train(params, dtr, num_boost_round=rounds)\n            it = rounds\n        return b, mean_pos, share, it\n\n    def sev_pred(b, mean_pos, share, m, it):\n        if b is None:\n            return np.full(m.sum(), mean_pos * share)\n        return np.exp(b.predict(X[m], raw_score=True, num_iteration=it) + np.log(mean_pos)) * share\n\n    mu_sel = np.zeros(sel.sum())\n    if "desde" in opts:\n        src = json.loads((WORK / opts["desde"] / "resumen.json").read_text())\n        best = {p: (int(src["perils"][p]["best_frec"]), int(src["perils"][p]["best_sev"])) for p in PERILS[line]}\n        rep.update({"desde": opts["desde"], "semilla_inicial": s0, "perils": {p: {"best_frec": best[p][0], "best_sev": best[p][1]} for p in PERILS[line]}})\n    for p in ([] if "desde" in opts else PERILS[line]):\n        n = df[f"n_{p}"].fillna(0).values.astype(float)\n        yp = df[f"y_{p}"].fillna(0).values\n        off = df[f"e_ef_{p}"].values\n        b, base = fit(X[tr], n[tr], off[tr], X[cal], n[cal], off[cal], rounds=6000, early=200, params=dict(pf))\n        bf = b.best_iteration or b.current_iteration()\n        lam = predict(b, base, X[sel], off[sel], bf)\n        if "partes" in opts:\n            np.savez(out / f"v23_{p}.npz", best=bf, lam=lam)\n        sb, mean_pos, share, sit = sev_fit(p, n, yp, tr, cal)\n        sv = sev_pred(sb, mean_pos, share, sel, sit if sb is not None else None)\n        best[p] = (bf, sit)\n        mu_sel += lam * sv\n        sv_const = mean_pos * share\n        rep["perils"][p] = {"best_frec": bf, "best_sev": sit, "sev_media": round(sv_const, 1),\n                            "d2_peligro_sev_modelo": round(d2(yp[sel], lam * sv), 5), "d2_peligro_sev_constante": round(d2(yp[sel], lam * sv_const), 5)}\n        print(name, p, rep["perils"][p], round(time.time() - t0), flush=True)\n    if "desde" not in opts:\n        rep["d2_sel"] = round(d2(y[sel], mu_sel), 5)\n        print(name, "V23 selección", rep["d2_sel"], flush=True)\n        pd.DataFrame({"row_id": df.row_id[sel].astype(str).values, "line": line, "y": y[sel], "mu": mu_sel}).to_parquet(WORK / f"sel_{name}.parquet", index=False)\n    (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n    if "prod" not in opts:\n        return\n    for tag, trm, evm, mult in (("v24", yr <= 2023, yr == 2024, 1.15), ("2025", yr <= 2024, yr == 2025, 1.3)):\n        mu = np.zeros(evm.sum())\n        for p in PERILS[line]:\n            n = df[f"n_{p}"].fillna(0).values.astype(float)\n            yp = df[f"y_{p}"].fillna(0).values\n            off = df[f"e_ef_{p}"].values\n            offe = off[evm] if tag == "v24" else expo[evm]\n            bf, sit = best[p]\n            if sit:\n                sb, mean_pos, share, _ = sev_fit(p, n, yp, trm, None, rounds=int(round(sit * mult)))\n            else:\n                pos = trm & (yp > 0) & (n > 0)\n                sb, mean_pos, share = None, float(yp[pos].sum() / n[pos].sum()), float((yp[trm] > 0).sum() / max((n[trm] > 0).sum(), 1))\n            sv = sev_pred(sb, mean_pos, share, evm, None)\n            freq = np.zeros(evm.sum())\n            for s in range(seeds):\n                b, base = fit(X[trm], n[trm], off[trm], rounds=int(round(bf * mult)), early=None, params=dict(pf, seed=20261017 + 101 * (s0 + s)))\n                freq += np.exp(b.predict(X[evm], raw_score=True) + base + np.log(offe)) / seeds\n            mu += freq * sv\n            if "partes" in opts:\n                # cat_frecsev format: predicted frequency times training cost/claim count.\n                np.save(out / f"{\'v24\' if tag == \'v24\' else \'prod\'}_{p}.npy", freq * (yp[trm].sum() / max(n[trm].sum(), 1.0)))\n        if tag == "v24":\n            rep["d2_v24"] = round(d2(y[evm], mu), 5)\n            pd.DataFrame({"row_id": df.row_id[evm].astype(str).values, "line": line, "y": y[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)\n            print(name, "V24", rep["d2_v24"], round(time.time() - t0), flush=True)\n        else:\n            rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)\n            pd.DataFrame({"row_id": df.row_id[evm].astype(str).values, "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)\n        (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n    print(name, json.dumps(rep), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/influencia_reservas.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/influencia_reservas.py'] = '"""Compare fixed 2024 predictions under nominal and +/-25% case reserves.\n\nReport candidate-minus-reference D2 by line and overall, with 95% paired customer\nand address intervals from 1,000 Poisson(1) replicates and a fixed denominator.\nShow the ten most favorable customers and the difference without them as an\ninfluence diagnostic, not a removal rule. Attribute the change in the paired\ndeviance numerator to changing one peril\'s reserve at a time; this is a row-wise\nidentity rather than a causal attribution. Keep predictions unchanged.\nUsage: python influencia_reservas.py <output.json> <cand>=<spec>@<shape>@<level> <other>=<spec>@<shape>@<level>\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import BASESDIR, LINES, WORK, d2, deviance\nfrom escenarios_reserva import CLAIMS, C\nfrom mezcla3 import labels\nfrom robustez import blend, parse\n\nB = 1000\n\n\ndef reservas():\n    c = pd.read_parquet(CLAIMS, columns=["claim_id", "status", "peril", "paid_to_date", "incurred_to_date", "audit_pre00"])\n    a = pd.read_parquet(BASESDIR / "tables" / "asignacion_por_siniestro.parquet", columns=["claim_id", "category", "row_id"])\n    m = a.merge(c, on="claim_id", how="left", validate="one_to_one")\n    ok = (m.status == "open") & (m.category == "unica") & (~m.audit_pre00.fillna(False).astype(bool)) & m.row_id.notna()\n    m = m[ok].assign(res=lambda d: d.incurred_to_date - d.paid_to_date, row_id=lambda d: d.row_id.astype(str))\n    return m.pivot_table(index="row_id", columns="peril", values="res", aggfunc="sum", fill_value=0.0)\n\n\ndef ic(dif, groups, null, rng):\n    codes, inv = np.unique(groups, return_inverse=True)\n    d = np.bincount(inv, weights=dif)\n    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null\n    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)], d\n\n\ndef main():\n    out = sys.argv[1]\n    (cn, cspec, cfm), (on, ospec, ofm) = parse(sys.argv[2]), parse(sys.argv[3])\n    Rp = reservas()\n    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam\n    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate\n    rng = np.random.default_rng(20261018)\n    cols = {"y": [], "cu": [], "ad": [], "ln": [], "R": [], cn: [], on: [], "H": [], "A_G": []}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "address_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        cols[cn].append(blend(cspec, cfm, line, ysel, y24, idx25, expos)[2]); cols[on].append(blend(ospec, ofm, line, ysel, y24, idx25, expos)[2])\n        cols["H"].append(hh.reindex(y24.index).values); cols["A_G"].append(ag.reindex(y24.index).values)\n        cols["y"].append(y24.values); cols["ln"].append(np.full(len(y24), line))\n        cols["cu"].append(fe.customer_id.reindex(y24.index).astype(str).values); cols["ad"].append(fe.address_id.reindex(y24.index).astype(str).values)\n        cols["R"].append(Rp.reindex(y24.index).fillna(0.0))\n    V = {k: (pd.concat(v) if k == "R" else np.concatenate(v)) for k, v in cols.items()}\n    R = V["R"]; Rtot = R.sum(axis=1).values\n    rep = {"candidato": cn, "comparadores": [on, "H", "A_G"], "escenarios": {}}\n    for a in (0.75, 1.0, 1.25):\n        y_all = V["y"] + (a - 1.0) * Rtot\n        res = {}\n        for scope in LINES + ["agregado"]:\n            msk = np.ones(len(y_all), bool) if scope == "agregado" else (V["ln"] == scope)\n            y = y_all[msk]; null = deviance(y, np.full_like(y, y.mean())).sum()\n            row = {"d2": {k: round(d2(y, V[k][msk]), 6) for k in (cn, on, "H", "A_G")}}\n            for k in (on, "H", "A_G"):\n                dif = deviance(y, V[k][msk]) - deviance(y, V[cn][msk])\n                ic_cu, by_cu = ic(dif, V["cu"][msk], null, rng)\n                ic_ad, _ = ic(dif, V["ad"][msk], null, rng)\n                top = np.sort(by_cu)[::-1][:10]\n                row[f"{cn}_menos_{k}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ic_cu, "ic95_direccion": ic_ad,\n                                          "diez_clientes_mas_favorables": round(float(top.sum() / null), 6),\n                                          "dif_sin_esos_diez": round(float((dif.sum() - top.sum()) / null), 6)}\n                if a == 1.25:\n                    # Row-wise change in paired deviance when increasing only this peril\'s reserve by 25%.\n                    base_num = dif.sum()\n                    por_peligro = {}\n                    for p in R.columns:\n                        rp = R[p].values[msk]\n                        if rp.sum() <= 0:\n                            continue\n                        y0 = V["y"][msk]; yp = y0 + 0.25 * rp\n                        d0 = deviance(y0, V[k][msk]) - deviance(y0, V[cn][msk])\n                        d1 = deviance(yp, V[k][msk]) - deviance(yp, V[cn][msk])\n                        por_peligro[p] = round(float((d1 - d0).sum()), 3)\n                    row[f"{cn}_menos_{k}"]["cambio_numerador_por_peligro_a_mas25"] = por_peligro\n            res[scope] = row\n        rep["escenarios"][str(a)] = res\n        print(f"a={a}", flush=True)\n        for scope in LINES + ["agregado"]:\n            r = res[scope]\n            print("  ", scope, "D2", r["d2"], flush=True)\n            for k in (on, "H", "A_G"):\n                v = r[f"{cn}_menos_{k}"]\n                extra = f"  por peligro {v[\'cambio_numerador_por_peligro_a_mas25\']}" if "cambio_numerador_por_peligro_a_mas25" in v else ""\n                print(f"     frente a {k}: {v[\'dif\']:+.6f} IC95 cliente {v[\'ic95_cliente\']} dirección {v[\'ic95_direccion\']} diez {v[\'diez_clientes_mas_favorables\']:+.6f} sin ellos {v[\'dif_sin_esos_diez\']:+.6f}{extra}", flush=True)\n    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/intervalos_region.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/intervalos_region.py'] = '"""Report paired 2024 D2 differences by customer, address and region.\n\nUse separate generators, 1,000 Poisson(1) group-bootstrap replicates and fixed\ndenominators. Evaluate T0+(a-1)*R at a=0.75, 1 and 1.25 by line and overall.\nAlso report differences within each of the 12 regions and count negative regions.\nRegional resampling is an additional uncertainty diagnostic.\nUsage: python intervalos_region.py <output.json> <cand>=<spec>@<shape>@<level> <other>=<spec>@<shape>@<level>\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK, deviance\nfrom escenarios_reserva import C, reserva_por_fila\nfrom mezcla3 import labels\nfrom robustez import blend, parse\n\nB = 1000\n\n\ndef ic(dif, groups, null, rng):\n    codes, inv = np.unique(groups, return_inverse=True)\n    d = np.bincount(inv, weights=dif)\n    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null\n    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)], round(float((bs > 0).mean()), 3)\n\n\ndef main():\n    out = sys.argv[1]\n    (cn, cspec, cfm), (on, ospec, ofm) = parse(sys.argv[2]), parse(sys.argv[3])\n    R = reserva_por_fila()\n    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam\n    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate\n    V = {k: [] for k in ("y", "r", "cu", "ad", "rg", "ln", cn, on, "H", "A_G")}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "address_id", "region", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        V[cn].append(blend(cspec, cfm, line, ysel, y24, idx25, expos)[2]); V[on].append(blend(ospec, ofm, line, ysel, y24, idx25, expos)[2])\n        V["H"].append(hh.reindex(y24.index).values); V["A_G"].append(ag.reindex(y24.index).values)\n        f = fe.reindex(y24.index)\n        V["y"].append(y24.values); V["r"].append(R.reindex(y24.index.astype(str)).fillna(0.0).values)\n        V["cu"].append(f.customer_id.astype(str).values); V["ad"].append(f.address_id.astype(str).values)\n        V["rg"].append(f.region.astype(str).values); V["ln"].append(np.full(len(y24), line))\n    V = {k: np.concatenate(v) for k, v in V.items()}\n    rep = {"candidato": cn, "comparadores": [on, "H", "A_G"], "escenarios": {}}\n    for a in (0.75, 1.0, 1.25):\n        y_all = V["y"] + (a - 1.0) * V["r"]\n        res = {}\n        for scope in LINES + ["agregado"]:\n            msk = np.ones(len(y_all), bool) if scope == "agregado" else (V["ln"] == scope)\n            y = y_all[msk]; null = deviance(y, np.full_like(y, y.mean())).sum()\n            row = {}\n            for k in (on, "H", "A_G"):\n                dif = deviance(y, V[k][msk]) - deviance(y, V[cn][msk])\n                rngs = {g: np.random.default_rng(20261018 + i) for i, g in enumerate(("cu", "ad", "rg"))}\n                (icu, pcu), (iad, pad), (irg, prg) = (ic(dif, V[g][msk], null, rngs[g]) for g in ("cu", "ad", "rg"))\n                por_region = pd.Series(dif).groupby(V["rg"][msk]).sum() / null\n                row[f"{cn}_menos_{k}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": icu, "ic95_direccion": iad, "ic95_region": irg,\n                                          "P_mayor_que_0_region": prg, "regiones_negativas": int((por_region < 0).sum()), "regiones": int(len(por_region))}\n            res[scope] = row\n            print(f"a={a} {scope} " + " ".join(f"{k}: {row[f\'{cn}_menos_{k}\'][\'dif\']:+.6f} cli {row[f\'{cn}_menos_{k}\'][\'ic95_cliente\']} dir {row[f\'{cn}_menos_{k}\'][\'ic95_direccion\']} reg {row[f\'{cn}_menos_{k}\'][\'ic95_region\']} neg {row[f\'{cn}_menos_{k}\'][\'regiones_negativas\']}/12" for k in (on, "H", "A_G")), flush=True)\n        rep["escenarios"][str(a)] = res\n    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/mezcla3.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/mezcla3.py'] = '"""Blend line-specific components after normalizing their prediction totals.\n\nEach component supplies V23-selection, V24 and 2025 predictions. Normalize to the\nweighted total of eligible internal components separately in each cut; weights\nthen control relativities rather than selecting a single model\'s level. The\nexternal S08 stack can contribute shape but is excluded from that reference.\nApply the V23 Tweedie factor to reported V24 predictions and the mean of V23/V24\nfactors to 2025. Optional shape, level and grid settings retain the frozen recipe.\nUsage: python mezcla3.py <name> <spec_json> [--rejilla=<line>:<index>]\n       [--alfa=<line>:<a>,...] [--forma=<line>:<rate_power>:<exposure_power>,...]\n       [--nivel=s17] [--escribir]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import BASE, LINES, WORK, d2, digest\n\nEXTERNAL_STACK = Path("/mnt/user-data/uploads/Hakaton/Preparación/artifacts/verification/improvement_D76")\n# External S08 and N01 contents-GAM components do not determine the level reference.\nEXTERNOS = {"EXTERNAL_STACK", "GAMC_contents"}\n# Neural-network levels were unstable between V24 and production: MLP3_contents had\n# a 2025/V24 total ratio of 0.897 versus 1.074-1.078 for trees; use its relativities only.\nNIVEL_INESTABLE = {"MLP3_contents", "MLP3_buildings", "MLP3_motor"}\n# Since S17, exclude any component whose unlabeled total ratio deviates by more than\n# 1.5% from its line anchor between V24 and 2025; this also covers peril-mean severity fits.\n# Recorded 18 October against the frozen anchors: CFS motor -4.3%,\n# buildings -2.7%, contents -2.9%; CFSS motor -4.4%, household lines -2.0%.\n# Earlier motor/buildings CFS values (-3.8%/-2.6%) used another reference; classification is unchanged.\n# S19 hybrid GLM/boosting severity: CFSG motor -3.9% and buildings -1.7% are excluded;\n# CFSG contents -1.0% remains within the threshold and enters the level reference.\nNIVEL_INESTABLE_S17 = NIVEL_INESTABLE | {"CFS_motor", "CFS_motor_4s", "CFSS_motor", "CFS_buildings", "CFSS_buildings", "CFS_contents", "CFSS_contents",\n                                          "CFSG5_motor", "CFSG2_motor", "CFSG5_buildings", "CFSG2_buildings", "FSAS_contents",\n                                          "FSAS4_contents", "FSAS_motor", "FSAX4_contents", "CFSX_contents", "CFSX_buildings",\n                                          "CFSX_motor", "FSAXc4_contents", "CFSXc_contents", "CFSXc_buildings", "CFSXc_motor",\n                                          "CFSS4_buildings", "FS2Xc_contents", "FS2S_motor",\n                                          "CFSSr_motor", "CFSS4r_buildings", "FSAXc4r_contents",\n                                          "CFSS6_motor", "CFSS4v_motor_tmp"}\n# Four-seed buildings CFSS (S21): -1.72%, excluded under the same rule.\n# Third frequency engine with a severity GLM: buildings FS2S\n# -1.46% is included; contents FS2Xc -1.58% and motor FS2S -3.86% are excluded.\n# Severity relativities capped at p99.5: motor CFSSr -4.4%, buildings CFSS4r\n# -1.75% and contents FSAXc4r -1.55% are excluded; buildings FSAS4r +0.02% is included.\n# Additional motor CFSS seeds retain exclusion; CFSS4v_motor_tmp was a V24-only diagnostic.\n# Physical-covariate severity (S21): contents FSAX4 -1.52%, CFSX -2.06% and\n# buildings CFSX -1.79% are excluded; buildings FSAX4 +0.2% is included.\n# Corrected unpenalized-base variant: contents -1.53%/-2.06%, buildings CFSXc -1.86% excluded; FSAXc4 +0.15% included.\n# Additive frequency with insured-value severity (S20): buildings FSAS +0.2% enters\n# the level reference; contents -1.53% is excluded. Four-seed buildings FSAS4 +0.06%\n# remains included; contents -1.57% and motor FSAS -3.85% remain excluded.\nSIN_REFERENCIA = EXTERNOS | NIVEL_INESTABLE\n\n\ndef sin_referencia(opts):\n    """Exclude components from the level reference; s17 applies the V24-to-2025 stability rule."""\n    return EXTERNOS | (NIVEL_INESTABLE_S17 if opts.get("nivel") == "s17" else NIVEL_INESTABLE)\n_cx = {}\n\n\ndef ftw(y, mu):\n    return float((y / np.sqrt(mu)).sum() / np.sqrt(mu).sum())\n\n\ndef external_stack(split):\n    if split not in _cx:\n        f = {"v24": "learned_predictions_V24.parquet", "sel": "learned_fit_V23.parquet", "p25": "learned_predictions_PROD.parquet"}[split]\n        _cx[split] = pd.read_parquet(EXTERNAL_STACK / f, columns=["row_id", "stack"]).set_index("row_id")["stack"]\n    return _cx[split]\n\n\ndef component(run, sel_name, line):\n    if run == "EXTERNAL_STACK":\n        return external_stack("sel"), external_stack("v24"), external_stack("p25")\n    s = pd.read_parquet(WORK / f"sel_{sel_name}.parquet"); s = s[s.line == line].set_index("row_id")["mu"]\n    v = pd.read_parquet(WORK / run / "v24.parquet"); v = v[v.line == line].set_index("row_id")["mu"]\n    p = pd.read_parquet(WORK / run / "prediccion_2025.parquet"); p = p[p.line == line].set_index("row_id")["prediction"]\n    return s, v, p\n\n\ndef labels(line):\n    ref = pd.read_parquet(WORK / "S04b_lgbm_F0124_F3autos" / "v24.parquet"); ref = ref[ref.line == line].set_index("row_id")["y"]\n    sel = pd.read_parquet(WORK / "sel_total_F0124.parquet"); sel = sel[sel.line == line].set_index("row_id")["y"]\n    p25 = pd.read_parquet(WORK / "S04b_lgbm_F0124_F3autos" / "prediccion_2025.parquet"); p25 = p25[p25.line == line].set_index("row_id").index\n    return sel, ref, p25\n\n\ndef shape(mu, a):\n    """Raise predictions around their geometric mean; a>1 increases dispersion before level calibration."""\n    if a == 1.0:\n        return mu\n    g = np.exp(np.log(mu).mean())\n    return g * (mu / g) ** a\n\n\ndef shape_rate(mu, e, a, b):\n    """Apply mu=e*(e/e_bar)^(b-1)*g*(r/g)^a with r=mu/e; a changes rate dispersion and b changes exposure proportionality."""\n    if a == 1.0 and b == 1.0:\n        return mu\n    r = mu / e\n    g = np.exp(np.log(r).mean()); eb = np.exp(np.log(e).mean())\n    return e * (e / eb) ** (b - 1.0) * g * (r / g) ** a\n\n\ndef evaluate(comps, ws, ysel, y24, a=1.0, forma=None, expos=None):\n    ws = np.asarray(ws, float) / np.sum(ws)\n    bsel = sum(w * c[0] for w, c in zip(ws, comps)); b24 = sum(w * c[1] for w, c in zip(ws, comps)); b25 = sum(w * c[2] for w, c in zip(ws, comps))\n    bsel, b24, b25 = shape(bsel, a), shape(b24, a), shape(b25, a)\n    if forma is not None:\n        bsel, b24, b25 = (shape_rate(bsel, expos[0], *forma), shape_rate(b24, expos[1], *forma), shape_rate(b25, expos[2], *forma))\n    f23, f24 = ftw(ysel, bsel), ftw(y24, b24)\n    return {"d2_sel": d2(ysel, bsel), "d2_v24_f23": d2(y24, b24 * f23), "f23": f23, "f24": f24, "f": (f23 + f24) / 2}, bsel, b24 * f23, b25 * (f23 + f24) / 2\n\n\ndef main():\n    name, spec_path = sys.argv[1], sys.argv[2]\n    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[3:] if a.startswith("--"))\n    spec = json.loads(open(spec_path).read())\n    grid = opts.get("rejilla")\n    alfa = {kv.split(":")[0]: float(kv.split(":")[1]) for kv in opts.get("alfa", "").split(",") if kv}\n    forma = {kv.split(":")[0]: (float(kv.split(":")[1]), float(kv.split(":")[2])) for kv in opts.get("forma", "").split(",") if kv}\n    rep = {"spec": spec, "lines": {}, "alfa": alfa, "forma": {k: list(v) for k, v in forma.items()}}\n    ys24, ms24, preds = [], [], []\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        raw = []\n        for k, (run, sel_name, w) in enumerate(spec[line]):\n            s, v, p = component(run, sel_name, line)\n            s = s.reindex(ysel.index).values; v = v.reindex(y24.index).values; p = p.reindex(idx25).values\n            assert np.isfinite(s).all() and np.isfinite(v).all() and np.isfinite(p).all(), (line, run)\n            raw.append((run, w, s, v, p))\n        # Use the weighted total of eligible internal components as the reference in each cut;\n        # external components have another level basis. Rescale before applying relative-shape weights.\n        own = [(w, s, v, p) for run, w, s, v, p in raw if run not in sin_referencia(opts)]\n        wo = sum(o[0] for o in own)\n        ref = [sum(o[0] * o[j].sum() for o in own) / wo for j in (1, 2, 3)]\n        comps = [(s * ref[0] / s.sum(), v * ref[1] / v.sum(), p * ref[2] / p.sum()) for run, w, s, v, p in raw]\n        ws = [c[2] for c in spec[line]]\n        expos = None\n        if line in forma:\n            fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])\n            fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n            # V23/V24 use effective exposure; 2025 uses nominal exposure, matching prediction.\n            expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        res, bsel, b24, b25 = evaluate(comps, ws, ysel.values, y24.values, alfa.get(line, 1.0), forma.get(line), expos)\n        rep["lines"][line] = {k: round(v, 5) for k, v in res.items()}\n        print(line, rep["lines"][line], flush=True)\n        if grid and grid.split(":")[0] == line:\n            j = int(grid.split(":")[1])\n            others = np.array(ws, float); others[j] = 0; others = others / others.sum()\n            for wj in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):\n                w2 = others * (1 - wj); w2[j] = wj\n                r2, *_ = evaluate(comps, w2, ysel.values, y24.values, alfa.get(line, 1.0), forma.get(line), expos)\n                print("  rejilla", spec[line][j][0], wj, {k: round(v, 5) for k, v in r2.items()}, flush=True)\n        ys24.append(y24.values); ms24.append(b24)\n        preds.append(pd.DataFrame({"row_id": idx25.values, "line": line, "prediction": b25}))\n    rep["d2_v24_global_f23"] = round(d2(np.concatenate(ys24), np.concatenate(ms24)), 5)\n    rep["publico_esperado"] = round(0.0099 + 0.8644 * rep["d2_v24_global_f23"], 5)\n    print("global V24", rep["d2_v24_global_f23"], "público esperado", rep["publico_esperado"], flush=True)\n    if "escribir" in opts:\n        pred = pd.concat(preds, ignore_index=True)\n        sample = pd.read_parquet(BASE / "Datos" / "sample_submission.parquet", columns=["row_id"])\n        sub = sample.merge(pred[["row_id", "prediction"]], on="row_id", how="left", validate="one_to_one")\n        assert len(sub) == 626639 and np.isfinite(sub.prediction).all() and (sub.prediction > 0).all()\n        out = WORK / name\n        out.mkdir(parents=True, exist_ok=True)\n        csv = out / f"{name}.csv"\n        sub.to_csv(csv, index=False, float_format="%.6f")\n        back = pd.read_csv(csv, dtype={"row_id": str})\n        assert (back.row_id.values == sample.row_id.values).all()\n        rep["file"] = {"path": str(csv), "sha256": digest(csv), "rows": 626639, "total_CU": round(float(back.prediction.sum()), 2),\n                       "by_line_CU": {l: round(float(pred[pred.line == l].prediction.sum()), 2) for l in LINES}}\n        pred.to_parquet(out / "prediccion_2025.parquet", index=False)\n        (out / "resumen.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n        print(json.dumps(rep["file"], indent=1), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/mlp_tweedie.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/mlp_tweedie.py'] = '"""Fit a CPU PyTorch MLP with embeddings and power-1.5 Tweedie loss.\n\nlog(mu)=network(x)+log(effective exposure)+base. Fit numeric quantile-to-normal\ntransforms within training folds and add missingness indicators. Use categorical\nembeddings, the 15,000 motor cap/uniform surcharge and motor F3 contract. A zero\nfinal layer initializes predictions at the line mean rate. Stop V23 on calibration,\ncompare on selection, reuse its epoch count for V24/production and average seeds.\nUsage: python mlp_tweedie.py <name> <line> [--semillas=3] [--v24] [--prod]\n       [--ancho=128] [--drop=0.1] [--lr=0.001]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.preprocessing import QuantileTransformer\nfrom torch import nn\n\nfrom comun import WORK, d2, to_model_frame\nfrom experimentos import features, load\n\ntorch.set_num_threads(2)\nCAPS = {"motor": 15000.0}\n\n\ndef prep(X: pd.DataFrame, fit_mask: np.ndarray):\n    cats = [c for c in X.columns if isinstance(X[c].dtype, pd.CategoricalDtype)]\n    nums = [c for c in X.columns if c not in cats]\n    num = X[nums].astype("float64").values\n    miss = np.isnan(num)\n    med = np.nanmedian(num[fit_mask], axis=0)\n    med = np.where(np.isnan(med), 0.0, med)\n    num = np.where(miss, med, num)\n    qt = QuantileTransformer(n_quantiles=200, output_distribution="normal", subsample=200000, random_state=0)\n    qt.fit(num[fit_mask])\n    num = qt.transform(num).astype("float32")\n    keep_miss = miss[fit_mask].mean(axis=0) > 0.001\n    num = np.concatenate([num, miss[:, keep_miss].astype("float32")], axis=1)\n    codes = np.stack([X[c].cat.codes.values.astype("int64") + 1 for c in cats], axis=1) if cats else np.zeros((len(X), 0), "int64")\n    cards = [len(X[c].cat.categories) + 1 for c in cats]\n    for j in range(len(cats)):\n        # Missing or rare training categories (<30 rows) map to unknown index 0.\n        cnt = np.bincount(codes[fit_mask, j], minlength=cards[j])\n        codes[:, j] = np.where(cnt[codes[:, j]] < 30, 0, codes[:, j])\n    return num, codes, cards\n\n\nclass Net(nn.Module):\n    def __init__(self, n_num, cards, width=128, drop=0.1):\n        super().__init__()\n        dims = [min(12, int(np.ceil(c ** 0.5)) + 1) for c in cards]\n        self.embs = nn.ModuleList([nn.Embedding(c, d) for c, d in zip(cards, dims)])\n        d_in = n_num + sum(dims)\n        self.mlp = nn.Sequential(nn.Linear(d_in, width), nn.SiLU(), nn.Dropout(drop), nn.Linear(width, width // 2), nn.SiLU(),\n                                 nn.Dropout(drop), nn.Linear(width // 2, 1))\n        nn.init.zeros_(self.mlp[-1].weight)\n        nn.init.zeros_(self.mlp[-1].bias)\n\n    def forward(self, xn, xc):\n        parts = [xn] + [e(xc[:, i]) for i, e in enumerate(self.embs)]\n        return self.mlp(torch.cat(parts, dim=1)).squeeze(1)\n\n\ndef tweedie_loss(eta, y):\n    # Power-1.5 Tweedie deviance up to constants: 2*y*exp(-eta/2) + 2*exp(eta/2).\n    return (2.0 * y * torch.exp(-0.5 * eta) + 2.0 * torch.exp(0.5 * eta)).mean()\n\n\ndef train(num, codes, cards, y, logoff, idx_tr, idx_cal=None, epochs=40, seed=0, width=128, drop=0.1, lr=1e-3, bs=2048, patience=4, wd=1e-5):\n    torch.manual_seed(seed)\n    rng = np.random.default_rng(seed)\n    net = Net(num.shape[1], cards, width, drop)\n    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)\n    xn = torch.from_numpy(num); xc = torch.from_numpy(codes)\n    yt = torch.from_numpy(y.astype("float32")); lo = torch.from_numpy(logoff.astype("float32"))\n    best, best_ep, best_state, bad = np.inf, 0, None, 0\n    for ep in range(epochs):\n        net.train()\n        perm = rng.permutation(idx_tr)\n        for s in range(0, len(perm), bs):\n            b = torch.from_numpy(perm[s:s + bs])\n            eta = net(xn[b], xc[b]) + lo[b]\n            loss = tweedie_loss(eta, yt[b])\n            opt.zero_grad(); loss.backward(); opt.step()\n        if idx_cal is not None:\n            net.eval()\n            with torch.no_grad():\n                b = torch.from_numpy(idx_cal)\n                vl = float(tweedie_loss(net(xn[b], xc[b]) + lo[b], yt[b]))\n            if vl < best - 1e-7:\n                best, best_ep, bad = vl, ep + 1, 0\n                best_state = {k: v.clone() for k, v in net.state_dict().items()}\n            else:\n                bad += 1\n                if bad >= patience:\n                    break\n    if best_state is not None:\n        net.load_state_dict(best_state)\n    return net, best_ep\n\n\ndef predict(net, num, codes, logoff, idx):\n    net.eval()\n    out = []\n    with torch.no_grad():\n        for s in range(0, len(idx), 65536):\n            b = torch.from_numpy(idx[s:s + 65536])\n            out.append((net(torch.from_numpy(num)[b], torch.from_numpy(codes)[b]) + torch.from_numpy(logoff.astype("float32"))[b]).numpy())\n    return np.exp(np.concatenate(out).astype("float64"))\n\n\ndef main():\n    name, line = sys.argv[1], sys.argv[2]\n    opts = dict((a[2:].split("=") + ["1"])[:2] for a in sys.argv[3:] if a.startswith("--"))\n    seeds = int(opts.get("semillas", 3)); width = int(opts.get("ancho", 128)); drop = float(opts.get("drop", 0.1)); lr = float(opts.get("lr", 1e-3))\n    bs = int(opts.get("bs", 2048)); pat = int(opts.get("pac", 4)); wd = float(opts.get("wd", 1e-5))\n    t0 = time.time()\n    df = load(line)\n    feats = features(line, df, "F01234" if line == "motor" else "F0124")\n    if line != "motor" and "f3s2" in opts:\n        # Seven approved household customer F3 features (D80).\n        feats += [c for c in ["prior_claims_5y", "credit_band", "missed_payments_12m", "household_size", "years_at_address", "marital_status", "licence_points"] if c in df.columns]\n    X = to_model_frame(df, feats)\n    y = df["T0"].fillna(0).values\n    off = df["e_ef_coste"].values\n    expo = df["exposure"].values\n    yr = df.year.values\n    cap = CAPS.get(line)\n    out = WORK / name\n    out.mkdir(parents=True, exist_ok=True)\n    rep = {"line": line, "seeds": seeds, "width": width, "drop": drop, "lr": lr, "bs": bs, "wd": wd, "f3s2": "f3s2" in opts}\n\n    def fold(tr_mask, ev_mask, cal_mask, ev_off, epochs_fixed=None):\n        num, codes, cards = prep(X, tr_mask)\n        yt = np.minimum(y, cap) if cap else y.copy()\n        loading = y[tr_mask].sum() / yt[tr_mask].sum() if cap else 1.0\n        base = float(np.log(yt[tr_mask].sum() / off[tr_mask].sum()))\n        logoff = np.log(np.where(ev_mask, ev_off, off)) + base\n        idx_tr = np.flatnonzero(tr_mask); idx_ev = np.flatnonzero(ev_mask)\n        idx_cal = np.flatnonzero(cal_mask) if cal_mask is not None else None\n        mu = np.zeros(len(idx_ev)); eps = []\n        for s in range(seeds):\n            net, be = train(num, codes, cards, yt, logoff, idx_tr, idx_cal, epochs=(80 if epochs_fixed is None else epochs_fixed), seed=20261017 + s,\n                            width=width, drop=drop, lr=lr, bs=bs, patience=(pat if epochs_fixed is None else 10 ** 6), wd=wd)\n            eps.append(be if epochs_fixed is None else epochs_fixed)\n            mu += predict(net, num, codes, logoff, idx_ev) * loading / seeds\n            print(name, "semilla", s, "épocas", eps[-1], round(time.time() - t0), flush=True)\n        return mu, eps\n\n    tr = yr <= 2022; cal = (yr == 2023) & (df.cal_half.values == 0); sel = (yr == 2023) & (df.cal_half.values == 1)\n    mu, eps = fold(tr, sel, cal, off)\n    ep_fix = int(round(np.mean(eps)))\n    rep["epocas_v23"] = eps\n    rep["d2_sel"] = round(d2(y[sel], mu), 5)\n    print(name, json.dumps(rep), flush=True)\n    pd.DataFrame({"row_id": df.row_id[sel].astype(str).values, "line": line, "y": y[sel], "mu": mu}).to_parquet(WORK / f"sel_{name}.parquet", index=False)\n    if "v24" in opts:\n        ev = yr == 2024\n        mu24, _ = fold(yr <= 2023, ev, None, off, epochs_fixed=ep_fix)\n        rep["d2_v24"] = round(d2(y[ev], mu24), 5)\n        pd.DataFrame({"row_id": df.row_id[ev].astype(str).values, "line": line, "y": y[ev], "mu": mu24}).to_parquet(out / "v24.parquet", index=False)\n        print(name, json.dumps(rep), flush=True)\n    if "prod" in opts:\n        te = yr == 2025\n        mu25, _ = fold(yr <= 2024, te, None, expo, epochs_fixed=ep_fix)\n        rep["pred_2025_total_CU"] = round(float(mu25.sum()), 2)\n        pd.DataFrame({"row_id": df.row_id[te].astype(str).values, "line": line, "prediction": mu25}).to_parquet(out / "prediccion_2025.parquet", index=False)\n    rep["secs"] = round(time.time() - t0)\n    (out / "resumen.json").write_text(json.dumps(rep, indent=1))\n    print(name, json.dumps(rep), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/produccion.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/produccion.py'] = '"""Fit frozen line models and write predictions in submission-template order.\n\nV24 trains on 2021-2023 using 1.15 times the V23-selected rounds and evaluates\n2024 T0 with effective exposure. Production trains on 2021-2024 using 1.3 times\nthe rounds and predicts 2025 with nominal exposure, already at ultimate level\nwithout another development factor. Average the requested seeds. Save V24 and\n2025 Parquet, summary JSON and row_id,prediction CSV files under WORK/<name>/.\nUsage: python produccion.py <name> <feature_set> <config_json> [--semillas=3]\n       [--tendencia=motor:1.0,buildings:1.0,contents:1.0]\nEach line configuration supplies params and rounds.\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import BASE, LINES, WORK, d2, digest, to_model_frame\nfrom experimentos import features, fit, load, predict\n\n\ndef main():\n    name, set_name, cfg_path = sys.argv[1], sys.argv[2], sys.argv[3]\n    cfg = json.loads(open(cfg_path).read())\n    seeds = 3\n    trend = {l: 1.0 for l in LINES}\n    only = list(LINES)\n    reuse = None\n    for a in sys.argv[4:]:\n        if a.startswith("--solo="):\n            only = a.split("=", 1)[1].split(",")\n        if a.startswith("--reusar="):\n            reuse = WORK / a.split("=", 1)[1]\n        if a.startswith("--semillas="):\n            seeds = int(a.split("=")[1])\n        if a.startswith("--tendencia="):\n            for kv in a.split("=", 1)[1].split(","):\n                k, v = kv.split(":")\n                trend[k] = float(v)\n    out = WORK / name\n    out.mkdir(parents=True, exist_ok=True)\n    preds, v24_rows, summary = [], [], {"set": set_name, "config": cfg, "seeds": seeds, "trend": trend, "lines": {}}\n    for line in LINES:\n        if line not in only:\n            rp = pd.read_parquet(reuse / "prediccion_2025.parquet")\n            rv = pd.read_parquet(reuse / "v24.parquet")\n            preds.append(rp[rp.line == line].copy())\n            v24_rows.append(rv[rv.line == line].copy())\n            summary["lines"][line] = json.loads((reuse / "resumen.json").read_text())["lines"][line] | {"reutilizado_de": reuse.name}\n            continue\n        t = time.time()\n        df = load(line)\n        feats = [f for f in features(line, df, set_name) if f not in set(cfg[line].get("drop", []))]\n        feats += [f for f in cfg[line].get("add", []) if f in df.columns and f not in feats]\n        X = to_model_frame(df, feats)\n        y = df["T0"].fillna(0).values\n        off = df["e_ef_coste"].values\n        expo = df["exposure"].values\n        tr24 = (df.year <= 2023).values\n        ev24 = (df.year == 2024).values\n        trp = (df.year <= 2024).values\n        te = (df.year == 2025).values\n        r = int(cfg[line]["rounds"])\n        mu24 = np.zeros(ev24.sum())\n        mu25 = np.zeros(te.sum())\n        for s in range(seeds):\n            params = dict(cfg[line]["params"], seed=20261017 + 101 * s)\n            b24, base24 = fit(X[tr24], y[tr24], off[tr24], rounds=int(round(r * 1.15)), early=None, params=dict(params))\n            mu24 += predict(b24, base24, X[ev24], off[ev24]) / seeds\n            bp, basep = fit(X[trp], y[trp], off[trp], rounds=int(round(r * 1.3)), early=None, params=dict(params))\n            mu25 += predict(bp, basep, X[te], expo[te]) / seeds\n        mu25 *= trend[line]\n        imp = pd.Series(bp.feature_importance("gain"), index=feats).sort_values(ascending=False)\n        summary["lines"][line] = {\n            "n_feats": len(feats), "rounds_v24": int(round(r * 1.15)), "rounds_prod": int(round(r * 1.3)),\n            "d2_v24": round(d2(y[ev24], mu24), 5), "balance_v24": round(float(mu24.sum() / y[ev24].sum()), 4),\n            "pred_2025_total_CU": round(float(mu25.sum()), 2), "pred_2025_rate_CU": round(float(mu25.sum() / expo[te].sum()), 4),\n            "train_T0_rate_dev_CU": round(float(y[trp].sum() / off[trp].sum()), 4), "loading": round(float(getattr(bp, "loading", 1.0)), 5),\n            "top_gain": {k: round(float(v / imp.sum()), 4) for k, v in imp.head(20).items()}, "secs": round(time.time() - t)}\n        print(line, summary["lines"][line], flush=True)\n        preds.append(pd.DataFrame({"row_id": df.row_id[te].astype(str).values, "line": line, "prediction": mu25}))\n        v24_rows.append(pd.DataFrame({"row_id": df.row_id[ev24].astype(str).values, "line": line, "y": y[ev24], "mu": mu24}))\n    pred = pd.concat(preds, ignore_index=True)\n    v24 = pd.concat(v24_rows, ignore_index=True)\n    summary["d2_v24_global"] = round(d2(v24.y, v24.mu), 5)\n    sample = pd.read_parquet(BASE / "Datos" / "sample_submission.parquet", columns=["row_id"])\n    sub = sample.merge(pred[["row_id", "prediction"]], on="row_id", how="left", validate="one_to_one")\n    assert len(sub) == 626639 and sub.prediction.notna().all() and np.isfinite(sub.prediction).all() and (sub.prediction > 0).all()\n    csv = out / f"{name}.csv"\n    sub.to_csv(csv, index=False, float_format="%.6f")\n    back = pd.read_csv(csv, dtype={"row_id": str})\n    assert len(back) == 626639 and (back.row_id.values == sample.row_id.values).all() and (back.prediction > 0).all()\n    summary["file"] = {"path": str(csv), "sha256": digest(csv), "rows": int(len(back)), "total_CU": round(float(back.prediction.sum()), 2),\n                       "by_line_CU": {l: round(float(pred[pred.line == l].prediction.sum()), 2) for l in LINES}}\n    pred.to_parquet(out / "prediccion_2025.parquet", index=False)\n    v24.to_parquet(out / "v24.parquet", index=False)\n    (out / "resumen.json").write_text(json.dumps(summary, indent=1, default=float))\n    print("V24 global", summary["d2_v24_global"], "archivo", summary["file"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/publico_simulado.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/publico_simulado.py'] = '"""Diagnose variability in comparisons on random customer subsets of V23/V24.\n\nDraw 1,000 customer subsets at 30% and 50%, retaining each candidate\'s own level.\nReport the mean global-D2 difference, P(difference>0) and a 90% interval. These\nlocal simulations do not reproduce Kaggle\'s unknown 2025 public subset.\nUsage: python publico_simulado.py <output.json> <ref>=<spec>[@<shape>] <cand>=<spec>[@<shape>] [...]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK, deviance\nfrom mezcla3 import labels\nfrom robustez import blend, parse\n\nB = 1000\n\n\ndef main():\n    out_path = sys.argv[1]\n    cands = [parse(a) for a in sys.argv[2:]]\n    rng = np.random.default_rng(20261019)\n    folds = {"V23": {"y": [], "cust": [], "pred": {c[0]: [] for c in cands}}, "V24": {"y": [], "cust": [], "pred": {c[0]: [] for c in cands}}}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        folds["V23"]["y"].append(ysel.values); folds["V24"]["y"].append(y24.values)\n        folds["V23"]["cust"].append(fe.customer_id.reindex(ysel.index).astype(str).values)\n        folds["V24"]["cust"].append(fe.customer_id.reindex(y24.index).astype(str).values)\n        for name, spec, forma in cands:\n            _, bsel, b24, _ = blend(spec, forma, line, ysel, y24, idx25, expos)\n            folds["V23"]["pred"][name].append(bsel); folds["V24"]["pred"][name].append(b24)\n    rep = {}\n    ref = cands[0][0]\n    for fold, d in folds.items():\n        y = np.concatenate(d["y"]); cu = np.concatenate(d["cust"])\n        codes, inv = np.unique(cu, return_inverse=True)\n        nc = len(codes)\n        dev = {name: deviance(y, np.concatenate(p)) for name, p in d["pred"].items()}\n        rep[fold] = {}\n        for frac in (0.3, 0.5):\n            res = {name: [] for name in dev if name != ref}\n            pairs = [(a, b) for i, a in enumerate(dev) for b in list(dev)[i + 1:] if a != ref and b != ref]\n            pres = {f"{a}-{b}": [] for a, b in pairs}\n            for _ in range(B):\n                keep = (rng.random(nc) < frac)[inv]\n                yk = y[keep]\n                null = deviance(yk, np.full_like(yk, yk.mean())).sum()\n                d2k = {name: 1.0 - v[keep].sum() / null for name, v in dev.items()}\n                for name in res:\n                    res[name].append(d2k[name] - d2k[ref])\n                for a, b in pairs:\n                    pres[f"{a}-{b}"].append(d2k[a] - d2k[b])\n            summ = {}\n            for k, v in list({f"{n}-{ref}": res[n] for n in res}.items()) + list(pres.items()):\n                v = np.array(v)\n                summ[k] = {"media": round(float(v.mean()), 5), "P_mayor_que_0": round(float((v > 0).mean()), 3),\n                           "ic90": [round(float(np.quantile(v, 0.05)), 5), round(float(np.quantile(v, 0.95)), 5)]}\n                print(fold, f"{int(frac * 100)} %", k, summ[k], flush=True)\n            rep[fold][f"{int(frac * 100)}%"] = summ\n    (WORK / out_path).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/rejilla_pesos.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/rejilla_pesos.py'] = '"""Inspect a line\'s component weights on a simplex grid with step 0.1.\n\nLoad each component once, normalize its cut-specific level and retain the chosen\nshape rule. Evaluate V23 selection without a next-year factor and V24 with the\nV23 factor. Report rankings by each cut, their mean, and the worst improvement\nover the starting weights (maximin), alongside that starting configuration.\nUsage: python rejilla_pesos.py <line> <shape a:b> <level|-> <run:selection:weight> [...]\n"""\nfrom __future__ import annotations\n\nimport itertools\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import WORK\nfrom mezcla3 import component, evaluate, labels, sin_referencia\n\n\ndef main():\n    line, forma, nivel = sys.argv[1], sys.argv[2], sys.argv[3]\n    a, b = (float(x) for x in forma.split(":"))\n    comps_in = [c.split(":") for c in sys.argv[4:]]\n    ysel, y24, idx25 = labels(line)\n    raw = []\n    for run, sel_name, w in comps_in:\n        s, v, p = component(run, sel_name, line)\n        raw.append((run, float(w), s.reindex(ysel.index).values, v.reindex(y24.index).values, p.reindex(idx25).values))\n    excl = sin_referencia({"nivel": None if nivel == "-" else nivel})\n    fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])\n    fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n    expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n\n    def run_ws(ws):\n        own = [(w, s, v, p) for (run, _, s, v, p), w in zip(raw, ws) if run not in excl and w > 0]\n        if not own:\n            own = [(1.0, s, v, p) for (run, _, s, v, p) in raw if run not in excl]\n        wo = sum(o[0] for o in own)\n        ref = [sum(o[0] * o[j].sum() for o in own) / wo for j in (1, 2, 3)]\n        comps = [(s * ref[0] / s.sum(), v * ref[1] / v.sum(), p * ref[2] / p.sum()) for run, _, s, v, p in raw]\n        r, *_ = evaluate(comps, ws, ysel.values, y24.values, 1.0, (a, b), expos)\n        return r["d2_sel"], r["d2_v24_f23"]\n\n    base = run_ws([c[1] for c in raw])\n    print(line, "partida", [c[1] for c in raw], "V23 %.5f V24 %.5f" % base, flush=True)\n    k = len(raw)\n    rows = []\n    for combo in itertools.product(range(11), repeat=k):\n        if sum(combo) != 10:\n            continue\n        ws = [c / 10 for c in combo]\n        s23, s24 = run_ws(ws)\n        rows.append((ws, s23, s24))\n    for nombre, key in (("V23", lambda r: r[1]), ("V24", lambda r: r[2]), ("media", lambda r: r[1] + r[2]),\n                        ("la peor de las dos mejoras", lambda r: min(r[1] - base[0], r[2] - base[1]))):\n        print("mejores por", nombre, flush=True)\n        for ws, s23, s24 in sorted(rows, key=key, reverse=True)[:5]:\n            print("   ", ws, "V23 %.5f (%+.5f)  V24 %.5f (%+.5f)" % (s23, s23 - base[0], s24, s24 - base[1]), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/rejilla_robusta.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/rejilla_robusta.py'] = '"""Inspect simplex weights using the worst of four historical improvements.\n\nUse step 0.1 and the same loading, shape and level contract as rejilla_pesos.py.\nCompare V23/V24 on all rows and after excluding the largest 0.1% of T0, without\nreleveling the subsets. Report the worst-of-four and worst-of-two rankings versus\nthe starting weights. The criterion was recorded before the historical calculation.\nUsage: python rejilla_robusta.py <line> <shape a:b> <level|-> <run:selection:weight> [...]\n"""\nfrom __future__ import annotations\n\nimport itertools\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import WORK, d2\nfrom mezcla3 import component, evaluate, labels, sin_referencia\n\n\ndef main():\n    line, forma, nivel = sys.argv[1], sys.argv[2], sys.argv[3]\n    a, b = (float(x) for x in forma.split(":"))\n    comps_in = [c.split(":") for c in sys.argv[4:]]\n    ysel, y24, idx25 = labels(line)\n    raw = []\n    for run, sel_name, w in comps_in:\n        s, v, p = component(run, sel_name, line)\n        raw.append((run, float(w), s.reindex(ysel.index).values, v.reindex(y24.index).values, p.reindex(idx25).values))\n    excl = sin_referencia({"nivel": None if nivel == "-" else nivel})\n    fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "e_ef_coste", "exposure"])\n    fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n    expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n    y23, y4 = ysel.values, y24.values\n    k23 = y23 <= np.quantile(y23, 0.999); k24 = y4 <= np.quantile(y4, 0.999)\n\n    def run_ws(ws):\n        own = [(w, s, v, p) for (run, _, s, v, p), w in zip(raw, ws) if run not in excl and w > 0]\n        if not own:\n            own = [(1.0, s, v, p) for (run, _, s, v, p) in raw if run not in excl]\n        wo = sum(o[0] for o in own)\n        ref = [sum(o[0] * o[j].sum() for o in own) / wo for j in (1, 2, 3)]\n        comps = [(s * ref[0] / s.sum(), v * ref[1] / v.sum(), p * ref[2] / p.sum()) for run, _, s, v, p in raw]\n        r, bsel, b24, _ = evaluate(comps, ws, y23, y4, 1.0, (a, b), expos)\n        return (r["d2_sel"], r["d2_v24_f23"], d2(y23[k23], bsel[k23]), d2(y4[k24], b24[k24]))\n\n    base = run_ws([c[1] for c in raw])\n    print(line, "partida", [c[1] for c in raw], "V23 %.5f V24 %.5f | sin top V23 %.5f V24 %.5f" % base, flush=True)\n    rows = []\n    for combo in itertools.product(range(11), repeat=len(raw)):\n        if sum(combo) != 10:\n            continue\n        ws = [c / 10 for c in combo]\n        rows.append((ws, run_ws(ws)))\n    for nombre, key in (("la peor de las cuatro mejoras", lambda r: min(x - y for x, y in zip(r[1], base))),\n                        ("la peor de las dos mejoras (completo)", lambda r: min(r[1][0] - base[0], r[1][1] - base[1]))):\n        print("mejores por", nombre, flush=True)\n        for ws, m in sorted(rows, key=key, reverse=True)[:5]:\n            print("   ", ws, " ".join("%+.5f" % (x - y) for x, y in zip(m, base)), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/reproducir_mezcla.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/reproducir_mezcla.py'] = '"""Rebuild a frozen blend from saved components and check its exact CSV SHA-256.\n\nStage selection, V24 and 2025 tables in a temporary workspace using symbolic\nlinks. Add the exposure extracts as feat_<line>.parquet; the blend reads only\nrow_id, e_ef_coste and exposure from them. Run the same specification, shape and\nlevel rule as mezcla3.py. Keep sources unchanged and compare final file bytes.\nUsage: python reproducir_mezcla.py <spec_json> <shape> <level|-> <name> <expected_sha256> [<temporary_directory>]\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\n\nAQUI = Path(__file__).resolve().parent\nSEMANA1 = AQUI.parent\n\n\ndef sha(p: Path) -> str:\n    h = hashlib.sha256()\n    with open(p, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef buscar_sel(nombre: str) -> Path:\n    cands = list((AQUI / "runs").glob(f"*/sel_{nombre}.parquet")) + list((AQUI / "selecciones_v23").glob(f"sel_{nombre}.parquet"))\n    if not cands:\n        raise SystemExit(f"No encuentro sel_{nombre}.parquet en runs/ ni en selecciones_v23/")\n    return cands[0]\n\n\ndef main():\n    spec_path, forma, nivel, nombre, esperado = sys.argv[1:6]\n    tmp = Path(sys.argv[6]) if len(sys.argv) > 6 else Path(tempfile.mkdtemp(prefix="repro_"))\n    work = tmp / "work"\n    work.mkdir(parents=True, exist_ok=True)\n    spec = json.loads((AQUI / spec_path).read_text(encoding="utf-8")) if not Path(spec_path).is_absolute() else json.loads(Path(spec_path).read_text(encoding="utf-8"))\n    corridas = {"S04b_lgbm_F0124_F3autos"}  # mezcla3.labels() reads the V24 labels and 2025 identifiers here.\n    sels = {"total_F0124"}  # V23 selection labels are read from this source.\n    for line in spec:\n        for run, sel_name, _ in spec[line]:\n            corridas.add(run); sels.add(sel_name)\n    for run in sorted(corridas):\n        d = work / run\n        d.mkdir(exist_ok=True)\n        for f in ("v24.parquet", "prediccion_2025.parquet"):\n            src = AQUI / "runs" / run / f\n            if not src.exists():\n                raise SystemExit(f"Falta {src}")\n            if not (d / f).exists():\n                (d / f).symlink_to(src)\n    for s in sorted(sels):\n        dst = work / f"sel_{s}.parquet"\n        if not dst.exists():\n            dst.symlink_to(buscar_sel(s))\n    for line in ("motor", "buildings", "contents"):\n        dst = work / f"feat_{line}.parquet"\n        if not dst.exists():\n            dst.symlink_to(AQUI / "insumos" / f"exposicion_{line}.parquet")\n    env = dict(os.environ, MODELOS_WORK=str(work), HAKATON_SEMANA1=str(SEMANA1))\n    cmd = [sys.executable, "-W", "ignore", str(AQUI / "mezcla3.py"), nombre, str(AQUI / spec_path), f"--forma={forma}", "--escribir"]\n    if nivel != "-":\n        cmd.append(f"--nivel={nivel}")\n    print("Ejecuto:", " ".join(cmd), flush=True)\n    subprocess.run(cmd, check=True, env=env, cwd=str(AQUI))\n    csv = work / nombre / f"{nombre}.csv"\n    obtenido = sha(csv)\n    rep = {"nombre": nombre, "sha256_esperado": esperado, "sha256_obtenido": obtenido, "coincide": obtenido == esperado, "carpeta": str(work)}\n    if obtenido != esperado:\n        # NumPy versions may change the last bit of sums; report numerical differences separately.\n        import numpy as np\n        import pandas as pd\n        ref = SEMANA1 / "Envios" / "archivos" / f"{nombre}.csv"\n        a = pd.read_csv(csv, dtype={"row_id": str}); r = pd.read_csv(ref, dtype={"row_id": str})\n        assert (a.row_id.values == r.row_id.values).all()\n        dif = np.abs(a.prediction.values - r.prediction.values)\n        rep.update({"referencia": str(ref), "filas_distintas": int((dif > 0).sum()), "dif_max": float(dif.max()),\n                    "total_obtenido": round(float(a.prediction.sum()), 2), "total_entregado": round(float(r.prediction.sum()), 2)})\n    print(json.dumps(rep, ensure_ascii=False, indent=1), flush=True)\n    if obtenido != esperado and rep["dif_max"] > 1.01e-6:\n        raise SystemExit("El CSV repetido no coincide con el entregado")\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/robustez.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/robustez.py'] = '"""Compare frozen blends with paired robustness diagnostics in V23 and V24.\n\nReconstruct the same normalized components, shape rule and V23-derived V24 factor\nas mezcla3.py. Report line/global D2 differences, paired customer Poisson(1)\nbootstrap intervals (1,000 replicates, 90% interval) and P(improvement>0).\nFor the global comparison, group a customer\'s rows across all lines. Also inspect\nresults without the largest 0.1% of observed losses in each line/cut.\nUsage: python robustez.py <output.json> <ref>=<spec>[@<shape>] <cand>=<spec>[@<shape>] [...]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK, d2, deviance\nfrom mezcla3 import component, evaluate, ftw, labels, sin_referencia\n\nB = 1000\n\n\ndef parse(arg):\n    """Parse <name>=<spec>[@<shape>[@<level_rule>]]; s17 excludes unstable level components."""\n    name, rest = arg.split("=", 1)\n    spec, _, rest2 = rest.partition("@")\n    forma, _, nivel = rest2.partition("@")\n    fm = {kv.split(":")[0]: (float(kv.split(":")[1]), float(kv.split(":")[2])) for kv in forma.split(",") if kv}\n    if nivel:\n        fm["_nivel"] = nivel\n    return name, json.loads(open(spec).read()), fm\n\n\ndef blend(spec, forma, line, ysel, y24, idx25, expos):\n    raw = []\n    for run, sel_name, w in spec[line]:\n        s, v, p = component(run, sel_name, line)\n        raw.append((run, w, s.reindex(ysel.index).values, v.reindex(y24.index).values, p.reindex(idx25).values))\n    own = [(w, s, v, p) for run, w, s, v, p in raw if run not in sin_referencia({"nivel": forma.get("_nivel")})]\n    wo = sum(o[0] for o in own)\n    ref = [sum(o[0] * o[j].sum() for o in own) / wo for j in (1, 2, 3)]\n    comps = [(s * ref[0] / s.sum(), v * ref[1] / v.sum(), p * ref[2] / p.sum()) for run, w, s, v, p in raw]\n    res, bsel, b24, b25 = evaluate(comps, [c[2] for c in spec[line]], ysel.values, y24.values, 1.0, forma.get(line), expos)\n    return res, bsel, b24, b25\n\n\ndef boot(dif_by_c, null_total, rng):\n    # Fix the denominator at total null deviance; bootstrap only the numerator, in batches of 50.\n    d = dif_by_c.astype(np.float64)\n    out = [rng.poisson(1.0, size=(50, len(d))).astype(np.float64) @ d for _ in range(B // 50)]\n    return np.concatenate(out) / null_total\n\n\ndef main():\n    out_path = sys.argv[1]\n    cands = [parse(a) for a in sys.argv[2:]]\n    rng = np.random.default_rng(20261018)\n    data = {}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "e_ef_coste", "exposure"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        cust = {"V23": fe.customer_id.reindex(ysel.index).astype(str).values, "V24": fe.customer_id.reindex(y24.index).astype(str).values}\n        preds = {}\n        for name, spec, forma in cands:\n            res, bsel, b24, b25 = blend(spec, forma, line, ysel, y24, idx25, expos)\n            preds[name] = {"V23": bsel, "V24": b24, "tot25": float(b25.sum()), "res": res}\n        data[line] = {"y": {"V23": ysel.values, "V24": y24.values}, "cust": cust, "preds": preds}\n    ref = cands[0][0]\n    rep = {"referencia": ref, "candidatos": {}}\n    for name, _, _ in cands[1:]:\n        r = {}\n        for fold in ("V23", "V24"):\n            rows = {}\n            ally, allr, allc, allcu, keep_all = [], [], [], [], []\n            for line in LINES:\n                y = data[line]["y"][fold]; pr = data[line]["preds"][ref][fold]; pc = data[line]["preds"][name][fold]\n                cu = data[line]["cust"][fold]\n                dr, dc = deviance(y, pr), deviance(y, pc)\n                null = deviance(y, np.full_like(y, y.mean())).sum()\n                codes, inv = np.unique(cu, return_inverse=True)\n                dif_c = np.bincount(inv, weights=dr - dc)\n                bs = boot(dif_c, null, rng)\n                keep = y <= np.quantile(y, 0.999)\n                rows[line] = {"d2_ref": round(d2(y, pr), 5), "d2_cand": round(d2(y, pc), 5), "dif": round(float((dr - dc).sum() / null), 6),\n                              "P_mejora": round(float((bs > 0).mean()), 3), "ic90": [round(float(np.quantile(bs, 0.05)), 6), round(float(np.quantile(bs, 0.95)), 6)],\n                              "dif_sin_top01": round(d2(y[keep], pc[keep]) - d2(y[keep], pr[keep]), 6),\n                              # Use the same filter with subset-optimal levels to separate shape from level.\n                              "dif_sin_top01_renivelado": round(d2(y[keep], pc[keep] * ftw(y[keep], pc[keep])) - d2(y[keep], pr[keep] * ftw(y[keep], pr[keep])), 6),\n                              "dif_renivelado": round(d2(y, pc * ftw(y, pc)) - d2(y, pr * ftw(y, pr)), 6),\n                              "nivel_cand_sobre_ref": round(float(pc.sum() / pr.sum()), 4)}\n                ally.append(y); allr.append(pr); allc.append(pc); allcu.append(cu); keep_all.append(keep)\n            y = np.concatenate(ally); pr = np.concatenate(allr); pc = np.concatenate(allc); cu = np.concatenate(allcu); keep = np.concatenate(keep_all)\n            dr, dc = deviance(y, pr), deviance(y, pc)\n            null = deviance(y, np.full_like(y, y.mean())).sum()\n            codes, inv = np.unique(cu, return_inverse=True)\n            bs = boot(np.bincount(inv, weights=dr - dc), null, rng)\n            rows["global"] = {"d2_ref": round(d2(y, pr), 5), "d2_cand": round(d2(y, pc), 5), "dif": round(float((dr - dc).sum() / null), 6),\n                              "P_mejora": round(float((bs > 0).mean()), 3), "ic90": [round(float(np.quantile(bs, 0.05)), 6), round(float(np.quantile(bs, 0.95)), 6)],\n                              "dif_sin_top01": round(d2(y[keep], pc[keep]) - d2(y[keep], pr[keep]), 6)}\n            r[fold] = rows\n        r["total_2025"] = {line: {"ref": round(data[line]["preds"][ref]["tot25"], 0), "cand": round(data[line]["preds"][name]["tot25"], 0)} for line in LINES}\n        rep["candidatos"][name] = r\n        print("==", name, "frente a", ref, flush=True)\n        for fold in ("V23", "V24"):\n            for k, v in r[fold].items():\n                extra = f"  renivelado {v[\'dif_renivelado\']:+.5f}  sin top renivelado {v[\'dif_sin_top01_renivelado\']:+.5f}  nivel {v[\'nivel_cand_sobre_ref\']}" if k != "global" else ""\n                print(f"  {fold} {k:9s} dif {v[\'dif\']:+.5f}  P>0 {v[\'P_mejora\']:.3f}  IC90 [{v[\'ic90\'][0]:+.5f}, {v[\'ic90\'][1]:+.5f}]  sin top 0,1 % {v[\'dif_sin_top01\']:+.5f}{extra}", flush=True)\n    (WORK / out_path).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/semestres.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/semestres.py'] = '"""Inspect 2024 paired D2 differences by policy-start half-year.\n\nH1 is January-June and H2 July-December. Compare the candidate with H, A_G and\nanother blend at nominal and +25% reserves, by line and overall. Use 1,000\nPoisson(1) customer/address bootstrap replicates and half-year fixed denominators,\nwith separate address RNG state. Report exposure weight and effective/nominal\nexposure maturity because H2 labels are less developed.\nUsage: python semestres.py <output.json> <cand>=<spec>@<shape>@<level> <other>=<spec>@<shape>@<level>\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import LINES, WORK, d2, deviance\nfrom escenarios_reserva import C, reserva_por_fila\nfrom mezcla3 import labels\nfrom robustez import blend, parse\n\nB = 1000\n\n\ndef ic(dif, groups, null, rng):\n    codes, inv = np.unique(groups, return_inverse=True)\n    d = np.bincount(inv, weights=dif)\n    bs = np.concatenate([rng.poisson(1.0, size=(50, len(d))) @ d for _ in range(B // 50)]) / null\n    return [round(float(np.quantile(bs, 0.025)), 6), round(float(np.quantile(bs, 0.975)), 6)]\n\n\ndef main():\n    out = sys.argv[1]\n    (cn, cspec, cfm), (on, ospec, ofm) = parse(sys.argv[2]), parse(sys.argv[3])\n    R = reserva_por_fila()\n    ag = pd.read_parquet(f"{C}/crossfit_meta_gam_guard/predictions_V24_crossfit.parquet", columns=["row_id", "meta_gam"]).set_index("row_id").meta_gam\n    hh = pd.read_parquet(f"{C}/geometric_meta_nested/V24.parquet", columns=["row_id", "candidate"]).set_index("row_id").candidate\n    rng = np.random.default_rng(20261018)\n    rng_ad = np.random.default_rng(20261019)\n    V = {k: [] for k in ("y", "r", "cu", "ad", "ln", "sem", "e", "eef", cn, on, "H", "A_G")}\n    for line in LINES:\n        ysel, y24, idx25 = labels(line)\n        fe = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=["row_id", "customer_id", "address_id", "e_ef_coste", "exposure", "period_start"])\n        fe["row_id"] = fe.row_id.astype(str); fe = fe.set_index("row_id")\n        expos = (fe.e_ef_coste.reindex(ysel.index).values, fe.e_ef_coste.reindex(y24.index).values, fe.exposure.reindex(idx25).values)\n        V[cn].append(blend(cspec, cfm, line, ysel, y24, idx25, expos)[2]); V[on].append(blend(ospec, ofm, line, ysel, y24, idx25, expos)[2])\n        V["H"].append(hh.reindex(y24.index).values); V["A_G"].append(ag.reindex(y24.index).values)\n        f = fe.reindex(y24.index)\n        V["y"].append(y24.values); V["r"].append(R.reindex(y24.index).fillna(0.0).values); V["cu"].append(f.customer_id.astype(str).values)\n        V["ad"].append(f.address_id.astype(str).values)\n        V["ln"].append(np.full(len(y24), line)); V["sem"].append(np.where(pd.to_datetime(f.period_start).dt.month.values <= 6, "H1", "H2"))\n        V["e"].append(f.exposure.values); V["eef"].append(f.e_ef_coste.values)\n    V = {k: np.concatenate(v) for k, v in V.items()}\n    rep = {"candidato": cn, "comparadores": [on, "H", "A_G"], "resultados": {}}\n    for a in (1.0, 1.25):\n        y_all = V["y"] + (a - 1.0) * V["r"]\n        res = {}\n        for scope in LINES + ["agregado"]:\n            for sem in ("H1", "H2"):\n                msk = (V["sem"] == sem) & (np.ones(len(y_all), bool) if scope == "agregado" else (V["ln"] == scope))\n                y = y_all[msk]; null = deviance(y, np.full_like(y, y.mean())).sum()\n                row = {"filas": int(msk.sum()), "peso_exposicion": round(float(V["e"][msk].sum() / V["e"][(np.ones(len(y_all), bool) if scope == "agregado" else (V["ln"] == scope))].sum()), 4),\n                       "madurez": round(float(V["eef"][msk].sum() / V["e"][msk].sum()), 3), "d2": {k: round(d2(y, V[k][msk]), 6) for k in (cn, on, "H", "A_G")}}\n                for k in (on, "H", "A_G"):\n                    dif = deviance(y, V[k][msk]) - deviance(y, V[cn][msk])\n                    row[f"{cn}_menos_{k}"] = {"dif": round(float(dif.sum() / null), 6), "ic95_cliente": ic(dif, V["cu"][msk], null, rng),\n                                              "ic95_direccion": ic(dif, V["ad"][msk], null, rng_ad)}\n                res[f"{scope}_{sem}"] = row\n                print(f"a={a} {scope} {sem} filas {row[\'filas\']} peso {row[\'peso_exposicion\']} madurez {row[\'madurez\']} " +\n                      " ".join(f"{k}: {row[f\'{cn}_menos_{k}\'][\'dif\']:+.6f} {row[f\'{cn}_menos_{k}\'][\'ic95_cliente\']} dir {row[f\'{cn}_menos_{k}\'][\'ic95_direccion\']}" for k in (on, "H", "A_G")), flush=True)\n        rep["resultados"][str(a)] = res\n    (WORK / out).write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/severidad_cfs.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/severidad_cfs.py'] = '"""Apply insured-value gamma severity relativities to saved peril frequencies.\n\nFit a log-link gamma GLM to positive training claims, weighted by claim count.\nHousehold predictors include log insured value relative to the training median\nand basement for flood. Motor own-damage, theft and glass use vehicle value and\nage; third-party property and injury retain constant severity. Normalize each\nrelativity to weighted training-claim mean 1, then multiply the saved frequency\ntimes mean-severity contribution. Do not refit frequency models.\nV23 trains on 2021-2022, V24 on 2021-2023 and production on 2021-2024. Optional\nseed-run lists are averaged. --tope=q caps training row severity for slope fitting\nonly; normalization and the peril\'s original mean severity remain unchanged.\nUsage: python severidad_cfs.py <CFS_run> <line> <output_run> [--v23=...]\n       [--v24=c1,c2] [--prod=c1,c2] [--tope=q]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import F0DIR, PERILS, WORK, d2\n\n\ndef glm_gamma(Z, y, w, iters=50):\n    """Fit log-link gamma by IRLS; with variance mu^2, working weights equal prior weights."""\n    X = np.column_stack([np.ones(len(y)), Z])\n    beta = np.zeros(X.shape[1]); beta[0] = np.log(np.average(y, weights=w))\n    for _ in range(iters):\n        eta = X @ beta; mu = np.exp(eta)\n        z = eta + (y - mu) / mu\n        new = np.linalg.solve((X * w[:, None]).T @ X, (X * w[:, None]).T @ z)\n        if np.max(np.abs(new - beta)) < 1e-9:\n            beta = new\n            break\n        beta = new\n    return beta\n\n\ndef main():\n    src, line, name = sys.argv[1:4]\n    opts = dict(a[2:].split("=", 1) for a in sys.argv[4:] if a.startswith("--"))\n    run_v23 = opts.get("v23", src); runs_v24 = opts.get("v24", src).split(","); runs_prod = opts.get("prod", src).split(",")\n    tope = float(opts["tope"]) if "tope" in opts else None\n    extra = ["sum_insured", "basement"] if line != "motor" else ["vehicle_value_group", "vehicle_age"]\n    cols = ["row_id", "year", "T0"] + extra + [f"{k}_{p}" for p in PERILS[line] for k in ("n", "y")]\n    df = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=cols)\n    part = pd.read_parquet(F0DIR / "tables" / "particiones.parquet", columns=["row_id", "cal_half"])\n    df = df.merge(part, on="row_id", how="left", validate="one_to_one")\n    df["row_id"] = df.row_id.astype(str)\n    yr = df.year.values\n    if line != "motor":\n        si = df.sum_insured.astype(float).values\n        si = np.where(np.isfinite(si) & (si > 0), si, np.nan)\n        bas = df.basement.astype(float).fillna(0).values\n    else:\n        vvg = df.vehicle_value_group.astype(float).values\n        vage = np.clip(df.vehicle_age.astype(float).fillna(df.vehicle_age.median()).values, 0, 30) / 10.0\n    folds = {"V23": (yr <= 2022, (yr == 2023) & (df.cal_half.values == 1)), "V24": (yr <= 2023, yr == 2024), "PROD": (yr <= 2024, yr == 2025)}\n    out = WORK / name\n    out.mkdir(parents=True, exist_ok=True)\n    rep = {"origen": src, "line": line, "tope": tope, "perils": {}}\n    parts = {}\n    for fold, (trm, evm) in folds.items():\n        if line != "motor":\n            med = np.nanmedian(si[trm])\n            zsi = np.log(np.where(np.isnan(si), med, si) / med)\n        else:\n            medv = np.nanmedian(vvg[trm])\n            zv = np.log(np.where(np.isnan(vvg), medv, vvg) / medv)\n        tot = np.zeros(evm.sum())\n        for p in PERILS[line]:\n            n = df[f"n_{p}"].fillna(0).values.astype(float); yp = df[f"y_{p}"].fillna(0).values\n            pos = trm & (n > 0) & (yp > 0)\n            if line != "motor":\n                covs = [zsi] + ([bas] if p == "flood" else [])\n            else:\n                covs = [zv, vage] if p in ("own_damage", "theft", "glass_hail") else []\n            if covs:\n                Z = np.column_stack(covs)\n                ys = yp[pos] / n[pos]\n                if tope is not None:\n                    ys = np.minimum(ys, np.quantile(ys, tope))\n                beta = glm_gamma(Z[pos], ys, n[pos])\n                rel = np.exp(Z @ beta[1:])\n                rel = rel / np.average(rel[pos], weights=n[pos])\n            else:\n                beta = np.zeros(1); rel = np.ones(len(df))\n            # Saved frequency times constant training severity for this peril and cut.\n            if fold == "V23":\n                lam = np.load(WORK / run_v23 / f"v23_{p}.npz")["lam"]\n                sev = yp[trm].sum() / max(n[trm].sum(), 1.0)\n                part = lam * sev\n            else:\n                rr = runs_v24 if fold == "V24" else runs_prod\n                part = sum(np.load(WORK / r / f"{\'v24\' if fold == \'V24\' else \'prod\'}_{p}.npy") for r in rr) / len(rr)\n            assert len(part) == evm.sum()\n            tot += part * rel[evm]\n            rep["perils"].setdefault(p, {})[fold] = {"betas": [round(float(b), 4) for b in beta[1:]], "siniestros": int(n[pos].sum())}\n        parts[fold] = (evm, tot)\n    t0 = df["T0"].fillna(0).values\n    evm, mu = parts["V23"]\n    rep["d2_sel"] = round(d2(t0[evm], mu), 5)\n    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "y": t0[evm], "mu": mu}).to_parquet(WORK / f"sel_{name}.parquet", index=False)\n    evm, mu = parts["V24"]\n    rep["d2_v24"] = round(d2(t0[evm], mu), 5)\n    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "y": t0[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)\n    evm, mu = parts["PROD"]\n    rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)\n    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)\n    base = json.loads((WORK / src / "resumen.json").read_text())\n    rep["referencia_cfs"] = {"d2_sel": base.get("d2_sel"), "d2_v24": base.get("d2_v24")}\n    (out / "resumen.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n    print(json.dumps({k: v for k, v in rep.items() if k != "perils"}, ensure_ascii=False), flush=True)\n    for p, v in rep["perils"].items():\n        print("  ", p, v, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/modelos/severidad_cfs3.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/modelos/severidad_cfs3.py'] = '"""Extend saved frequency contributions with prespecified physical severity covariates.\n\nFit positive-claim, count-weighted gamma GLMs by peril/cut. Apply ridge LAMBDA\nonly to additional covariates; leave the intercept and base insured-value,\nbasement, vehicle-value and vehicle-age terms unpenalized. Normalize relativities\nto weighted training mean 1 before applying them to saved contributions.\nPrespecified buildings features cover roof, dwelling type, urban density, flood\nzone, river distance, vacancy, occupancy, soil, nearby trees, construction and\nlisted status as appropriate to each peril. Contents uses security, valuables,\nurban density and dwelling/flood attributes. Motor uses body/fuel/engine type,\ndriver age, young-driver status, use, assistance and overnight parking by peril.\nCategorical levels come from training; the most common is the base, missing is\nexplicit, and levels below MIN_NIVEL claims merge with the base. --tope=q caps\nseverity for slope estimation without changing the original mean-severity level.\nUsage: python severidad_cfs3.py <V23_run> <line> <output_run> [--v24=c1,c2]\n       [--prod=c1,c2] [--lambda=20] [--tope=q]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom comun import PERILS, WORK, d2\nfrom experimentos import load\n\nLAMBDA = 20.0\nMIN_NIVEL = 30\n# (type, column): num is transformed in preparar(); cat is categorical.\nCOV = {\n    "buildings": {\n        "storm": [("num", "zsi"), ("cat", "roof_type"), ("cat", "property_type"), ("cat", "urban_density")],\n        "flood": [("num", "zsi"), ("num", "basement"), ("cat", "flood_zone"), ("num", "log_dist_rio"), ("cat", "property_type")],\n        "escape_of_water": [("num", "zsi"), ("cat", "property_type"), ("num", "antiguedad"), ("num", "log_desocupada"), ("cat", "occupancy")],\n        "subsidence": [("num", "zsi"), ("cat", "soil_type"), ("num", "arboles"), ("num", "antiguedad"), ("cat", "property_type")],\n        "fire": [("num", "zsi"), ("cat", "construction_type_harmonized"), ("cat", "property_type"), ("num", "protegido"), ("num", "antiguedad")],\n        "accidental_damage": [("num", "zsi"), ("cat", "property_type")],\n    },\n    "contents": {\n        "theft": [("num", "zsi"), ("cat", "security_level"), ("num", "objetos_valor"), ("cat", "urban_density")],\n        "escape_of_water": [("num", "zsi"), ("cat", "property_type"), ("num", "antiguedad")],\n        "accidental_damage": [("num", "zsi"), ("num", "objetos_valor")],\n        "flood": [("num", "zsi"), ("num", "basement"), ("cat", "flood_zone")],\n    },\n    "motor": {\n        "own_damage": [("num", "zv"), ("num", "vage"), ("cat", "body_type"), ("cat", "fuel_type"), ("num", "asistencia")],\n        "third_party_prop": [("num", "zv"), ("cat", "urban_density"), ("cat", "body_type"), ("num", "log_cc")],\n        "bodily_injury": [("num", "edad"), ("num", "joven"), ("cat", "urban_density"), ("cat", "vehicle_use")],\n        "theft": [("num", "zv"), ("num", "vage"), ("cat", "overnight_parking")],\n        "glass_hail": [("num", "zv"), ("num", "vage"), ("cat", "body_type")],\n    },\n}\n\n\n# Leave base-severity terms unpenalized so the extended model contains the base model.\nLIBRES = {"buildings": {p: ("zsi",) for p in COV["buildings"]} | {"flood": ("zsi", "basement")},\n          "contents": {p: ("zsi",) for p in COV["contents"]} | {"flood": ("zsi", "basement")},\n          "motor": {"own_damage": ("zv", "vage"), "theft": ("zv", "vage"), "glass_hail": ("zv", "vage")}}\n\n\ndef glm_gamma_ridge(X, y, w, lam=LAMBDA, iters=60, libres=None):\n    """Fit log-link gamma IRLS with ridge except on the intercept and free base-severity columns."""\n    beta = np.zeros(X.shape[1]); beta[0] = np.log(np.average(y, weights=w))\n    P = np.eye(X.shape[1]) * lam; P[0, 0] = 0.0\n    for j in (libres or []):\n        P[j, j] = 0.0\n    for _ in range(iters):\n        eta = X @ beta; mu = np.exp(eta)\n        z = eta + (y - mu) / mu\n        XtW = (X * w[:, None]).T\n        new = np.linalg.solve(XtW @ X + P, XtW @ z)\n        if np.max(np.abs(new - beta)) < 1e-9:\n            return new\n        beta = new\n    return beta\n\n\ndef preparar(df, line, trm):\n    """Transform numeric columns and impute missing values with cut-specific training medians."""\n    out = {}\n    yr = df.year.values.astype(float)\n\n    def med(v):\n        return np.nanmedian(v[trm])\n    if line != "motor":\n        si = df.sum_insured.astype(float).values; si = np.where(np.isfinite(si) & (si > 0), si, np.nan)\n        m = med(si); out["zsi"] = np.log(np.where(np.isnan(si), m, si) / m)\n        out["basement"] = df.basement.astype(float).fillna(0).values\n        ant = np.clip(yr - df.year_built.astype(float).values, 0, 150); ant = np.where(np.isnan(ant), med(ant), ant)\n        out["antiguedad"] = ant / 50.0\n        if line == "buildings":\n            d = np.log1p(df.distance_to_river_km.astype(float).values); out["log_dist_rio"] = np.where(np.isnan(d), med(d), d)\n            u = np.log1p(df.days_unoccupied.astype(float).values); out["log_desocupada"] = np.where(np.isnan(u), med(u), u)\n            out["arboles"] = df.trees_within_5m.astype(float).fillna(0).values\n            out["protegido"] = df.listed_building.astype(float).fillna(0).values\n        else:\n            h = df.high_value_items.astype(float).values; out["objetos_valor"] = np.where(np.isnan(h), med(h), h)\n    else:\n        v = df.vehicle_value_group.astype(float).values; m = med(v); out["zv"] = np.log(np.where(np.isnan(v), m, v) / m)\n        va = df.vehicle_age.astype(float).values; out["vage"] = np.clip(np.where(np.isnan(va), med(va), va), 0, 30) / 10.0\n        out["asistencia"] = df.driver_assist.astype(float).fillna(0).values\n        cc = np.log(df.engine_size_cc.astype(float).values / 2000.0); out["log_cc"] = np.where(np.isnan(cc), med(cc), cc)\n        age = df.driver_age.astype(float).values\n        out["edad"] = (age - 45.0) / 20.0\n        out["joven"] = (age < 25).astype(float)\n    return out\n\n\ndef disenio(df, nums, spec, pos, n, libres_cols=()):\n    """Build a peril design matrix from training levels; return the unpenalized base-column indices too."""\n    cols = [np.ones(len(df))]\n    libres = []\n    for kind, c in spec:\n        if kind == "num":\n            if c in libres_cols:\n                libres.append(len(cols))\n            cols.append(nums[c])\n            continue\n        v = df[c].astype("string").fillna("NA").values.astype(str)\n        cnt = pd.Series(n[pos]).groupby(v[pos]).sum().sort_values(ascending=False)\n        base = cnt.index[0]\n        for lev in cnt.index[1:]:\n            if cnt[lev] >= MIN_NIVEL:\n                cols.append((v == lev).astype(float))\n    return np.column_stack(cols), libres\n\n\ndef main():\n    src, line, name = sys.argv[1:4]\n    opts = dict(a[2:].split("=", 1) for a in sys.argv[4:] if a.startswith("--"))\n    runs_v24 = opts.get("v24", src).split(","); runs_prod = opts.get("prod", src).split(","); run_v23 = opts.get("v23", src)\n    global LAMBDA\n    LAMBDA = float(opts.get("lambda", LAMBDA))\n    tope = float(opts["tope"]) if "tope" in opts else None\n    df = load(line)\n    yr = df.year.values\n    folds = {"V23": (yr <= 2022, (yr == 2023) & (df.cal_half.values == 1)), "V24": (yr <= 2023, yr == 2024), "PROD": (yr <= 2024, yr == 2025)}\n    out = WORK / name\n    out.mkdir(parents=True, exist_ok=True)\n    rep = {"origen": {"v23": run_v23, "v24": runs_v24, "prod": runs_prod}, "line": line, "lambda": LAMBDA, "tope": tope, "perils": {}}\n    parts = {}\n    for fold, (trm, evm) in folds.items():\n        nums = preparar(df, line, trm)\n        tot = np.zeros(evm.sum())\n        for p in PERILS[line]:\n            n = df[f"n_{p}"].fillna(0).values.astype(float); yp = df[f"y_{p}"].fillna(0).values\n            pos = trm & (n > 0) & (yp > 0)\n            X, libres = disenio(df, nums, COV[line][p], pos, n, LIBRES[line].get(p, ()))\n            ys = yp[pos] / n[pos]\n            if tope is not None:\n                ys = np.minimum(ys, np.quantile(ys, tope))\n            beta = glm_gamma_ridge(X[pos], ys, n[pos], lam=LAMBDA, libres=libres)\n            rel = np.exp(X @ beta - beta[0])\n            rel = rel / np.average(rel[pos], weights=n[pos])\n            if fold == "V23":\n                lam = np.load(WORK / run_v23 / f"v23_{p}.npz")["lam"]\n                part = lam * (yp[trm].sum() / max(n[trm].sum(), 1.0))\n            else:\n                rr = runs_v24 if fold == "V24" else runs_prod\n                part = sum(np.load(WORK / r / f"{\'v24\' if fold == \'V24\' else \'prod\'}_{p}.npy") for r in rr) / len(rr)\n            assert len(part) == evm.sum()\n            tot += part * rel[evm]\n            rep["perils"].setdefault(p, {})[fold] = {"parametros": int(X.shape[1] - 1), "siniestros": int(n[pos].sum()),\n                                                     "rel_p5_p95": [round(float(np.quantile(rel[evm], 0.05)), 3), round(float(np.quantile(rel[evm], 0.95)), 3)]}\n        parts[fold] = (evm, tot)\n    t0 = df["T0"].fillna(0).values; row_ids = df.row_id.astype(str).values\n    evm, mu = parts["V23"]; rep["d2_sel"] = round(d2(t0[evm], mu), 5)\n    pd.DataFrame({"row_id": row_ids[evm], "line": line, "y": t0[evm], "mu": mu}).to_parquet(WORK / f"sel_{name}.parquet", index=False)\n    evm, mu = parts["V24"]; rep["d2_v24"] = round(d2(t0[evm], mu), 5)\n    pd.DataFrame({"row_id": row_ids[evm], "line": line, "y": t0[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)\n    evm, mu = parts["PROD"]; rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)\n    pd.DataFrame({"row_id": row_ids[evm], "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)\n    (out / "resumen.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n    print(json.dumps({k: v for k, v in rep.items() if k != "perils"}, ensure_ascii=False), flush=True)\n    for p, v in rep["perils"].items():\n        print("  ", p, v, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/linea_b/mezcla3_b.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/linea_b/mezcla3_b.py'] = '"""Extend the main blend\'s s17 level rule to B_ runs through nivel_b.\nUsage: python mezcla3_b.py <name> <spec_json> [--forma=...] [--nivel=s17] [--escribir]\n"""\nimport json\nimport os as _os\nimport sys\nfrom pathlib import Path as _Path\n\n_MC = _os.environ.get("MODELOS_BASE") or next((str(p) for p in (_Path(__file__).resolve().parent.parent / "Modelos_base", _Path("/home/participant/mc")) if p.exists()), "/home/participant/mc")\nsys.path.insert(0, _MC)\nsys.path.insert(0, str(_Path(__file__).resolve().parent))\nimport nivel_b  # noqa: E402\n\nspec = json.loads(open(sys.argv[2]).read())\nnivel_b.clasificar({line: [c[0] for c in comps] for line, comps in spec.items()})\nextra = nivel_b.aplicar()\nprint("fuera de la referencia de nivel (corridas B):", sorted(extra), flush=True)\nnivel_b.mezcla3.main()\n'


### Embedded source: `src/linea_b/nivel_b.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/linea_b/nivel_b.py'] = '"""Apply the existing unlabeled s17 stability rule to additional line-B runs.\n\nCompare each run\'s 2025/V24 total ratio with its frozen line anchor. Exclude\ndeviations above 1.5% from the level reference, cache the result in\nWORK/nivel_inestable_b.json and extend the fixed exclusion set. Call aplicar()\nbefore using mezcla3, robustez or publico_simulado.\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport pandas as pd\n\nimport os as _os\nfrom pathlib import Path as _Path\n# Resolve the main model source directory from runtime bindings or a sibling checkout.\n_MC = _os.environ.get("MODELOS_BASE") or next((str(p) for p in (_Path(__file__).resolve().parent.parent / "Modelos_base", _Path("/home/participant/mc")) if p.exists()), "/home/participant/mc")\nsys.path.insert(0, _MC)\nimport mezcla3  # noqa: E402\nfrom comun import WORK  # noqa: E402\n\nANCLA = {"motor": "CAT_motor_4s", "buildings": "S12h_edif_md1000", "contents": "S08h_hogar_F3s2"}\nUMBRAL = 0.015\nCACHE = WORK / "nivel_inestable_b.json"\n\n\ndef razon(run, line):\n    v = pd.read_parquet(WORK / run / "v24.parquet"); v = v[v.line == line]["mu"].sum()\n    p = pd.read_parquet(WORK / run / "prediccion_2025.parquet"); p = p[p.line == line]["prediction"].sum()\n    return float(p / v)\n\n\ndef clasificar(runs_por_ramo: dict) -> dict:\n    """Map {line: [runs]} to cached {run: {desvio: value, fuera: boolean}} decisions."""\n    cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}\n    for line, runs in runs_por_ramo.items():\n        ra = razon(ANCLA[line], line)\n        for run in runs:\n            if run in cache or run in mezcla3.NIVEL_INESTABLE_S17 or run in mezcla3.EXTERNOS or run == ANCLA[line]:\n                continue\n            d = razon(run, line) / ra - 1.0\n            cache[run] = {"ramo": line, "desvio_2025_v24_frente_ancla": round(d, 5), "fuera": bool(abs(d) > UMBRAL)}\n    CACHE.write_text(json.dumps(cache, indent=1, ensure_ascii=False))\n    return cache\n\n\ndef aplicar(spec: dict | None = None):\n    """Apply cached unstable-run exclusions to mezcla3 and, when imported, robustez."""\n    if spec is not None:\n        clasificar({line: [c[0] for c in comps] for line, comps in spec.items()})\n    cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}\n    extra = {run for run, v in cache.items() if v["fuera"]}\n    base = mezcla3.sin_referencia\n\n    def patched(opts):\n        s = base(opts)\n        return (s | extra) if opts.get("nivel") == "s17" else s\n\n    mezcla3.sin_referencia = patched\n    try:\n        import robustez\n        robustez.sin_referencia = patched\n    except ImportError:\n        pass\n    return extra\n'


### Embedded source: `src/linea_b/reproducir_s30_b.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/linea_b/reproducir_s30_b.py'] = '"""Rebuild a line-B blend from verified saved component and exposure tables.\n\nStage selection, V24 and production files from both model families in a private\nworkspace. Apply the unlabeled s17 rule extended through nivel_b.py, then the\nfrozen blend specification and shape. Require the delivered CSV\'s exact SHA-256.\nUsage: python reproducir_s30_b.py <spec_json> <shape> <name> <expected_sha256> [<temporary_directory>]\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\n\nAQUI = Path(__file__).resolve().parent\nMC = AQUI.parent / "Modelos_base"\nSEMANA1 = AQUI.parent\n\n\ndef sha(p: Path) -> str:\n    h = hashlib.sha256()\n    with open(p, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef buscar(nombre: str, patron: str) -> Path:\n    for base in (AQUI, MC):\n        cands = list((base / "runs").glob(patron.format(nombre))) + list((base / "selecciones_v23").glob(patron.format(nombre)) if (base / "selecciones_v23").exists() else [])\n        if cands:\n            return cands[0]\n    raise SystemExit(f"No encuentro {patron.format(nombre)} en runs/ ni selecciones_v23/ de las dos carpetas")\n\n\ndef main():\n    spec_path, forma, nombre, esperado = sys.argv[1:5]\n    tmp = Path(sys.argv[5]) if len(sys.argv) > 5 else Path(tempfile.mkdtemp(prefix="repro_b_"))\n    work = tmp / "work"\n    work.mkdir(parents=True, exist_ok=True)\n    sp = Path(spec_path) if Path(spec_path).is_absolute() else AQUI / spec_path\n    spec = json.loads(sp.read_text(encoding="utf-8"))\n    corridas = {"S04b_lgbm_F0124_F3autos"}\n    sels = {"total_F0124"}\n    for line in spec:\n        for run, sel_name, _ in spec[line]:\n            corridas.add(run); sels.add(sel_name)\n    for run in sorted(corridas):\n        d = work / run\n        d.mkdir(exist_ok=True)\n        for f in ("v24.parquet", "prediccion_2025.parquet"):\n            src = next((b / "runs" / run / f for b in (AQUI, MC) if (b / "runs" / run / f).exists()), None)\n            if src is None:\n                raise SystemExit(f"Falta runs/{run}/{f}")\n            if not (d / f).exists():\n                (d / f).symlink_to(src)\n    for s in sorted(sels):\n        dst = work / f"sel_{s}.parquet"\n        if not dst.exists():\n            dst.symlink_to(buscar(s, "*/sel_{}.parquet") if not (MC / "selecciones_v23" / f"sel_{s}.parquet").exists() else MC / "selecciones_v23" / f"sel_{s}.parquet")\n    for line in ("motor", "buildings", "contents"):\n        dst = work / f"feat_{line}.parquet"\n        if not dst.exists():\n            dst.symlink_to(MC / "insumos" / f"exposicion_{line}.parquet")\n    env = dict(os.environ, MODELOS_WORK=str(work), HAKATON_SEMANA1=str(SEMANA1), MODELOS_BASE=str(MC))\n    # Cache the level classification before mezcla3 reads it through nivel_b.\n    cmd = [sys.executable, "-W", "ignore", str(AQUI / "mezcla3_b.py"), nombre, str(sp), f"--forma={forma}", "--nivel=s17", "--escribir"]\n    print("Ejecuto:", " ".join(cmd), flush=True)\n    subprocess.run(cmd, check=True, env=env, cwd=str(AQUI))\n    csv = work / nombre / f"{nombre}.csv"\n    obtenido = sha(csv)\n    rep = {"nombre": nombre, "sha256_esperado": esperado, "sha256_obtenido": obtenido, "coincide": obtenido == esperado, "carpeta": str(work)}\n    if obtenido != esperado:\n        import numpy as np\n        import pandas as pd\n        ref = SEMANA1 / "Envios" / "archivos" / f"{nombre}.csv"\n        a = pd.read_csv(csv, dtype={"row_id": str}); r = pd.read_csv(ref, dtype={"row_id": str})\n        assert (a.row_id.values == r.row_id.values).all()\n        dif = np.abs(a.prediction.values - r.prediction.values)\n        rep.update({"referencia": str(ref), "filas_distintas": int((dif > 0).sum()), "dif_max": float(dif.max()),\n                    "total_obtenido": round(float(a.prediction.sum()), 2), "total_entregado": round(float(r.prediction.sum()), 2)})\n    print(json.dumps(rep, ensure_ascii=False, indent=1), flush=True)\n    if obtenido != esperado and rep["dif_max"] > 1.01e-6:\n        raise SystemExit("El CSV repetido no coincide con el entregado")\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/linea_b/severidad_tendencia.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/linea_b/severidad_tendencia.py'] = '"""Apply annual peril-severity levels to saved frequency contributions.\n\nCFS and additive FSAP runs save frequency times pooled training mean severity.\nRetain the insured-value, basement and vehicle-value/age gamma relativities from\nseveridad_cfs.py, normalized to mean 1, and change only the target-year level.\nEstimate annual severities from training data with trimming to limit single-claim\ninfluence. Supported rules are pooled (all training years), ultimo (last year),\ntendencia (log-linear annual trend, capped at +/-15%, requiring three years),\nmixto (geometric mean of last-year and trend), and ruptura (fit after the latest\nannual change exceeding 15%; with two points, use the last year).\nWrite selection, V24 and 2025 predictions, summary and peril-level diagnostics.\nOptional --frec factors apply recorded out-of-sample frequency drift: V23 uses\nr22, V24 the geometric mean of r22/r23, and 2025 the mean of r23/r24, capped at\n15% in either direction.\nUsage: python severidad_tendencia.py <line> <output> --regla=<rule>\n       --v23=<run> --v24=<c1[,c2]> --prod=<c1[,c2]> [--frec=<json>[:<family>]]\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nimport os as _os\nfrom pathlib import Path as _Path\n# Resolve main-model sources from the runtime binding or a sibling checkout.\n_MC = _os.environ.get("MODELOS_BASE") or next((str(p) for p in (_Path(__file__).resolve().parent.parent / "Modelos_base", _Path("/home/participant/mc")) if p.exists()), "/home/participant/mc")\nsys.path.insert(0, _MC)\nfrom comun import F0DIR, PERILS, WORK, d2  # noqa: E402\nfrom severidad_cfs import glm_gamma  # noqa: E402\n\nTRIM_Q = 0.999\nCAP_GROWTH = 0.15\n\n\ndef nivel_anual(df, p, years):\n    """Return annual peril mean severity capped at training TRIM_Q, plus the pooled uncapped/capped ratio."""\n    n = df[f"n_{p}"].fillna(0).values.astype(float)\n    yp = df[f"y_{p}"].fillna(0).values\n    yr = df.year.values\n    m = np.isin(yr, years) & (n > 0)\n    sev_fila = yp[m] / n[m]          # Row mean cost per claim approximates individual severity when a row has multiple claims.\n    cap = np.quantile(np.repeat(sev_fila, n[m].astype(int)), TRIM_Q)\n    sev_trim = np.minimum(sev_fila, cap)\n    serie = {}\n    for y in years:\n        k = yr[m] == y\n        serie[int(y)] = float(np.average(sev_trim[k], weights=n[m][k])) if n[m][k].sum() > 0 else np.nan\n    pooled_full = float(yp[m].sum() / n[m].sum())\n    pooled_trim = float(np.average(sev_trim, weights=n[m]))\n    return serie, pooled_full, pooled_trim\n\n\ndef nivel_objetivo(serie, regla):\n    ys = sorted(serie)\n    vals = np.array([serie[y] for y in ys], float)\n    if regla == "pooled":\n        return None\n    if regla == "ruptura":\n        saltos = np.abs(np.diff(np.log(vals)))\n        corte = int(np.where(saltos > np.log(1 + CAP_GROWTH))[0].max()) + 1 if (saltos > np.log(1 + CAP_GROWTH)).any() else 0\n        return nivel_objetivo({y: serie[y] for y in ys[corte:]}, "tendencia")\n    ultimo = vals[-1]\n    if len(ys) >= 3:\n        b = np.polyfit(np.array(ys, float), np.log(vals), 1)[0]\n        b = float(np.clip(b, np.log(1 - CAP_GROWTH), np.log(1 + CAP_GROWTH)))\n        # Anchor the trend to the fitted last-year value, then apply annual growth.\n        fit = np.polyval(np.polyfit(np.array(ys, float), np.log(vals), 1), ys[-1])\n        tend = float(np.exp(fit + b))\n    else:\n        tend = ultimo\n    if regla == "ultimo":\n        return ultimo\n    if regla == "tendencia":\n        return tend\n    if regla == "mixto":\n        return float(np.sqrt(ultimo * tend))\n    raise SystemExit(regla)\n\n\ndef main():\n    line, name = sys.argv[1:3]\n    opts = dict(a[2:].split("=", 1) for a in sys.argv[3:] if a.startswith("--"))\n    regla = opts.get("regla", "tendencia")\n    run_v23 = opts["v23"]; runs_v24 = opts["v24"].split(","); runs_prod = opts["prod"].split(",")\n    frec = {}\n    if "frec" in opts:\n        ruta, _, familia = opts["frec"].partition(":")\n        fj = json.load(open(ruta))\n        frec = fj["factores_" + familia] if familia else fj["factores"]\n    extra = ["sum_insured", "basement"] if line != "motor" else ["vehicle_value_group", "vehicle_age"]\n    cols = ["row_id", "year", "T0"] + extra + [f"{k}_{p}" for p in PERILS[line] for k in ("n", "y")]\n    df = pd.read_parquet(WORK / f"feat_{line}.parquet", columns=cols)\n    part = pd.read_parquet(F0DIR / "tables" / "particiones.parquet", columns=["row_id", "cal_half"])\n    df = df.merge(part, on="row_id", how="left", validate="one_to_one")\n    df["row_id"] = df.row_id.astype(str)\n    yr = df.year.values\n    if line != "motor":\n        si = df.sum_insured.astype(float).values\n        si = np.where(np.isfinite(si) & (si > 0), si, np.nan)\n        bas = df.basement.astype(float).fillna(0).values\n    else:\n        vvg = df.vehicle_value_group.astype(float).values\n        vage = np.clip(df.vehicle_age.astype(float).fillna(df.vehicle_age.median()).values, 0, 30) / 10.0\n    folds = {"V23": (yr <= 2022, (yr == 2023) & (df.cal_half.values == 1), [2021, 2022]),\n             "V24": (yr <= 2023, yr == 2024, [2021, 2022, 2023]),\n             "PROD": (yr <= 2024, yr == 2025, [2021, 2022, 2023, 2024])}\n    out = WORK / name\n    out.mkdir(parents=True, exist_ok=True)\n    rep = {"line": line, "regla": regla, "v23": run_v23, "v24": runs_v24, "prod": runs_prod, "frec": opts.get("frec"), "perils": {}}\n    parts = {}\n    for fold, (trm, evm, years) in folds.items():\n        if line != "motor":\n            med = np.nanmedian(si[trm])\n            zsi = np.log(np.where(np.isnan(si), med, si) / med)\n        else:\n            medv = np.nanmedian(vvg[trm])\n            zv = np.log(np.where(np.isnan(vvg), medv, vvg) / medv)\n        tot = np.zeros(evm.sum())\n        for p in PERILS[line]:\n            n = df[f"n_{p}"].fillna(0).values.astype(float); yp = df[f"y_{p}"].fillna(0).values\n            pos = trm & (n > 0) & (yp > 0)\n            if line != "motor":\n                covs = [zsi] + ([bas] if p == "flood" else [])\n            else:\n                covs = [zv, vage] if p in ("own_damage", "theft", "glass_hail") else []\n            if covs:\n                Z = np.column_stack(covs)\n                beta = glm_gamma(Z[pos], yp[pos] / n[pos], n[pos])\n                rel = np.exp(Z @ beta[1:])\n                rel = rel / np.average(rel[pos], weights=n[pos])\n            else:\n                beta = np.zeros(1); rel = np.ones(len(df))\n            serie, pooled_full, pooled_trim = nivel_anual(df, p, years)\n            objetivo = nivel_objetivo(serie, regla)\n            # Saved contributions contain uncapped pooled training severity; move their level\n            # to the target while retaining the pooled uncapped/capped ratio.\n            factor = 1.0 if objetivo is None else float(objetivo * (pooled_full / pooled_trim) / pooled_full)\n            if fold == "V23":\n                lam = np.load(WORK / run_v23 / f"v23_{p}.npz")["lam"]\n                sev = yp[trm].sum() / max(n[trm].sum(), 1.0)\n                base_part = lam * sev\n            else:\n                rr = runs_v24 if fold == "V24" else runs_prod\n                base_part = sum(np.load(WORK / r / f"{\'v24\' if fold == \'V24\' else \'prod\'}_{p}.npy") for r in rr) / len(rr)\n            assert len(base_part) == evm.sum(), (fold, p, len(base_part), evm.sum())\n            ffrec = float(frec.get(fold, {}).get(p, 1.0))\n            tot += base_part * rel[evm] * factor * ffrec\n            rep["perils"].setdefault(p, {})[fold] = {"serie_recortada": {k: round(v, 1) for k, v in serie.items()}, "pooled": round(pooled_full, 1),\n                                                     "objetivo_recortado": None if objetivo is None else round(objetivo, 1), "factor": round(factor, 4), "factor_frecuencia": round(ffrec, 4),\n                                                     "betas": [round(float(b), 4) for b in beta[1:]], "siniestros": int(n[pos].sum())}\n        parts[fold] = (evm, tot)\n    t0 = df["T0"].fillna(0).values\n    evm, mu = parts["V23"]\n    rep["d2_sel"] = round(d2(t0[evm], mu), 5)\n    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "y": t0[evm], "mu": mu}).to_parquet(WORK / f"sel_{name}.parquet", index=False)\n    evm, mu = parts["V24"]\n    rep["d2_v24"] = round(d2(t0[evm], mu), 5)\n    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "y": t0[evm], "mu": mu}).to_parquet(out / "v24.parquet", index=False)\n    evm, mu = parts["PROD"]\n    rep["pred_2025_total_CU"] = round(float(mu.sum()), 2)\n    pd.DataFrame({"row_id": df.row_id[evm].values, "line": line, "prediction": mu}).to_parquet(out / "prediccion_2025.parquet", index=False)\n    (out / "resumen.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False))\n    print(json.dumps({k: v for k, v in rep.items() if k != "perils"}, ensure_ascii=False), flush=True)\n    for p, v in rep["perils"].items():\n        print("  ", p, {f: (w["serie_recortada"], w["objetivo_recortado"], w["factor"], w["factor_frecuencia"]) for f, w in v.items()}, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/datos/bases_actuariales.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/datos/bases_actuariales.py'] = '"""Build actuarial exposure, claims assignment and development tables from prepared data.\n\nVerify the seven input-table SHA-256 values against the active catalog. Derive\nexposure, reporting delay/IBNR, frequency, severity and constant-rate M00\nreferences. Read existing data and indices without modifying them. Each run\ncreates a new directory with evidence, CSV/Parquet tables, source snapshot and\nhash manifest; current_run.json is selected separately after verification.\nM00 references are line-specific constant rates evaluated on the next-year cut,\nnot trained machine-learning models. The isolated runner supplies project paths.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nimport sys\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\n\n\ndef find_root(start: Path) -> Path:\n    for candidate in [start, *start.parents]:\n        if (candidate / "RUTAS_PROYECTO.json").is_file():\n            return candidate\n    raise FileNotFoundError("No se encontró RUTAS_PROYECTO.json hacia arriba.")\n\n\nROOT = find_root(HERE)\ntry:  # Use the project\'s isolated Parquet reader if the active environment lacks PyArrow.\n    import pyarrow  # noqa: F401\nexcept ImportError:  # pragma: no cover\n    sys.path.insert(0, str(ROOT / "Preparación" / ".cache" / "parquet-reader"))\n\nimport numpy as np  # noqa: E402\nimport pandas as pd  # noqa: E402\n\nLINES = ["motor", "buildings", "contents"]\nLINE_ES = {"motor": "autos", "buildings": "edificios", "contents": "contenidos"}\nCUTOFF = pd.Timestamp("2024-12-30")  # Last observed reporting date, checked against the data.\nPOWER = 1.5\nLONG_TAIL = {("motor", "bodily_injury"), ("buildings", "subsidence")}\nMONTHS = pd.period_range("2021-01", "2025-12", freq="M")\n\n\ndef digest(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef tweedie_deviance(y: np.ndarray, mu: np.ndarray, p: float = POWER) -> np.ndarray:\n    """Unit Tweedie deviance; at p=1.5 it equals 4*(sqrt(mu)-sqrt(y))**2/sqrt(mu)."""\n    y = np.asarray(y, dtype=float)\n    mu = np.asarray(mu, dtype=float)\n    return 2.0 * (np.power(y, 2 - p) / ((1 - p) * (2 - p)) - y * np.power(mu, 1 - p) / (1 - p) + np.power(mu, 2 - p) / (2 - p))\n\n\ndef d2_tweedie(y: np.ndarray, mu: np.ndarray, p: float = POWER) -> float:\n    """Compute scikit-learn-style D2: 1-dev(y,mu)/dev(y,mean(y))."""\n    y = np.asarray(y, dtype=float)\n    mu = np.asarray(mu, dtype=float)\n    null = np.full_like(y, y.mean())\n    return float(1.0 - tweedie_deviance(y, mu, p).sum() / tweedie_deviance(y, null, p).sum())\n\n\ndef r(x, nd=4):\n    if x is None:\n        return None\n    if isinstance(x, (np.floating, float)):\n        if not np.isfinite(x):\n            return None\n        return round(float(x), nd)\n    if isinstance(x, (np.integer,)):\n        return int(x)\n    return x\n\n\ndef main() -> None:\n    started = datetime.now(timezone.utc)\n    run_id = f"bases_{started.strftime(\'%Y%m%dT%H%M%SZ\')}_{uuid.uuid4().hex[:8]}"\n    run_dir = HERE / "runs" / run_id\n    tables_dir = run_dir / "tables"\n    tables_dir.mkdir(parents=True, exist_ok=False)\n    checks: list[dict] = []\n\n    def check(code: str, name: str, ok: bool, detail=None):\n        checks.append({"code": code, "name": name, "passed": bool(ok), "detail": detail})\n\n    # Stage 1: current prepared-data layer and fingerprints.\n    index_path = ROOT / "Hackaton/Semana 1/Limpieza/current_dataset.json"\n    current = json.loads(index_path.read_text(encoding="utf-8"))\n    catalog_path = ROOT / current["catalog_workspace_relative"]\n    catalog = json.loads(catalog_path.read_text(encoding="utf-8"))\n    verification = json.loads((catalog_path.parent / "verification.json").read_text(encoding="utf-8"))\n    check("K01", "la corrida vigente de Limpieza tiene verificación con cero fallos", verification.get("failed") == 0 and verification.get("passed", 0) > 0,\n          {"status": verification.get("status"), "passed": verification.get("passed"), "failed": verification.get("failed")})\n    used = ["claims", "policies_motor", "policies_buildings", "policies_contents", "test_motor", "test_buildings", "test_contents"]\n    hashes = {}\n    bad = []\n    for name in used:\n        path = ROOT / catalog[name]["workspace_relative_path"]\n        hashes[name] = digest(path)\n        if hashes[name] != catalog[name]["sha256"]:\n            bad.append(name)\n    check("K02", "los SHA-256 de las siete tablas usadas coinciden con el catálogo activo", not bad, bad)\n\n    def read(name: str, columns: list[str]) -> pd.DataFrame:\n        path = ROOT / catalog[name]["workspace_relative_path"]\n        df = pd.read_parquet(path, columns=columns)\n        for col in df.columns:  # Compact strings reduce memory use in constrained environments.\n            if df[col].dtype == object:\n                df[col] = df[col].astype("string[pyarrow]")\n        return df\n\n    base_cols = ["row_id", "policy_id", "customer_id", "period_start", "period_end", "exposure", "new_business", "written_premium"]\n    frames = []\n    for line in LINES:\n        extra = ["cover_type"] if line == "motor" else []\n        for kind, table in (("train", f"policies_{line}"), ("test", f"test_{line}")):\n            df = read(table, base_cols + extra)\n            df["line"] = line\n            df["is_test"] = kind == "test"\n            if line != "motor":\n                df["cover_type"] = "n/a"\n            frames.append(df)\n    periods = pd.concat(frames, ignore_index=True)\n    del frames\n    periods["period_start"] = pd.to_datetime(periods["period_start"])\n    periods["period_end"] = pd.to_datetime(periods["period_end"])\n    periods["year"] = periods["period_start"].dt.year.astype(int)\n    days = ((periods["period_end"] - periods["period_start"]).dt.days + 1).values\n    max_abs_diff = float(np.abs(periods["exposure"].values - days / 365.0).max())\n    check("K03", "exposición EXP01 = días inclusivos / 365 en todas las filas (tolerancia 1e-4)", max_abs_diff < 1e-4, {"max_abs_diff": r(max_abs_diff, 6)})\n    del days\n    check("K04", "626.639 filas de test con row_id único", int(periods["is_test"].sum()) == 626639 and periods.loc[periods.is_test, "row_id"].is_unique,\n          {"test_rows": int(periods["is_test"].sum())})\n    check("K05", "los periodos de test son todos de 2025 y los de entrenamiento de 2021-2024",\n          set(periods.loc[periods.is_test, "year"].unique()) == {2025} and set(periods.loc[~periods.is_test, "year"].unique()) == {2021, 2022, 2023, 2024})\n\n    # Stage 2: exposure.\n    def exposure_block(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:\n        g = df.groupby(keys)\n        out = g.agg(filas=("row_id", "size"), expuestos_anios=("exposure", "sum"), exposicion_media=("exposure", "mean"),\n                    polizas=("policy_id", "nunique"), clientes=("customer_id", "nunique"),\n                    prima_escrita_CU=("written_premium", "sum")).reset_index()\n        out["pct_filas_anio_completo"] = g["exposure"].apply(lambda s: float((s >= 0.99).mean())).values * 100\n        out["pct_filas_menos_medio_anio"] = g["exposure"].apply(lambda s: float((s < 0.5).mean())).values * 100\n        out["pct_filas_nuevo_negocio"] = g["new_business"].apply(lambda s: float(s.mean())).values * 100\n        out["prima_escrita_por_anio_expuesto_CU"] = out["prima_escrita_CU"] / out["expuestos_anios"]\n        return out\n\n    exp_line_year = exposure_block(periods, ["line", "year"])\n    exp_line_year.to_csv(tables_dir / "expuestos_ramo_anio.csv", index=False)\n    exp_cover = exposure_block(periods[periods.line == "motor"], ["year", "cover_type"])\n    exp_cover.to_csv(tables_dir / "expuestos_autos_cobertura.csv", index=False)\n    train_exposure_total = float(periods.loc[~periods.is_test, "exposure"].sum())\n    test_exposure_total = float(periods.loc[periods.is_test, "exposure"].sum())\n    check("K12", "exposición total del entrenamiento igual a la de EXP01 (1.968.577,0591 años, tolerancia 0,01)", abs(train_exposure_total - 1968577.0591) < 0.01, r(train_exposure_total, 4))\n\n    # Calendar occurrence-month exposure: covered days in the month / 365.\n    ps = periods["period_start"].values.astype("datetime64[D]")\n    pe = periods["period_end"].values.astype("datetime64[D]")\n    line_codes = periods["line"].map({l: i for i, l in enumerate(LINES)}).values.astype(int)\n    month_rows = []\n    month_exposure = np.zeros((len(LINES), len(MONTHS)))\n    for j, m in enumerate(MONTHS):\n        ms = np.datetime64(m.start_time.date(), "D")\n        me = np.datetime64(m.end_time.date(), "D")\n        overlap = (np.minimum(pe, me) - np.maximum(ps, ms)).astype("timedelta64[D]").astype(int) + 1\n        overlap = np.clip(overlap, 0, None) / 365.0\n        for i in range(len(LINES)):\n            month_exposure[i, j] = overlap[line_codes == i].sum()\n            month_rows.append({"line": LINES[i], "mes": str(m), "expuestos_anios": month_exposure[i, j]})\n    del ps, pe, line_codes, overlap\n    month_df = pd.DataFrame(month_rows)\n    month_df.to_csv(tables_dir / "expuestos_ramo_mes.csv", index=False)\n    year_from_months = month_df.assign(anio=month_df.mes.str[:4].astype(int)).groupby(["line", "anio"])["expuestos_anios"].sum()\n    ely = exp_line_year.set_index(["line", "year"])["expuestos_anios"]\n    diff_month = max(abs(year_from_months[(l, y)] - ely[(l, y)]) for l in LINES for y in range(2021, 2026))\n    check("K13", "la exposición por meses naturales suma la exposición por fila en cada ramo y año (tolerancia 0,5 años)", diff_month < 0.5, r(float(diff_month), 4))\n\n    train = periods.loc[~periods.is_test, ["row_id", "policy_id", "line", "period_start", "period_end", "year", "exposure", "cover_type"]].reset_index(drop=True)\n    del periods\n\n    claims = read("claims", ["claim_id", "policy_id", "line", "peril", "loss_date", "report_date", "status", "paid_to_date", "incurred_to_date", "recoveries", "audit_pre00"])\n    claims["loss_date"] = pd.to_datetime(claims["loss_date"])\n    claims["report_date"] = pd.to_datetime(claims["report_date"])\n    claims["accepted"] = claims["status"].isin(["closed", "open"])\n    claims["net"] = claims["incurred_to_date"] - claims["recoveries"]\n    claims["reserve"] = claims["incurred_to_date"] - claims["paid_to_date"]\n    claims["delay_days"] = (claims["report_date"] - claims["loss_date"]).dt.days\n    observed_cutoff = claims["report_date"].max()\n    check("K06", "la última fecha de reporte observada es el 30-12-2024 (corte de valoración)", observed_cutoff == CUTOFF, str(observed_cutoff.date()))\n    check("K07", "claims vigente: 162.546 registros, 413 marcados audit_pre00, ningún reporte anterior a la ocurrencia",\n          len(claims) == 162546 and int(claims["audit_pre00"].sum()) == 413 and int((claims["delay_days"] < 0).sum()) == 0,\n          {"rows": len(claims), "pre00": int(claims["audit_pre00"].sum()), "negative_delays": int((claims["delay_days"] < 0).sum())})\n\n    # Stage 3: assign claims to policy periods.\n    cand = claims.loc[~claims.audit_pre00, ["claim_id", "policy_id", "line", "loss_date"]].merge(train[["row_id", "policy_id", "line", "period_start", "period_end"]], on=["policy_id", "line"], how="left")\n    cand["inside"] = (cand["loss_date"] >= cand["period_start"]) & (cand["loss_date"] <= cand["period_end"])\n    has_policy = cand.groupby("claim_id")["row_id"].count() > 0\n    n_inside = cand[cand.inside].groupby("claim_id")["row_id"].nunique()\n    assign = pd.DataFrame({"claim_id": claims.loc[~claims.audit_pre00, "claim_id"].values})\n    assign["has_policy"] = assign["claim_id"].map(has_policy).fillna(False).astype(bool)\n    assign["n_periods"] = assign["claim_id"].map(n_inside).fillna(0).astype(int)\n    assign["category"] = np.select([~assign.has_policy, assign.n_periods == 1, assign.n_periods == 0, assign.n_periods > 1],\n                                   ["sin_correspondencia", "unica", "fuera_de_ventana", "ambigua"], default="?")\n    unique_map = cand[cand.inside & cand.claim_id.isin(assign.loc[assign.category == "unica", "claim_id"])][["claim_id", "row_id"]]\n    pre = pd.DataFrame({"claim_id": claims.loc[claims.audit_pre00, "claim_id"].values, "has_policy": True, "n_periods": 0, "category": "prehistoria_PRE00"})\n    assign = pd.concat([assign, pre], ignore_index=True).merge(unique_map, on="claim_id", how="left")\n    claims = claims.merge(assign[["claim_id", "category", "row_id"]], on="claim_id", how="left")\n    check("K08", "PRE00: ningún siniestro audit_pre00 queda asignado a un periodo", claims.loc[claims.audit_pre00, "row_id"].isna().all())\n    cat_counts = claims.groupby(["line", "category"]).size().unstack(fill_value=0)\n    acc_counts = claims[claims.accepted].groupby(["line", "category"]).size().unstack(fill_value=0)\n    check("K09", "sin siniestros ambiguos ni sin póliza", int(cat_counts.get("ambigua", pd.Series(0, index=cat_counts.index)).sum()) == 0 and int(cat_counts.get("sin_correspondencia", pd.Series(0, index=cat_counts.index)).sum()) == 0)\n    expected_unique_accepted = {"motor": 110496, "buildings": 26266, "contents": 23670}\n    got_unique_accepted = {line: int(acc_counts.loc[line].get("unica", 0)) for line in LINES}\n    check("K10", "asignación única de aceptados igual a D42/D50 (110.496, 26.266, 23.670)", got_unique_accepted == expected_unique_accepted, got_unique_accepted)\n    fuera = claims[(claims.category == "fuera_de_ventana")]\n    fuera_desc = {\n        "total": int(len(fuera)), "aceptados": int(fuera.accepted.sum()),\n        "net_aceptado_CU": r(float(fuera.loc[fuera.accepted, "net"].sum()), 2),\n        "loss_31_12_2024": int((fuera.loss_date == pd.Timestamp("2024-12-31")).sum()),\n        "loss_dec_2024": int(((fuera.loss_date >= pd.Timestamp("2024-12-01")) & (fuera.loss_date <= pd.Timestamp("2024-12-31"))).sum()),\n        "loss_after_last_period_of_policy": None,\n    }\n    last_end = train.groupby("policy_id")["period_end"].max()\n    first_start = train.groupby("policy_id")["period_start"].min()\n    fuera_after = fuera.loss_date > fuera.policy_id.map(last_end)\n    fuera_before = fuera.loss_date < fuera.policy_id.map(first_start)\n    fuera_desc["loss_after_last_period_of_policy"] = int(fuera_after.sum())\n    fuera_desc["loss_before_first_period_of_policy"] = int(fuera_before.sum())\n    fuera_desc["loss_in_gap_between_periods"] = int((~fuera_after & ~fuera_before).sum())\n    assignment_table = claims.groupby(["line", "category", "status"]).agg(n=("claim_id", "size"), net_CU=("net", "sum")).reset_index()\n    assignment_table.to_csv(tables_dir / "asignacion_siniestros.csv", index=False)\n    claims[["claim_id", "policy_id", "line", "peril", "loss_date", "report_date", "status", "category", "row_id"]].to_parquet(tables_dir / "asignacion_por_siniestro.parquet", index=False)\n\n    # Stage 4: row-level T0 labels.\n    acc = claims[claims.accepted & (claims.category == "unica")].copy()\n    acc["positive"] = acc["net"] > 0\n    acc["is_open"] = acc["status"] == "open"\n    t0 = acc.groupby("row_id").agg(T0=("net", "sum"), n_accepted=("claim_id", "size"), n_positive=("positive", "sum"), n_open=("is_open", "sum"),\n                                  paid=("paid_to_date", "sum"), incurred=("incurred_to_date", "sum"), recoveries=("recoveries", "sum"), reserve=("reserve", "sum"))\n    t0_rows = train[["row_id", "line", "year", "exposure", "cover_type"]].merge(t0, on="row_id", how="left")\n    for col in ["T0", "n_accepted", "n_positive", "n_open", "paid", "incurred", "recoveries", "reserve"]:\n        t0_rows[col] = t0_rows[col].fillna(0)\n    for col in ["n_accepted", "n_positive", "n_open"]:\n        t0_rows[col] = t0_rows[col].astype(int)\n    check("K11", "la suma de T0 por fila iguala el neto de los aceptados asignados y no hay netos negativos",\n          abs(t0_rows["T0"].sum() - acc["net"].sum()) < 1e-6 and int((acc["net"] < 0).sum()) == 0,\n          {"T0_total_CU": r(float(t0_rows["T0"].sum()), 2), "net_zero_accepted": int((acc["net"] == 0).sum())})\n    t0_rows[["row_id", "line", "year", "exposure", "T0", "n_accepted", "n_positive", "n_open", "paid", "incurred", "recoveries", "reserve"]].to_parquet(tables_dir / "t0_por_row_id.parquet", index=False)\n\n    # Stage 5: reporting delays and count chain-ladder.\n    rep = claims[(claims.category == "unica")].copy()  # All assigned notifications, including accepted and declined claims.\n    rep["occ_m"] = (rep.loss_date.dt.year - 2021) * 12 + rep.loss_date.dt.month - 1\n    rep["rep_m"] = (rep.report_date.dt.year - 2021) * 12 + rep.report_date.dt.month - 1\n    rep["lag"] = rep["rep_m"] - rep["occ_m"]\n    N = 48  # Monthly cohorts from January 2021 through December 2024.\n    delay_rows, cl_rows, tail_rows, month_complete_rows, factor_rows = [], [], [], [], []\n    completeness = {}  # (line, peril) maps to 12 occurrence-month completeness values for 2024.\n    cdf_by_group = {}\n    for (line, peril), g in rep.groupby(["line", "peril"]):\n        # Cumulative count triangle by monthly cohort and lag.\n        tri = np.zeros((N, N))\n        for occ, lag in zip(g.occ_m.values, g.lag.values):\n            if 0 <= occ < N and 0 <= lag < N:\n                tri[occ, lag:] += 1  # Cumulative counts.\n        factors = []\n        for k in range(N - 1):\n            rows = [i for i in range(N) if i + k + 1 <= N - 1]  # Cohorts with observed development through lag k+1.\n            num = sum(tri[i, k + 1] for i in rows)\n            den = sum(tri[i, k] for i in rows)\n            factors.append(num / den if den > 0 else 1.0)\n        factors = np.array(factors)\n        # Exponential tail fitted to the last factors >1 (lags 12-46) for long-tail perils.\n        tail = 1.0\n        tail_fit = None\n        if (line, peril) in LONG_TAIL:\n            ks = np.arange(12, N - 1)\n            fk = factors[12:N - 1]\n            mask = fk > 1.0\n            if mask.sum() >= 6:\n                a, b = np.polyfit(ks[mask], np.log(fk[mask] - 1.0), 1)\n                if a < 0:\n                    future = np.arange(N - 1, N - 1 + 120)\n                    tail = float(np.prod(1.0 + np.exp(a * future + b)))\n                    tail_fit = {"pendiente": r(float(a), 5), "intercepto": r(float(b), 4), "puntos": int(mask.sum()), "cola_120_meses": r(tail, 4)}\n                else:\n                    tail_fit = {"pendiente": r(float(a), 5), "nota": "sin decaimiento; cola fijada en 1"}\n        cdf = np.ones(N)  # Age k to ultimate development factor.\n        for k in range(N - 2, -1, -1):\n            cdf[k] = cdf[k + 1] * factors[k]\n        cdf = cdf * tail\n        cdf_by_group[(line, peril)] = cdf\n        reported_2024 = float(tri[36:48, :].max(axis=1).sum())\n        ult_2024 = float(sum(tri[i, N - 1 - i] * cdf[N - 1 - i] for i in range(36, 48)))\n        rep_by_year = {y: float(sum(tri[i, N - 1 - i] for i in range((y - 2021) * 12, (y - 2021) * 12 + 12))) for y in range(2021, 2025)}\n        ult_by_year = {y: float(sum(tri[i, N - 1 - i] * cdf[N - 1 - i] for i in range((y - 2021) * 12, (y - 2021) * 12 + 12))) for y in range(2021, 2025)}\n        comp_months = np.array([1.0 / cdf[N - 1 - i] for i in range(36, 48)])\n        completeness[(line, peril)] = comp_months\n        for i in range(36, 48):\n            month_complete_rows.append({"line": line, "peril": peril, "mes_ocurrencia": str(MONTHS[i]), "edad_meses": N - 1 - i,\n                                        "reportados": int(tri[i, N - 1 - i]), "factor_a_ultimo": r(float(cdf[N - 1 - i]), 4), "completitud": r(float(1.0 / cdf[N - 1 - i]), 4)})\n        for k in range(N - 1):\n            factor_rows.append({"line": line, "peril": peril, "lag_meses": k, "factor": r(float(factors[k]), 5), "factor_a_ultimo": r(float(cdf[k]), 5)})\n        # Empirical reporting delay in days by occurrence cohort, accepted and declined.\n        for y in range(2021, 2025):\n            gy = g[g.loss_date.dt.year == y]\n            d = gy.delay_days\n            delay_rows.append({"line": line, "peril": peril, "cohorte": y, "n": int(len(d)), "p50": r(float(d.quantile(0.5)), 1) if len(d) else None,\n                               "p90": r(float(d.quantile(0.9)), 1) if len(d) else None, "p99": r(float(d.quantile(0.99)), 1) if len(d) else None,\n                               "pct_30d": r(float((d <= 30).mean() * 100), 2) if len(d) else None, "pct_90d": r(float((d <= 90).mean() * 100), 2) if len(d) else None,\n                               "pct_365d": r(float((d <= 365).mean() * 100), 2) if len(d) else None, "max": int(d.max()) if len(d) else None})\n        cl_rows.append({"line": line, "peril": peril, "notificaciones_2021_2024": int(len(g)), "cola": r(tail, 4), "cola_ajuste": json.dumps(tail_fit, ensure_ascii=False) if tail_fit else "",\n                        **{f"reportados_{y}": int(rep_by_year[y]) for y in range(2021, 2025)},\n                        **{f"ultimo_{y}": r(ult_by_year[y], 1) for y in range(2021, 2025)},\n                        **{f"completitud_{y}": r(rep_by_year[y] / ult_by_year[y], 4) if ult_by_year[y] else None for y in range(2021, 2025)},\n                        "ibnr_conteo_2024": r(ult_2024 - reported_2024, 1), "factor_1_mes": r(float(factors[0]), 4), "factor_a_ultimo_edad_0": r(float(cdf[0]), 4), "factor_a_ultimo_edad_11": r(float(cdf[11]), 4)})\n    pd.DataFrame(delay_rows).to_csv(tables_dir / "retraso_reporte_cohortes.csv", index=False)\n    cl = pd.DataFrame(cl_rows)\n    cl.to_csv(tables_dir / "chain_ladder_conteos.csv", index=False)\n    pd.DataFrame(month_complete_rows).to_csv(tables_dir / "completitud_2024_por_mes.csv", index=False)\n    pd.DataFrame(factor_rows).to_csv(tables_dir / "factores_desarrollo_conteos.csv", index=False)\n    short = cl[~cl.apply(lambda row: (row.line, row.peril) in LONG_TAIL, axis=1)]\n    check("K14", "completitud de reporte 2021 >= 0,995 en todos los peligros de cola corta", bool((short["completitud_2021"] >= 0.995).all()), r(float(short["completitud_2021"].min()), 4))\n    check("K15", "completitud 2024 < completitud 2021 en lesiones corporales y hundimiento",\n          all(cl.loc[(cl.line == l) & (cl.peril == p), "completitud_2024"].iloc[0] < cl.loc[(cl.line == l) & (cl.peril == p), "completitud_2021"].iloc[0] for l, p in LONG_TAIL))\n\n    # Stage 6: severity.\n    sev_rows = []\n    pos = acc[acc.positive]\n    for (line, peril), g in pos.groupby(["line", "peril"]):\n        for y in [2021, 2022, 2023, 2024, "2021-2022", "2021-2024"]:\n            if isinstance(y, int):\n                gy = g[g.loss_date.dt.year == y]\n            elif y == "2021-2022":\n                gy = g[g.loss_date.dt.year <= 2022]\n            else:\n                gy = g\n            s = gy["net"]\n            if len(s) == 0:\n                continue\n            closed = gy[gy.status == "closed"]["net"]\n            opened = gy[gy.status == "open"]["net"]\n            late = gy[gy.delay_days > 365]["net"]\n            sev_rows.append({"line": line, "peril": peril, "ocurrencia": y, "n_positivos": int(len(s)), "media": r(float(s.mean()), 2), "mediana": r(float(s.median()), 2),\n                             "cv": r(float(s.std(ddof=1) / s.mean()), 3) if len(s) > 1 else None, "p90": r(float(s.quantile(0.9)), 2), "p99": r(float(s.quantile(0.99)), 2), "max": r(float(s.max()), 2),\n                             "pct_abiertos": r(float((gy.status == "open").mean() * 100), 2), "media_cerrados": r(float(closed.mean()), 2) if len(closed) else None,\n                             "media_abiertos": r(float(opened.mean()), 2) if len(opened) else None, "reserva_sobre_incurrido_pct": r(float(gy.reserve.sum() / gy.incurred_to_date.sum() * 100), 2),\n                             "n_reportados_mas_365d": int(len(late)), "media_reportados_mas_365d": r(float(late.mean()), 2) if len(late) else None,\n                             "coste_1pct_mayor_pct": r(float(s.nlargest(max(1, int(np.ceil(len(s) * 0.01)))).sum() / s.sum() * 100), 2)})\n    sev = pd.DataFrame(sev_rows)\n    sev.to_csv(tables_dir / "severidad_ramo_peligro.csv", index=False)\n    share_positive = (acc.groupby(["line", "peril"])["positive"].mean()).to_dict()\n    notif_positive = (rep.merge(acc[["claim_id", "positive"]], on="claim_id", how="left").assign(positive=lambda d: d.positive.fillna(False)).groupby(["line", "peril"])["positive"].mean()).to_dict()\n\n    # Stage 7: frequency.\n    freq_rows = []\n    exp_ly = exp_line_year.set_index(["line", "year"])["expuestos_anios"]\n    for line in LINES:\n        for y in range(2021, 2025):\n            e = float(exp_ly[(line, y)])\n            rows = t0_rows[(t0_rows.line == line) & (t0_rows.year == y)]\n            n_pos = int(rows.n_positive.sum())\n            n_acc = int(rows.n_accepted.sum())\n            lam = rows.n_positive.values\n            disp = float(((lam - lam.mean()) ** 2).sum() / (len(lam) - 1) / lam.mean()) if lam.mean() > 0 else None\n            freq_rows.append({"line": line, "year": y, "peril": "todos", "expuestos_anios": r(e, 2), "aceptados": n_acc, "positivos": n_pos,\n                              "frecuencia_aceptados": r(n_acc / e, 5), "frecuencia_positivos": r(n_pos / e, 5), "dispersion_varianza_media": r(disp, 3),\n                              "pct_periodos_sin_coste": r(float((rows.T0 == 0).mean() * 100), 2)})\n            for peril, g in acc[(acc.line == line) & (acc.loss_date.dt.year == y)].groupby("peril"):\n                freq_rows.append({"line": line, "year": y, "peril": peril, "expuestos_anios": r(e, 2), "aceptados": int(len(g)), "positivos": int(g.positive.sum()),\n                                  "frecuencia_aceptados": r(len(g) / e, 5), "frecuencia_positivos": r(float(g.positive.sum()) / e, 5), "dispersion_varianza_media": None, "pct_periodos_sin_coste": None})\n    freq = pd.DataFrame(freq_rows)\n    freq.to_csv(tables_dir / "frecuencia_ramo_anio_peligro.csv", index=False)\n    freq_cover = []\n    for (y, cov), rows in t0_rows[t0_rows.line == "motor"].groupby(["year", "cover_type"]):\n        e = float(rows.exposure.sum())\n        freq_cover.append({"year": y, "cover_type": cov, "filas": int(len(rows)), "expuestos_anios": r(e, 2), "frecuencia_positivos": r(rows.n_positive.sum() / e, 5),\n                           "prima_riesgo_T0_CU": r(float(rows.T0.sum() / e), 2)})\n    pd.DataFrame(freq_cover).to_csv(tables_dir / "frecuencia_autos_cobertura.csv", index=False)\n\n    # Ultimate 2024 frequency: count IBNR on positive claims and effective exposure.\n    ult_rows = []\n    for (line, peril), comp in completeness.items():\n        g24 = acc[(acc.line == line) & (acc.peril == peril) & (acc.loss_date.dt.year == 2024)]\n        n_pos = int(g24.positive.sum())\n        n_all = int(len(rep[(rep.line == line) & (rep.peril == peril) & (rep.loss_date.dt.year == 2024)]))\n        e24 = month_exposure[LINES.index(line), 36:48]\n        e_eff = float((e24 * comp).sum())\n        e_nom = float(e24.sum())\n        ibnr_notif = float(cl.loc[(cl.line == line) & (cl.peril == peril), "ibnr_conteo_2024"].iloc[0])\n        p_pos = float(notif_positive.get((line, peril), 0.0))\n        ibnr_pos = ibnr_notif * p_pos\n        sev_mature = sev[(sev.line == line) & (sev.peril == peril) & (sev.ocurrencia == "2021-2022")]\n        mean_sev = float(sev_mature["media"].iloc[0]) if len(sev_mature) else float("nan")\n        late_sev = sev_mature["media_reportados_mas_365d"].iloc[0] if len(sev_mature) else None\n        sev_for_ibnr = float(late_sev) if ((line, peril) in LONG_TAIL and late_sev is not None and not pd.isna(late_sev)) else mean_sev\n        t0_24 = float(g24["net"].sum())\n        ult_rows.append({"line": line, "peril": peril, "notificaciones_2024": n_all, "positivos_2024": n_pos, "expuestos_2024": r(e_nom, 2), "expuestos_efectivos_2024": r(e_eff, 2),\n                         "completitud_ponderada_2024": r(e_eff / e_nom, 4), "frecuencia_positivos_reportada": r(n_pos / e_nom, 5), "frecuencia_positivos_ultima_por_exposicion_efectiva": r(n_pos / e_eff, 5),\n                         "ibnr_notificaciones": r(ibnr_notif, 1), "prob_positivo_por_notificacion": r(p_pos, 4), "ibnr_positivos": r(ibnr_pos, 1),\n                         "severidad_media_2021_2022": r(mean_sev, 2), "severidad_usada_ibnr": r(sev_for_ibnr, 2), "ibnr_coste_CU": r(ibnr_pos * sev_for_ibnr, 2),\n                         "T0_2024_CU": r(t0_24, 2), "factor_ibnr_coste": r((t0_24 + ibnr_pos * sev_for_ibnr) / t0_24, 4) if t0_24 > 0 else None})\n    ult = pd.DataFrame(ult_rows)\n    ult.to_csv(tables_dir / "ibnr_2024_por_peligro.csv", index=False)\n    ult_line = ult.groupby("line").agg(T0_2024_CU=("T0_2024_CU", "sum"), ibnr_coste_CU=("ibnr_coste_CU", "sum"), ibnr_positivos=("ibnr_positivos", "sum"), positivos_2024=("positivos_2024", "sum")).reset_index()\n    ult_line["factor_ibnr_coste"] = (ult_line.T0_2024_CU + ult_line.ibnr_coste_CU) / ult_line.T0_2024_CU\n    ult_line["factor_ibnr_conteo"] = (ult_line.positivos_2024 + ult_line.ibnr_positivos) / ult_line.positivos_2024\n    ult_line.to_csv(tables_dir / "ibnr_2024_por_ramo.csv", index=False)\n\n    # Apply the same peril/year IBNR rule for 2021-2024 and derive ultimate risk premiums.\n    sev_used = {(row.line, row.peril): row.severidad_usada_ibnr for row in ult.itertuples()}\n    ibnr_year_rows = []\n    for row in cl.itertuples():\n        for y in range(2021, 2025):\n            reported = getattr(row, f"reportados_{y}")\n            ultimate = getattr(row, f"ultimo_{y}")\n            ibnr_notif = float(ultimate) - float(reported)\n            p_pos = float(notif_positive.get((row.line, row.peril), 0.0))\n            gy = acc[(acc.line == row.line) & (acc.peril == row.peril) & (acc.loss_date.dt.year == y)]\n            t0_y = float(gy.net.sum())\n            cost = ibnr_notif * p_pos * float(sev_used[(row.line, row.peril)])\n            ibnr_year_rows.append({"line": row.line, "peril": row.peril, "year": y, "notificaciones": int(reported), "ultimo": r(float(ultimate), 1), "ibnr_notificaciones": r(ibnr_notif, 1),\n                                   "ibnr_positivos": r(ibnr_notif * p_pos, 1), "ibnr_coste_CU": r(cost, 2), "T0_CU": r(t0_y, 2), "factor_ibnr_coste": r((t0_y + cost) / t0_y, 4) if t0_y > 0 else None})\n    ibnr_year = pd.DataFrame(ibnr_year_rows)\n    ibnr_year.to_csv(tables_dir / "ibnr_por_peligro_anio.csv", index=False)\n    prima_ult_rows = []\n    for line in LINES:\n        for y in range(2021, 2025):\n            e = float(exp_ly[(line, y)])\n            sub = ibnr_year[(ibnr_year.line == line) & (ibnr_year.year == y)]\n            t0_y = float(sub.T0_CU.sum())\n            cost = float(sub.ibnr_coste_CU.sum())\n            res_y = float(acc[(acc.line == line) & (acc.loss_date.dt.year == y) & acc.is_open].reserve.sum())\n            prima_ult_rows.append({"line": line, "year": y, "expuestos_anios": r(e, 2), "T0_CU": r(t0_y, 2), "ibnr_coste_CU": r(cost, 2), "reserva_abiertos_CU": r(res_y, 2),\n                                   "prima_riesgo_T0_CU": r(t0_y / e, 2), "prima_riesgo_T0_mas_IBNR_CU": r((t0_y + cost) / e, 2), "factor_ibnr_coste": r((t0_y + cost) / t0_y, 4),\n                                   "prima_T1_a075_mas_IBNR_CU": r((t0_y - 0.25 * res_y + cost) / e, 2), "prima_T1_a125_mas_IBNR_CU": r((t0_y + 0.25 * res_y + cost) / e, 2)})\n    prima_ult = pd.DataFrame(prima_ult_rows)\n    prima_ult.to_csv(tables_dir / "prima_riesgo_ultima_ramo_anio.csv", index=False)\n\n    # Stage 8: case reserves and T1 sensitivity.\n    res_rows = []\n    for (line, peril, y), g in acc.groupby(["line", "peril", acc.loss_date.dt.year.rename("year")]):\n        op = g[g.is_open]\n        res_rows.append({"line": line, "peril": peril, "year": int(y), "aceptados": int(len(g)), "abiertos": int(len(op)), "pct_abiertos": r(float(len(op) / len(g) * 100), 2),\n                         "T0_CU": r(float(g.net.sum()), 2), "reserva_abiertos_CU": r(float(op.reserve.sum()), 2), "reserva_sobre_T0_pct": r(float(op.reserve.sum() / g.net.sum() * 100), 2) if g.net.sum() > 0 else None,\n                         "T1_a075_CU": r(float(g.net.sum() - 0.25 * op.reserve.sum()), 2), "T1_a125_CU": r(float(g.net.sum() + 0.25 * op.reserve.sum()), 2)})\n    res = pd.DataFrame(res_rows)\n    res.to_csv(tables_dir / "reservas_caso_T1.csv", index=False)\n    res_line_year = res.groupby(["line", "year"]).agg(T0_CU=("T0_CU", "sum"), reserva_abiertos_CU=("reserva_abiertos_CU", "sum"), abiertos=("abiertos", "sum"), aceptados=("aceptados", "sum")).reset_index()\n    res_line_year["reserva_sobre_T0_pct"] = res_line_year.reserva_abiertos_CU / res_line_year.T0_CU * 100\n    res_line_year["T1_rango_pct"] = 0.25 * res_line_year["reserva_sobre_T0_pct"]\n    res_line_year.to_csv(tables_dir / "reservas_caso_ramo_anio.csv", index=False)\n\n    # Stage 9: risk premiums and M00 references.\n    pr_rows = []\n    for line in LINES:\n        for y in range(2021, 2025):\n            rows = t0_rows[(t0_rows.line == line) & (t0_rows.year == y)]\n            e = float(rows.exposure.sum())\n            n_pos = float(rows.n_positive.sum())\n            pr_rows.append({"line": line, "year": y, "expuestos_anios": r(e, 2), "T0_CU": r(float(rows.T0.sum()), 2), "prima_riesgo_T0_CU": r(float(rows.T0.sum() / e), 2),\n                            "frecuencia_positivos": r(n_pos / e, 5), "severidad_media_positivos_CU": r(float(rows.T0.sum() / n_pos), 2) if n_pos else None,\n                            "identidad_freq_x_sev": r(float(n_pos / e * rows.T0.sum() / n_pos), 2) if n_pos else None})\n    pr = pd.DataFrame(pr_rows)\n    pr.to_csv(tables_dir / "prima_riesgo_ramo_anio.csv", index=False)\n    check("K16", "identidad prima de riesgo = frecuencia positiva x severidad media positiva en todos los ramos y años",\n          bool(np.allclose(pr["prima_riesgo_T0_CU"], pr["identidad_freq_x_sev"], rtol=1e-6)))\n\n    # Check the closed-form power-1.5 deviance.\n    yy = np.array([0.0, 0.0, 12.5, 3000.0, 1.0])\n    mm = np.array([100.0, 2.0, 10.0, 2500.0, 0.5])\n    closed_form = 4.0 * (np.sqrt(mm) - np.sqrt(yy)) ** 2 / np.sqrt(mm)\n    check("K17", "la desviación Tweedie p=1,5 coincide con la forma cerrada 4(sqrt(mu)-sqrt(y))^2/sqrt(mu)", bool(np.allclose(tweedie_deviance(yy, mm), closed_form)))\n\n    m00_rows = []\n    splits = {"V22": ([2021], 2022), "V23": ([2021, 2022], 2023), "V24": ([2021, 2022, 2023], 2024)}\n    for split, (train_years, eval_year) in splits.items():\n        tr = t0_rows[t0_rows.year.isin(train_years)]\n        ev = t0_rows[t0_rows.year == eval_year].copy()\n        q_bal = tr.groupby("line")["T0"].sum() / tr.groupby("line")["exposure"].sum()\n        q_tw = (tr.T0 / np.sqrt(tr.exposure)).groupby(tr.line).sum() / np.sqrt(tr.exposure).groupby(tr.line).sum()\n        q_glob_bal = tr.T0.sum() / tr.exposure.sum()\n        ev["m_bal"] = ev.exposure * ev.line.map(q_bal)\n        ev["m_tw"] = ev.exposure * ev.line.map(q_tw)\n        ev["m_glob"] = ev.exposure * q_glob_bal\n        ev["m_const"] = tr.T0.mean()\n        rowd = {"split": split, "train": "-".join(map(str, train_years)), "eval": eval_year, "filas_eval": int(len(ev)),\n                "d2_constante_por_fila": r(d2_tweedie(ev.T0.values, ev.m_const.values), 5),\n                "d2_tasa_global_x_exposicion": r(d2_tweedie(ev.T0.values, ev.m_glob.values), 5),\n                "d2_tasa_ramo_balance_x_exposicion": r(d2_tweedie(ev.T0.values, ev.m_bal.values), 5),\n                "d2_tasa_ramo_tweedie_x_exposicion": r(d2_tweedie(ev.T0.values, ev.m_tw.values), 5)}\n        for line in LINES:\n            evl = ev[ev.line == line]\n            rowd[f"q_balance_{line}"] = r(float(q_bal[line]), 4)\n            rowd[f"q_tweedie_{line}"] = r(float(q_tw[line]), 4)\n            rowd[f"d2_ramo_{line}_balance"] = r(d2_tweedie(evl.T0.values, evl.m_bal.values), 5)\n            rowd[f"d2_ramo_{line}_tweedie"] = r(d2_tweedie(evl.T0.values, evl.m_tw.values), 5)\n            rowd[f"balance_{line}_pred_sobre_obs"] = r(float(evl.m_bal.sum() / evl.T0.sum()), 4)\n        m00_rows.append(rowd)\n    prod = t0_rows\n    q_bal_prod = prod.groupby("line")["T0"].sum() / prod.groupby("line")["exposure"].sum()\n    q_tw_prod = (prod.T0 / np.sqrt(prod.exposure)).groupby(prod.line).sum() / np.sqrt(prod.exposure).groupby(prod.line).sum()\n    m00 = pd.DataFrame(m00_rows)\n    m00.to_csv(tables_dir / "referencias_M00.csv", index=False)\n    check("K18", "en los tres cortes: D² de la constante por fila dentro de ±0,01 y D² de tasa por ramo x exposición > tasa global x exposición > 0",\n          bool((m00["d2_constante_por_fila"].abs() < 0.01).all() and (m00["d2_tasa_ramo_balance_x_exposicion"] > m00["d2_tasa_global_x_exposicion"]).all() and (m00["d2_tasa_global_x_exposicion"] > 0).all()),\n          m00[["split", "d2_constante_por_fila", "d2_tasa_global_x_exposicion", "d2_tasa_ramo_balance_x_exposicion"]].to_dict(orient="records"))\n\n    # Stage 10: evidence and manifest.\n    evidence = {\n        "run_id": run_id,\n        "created_utc": started.isoformat(timespec="seconds"),\n        "author": "AI-assisted calculation requested by the participant; no human or independent professional review",\n        "scope": {"data_modified": False, "indices_modified": False, "models_trained": False, "remote_execution": False, "external_providers": False,\n                  "description": "Bases actuariales (expuestos, asignación, T0, retraso e IBNR de conteos, frecuencia, severidad, reservas de caso, referencias M00) sobre la capa preparada vigente"},\n        "provenance": {"active_index": "Hackaton/Semana 1/Limpieza/current_dataset.json", "active_run": current["workspace_relative_run"], "active_status": current["status"],\n                       "catalog_sha256": digest(catalog_path), "tables_sha256": hashes, "cutoff_observed": str(observed_cutoff.date()), "power_eval": POWER},\n        "assignment": {"by_line_category": {line: {cat: int(cat_counts.loc[line].get(cat, 0)) for cat in cat_counts.columns} for line in LINES},\n                       "accepted_unique_by_line": got_unique_accepted, "accepted_unique_total": int(sum(got_unique_accepted.values())),\n                       "accepted_reported_by_line": {line: int(claims[(claims.line == line) & claims.accepted].shape[0]) for line in LINES},\n                       "out_of_window": fuera_desc, "pre00_claims": 413, "pre00_accepted_net_CU": r(float(claims.loc[claims.audit_pre00 & claims.accepted, "net"].sum()), 2)},\n        "t0": {"total_CU": r(float(t0_rows.T0.sum()), 2), "by_line_CU": {line: r(float(t0_rows.loc[t0_rows.line == line, "T0"].sum()), 2) for line in LINES},\n               "rows_train": int(len(t0_rows)), "rows_with_positive_cost": int((t0_rows.T0 > 0).sum()), "accepted_net_zero": int((acc.net == 0).sum()),\n               "open_claims_assigned": int(acc.is_open.sum()), "reserve_open_total_CU": r(float(acc.loc[acc.is_open, "reserve"].sum()), 2)},\n        "exposure": {"train_total_years": r(train_exposure_total, 4), "test_total_years": r(test_exposure_total, 4),\n                     "by_line_year": exp_line_year.round(4).to_dict(orient="records"), "motor_by_cover": exp_cover.round(4).to_dict(orient="records"),\n                     "by_line_month_2024": {line: [r(float(v), 2) for v in month_exposure[LINES.index(line), 36:48]] for line in LINES},\n                     "by_line_month_2025": {line: [r(float(v), 2) for v in month_exposure[LINES.index(line), 48:60]] for line in LINES}},\n        "reporting": {"chain_ladder_by_peril": cl.to_dict(orient="records"), "delay_by_cohort": delay_rows, "long_tail_perils": [list(x) for x in sorted(LONG_TAIL)],\n                      "method": "chain-ladder de conteos acumulados por cohorte mensual de ocurrencia y lag mensual de reporte, factores ponderados por volumen, cola exponencial solo en lesiones y hundimiento"},\n        "ibnr_2024": {"by_peril": ult.to_dict(orient="records"), "by_line": ult_line.round(4).to_dict(orient="records"),\n                      "note": "IBNR de notificaciones por chain-ladder; positivos = notificaciones x probabilidad de coste positivo (2021-2024); coste = positivos x severidad media 2021-2022 (en lesiones y hundimiento, media de los reportados a más de 365 días)"},\n        "ibnr_by_year": {"by_peril_year": ibnr_year.to_dict(orient="records"), "risk_premium_ultimate_by_line_year": prima_ult.to_dict(orient="records")},\n        "frequency": {"by_line_year_peril": freq.to_dict(orient="records"), "motor_by_cover": freq_cover},\n        "severity": {"by_line_peril": sev.to_dict(orient="records"), "share_positive_among_accepted": {f"{k[0]}|{k[1]}": r(float(v), 4) for k, v in share_positive.items()}},\n        "case_reserves": {"by_line_peril_year": res.to_dict(orient="records"), "by_line_year": res_line_year.round(4).to_dict(orient="records")},\n        "risk_premium": {"by_line_year": pr.to_dict(orient="records")},\n        "m00": {"splits": m00.to_dict(orient="records"), "production_2021_2024": {line: {"q_balance": r(float(q_bal_prod[line]), 4), "q_tweedie": r(float(q_tw_prod[line]), 4)} for line in LINES},\n                "deviance_closed_form_p15": "4*(sqrt(mu)-sqrt(y))^2/sqrt(mu)"},\n        "checks": checks,\n        "checks_summary": {"checks_total": len(checks), "checks_passed": sum(c["passed"] for c in checks), "all_passed": all(c["passed"] for c in checks)},\n    }\n    (run_dir / "evidencia.json").write_text(json.dumps(evidence, ensure_ascii=False, indent=1, default=lambda o: r(o) if isinstance(o, (np.floating, np.integer)) else str(o)), encoding="utf-8")\n    pd.DataFrame(checks).to_csv(tables_dir / "controles.csv", index=False)\n    shutil.copy2(Path(__file__), run_dir / Path(__file__).name)\n    manifest = {}\n    for path in sorted(run_dir.rglob("*")):\n        if path.is_file() and path.name != "manifest.json":\n            manifest[str(path.relative_to(run_dir)).replace("\\\\", "/")] = digest(path)\n    (run_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")\n    summary = evidence["checks_summary"]\n    print(f"Corrida {run_id}: {summary[\'checks_passed\']}/{summary[\'checks_total\']} controles; T0 total {evidence[\'t0\'][\'total_CU\']:,} CU")\n    for c in checks:\n        print(("OK  " if c["passed"] else "FALLA ") + c["code"] + " " + c["name"] + ("" if c["passed"] else f" -> {c[\'detail\']}"))\n    if not summary["all_passed"]:\n        sys.exit(1)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/datos/construir_f0.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/datos/construir_f0.py'] = '"""Build line-specific F0 predictors, labels and persistent temporal partitions.\n\nVerify the ten prepared input tables and the actuarial-base manifest. Write\n2021-2024 training/2025 test tables, row/peril T0 labels, partition keys,\navailability contract, evidence and source/hash manifest into a new run.\n\nFeature blocks: F0 contractual/physical attributes; F1 policy notifications known\nbefore 1 January of the period year; F2 prior-year address weather; F3 conditional\nattributes without reference dates; F4 the existing tariff. Keys, labels and audit\nflags are excluded from predictors. F3/F4 require their documented separate checks.\nScale insured values above each line threshold by 1,000. Reconstructed capital\naudit flags remain excluded. Harmonize construction t/w into tw_code and exclude\nthe unharmonized original.\nPeril effective exposure weights covered days by occurrence-month completeness\nat the historical valuation date. Completeness is inverse chain-ladder development\nand equals 1 for 2025; e_ef_coste uses line-level developed-cost peril shares.\nV22 trains on 2021 for 2022; V23 on 2021-2022 for customer-hash-split 2023;\nV24 on 2021-2023 for 2024; production on 2021-2024 for 2025.\n\nRun all lines through the isolated runner, or use --ramo motor, then\n--run <id> --ramo buildings,contents and --run <id> --finalizar for staged execution.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport shutil\nimport sys\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\n\n\ndef find_root(start: Path) -> Path:\n    for candidate in [start, *start.parents]:\n        if (candidate / "RUTAS_PROYECTO.json").is_file():\n            return candidate\n    raise FileNotFoundError("No se encontró RUTAS_PROYECTO.json hacia arriba.")\n\n\nROOT = find_root(HERE)\ntry:\n    import pyarrow  # noqa: F401\nexcept ImportError:  # pragma: no cover\n    sys.path.insert(0, str(ROOT / "Preparación" / ".cache" / "parquet-reader"))\n\nimport numpy as np  # noqa: E402\nimport pandas as pd  # noqa: E402\nimport pyarrow as pa  # noqa: E402\nimport pyarrow.parquet as pq  # noqa: E402\n\nLINES = ["motor", "buildings", "contents"]\nPERILS = {"motor": ["own_damage", "third_party_prop", "bodily_injury", "theft", "glass_hail"],\n          "buildings": ["storm", "flood", "escape_of_water", "subsidence", "fire", "accidental_damage"],\n          "contents": ["theft", "escape_of_water", "accidental_damage", "flood"]}\nTHRESHOLD = {"buildings": 10_000_000.0, "contents": 1_000_000.0}\nMONTHS = pd.period_range("2021-01", "2025-12", freq="M")\nCUTOFF_MONTH_INDEX = 47  # December 2024 with January 2021 indexed as zero.\nCAL_SALT = "cal_v1"\nTRAIN_ROWS = 2409311\nTEST_ROWS = 626639\nSTRING_MAPPER = {pa.string(): pd.StringDtype("pyarrow"), pa.large_string(): pd.StringDtype("pyarrow")}.get\nAMOUNT_LABELS = ["T0", "paid", "incurred", "recoveries", "reserve"]\nCOUNT_LABELS = ["n_accepted", "n_positive", "n_open"]\nEXCLUDED_ALWAYS = ["customer_service_contacts", "renewal_decision", "audit_ren00", "surname", "construction_type", "sum_insured_raw",\n                   "audit_capital_scale_adjusted", "audit_capital_scale_confirmed", "audit_capital_cap01"]\nBLOCKS = ["F0", "F1", "F2", "F3", "F4", "excluida", "clave", "etiqueta"]\n\nCONTRACT_COMMON = {\n    "row_id": ("clave", "identificador de la fila; nunca predictor"),\n    "policy_id": ("clave", "identificador; nunca predictor"),\n    "customer_id": ("clave", "identificador; solo para hash de calibración y agrupación"),\n    "address_id": ("clave", "identificador; solo para agrupación y cruces"),\n    "period_start": ("clave", "fecha de inicio; define año y mes de ocurrencia"),\n    "period_end": ("clave", "fecha de fin"),\n    "is_test": ("clave", "indicador de fila de test 2025"),\n    "exposure": ("F0", "exposición nominal EXP01 en años; offset o multiplicador en predicción"),\n    "year": ("F0", "año del periodo; efecto de nivel con extrapolación declarada a 2025"),\n    "mes_inicio": ("F0", "mes de inicio del periodo; estacionalidad y duración"),\n    "periodo_corto": ("F0", "indicador de exposición menor que 0,5 años; efecto de duración (D37)"),\n    "new_business": ("F0", "nuevo negocio frente a renovación; conocido al inicio"),\n    "renewal_count": ("F0", "número de renovaciones; conocido al inicio"),\n    "channel": ("F0", "canal de venta; contractual"),\n    "payment_method": ("F0", "forma de pago; contractual"),\n    "voluntary_excess": ("F0", "franquicia voluntaria; contractual"),\n    "compulsory_excess": ("F0", "franquicia obligatoria; contractual"),\n    "days_quote_to_inception": ("F0", "días de cotización a inicio; conocido al inicio"),\n    "missed_payments_12m": ("F3", "impagos de los últimos doce meses sin fecha de referencia; prueba aparte"),\n    "discount_applied": ("F4", "descuento comercial; ablación del tarifario"),\n    "written_premium": ("F4", "prima escrita del tarifario existente; ablación aparte, nunca en la matriz base"),\n    "customer_service_contacts": ("excluida", "mide el siniestro del propio periodo y cambia de distribución en el test (D50)"),\n    "renewal_decision": ("excluida", "solo existe en entrenamiento; decisión posterior al periodo (D45)"),\n    "audit_ren00": ("excluida", "marca de auditoría REN00"),\n    "anios_como_cliente": ("F0", "año del periodo menos customer_since_year; conocido al inicio"),\n    "surname": ("excluida", "nombre personal"),\n    "occupation": ("F3", "atributo del maestro de clientes sin fecha de referencia"),\n    "marital_status": ("F3", "atributo del maestro sin fecha de referencia"),\n    "household_size": ("F3", "atributo del maestro sin fecha de referencia"),\n    "credit_band": ("F3", "banda de crédito sin fecha de referencia"),\n    "prior_claims_5y": ("F3", "siniestros previos declarados sin fecha de referencia; posible solape con F1"),\n    "licence_points": ("F3", "puntos de licencia sin fecha de referencia"),\n    "years_at_address": ("F3", "años en el domicilio sin fecha de referencia"),\n    "email_domain": ("F3", "atributo comercial"),\n    "marketing_consent": ("F3", "atributo comercial"),\n    "document_preference": ("F3", "atributo comercial"),\n    "previous_insurer": ("F3", "atributo comercial sin fecha"),\n    "campaign_code": ("F3", "atributo comercial sin fecha"),\n    "branch_id": ("F3", "sucursal; atributo comercial"),\n    "marketing_segment": ("F3", "segmento comercial de la aseguradora; posible proxy de edad y renta"),\n    "region": ("F0", "región de la dirección; física y estable"),\n    "urban_density": ("F0", "densidad urbana; física y estable"),\n    "flood_zone": ("F0", "zona de inundación 1-5; nula en 2,9 % de direcciones, el nulo se conserva como ausencia"),\n    "distance_to_river_km": ("F0", "distancia al río; física"),\n    "elevation_m": ("F0", "elevación; física"),\n    "soil_type": ("F0", "tipo de suelo; físico"),\n    "year_built": ("F0", "año de construcción; estable"),\n    "property_type": ("F0", "tipo de inmueble; estable"),\n    "roof_type": ("F0", "tipo de tejado; estable"),\n    "basement": ("F0", "sótano; estable"),\n    "trees_within_5m": ("F0", "árboles a menos de 5 m; físico"),\n    "listed_building": ("F0", "edificio protegido; estable"),\n    "bedrooms": ("F0", "habitaciones; estable"),\n    "energy_rating": ("F0", "calificación energética; estable bajo supuesto de maestro"),\n    "solar_panels": ("F0", "paneles solares; estable bajo supuesto de maestro"),\n}\nCONTRACT_LINE = {\n    "motor": {\n        "driver_age": ("F0", "edad del conductor principal; contractual"),\n        "years_licensed": ("F0", "años de licencia (VD34 abierto: 3.717 edades implícitas menores de 16)"),\n        "ncb_years": ("F0", "años de bonificación (161.822 filas con ncb mayor que licencia, sin recortar)"),\n        "vehicle_value_group": ("F0", "grupo de valor del vehículo"),\n        "vehicle_age": ("F0", "antigüedad del vehículo"),\n        "make_model": ("F0", "marca y modelo; alta cardinalidad, codificar dentro del fold"),\n        "engine_size_cc": ("F0", "cilindrada; nula en 2.533 eléctricos (VD35), se conserva nula"),\n        "body_type": ("F0", "carrocería"),\n        "transmission": ("F0", "transmisión"),\n        "seats": ("F0", "plazas"),\n        "vehicle_colour": ("F0", "color; señal dudosa, candidata a ablación"),\n        "fuel_type": ("F0", "combustible"),\n        "driver_assist": ("F0", "asistencia a la conducción"),\n        "cover_type": ("F0", "cobertura; estructura contractual con elegibilidad por peligro"),\n        "annual_mileage": ("F0", "kilometraje declarado"),\n        "overnight_parking": ("F0", "estacionamiento nocturno"),\n        "vehicle_use": ("F0", "uso del vehículo"),\n        "named_young_driver": ("F0", "conductor joven designado"),\n        "telematics_opt_in": ("F3", "adhesión a telemática; bloque condicional"),\n        "telematics_score": ("F3", "score telemático; puede generarse durante la cobertura; ausente = sin adhesión, no cero"),\n    },\n    "buildings": {\n        "policyholder_age": ("F0", "edad AGE01 al inicio del periodo"),\n        "sum_insured_raw": ("excluida", "capital leído de la capa vigente; se conserva para auditoría"),\n        "sum_insured": ("F0", "capital bajo el contrato de F0 de D66 (entre mil sobre 10.000.000 CU)"),\n        "log_sum_insured": ("F0", "logaritmo del capital del contrato; curva suave candidata"),\n        "construction_type": ("excluida", "t y w son alias temporales (D39); usar construction_type_harmonized"),\n        "construction_type_harmonized": ("F0", "construcción con t y w unificados en tw_code (equivalente a CON01, no activado en Limpieza)"),\n        "occupancy": ("F0", "ocupación del inmueble; contractual"),\n        "days_unoccupied": ("F0", "días desocupado declarados; contractual"),\n        "smoke_alarms": ("F0", "detectores de humo; nulo en 2,5 %, nunca cambia dentro de una dirección (D63)"),\n        "accidental_damage_cover": ("F0", "cobertura opcional de daño accidental"),\n        "mortgage": ("F0", "hipoteca; contractual"),\n        "audit_capital_scale_adjusted": ("excluida", "marca D66: capital dividido entre mil en F0"),\n        "audit_capital_scale_confirmed": ("excluida", "marca D66: división respaldada por historia (rama CAP02R-CAP06R o lista D66)"),\n        "audit_capital_cap01": ("excluida", "fila corregida por CAP01 en la base"),\n    },\n    "contents": {\n        "policyholder_age": ("F0", "edad AGE01 al inicio del periodo"),\n        "sum_insured_raw": ("excluida", "capital leído de la capa vigente; se conserva para auditoría"),\n        "sum_insured": ("F0", "capital bajo el contrato de F0 de D66 (entre mil sobre 1.000.000 CU)"),\n        "log_sum_insured": ("F0", "logaritmo del capital del contrato; curva suave candidata"),\n        "tenure": ("F0", "tenencia; contractual"),\n        "security_level": ("F0", "nivel de seguridad; nulo en 2,5 %"),\n        "high_value_items": ("F0", "bienes de alto valor; nulo en 2,5 %"),\n        "accidental_damage_cover": ("F0", "cobertura opcional"),\n        "works_from_home": ("F0", "trabajo desde casa; contractual"),\n        "lodgers": ("F0", "huéspedes; contractual"),\n        "bicycle_cover": ("F0", "cobertura de bicicleta"),\n        "audit_capital_scale_adjusted": ("excluida", "marca D66"),\n        "audit_capital_scale_confirmed": ("excluida", "marca D66"),\n        "audit_capital_cap01": ("excluida", "fila corregida por CAP01 en la base"),\n    },\n}\nWEATHER_VARS = ["hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index", "wildfire_index", "rainfall_mm", "mean_temperature_c"]\n\n\ndef digest(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef read_table(path: Path, columns=None, filters=None) -> pd.DataFrame:\n    return pq.read_table(path, columns=columns, filters=filters).to_pandas(types_mapper=STRING_MAPPER)\n\n\ndef cal_half(customer_id: str) -> int:\n    return int(hashlib.sha256(f"{CAL_SALT}|{customer_id}".encode("utf-8")).hexdigest()[:16], 16) % 2\n\n\ndef classify(col: str, line: str):\n    if col in CONTRACT_LINE[line]:\n        return CONTRACT_LINE[line][col]\n    if col in CONTRACT_COMMON:\n        return CONTRACT_COMMON[col]\n    if col.endswith("_prev1"):\n        return ("F2", "clima del año anterior en la dirección (nulo en 2021); años posteriores al corte nunca")\n    if col.startswith("f1_"):\n        return ("F1", "historia de notificaciones de la póliza conocida al 1 de enero del año del periodo, sin filtrar por estado final")\n    if col.startswith("e_ef_"):\n        return ("F0", "exposición efectiva por completitud de reporte: offset de entrenamiento; en predicción se usa exposure")\n    if col in AMOUNT_LABELS + COUNT_LABELS or col.startswith("y_") or col.startswith("n_"):\n        return ("etiqueta", "coste o conteo del propio periodo valorado al 30-12-2024; nunca predictor; nulo en test")\n    return ("revisar", "columna sin contrato declarado")\n\n\nclass Context:\n    """Load prepared data, hashes, actuarial bases, capital audit context and peril completeness."""\n\n    def __init__(self):\n        self.checks = []\n        current = json.loads((ROOT / "Hackaton/Semana 1/Limpieza/current_dataset.json").read_text(encoding="utf-8"))\n        self.current = current\n        self.catalog_path = ROOT / current["catalog_workspace_relative"]\n        self.catalog = json.loads(self.catalog_path.read_text(encoding="utf-8"))\n        used = ["policies_motor", "policies_buildings", "policies_contents", "test_motor", "test_buildings", "test_contents", "customers", "addresses", "weather", "claims"]\n        self.hashes = {n: digest(ROOT / self.catalog[n]["workspace_relative_path"]) for n in used}\n        bad = [n for n in used if self.hashes[n] != self.catalog[n]["sha256"]]\n        self.check("K01", "SHA-256 de las diez tablas usadas iguales al catálogo activo", not bad, bad)\n\n        self.bases_index = json.loads((ROOT / "Hackaton/Semana 1/Bases_actuariales/current_run.json").read_text(encoding="utf-8"))\n        self.bases_run = ROOT / self.bases_index["workspace_relative_run"]\n        manifest = json.loads((self.bases_run / "manifest.json").read_text(encoding="utf-8"))\n        self.bases_evidence = json.loads((self.bases_run / "evidencia.json").read_text(encoding="utf-8"))\n        needed = ["tables/t0_por_row_id.parquet", "tables/asignacion_por_siniestro.parquet", "tables/factores_desarrollo_conteos.csv", "tables/ibnr_por_peligro_anio.csv"]\n        self.check("K02", "la corrida vigente de Bases_actuariales usa la misma capa y sus tablas conservan el hash",\n                   self.bases_evidence["provenance"]["active_run"] == current["workspace_relative_run"] and all(digest(self.bases_run / n) == manifest[n] for n in needed),\n                   self.bases_index["workspace_relative_run"])\n\n        # The historical ledgers are not redistributed. This is an explicit retrospective\n        # reconstruction of the audit flags, not the original authorization process.\n        self.branch_index = {"workspace_relative_run": "audit-ledger-not-included"}\n        self.cap01_index = {"workspace_relative_run": current["workspace_relative_run"]}\n        self.d66_index = {"workspace_relative_run": "audit-ledger-not-included"}\n        reconstructed = json.loads((ROOT / current["workspace_relative_run"] / "evidence/cap01_row_ids.json").read_text(encoding="utf-8"))\n        self.cap01_ids = {str(row) for rows in reconstructed.values() for row in rows}\n        self.confirmed_ids = set()\n        for line, threshold in (("buildings", 10000000), ("contents", 1000000)):\n            frames = [read_table(ROOT / self.catalog[f"{kind}_{line}"]["workspace_relative_path"], columns=["row_id", "policy_id", "address_id", "sum_insured"]) for kind in ("policies", "test")]\n            rows = pd.concat(frames, ignore_index=True)\n            normal = rows.sum_insured.le(threshold)\n            grouping = [rows.policy_id, rows.address_id] if line == "buildings" else rows.policy_id\n            has_normal = normal.groupby(grouping, dropna=False).transform("sum").gt(0)\n            self.confirmed_ids.update(rows.loc[rows.sum_insured.gt(threshold) & has_normal, "row_id"].astype(str))\n        self.check("K03_CAP01", "CAP01 identifiers reconstructed from originals", len(self.cap01_ids) == 2073)\n        self.check("K03_RECONSTRUCTED", "Retrospective reconstruction of confirmation flag", len(self.confirmed_ids) == 1305)\n\n        factors = pd.read_csv(self.bases_run / "tables/factores_desarrollo_conteos.csv")\n        self.comp = {}\n        for (line, peril), g in factors.groupby(["line", "peril"]):\n            cdf = np.ones(48)\n            for row in g.itertuples():\n                cdf[int(row.lag_meses)] = float(row.factor_a_ultimo)\n            c = np.ones(len(MONTHS))\n            for m in range(CUTOFF_MONTH_INDEX + 1):\n                c[m] = 1.0 / cdf[CUTOFF_MONTH_INDEX - m]\n            self.comp[(line, peril)] = c\n        ibnr_year = pd.read_csv(self.bases_run / "tables/ibnr_por_peligro_anio.csv")\n        self.weights = {}\n        for line in LINES:\n            sub = ibnr_year[ibnr_year.line == line].groupby("peril").agg(T0=("T0_CU", "sum"), ibnr=("ibnr_coste_CU", "sum"))\n            tot = sub.T0 + sub.ibnr\n            self.weights[line] = (tot / tot.sum()).to_dict()\n\n        self.customers = read_table(ROOT / self.catalog["customers"]["workspace_relative_path"])\n        self.addresses = read_table(ROOT / self.catalog["addresses"]["workspace_relative_path"])\n        weather = read_table(ROOT / self.catalog["weather"]["workspace_relative_path"])\n        weather = weather.rename(columns={v: f"{v}_prev1" for v in WEATHER_VARS})\n        weather["year"] = weather["year"].astype(int) + 1  # Weather year Y supplies features for policy periods in Y+1.\n        self.weather = weather\n        claims = read_table(ROOT / self.catalog["claims"]["workspace_relative_path"], columns=["claim_id", "policy_id", "line", "peril", "report_date", "status", "incurred_to_date", "recoveries"])\n        claims["report_date"] = pd.to_datetime(claims["report_date"])\n        claims["net"] = claims["incurred_to_date"] - claims["recoveries"]\n        self.claims = claims\n        assignment = read_table(self.bases_run / "tables/asignacion_por_siniestro.parquet", columns=["claim_id", "row_id", "category"])\n        cr = claims.merge(assignment[assignment.category == "unica"][["claim_id", "row_id"]], on="claim_id", how="inner")\n        cr = cr[cr.status.isin(["closed", "open"])].copy()\n        cr["positive"] = (cr.net > 0).astype(int)\n        self.claim_rows = cr\n        self.half_cache: dict[str, int] = {}\n\n    def check(self, code, name, ok, detail=None):\n        self.checks.append({"code": code, "name": name, "passed": bool(ok), "detail": detail})\n\n\ndef process_line(line: str, ctx: Context, run_dir: Path) -> dict:\n    tables_dir = run_dir / "tables"\n    (tables_dir / "partes").mkdir(parents=True, exist_ok=True)\n    checks = []\n\n    def check(code, name, ok, detail=None):\n        checks.append({"code": code, "name": name, "passed": bool(ok), "detail": detail})\n\n    # F1: policy notifications known before 1 January of each period year.\n    cl = ctx.claims[ctx.claims.line == line][["policy_id", "report_date"]].copy()\n    cl["report_year"] = cl.report_date.dt.year.astype(int)\n    hist_parts = []\n    for y in range(2021, 2026):\n        prev = cl[cl.report_year < y]\n        h = prev.groupby("policy_id").agg(f1_notificaciones_previas=("report_date", "size"), f1_ultima_notificacion=("report_date", "max")).reset_index()\n        h["year"] = y\n        hist_parts.append(h)\n    hist = pd.concat(hist_parts, ignore_index=True)\n\n    # Line-specific labels.\n    t0 = read_table(ctx.bases_run / "tables/t0_por_row_id.parquet", filters=[("line", "==", line)]).drop(columns=["line", "year", "exposure"])\n    per = ctx.claim_rows[ctx.claim_rows.line == line].groupby(["row_id", "peril"]).agg(y=("net", "sum"), n=("positive", "sum"))\n    wide = per.unstack("peril")\n    wide.columns = [f"{a}_{b}" for a, b in wide.columns]\n    for p in PERILS[line]:\n        for pre in ("y", "n"):\n            if f"{pre}_{p}" not in wide.columns:\n                wide[f"{pre}_{p}"] = 0\n    wide = wide.reset_index()\n    labels = t0.merge(wide, on="row_id", how="left")\n    amount_cols = AMOUNT_LABELS + [f"y_{p}" for p in PERILS[line]]\n    count_cols = COUNT_LABELS + [f"n_{p}" for p in PERILS[line]]\n    for c in amount_cols:\n        labels[c] = labels[c].fillna(0.0).astype("float64")\n    for c in count_cols:\n        labels[c] = labels[c].fillna(0).astype("int64").astype("Int64")\n    del t0, per, wide\n\n    acc = {"rows": {"train": 0, "test": 0}, "nulls": {}, "n_total": 0, "cap_ok": True, "cap": {"adjusted": 0, "confirmed": 0, "cap01": 0, "adjusted_test": 0, "confirmed_test": 0},\n           "tw_ok": True, "tw_values": set(), "leak": 0, "f1_rows": 0, "f1_test_rows": 0, "ef_ok": True, "test_ef_ok": True, "ef_sum": {}, "exp_sum": {},\n           "label_ok": True, "T0_train": 0.0, "test_label_null_ok": True, "weather_rows": 0, "chunk_unique_ok": True}\n    out_path = tables_dir / f"f0_{line}.parquet"\n    writer = None\n    schema = None\n    columns = None\n    part_pieces = []\n    for kind, table in (("train", f"policies_{line}"), ("test", f"test_{line}")):\n        tab = pq.read_table(ROOT / ctx.catalog[table]["workspace_relative_path"])\n        years_arr = pd.to_datetime(tab.column("period_start").to_pandas()).dt.year.to_numpy()\n        for y in sorted(set(years_arr.tolist())):\n            df = tab.filter(pa.array(years_arr == y)).to_pandas(types_mapper=STRING_MAPPER)\n            is_test = kind == "test"\n            df["is_test"] = is_test\n            if "renewal_decision" not in df.columns:\n                df["renewal_decision"] = pd.array([None] * len(df), dtype="string[pyarrow]")\n            df["period_start"] = pd.to_datetime(df["period_start"])\n            df["period_end"] = pd.to_datetime(df["period_end"])\n            df["year"] = df.period_start.dt.year.astype("int64")\n            df["mes_inicio"] = df.period_start.dt.month.astype("int64")\n            df["periodo_corto"] = df.exposure < 0.5\n            df = df.merge(ctx.customers, on="customer_id", how="left")\n            df = df.merge(ctx.addresses, on="address_id", how="left")\n            df["anios_como_cliente"] = (df.year - df.customer_since_year).astype("float64")\n            df = df.merge(ctx.weather[ctx.weather.year == y], on=["address_id", "year"], how="left")\n            df = df.drop(columns=["date_of_birth", "customer_since_year"])\n            acc["weather_rows"] += int(df[f"{WEATHER_VARS[0]}_prev1"].notna().sum())\n\n            if line in THRESHOLD:\n                thr = THRESHOLD[line]\n                df["sum_insured_raw"] = df["sum_insured"]\n                adj = df["sum_insured_raw"] > thr\n                df["audit_capital_scale_adjusted"] = adj\n                df["sum_insured"] = np.where(adj, df["sum_insured_raw"] / 1000.0, df["sum_insured_raw"])\n                ids = df["row_id"].astype(str)\n                df["audit_capital_scale_confirmed"] = adj & ids.isin(ctx.confirmed_ids)\n                df["audit_capital_cap01"] = ids.isin(ctx.cap01_ids)\n                df["log_sum_insured"] = np.log(df["sum_insured"])\n                idem = np.where(df["sum_insured"] > thr, df["sum_insured"] / 1000.0, df["sum_insured"])\n                acc["cap_ok"] &= bool(np.allclose(idem, df["sum_insured"])) and bool((df["sum_insured"] <= thr).all()) and bool((df.audit_capital_scale_confirmed <= adj).all())\n                acc["cap"]["adjusted"] += int(adj.sum())\n                acc["cap"]["confirmed"] += int(df.audit_capital_scale_confirmed.sum())\n                acc["cap"]["cap01"] += int(df.audit_capital_cap01.sum())\n                if is_test:\n                    acc["cap"]["adjusted_test"] += int(adj.sum())\n                    acc["cap"]["confirmed_test"] += int(df.audit_capital_scale_confirmed.sum())\n            if line == "buildings":\n                df["construction_type_harmonized"] = df["construction_type"].replace({"t": "tw_code", "w": "tw_code"})\n                acc["tw_ok"] &= not bool(df["construction_type_harmonized"].isin(["t", "w"]).any())\n                acc["tw_values"] |= set(df["construction_type_harmonized"].dropna().astype(str).unique())\n\n            df = df.merge(hist[hist.year == y], on=["policy_id", "year"], how="left")\n            df["f1_notificaciones_previas"] = df["f1_notificaciones_previas"].fillna(0).astype("int64")\n            df["f1_dias_desde_ultima_notificacion"] = (df.period_start - df.f1_ultima_notificacion).dt.days.astype("float64")\n            acc["leak"] += int((df.f1_ultima_notificacion.notna() & (df.f1_ultima_notificacion >= pd.Timestamp(f"{y}-01-01"))).sum())\n            acc["f1_rows"] += int((df.f1_notificaciones_previas > 0).sum())\n            if is_test:\n                acc["f1_test_rows"] += int((df.f1_notificaciones_previas > 0).sum())\n            df = df.drop(columns=["f1_ultima_notificacion"])\n\n            ps = df["period_start"].values.astype("datetime64[D]")\n            pe = df["period_end"].values.astype("datetime64[D]")\n            # Apply day-weighted completeness to EXP01 exposure, preserving its rounding.\n            e_ef = {p: np.zeros(len(df)) for p in PERILS[line]}\n            e_cost = np.zeros(len(df))\n            covered = np.zeros(len(df))\n            for j, m in enumerate(MONTHS):\n                ms = np.datetime64(m.start_time.date(), "D")\n                me = np.datetime64(m.end_time.date(), "D")\n                if ms > pe.max() or me < ps.min():\n                    continue\n                overlap = np.clip((np.minimum(pe, me) - np.maximum(ps, ms)).astype("timedelta64[D]").astype(int) + 1, 0, None).astype(float)\n                covered += overlap\n                for p in PERILS[line]:\n                    c = ctx.comp[(line, p)][j]\n                    e_ef[p] += overlap * c\n                    e_cost += overlap * c * ctx.weights[line][p]\n            expo = df["exposure"].to_numpy(dtype=float)\n            for p in PERILS[line]:\n                df[f"e_ef_{p}"] = expo * e_ef[p] / covered\n            df["e_ef_coste"] = expo * e_cost / covered\n            ef_cols = [f"e_ef_{p}" for p in PERILS[line]] + ["e_ef_coste"]\n            acc["ef_ok"] &= all(bool(((df[c] <= df.exposure * (1 + 1e-12)) & (df[c] > 0)).all()) for c in ef_cols) and bool((covered > 0).all())\n            if is_test:\n                acc["test_ef_ok"] &= all(bool(np.allclose(df[c], df["exposure"], rtol=1e-12, atol=0)) for c in ef_cols)\n            acc["ef_sum"][y] = acc["ef_sum"].get(y, 0.0) + float(df.e_ef_coste.sum())\n            acc["exp_sum"][y] = acc["exp_sum"].get(y, 0.0) + float(df.exposure.sum())\n\n            if is_test:\n                for c in amount_cols:\n                    df[c] = pd.array([np.nan] * len(df), dtype="float64")\n                for c in count_cols:\n                    df[c] = pd.array([pd.NA] * len(df), dtype="Int64")\n                acc["test_label_null_ok"] &= bool(df["T0"].isna().all())\n            else:\n                df = df.merge(labels, on="row_id", how="left")\n                for c in amount_cols:\n                    df[c] = df[c].fillna(0.0).astype("float64")\n                for c in count_cols:\n                    df[c] = df[c].fillna(0).astype("Int64")\n                ysum = sum(df[f"y_{p}"] for p in PERILS[line])\n                acc["label_ok"] &= bool(np.allclose(ysum, df["T0"], atol=1e-6))\n                acc["T0_train"] += float(df["T0"].sum())\n\n            acc["chunk_unique_ok"] &= bool(df.row_id.is_unique)\n            acc["rows"][kind] += len(df)\n            acc["n_total"] += len(df)\n            for c in df.columns:\n                acc["nulls"][c] = acc["nulls"].get(c, 0) + int(df[c].isna().sum())\n\n            uniq = df.customer_id.astype(str).unique()\n            for cid in uniq:\n                if cid not in ctx.half_cache:\n                    ctx.half_cache[cid] = cal_half(cid)\n            piece = pd.DataFrame({"row_id": df.row_id.astype(str).values, "line": line, "year": df.year.values, "is_test": is_test,\n                                  "cal_half": df.customer_id.astype(str).map(ctx.half_cache).astype("int8").values})\n            part_pieces.append(piece)\n\n            if columns is None:\n                columns = list(df.columns)\n            df = df[columns]\n            table_out = pa.Table.from_pandas(df, preserve_index=False)\n            if writer is None:\n                schema = table_out.schema\n                writer = pq.ParquetWriter(out_path, schema)\n            else:\n                table_out = table_out.cast(schema)\n            writer.write_table(table_out)\n            del df, table_out, piece, e_ef, e_cost, ps, pe, covered, expo\n            gc.collect()\n        del tab, years_arr\n        gc.collect()\n    writer.close()\n    pd.concat(part_pieces, ignore_index=True).to_parquet(tables_dir / "partes" / f"particiones_{line}.parquet", index=False)\n\n    if line in THRESHOLD:\n        check(f"K04_{line}", f"{line}: contrato de capital idempotente, ningún capital del modelo supera el umbral y confirmadas contenidas en ajustadas", acc["cap_ok"], acc["cap"])\n    if line == "buildings":\n        check("K05", "buildings: construction_type_harmonized sin t ni w y con cuatro categorías", acc["tw_ok"] and len(acc["tw_values"]) == 4, sorted(acc["tw_values"]))\n    check(f"K06_{line}", f"{line}: ninguna notificación de F1 posterior o igual al 1 de enero del año del periodo", acc["leak"] == 0,\n          {"filas_con_historia": acc["f1_rows"], "test_con_historia": acc["f1_test_rows"]})\n    ratio = {int(y): round(acc["ef_sum"][y] / acc["exp_sum"][y], 4) for y in sorted(acc["ef_sum"])}\n    check(f"K07_{line}", f"{line}: exposición efectiva positiva, nunca mayor que la nominal e igual a la nominal en 2025", acc["ef_ok"] and acc["test_ef_ok"], ratio)\n    t0_expected = ctx.bases_evidence["t0"]["by_line_CU"][line]\n    check(f"K08_{line}", f"{line}: etiquetas por peligro suman T0 en cada fila, T0 total igual a D67 y etiquetas nulas en test",\n          acc["label_ok"] and abs(acc["T0_train"] - t0_expected) < 0.01 and acc["test_label_null_ok"], {"T0_CU": round(acc["T0_train"], 2), "esperado": t0_expected})\n    contract = {}\n    for field in schema:\n        block, reason = classify(field.name, line)\n        contract[field.name] = {"bloque": block, "motivo": reason, "tipo": str(field.type), "nulos_pct": round(acc["nulls"].get(field.name, 0) / acc["n_total"] * 100, 3)}\n    blocks = {b: sum(1 for v in contract.values() if v["bloque"] == b) for b in BLOCKS}\n    check(f"K09_{line}", f"{line}: todas las columnas con contrato y las de exclusión obligatoria marcadas como excluidas",\n          not any(v["bloque"] == "revisar" for v in contract.values()) and all(contract[c]["bloque"] == "excluida" for c in EXCLUDED_ALWAYS if c in contract),\n          {"bloques": blocks, "revisar": [k for k, v in contract.items() if v["bloque"] == "revisar"]})\n    src_train = pq.read_metadata(ROOT / ctx.catalog[f"policies_{line}"]["workspace_relative_path"]).num_rows\n    src_test = pq.read_metadata(ROOT / ctx.catalog[f"test_{line}"]["workspace_relative_path"]).num_rows\n    check(f"K10_{line}", f"{line}: filas de entrenamiento y test iguales a la capa vigente y row_id único por año",\n          acc["rows"]["train"] == src_train and acc["rows"]["test"] == src_test and acc["chunk_unique_ok"], acc["rows"])\n    evidence = {"rows_train": acc["rows"]["train"], "rows_test": acc["rows"]["test"], "columns": len(schema), "blocks": blocks,\n                "weather_prev1_rows": acc["weather_rows"], "f1_rows_with_history": acc["f1_rows"], "f1_test_rows_with_history": acc["f1_test_rows"],\n                "e_ef_coste_over_exposure_by_year": ratio, "capital": acc["cap"] if line in THRESHOLD else None,\n                "peril_cost_weights": {k: round(v, 4) for k, v in ctx.weights[line].items()}, "T0_train_CU": round(acc["T0_train"], 2),\n                "file": f"tables/f0_{line}.parquet", "row_groups_by_year": True}\n    partial = {"line": line, "common_checks": ctx.checks, "checks": checks, "evidence": evidence, "contract": contract}\n    (run_dir / f"parcial_{line}.json").write_text(json.dumps(partial, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n    print(f"{line}: {sum(c[\'passed\'] for c in checks)}/{len(checks)} controles del ramo; {acc[\'rows\']}")\n    return partial\n\n\ndef finalize(run_dir: Path, ctx_meta: dict) -> dict:\n    tables_dir = run_dir / "tables"\n    partials = {}\n    for line in LINES:\n        path = run_dir / f"parcial_{line}.json"\n        if not path.is_file():\n            raise FileNotFoundError(f"Falta {path.name}; ejecute --ramo {line} antes de --finalizar")\n        partials[line] = json.loads(path.read_text(encoding="utf-8"))\n    common = partials["motor"]["common_checks"]\n    checks = list(common)\n    same_common = all([(c["code"], c["passed"]) for c in partials[l]["common_checks"]] == [(c["code"], c["passed"]) for c in common] for l in LINES)\n    for line in LINES:\n        checks.extend(partials[line]["checks"])\n    part = pd.concat([pd.read_parquet(tables_dir / "partes" / f"particiones_{l}.parquet") for l in LINES], ignore_index=True)\n    part["V22"] = np.select([part.year == 2021, part.year == 2022], ["train", "eval"], default="none")\n    part["V23"] = np.select([part.year <= 2022, (part.year == 2023) & (part.cal_half == 0), (part.year == 2023) & (part.cal_half == 1)], ["train", "eval_calibracion", "eval_seleccion"], default="none")\n    part["V24"] = np.select([part.year <= 2023, part.year == 2024], ["train", "eval"], default="none")\n    part["PROD"] = np.select([part.year <= 2024, part.year == 2025], ["train", "predict"], default="none")\n    part.to_parquet(tables_dir / "particiones.parquet", index=False)\n    half_share = float(part.loc[part.year == 2023, "cal_half"].mean())\n\n    def check(code, name, ok, detail=None):\n        checks.append({"code": code, "name": name, "passed": bool(ok), "detail": detail})\n\n    check("K11", "las tres partes usaron la misma capa y los mismos controles comunes", same_common)\n    check("K12", "particiones: V24 entrena 2021-2023 y evalúa 2024; PROD predice las 626.639 filas de 2025; row_id único en 3.035.950 filas",\n          int((part.V24 == "train").sum()) == int((part.year <= 2023).sum()) and int((part.V24 == "eval").sum()) == int((part.year == 2024).sum())\n          and int((part.PROD == "predict").sum()) == TEST_ROWS and int((part.PROD == "train").sum()) == TRAIN_ROWS and part.row_id.is_unique,\n          {"filas": int(len(part)), "predict": int((part.PROD == "predict").sum()), "train": int((part.PROD == "train").sum())})\n    check("K13", "mitades de calibración y selección de V23 entre 45 % y 55 % de las filas de 2023", 0.45 <= half_share <= 0.55, round(half_share, 4))\n    known = "05180324"  # Master-table customer; recompute the split with the published hash rule.\n    check("K14", "regla de hash reproducible: SHA-256(\'cal_v1|customer_id\'), primeros 16 hex, módulo 2", cal_half(known) in (0, 1),\n          {"customer_id": known, "half": cal_half(known), "sha256_prefix": hashlib.sha256(f"{CAL_SALT}|{known}".encode()).hexdigest()[:16]})\n\n    contract = {line: partials[line]["contract"] for line in LINES}\n    (run_dir / "contrato_disponibilidad.json").write_text(json.dumps({\n        "version": "F0 v1 (D68)",\n        "bloques": {"F0": "contractual y físico, disponible al inicio del periodo bajo estabilidad de maestros; matriz base",\n                    "F1": "historia de notificaciones de la póliza al 1 de enero del año del periodo; bloque adicional",\n                    "F2": "clima del año anterior; bloque adicional", "F3": "información condicionada sin fecha de referencia; prueba separada",\n                    "F4": "tarifario existente; ablación aparte", "excluida": "nunca predictor", "clave": "identificadores y fechas",\n                    "etiqueta": "T0 y componentes; solo entrenamiento, nulo en test"},\n        "capital": "sum_insured = sum_insured_raw/1000 si supera el umbral del ramo (10.000.000 CU edificios, 1.000.000 contenidos); marcas audit_* fuera de los predictores (D66)",\n        "exposicion_efectiva": "exposure x completitud media del periodo ponderada por días; completitud del peligro para el mes de ocurrencia al 30-12-2024 (chain-ladder D67); 1 en 2025",\n        "lectura": "las columnas de predictores se toman por bloque desde este contrato; el selector debe rechazar \'excluida\', \'clave\' y \'etiqueta\' además del guard 1.0",\n        "columnas": contract}, ensure_ascii=False, indent=1), encoding="utf-8")\n    evidence = {\n        "run_id": run_dir.name, "created_utc": ctx_meta["started"],\n        "author": "AI-assisted calculation requested by the participant; no human or independent professional review",\n        "scope": {"data_modified": False, "indices_modified": False, "models_trained": False, "remote_execution": False, "external_providers": False,\n                  "description": "Matriz F0 por ramo con contrato de disponibilidad, etiquetas T0 y por peligro, exposición efectiva y particiones V22/V23/V24/PROD"},\n        "provenance": ctx_meta["provenance"],\n        "lines": {line: partials[line]["evidence"] for line in LINES},\n        "partitions": {"rows": int(len(part)), "by_year": {int(y): int(n) for y, n in part.year.value_counts().sort_index().items()}, "cal_half_share_2023": round(half_share, 4),\n                       "rule": "SHA-256 del texto UTF-8 \'cal_v1|customer_id\', primeros 16 hex como entero, módulo 2: 0 calibración, 1 selección",\n                       **{v: {k: int(n) for k, n in part[v].value_counts().items()} for v in ["V22", "V23", "V24", "PROD"]}},\n        "checks": checks,\n        "checks_summary": {"checks_total": len(checks), "checks_passed": sum(c["passed"] for c in checks), "all_passed": all(c["passed"] for c in checks)},\n    }\n    (run_dir / "evidencia.json").write_text(json.dumps(evidence, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n    pd.DataFrame(checks).to_csv(tables_dir / "controles.csv", index=False)\n    shutil.copy2(Path(__file__), run_dir / Path(__file__).name)\n    manifest = {str(p.relative_to(run_dir)).replace("\\\\", "/"): digest(p) for p in sorted(run_dir.rglob("*")) if p.is_file() and p.name != "manifest.json"}\n    (run_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")\n    s = evidence["checks_summary"]\n    print(f"Corrida {run_dir.name}: {s[\'checks_passed\']}/{s[\'checks_total\']} controles")\n    for c in checks:\n        print(("OK  " if c["passed"] else "FALLA ") + c["code"] + " " + c["name"] + ("" if c["passed"] else f" -> {c[\'detail\']}"))\n    return evidence\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    parser.add_argument("--run", help="id de una corrida existente en runs/")\n    parser.add_argument("--ramo", help="ramos a procesar, separados por comas (motor, buildings, contents)")\n    parser.add_argument("--finalizar", action="store_true", help="reúne los parciales, construye particiones, evidencia y manifiesto")\n    args = parser.parse_args()\n    if args.run:\n        run_dir = HERE / "runs" / args.run\n        if not run_dir.is_dir():\n            raise FileNotFoundError(run_dir)\n    else:\n        started = datetime.now(timezone.utc)\n        run_dir = HERE / "runs" / f"f0_{started.strftime(\'%Y%m%dT%H%M%SZ\')}_{uuid.uuid4().hex[:8]}"\n        (run_dir / "tables").mkdir(parents=True, exist_ok=False)\n        (run_dir / "inicio.json").write_text(json.dumps({"started": started.isoformat(timespec="seconds")}), encoding="utf-8")\n        print(f"Corrida creada: {run_dir.name}")\n    started_iso = json.loads((run_dir / "inicio.json").read_text(encoding="utf-8"))["started"]\n    lines = LINES if not (args.ramo or args.finalizar) else ([l.strip() for l in args.ramo.split(",")] if args.ramo else [])\n    ctx = None\n    if lines or not args.finalizar:\n        ctx = Context()\n        for line in lines:\n            process_line(line, ctx, run_dir)\n            gc.collect()\n    if args.finalizar or not (args.ramo or args.run):\n        if ctx is None:\n            ctx = Context()\n        meta = {"started": started_iso, "provenance": {\n            "active_index": "Hackaton/Semana 1/Limpieza/current_dataset.json", "active_run": ctx.current["workspace_relative_run"], "catalog_sha256": digest(ctx.catalog_path),\n            "tables_sha256": ctx.hashes, "bases_run": ctx.bases_index["workspace_relative_run"], "branch_run": ctx.branch_index["workspace_relative_run"],\n            "cap01_run": ctx.cap01_index["workspace_relative_run"], "d66_run": ctx.d66_index["workspace_relative_run"]}}\n        evidence = finalize(run_dir, meta)\n        if not evidence["checks_summary"]["all_passed"]:\n            sys.exit(1)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `src/datos/n62/rebuild_base.py`

This is stored in the notebook and materialized inside the new run.


In [ ]:
SOURCES['src/datos/n62/rebuild_base.py'] = '"""Build the frozen eight-rule preaudit base using only eleven raw Parquet inputs.\n\nNo legacy run, target catalog, cohort list, model, network or project import is used.\nApproved cohort fingerprints are guards, not row selections.\n"""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport argparse,hashlib,json,sys,os,shutil,gc\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.compute as pc\nimport pyarrow.parquet as pq\n\nLINES=[\'motor\',\'buildings\',\'contents\']\nPOLICIES=[p+\'_\'+l for p in [\'policies\',\'test\'] for l in LINES]\ndef utc():return datetime.now(timezone.utc).isoformat(timespec=\'seconds\')\ndef sha(p):\n h=hashlib.sha256()\n with Path(p).open(\'rb\') as f:\n  for b in iter(lambda:f.read(1<<20),b\'\'):h.update(b)\n return h.hexdigest()\ndef fp(v):return hashlib.sha256(json.dumps(sorted(v),ensure_ascii=True,separators=(\',\',\':\')).encode()).hexdigest()\ndef dump(p,v):Path(p).write_text(json.dumps(v,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf-8\')\ndef replace(t,name,v):\n field=t.schema.field(name)\n return t.set_column(t.schema.get_field_index(name),field,pa.array(v,type=field.type))\n\ndef main():\n ap=argparse.ArgumentParser()\n ap.add_argument(\'--raw\',required=True,type=Path);ap.add_argument(\'--rules\',required=True,type=Path);ap.add_argument(\'--out\',required=True,type=Path)\n a=ap.parse_args();raw=a.raw.resolve();rulesfile=a.rules.resolve();out=a.out.resolve()\n assert raw.is_dir() and not out.exists() and raw!=out and not raw.is_relative_to(out) and not out.is_relative_to(raw)\n out.mkdir(parents=True,exist_ok=False);(out/\'data\').mkdir();(out/\'evidence\').mkdir()\n reads=[]\n def audit(event,args):\n  if event!=\'open\' or not isinstance(args[0],(str,bytes,os.PathLike)):return\n  p=Path(os.fsdecode(args[0])).resolve()\n  if p.suffix.lower() not in [\'.json\',\'.parquet\',\'.csv\']:return\n  mode=args[1];flags=args[2] if len(args)>2 else 0\n  writing=(isinstance(mode,str) and any(x in mode for x in \'wax+\')) or (isinstance(flags,int) and bool(flags&(os.O_WRONLY|os.O_RDWR|os.O_CREAT)))\n  if not writing:\n   if not (p.parent==raw or p==rulesfile or p.is_relative_to(out)):raise RuntimeError(\'Unexpected data read: \'+str(p))\n   reads.append(str(p))\n sys.addaudithook(audit)\n rules=json.loads(rulesfile.read_text(encoding=\'utf-8\'))\n expected=rules[\'originals_sha256\']\n assert len(expected)==11 and set(expected)=={p.name for p in raw.glob(\'*.parquet\')}\n before={name:sha(raw/name) for name in sorted(expected)};assert before==expected,\'Raw input version changed\'\n dump(out/\'RUNNING.json\',{\'utc\':utc(),\'pid\':os.getpid(),\'source_sha256\':sha(Path(__file__)),\'rules_sha256\':sha(rulesfile)})\n def load(name,columns=None):\n  p=raw/(name+\'.parquet\');assert p.name in expected\n  with p.open(\'rb\') as f:return pq.read_table(f,columns=columns)\n def frame(name,columns=None):return load(name,columns).to_pandas()\n def check_set(v,c,key):\n  values=list(v);assert len(values)==len(set(values))==c[\'count\'] and fp(values)==c[key],\'Approved cohort fingerprint mismatch\'\n stats={};caps={};ren={};first={}\n # CAP01 selection from original, uncorrected household data; no cascading.\n for line in [\'buildings\',\'contents\']:\n  fields=rules[\'CAP01\'][\'context_fields\'][line]\n  cols=[\'row_id\',\'policy_id\',\'period_start\',\'period_end\',\'sum_insured\']+fields\n  parts=[]\n  for p in [\'policies\',\'test\']:\n   f=frame(p+\'_\'+line,cols);f[\'source_table\']=p+\'_\'+line;parts.append(f)\n  f=pd.concat(parts,ignore_index=True).sort_values([\'policy_id\',\'period_start\',\'period_end\',\'row_id\']).reset_index(drop=True)\n  f[\'period_start\']=pd.to_datetime(f.period_start);f[\'period_end\']=pd.to_datetime(f.period_end)\n  extreme=f.sum_insured.gt(rules[\'CAP01\'][\'thresholds\'][line])\n  positions=pd.Series(np.where(~extreme,np.arange(len(f)),np.nan),index=f.index)\n  previous=positions.groupby(f.policy_id).ffill().groupby(f.policy_id).shift()\n  exists=previous.notna();anchor=f.reindex(previous.fillna(-1).astype(int)).reset_index(drop=True)\n  known=pd.Series(True,index=f.index);same=pd.Series(True,index=f.index)\n  for name in fields:\n   x,y=f[name],anchor[name]\n   if name==\'construction_type\':x=x.astype(\'string\').str.casefold().str.strip();y=y.astype(\'string\').str.casefold().str.strip()\n   known &= x.notna()&y.notna()&exists;same &= x.eq(y).fillna(False)\n  selected=extreme&exists&known&same&(anchor.period_end<f.period_start)&(f.period_start-anchor.period_start).dt.days.between(1,400)&((f.sum_insured/1000)/anchor.sum_insured).between(.9,1.1)\n  for name,g in f.loc[selected].groupby(\'source_table\'):\n   ids=g.row_id.tolist();check_set(ids,rules[\'CAP01\'][\'approved_sets\'][name],\'row_ids_sha256\');caps[name]=set(ids)\n  print(json.dumps({\'rule\':\'CAP01\',\'line\':line,\'selected\':int(selected.sum()),\'cohort_guard\':\'exact\'}),flush=True)\n  del parts,f,anchor,positions,previous;gc.collect()\n assert set(caps)==set(rules[\'CAP01\'][\'approved_sets\'])\n dump(out/\'evidence/cap01_row_ids.json\',{key:sorted(value) for key,value in caps.items()})\n # REN00 membership and earliest train start, separately by line to bound RAM.\n for line in LINES:\n  tr=frame(\'policies_\'+line,[\'row_id\',\'policy_id\',\'period_start\',\'period_end\',\'renewal_decision\'])\n  first[line]=pd.to_datetime(tr.period_start).groupby(tr.policy_id).min()\n  te=frame(\'test_\'+line,[\'row_id\',\'policy_id\',\'period_start\',\'period_end\']);te[\'renewal_decision\']=pd.NA\n  f=pd.concat([tr,te],ignore_index=True);f[\'period_start\']=pd.to_datetime(f.period_start);f[\'period_end\']=pd.to_datetime(f.period_end)\n  f=f.sort_values([\'policy_id\',\'period_start\',\'period_end\']).reset_index(drop=True)\n  year=f.period_start.dt.year;ny=year.groupby(f.policy_id).shift(-1)\n  ids=sorted(f.loc[f.renewal_decision.eq(\'declined_by_insurer\')&year.eq(2024)&ny.eq(2025),\'policy_id\'].unique().tolist())\n  check_set(ids,rules[\'REN00\'][\'approved_sets\'][line],\'policy_ids_sha256\');ren[line]=set(ids)\n  print(json.dumps({\'rule\':\'REN00\',\'line\':line,\'policies\':len(ids),\'cohort_guard\':\'exact\'}),flush=True)\n  del tr,te,f;gc.collect()\n customers=frame(\'customers\',[\'customer_id\',\'date_of_birth\'])\n master=customers.customer_id.astype(\'string\')\n assert master.is_unique and master.str.fullmatch(\'[0-9]{8}\').all()\n births=customers.set_index(\'customer_id\').date_of_birth\n renfield=pa.field(\'audit_ren00\',pa.bool_(),nullable=False,metadata={\n  b\'role\':b\'audit_only\',b\'rule\':b\'REN00\',b\'excluded_from_predictors\':b\'true\',\n  b\'cohort_sha256\':rules[\'REN00\'][\'cohort_source_file_sha256\'].encode(),\n  b\'meaning\':b\'Policy membership in frozen 2024-declined/2025-successor cohort; not a row-level decision.\'})\n for name in POLICIES:\n  t=load(name);line=name.split(\'_\',1)[1]\n  f=t.select([\'row_id\',\'customer_id\',\'period_start\',\'period_end\',\'exposure\',\'policy_id\']).to_pandas()\n  ids=f.customer_id.astype(\'string\')\n  assert ids.str.fullmatch(\'[0-9]{1,8}\').fillna(False).all()\n  ids=ids.str.zfill(8);assert ids.isin(master).all()\n  start=pd.to_datetime(f.period_start);end=pd.to_datetime(f.period_end)\n  assert start.notna().all() and end.notna().all() and end.ge(start).all()\n  exp=((end-start).dt.days.add(1)/rules[\'EXP01\'][\'denominator\']).round(rules[\'EXP01\'][\'decimals\'])\n  q={\'rows\':t.num_rows,\'ID01\':int(ids.ne(f.customer_id).sum()),\'EXP01\':int(exp.ne(f.exposure).sum()),\'CAP01\':0,\'CAT01\':0,\'AGE01\':0}\n  t=replace(t,\'customer_id\',ids);t=replace(t,\'exposure\',exp)\n  if name in caps:\n   values=t.column(\'sum_insured\').to_pandas().to_numpy(copy=True);mask=f.row_id.isin(caps[name]).to_numpy()\n   q[\'CAP01\']=int(mask.sum());values[mask]=values[mask]/rules[\'CAP01\'][\'scale\'];t=replace(t,\'sum_insured\',values)\n  if line!=\'motor\':\n   b=pd.to_datetime(ids.map(births));assert b.notna().all() and start.ge(b).all()\n   ambiguous=b.dt.month.eq(2)&b.dt.day.eq(29)&~start.dt.is_leap_year&start.dt.month.eq(2)&start.dt.day.eq(28)\n   assert not ambiguous.any(),\'Leap birthday convention requires review\'\n   age=start.dt.year-b.dt.year-((start.dt.month<b.dt.month)|((start.dt.month==b.dt.month)&(start.dt.day<b.dt.day))).astype(\'int64\')\n   prior=t.column(\'policyholder_age\').to_pandas();assert (age-prior).abs().le(1).all()\n   q[\'AGE01\']=int(age.ne(prior).sum());t=replace(t,\'policyholder_age\',age)\n   if line==\'buildings\':\n    prior=t.column(\'construction_type\').to_pandas();cat=prior.replace(rules[\'CAT01\'][\'map\'])\n    q[\'CAT01\']=int(cat.ne(prior).sum());t=replace(t,\'construction_type\',cat)\n  mark=pc.is_in(t[\'policy_id\'],value_set=pa.array(sorted(ren[line]),type=t.schema.field(\'policy_id\').type))\n  q[\'REN00\']=int(pc.sum(mark).as_py());t=t.append_column(renfield,mark)\n  with (out/\'data\'/f\'{name}.parquet\').open(\'wb\') as dest:pq.write_table(t,dest,compression=\'zstd\')\n  stats[name]=q;print(json.dumps({\'table\':name,**q}),flush=True);del t,f;gc.collect()\n # DUP01: complete-payload identical pairs; retain short reviewed representative.\n original=load(\'claims\');c=original.to_pandas();assert c.claim_id.is_unique\n payload=[x for x in c.columns if x!=\'claim_id\']\n duplicates=c.loc[c.duplicated(payload,keep=False)]\n pairs=[];excluded=[]\n for values in duplicates.groupby(payload,dropna=False,sort=False,observed=True).claim_id.agg(list):\n  assert len(values)==2\n  short=[x for x in values if pd.Series([x]).str.fullmatch(\'CL[0-9]{8}\').iloc[0]]\n  long=[x for x in values if pd.Series([x]).str.fullmatch(\'CL[0-9]{9}\').iloc[0]]\n  assert len(short)==len(long)==1;pairs.append((short[0],long[0]));excluded.append(long[0])\n assert len(pairs)==rules[\'DUP01\'][\'count\'] and fp(pairs)==rules[\'DUP01\'][\'pairs_sha256\']\n assert len(excluded)==len(set(excluded))\n keep=pc.invert(pc.is_in(original[\'claim_id\'],value_set=pa.array(excluded,type=original.schema.field(\'claim_id\').type)))\n t=original.filter(keep);c=t.to_pandas();preids=[]\n for line in LINES:\n  z=c.loc[c.line.eq(line)];starts=z.policy_id.map(first[line]);assert starts.notna().all()\n  preids+=z.loc[pd.to_datetime(z.loss_date).lt(starts),\'claim_id\'].tolist()\n check_set(preids,rules[\'PRE00\'],\'claim_ids_sha256\')\n prefield=pa.field(\'audit_pre00\',pa.bool_(),nullable=False,metadata={\n  b\'role\':b\'audit_only\',b\'rule\':b\'PRE00\',b\'excluded_from_predictors\':b\'true\',\n  b\'excluded_from_current_period_target\':b\'true\',b\'cohort_sha256\':rules[\'PRE00\'][\'cohort_source_file_sha256\'].encode(),\n  b\'meaning\':b\'Frozen pre-first-observed-period claim list; not a coverage or repudiation decision.\'})\n marks=pc.is_in(t[\'claim_id\'],value_set=pa.array(sorted(preids),type=t.schema.field(\'claim_id\').type))\n t=t.append_column(prefield,marks)\n with (out/\'data/claims.parquet\').open(\'wb\') as dest:pq.write_table(t,dest,compression=\'zstd\')\n stats[\'claims\']={\'rows_before\':original.num_rows,\'rows\':t.num_rows,\'DUP01\':len(excluded),\'PRE00\':int(pc.sum(marks).as_py())}\n del original,t,c;gc.collect()\n for name in sorted(expected):\n  stem=Path(name).stem\n  if stem not in POLICIES and stem!=\'claims\':\n   shutil.copyfile(raw/name,out/\'data\'/name);stats[stem]={\'operation\':\'byte-copy unchanged original\'}\n after={name:sha(raw/name) for name in sorted(expected)};assert before==after\n raw_reads=sorted({p for p in reads if Path(p).parent==raw})\n assert {Path(p).name for p in raw_reads}==set(expected)\n outputs={p.name:sha(p) for p in sorted((out/\'data\').glob(\'*.parquet\'))};assert set(outputs)==set(expected)\n result={\'utc\':utc(),\'all_builder_guards_passed\':True,\'tables\':stats,\'input_sha256\':before,\'inputs_unchanged\':True,\n  \'output_sha256\':outputs,\'rules_sha256\':sha(rulesfile),\'source_sha256\':sha(Path(__file__)),\n  \'raw_read_paths\':raw_reads,\'data_read_paths\':sorted(set(reads)),\'reference_data_reads\':0,\n  \'environment\':{\'python\':sys.version,\'pandas\':pd.__version__,\'numpy\':np.__version__,\'pyarrow\':pa.__version__},\n  \'new_models\':0,\'Kaggle_calls\':0}\n dump(out/\'build_receipt.json\',result)\n dump(out/\'COMPLETE.json\',{\'utc\':utc(),\'build_receipt_sha256\':sha(out/\'build_receipt.json\'),\'tables\':11})\n print(\'RAW_ONLY_BASE_BUILD_COMPLETE\',flush=True)\nif __name__==\'__main__\':main()\n'


## Materialize and run the frozen recipe

I reconstruct the cleaned tables, actuarial bases and F0, then train the 30 S30 stages. The full F0 content and feature-block contract are checked against the original fingerprints, including both reconstructed audit flags. Matching the confirmation flag on this dataset does not recover or independently validate the original audit decisions. The runner records intermediate differences and insists on the historical S30 CSV hash. Its return code 3 means that the final CSV matches while some intermediate hashes differ. I investigate those differences separately; a matching final CSV does not establish equality of every intermediate artifact.

If a run is interrupted, the same run folder can reuse completed nodes only after checking their required hashes. In the same session, rerun the execution cell and inspect the log if the runner stops on partial outputs. Starting a fresh session creates a fresh run folder; retaining or transferring any checkpoint is a separate operation subject to the competition's data rules.


In [ ]:
DATAFILES = {'receta/codigo.json': '{\n  "description": "SHA-256 fingerprints of the English delivery runtime. Documentation and runtime-path names differ from historical source; model behavior is checked separately against the frozen execution snapshots.",\n  "files": {\n    "reproducir.py": {\n      "sha256": "4a2c2b8414f64a4975afe76e6a8a993eab966ea691db3fd9579e6246560bea33"\n    },\n    "herramientas/huella.py": {\n      "sha256": "7e38a9803075d5dc5830d801f00bb7f0186a1d29e5dc327a0c40e173315f2ce0"\n    },\n    "herramientas/etapas_datos.py": {\n      "sha256": "2ca891b066b6f639c9cb68ff2ed7e6de9e961e3a60c0419c788233ca10f8e09a"\n    },\n    "herramientas/ejecutar_csv_lf.py": {\n      "sha256": "aabdd55c8c42efb7463442f2996ad8790fa1447150a49556828a544866e6fd6c"\n    },\n    "herramientas/inventario.py": {\n      "sha256": "1061b7f9ef94f7a6bb0d9f4fccb34b8d9e15365e93ab8ea53fdb5fbeecd85bbc"\n    },\n    "src/modelos/anexar_entrega.py": {\n      "sha256": "0294fdb2a38a48f3afc1b5d95fa7c660a1a37a11e9cadf47891e020bc0ee847d"\n    },\n    "src/modelos/bootstrap_reference.py": {\n      "sha256": "6fb203a1c65e6ba231d2206d6fd76bbf9da33ebc3c524e357dfc914f2360f5df"\n    },\n    "src/modelos/calibracion_deciles.py": {\n      "sha256": "a88ea77d7c0509ce30a6ddb7209c661a79e17dd48f604f649c07508cff18e904"\n    },\n    "src/modelos/calibrar_siguiente_anio.py": {\n      "sha256": "8cc30e193d1f2d05a64d830a57f6c11b3a8d104a3771b841586aec66effe9fbe"\n    },\n    "src/modelos/cat_experimento.py": {\n      "sha256": "b887fd4f64636c49afc2d4cbf33853e17878ad33886fd2785cee8263c5dd0bbb"\n    },\n    "src/modelos/cat_frecsev.py": {\n      "sha256": "5e5b8c09cd803639eb5b5284cb2f3383c1fb7d5b012ee9bdbbeabc92b0d5abd5"\n    },\n    "src/modelos/cat_produccion.py": {\n      "sha256": "fa1d4725a298606d9522847540d5336d5e45785be11b3c9e1c087e84bba3ffe4"\n    },\n    "src/modelos/combinar_semillas.py": {\n      "sha256": "539b4b1cb6ef37c24236efa0b973539c5be04ffbb01c560e08f63a899a6bc216"\n    },\n    "src/modelos/compare_reference.py": {\n      "sha256": "bdc279ef294661ba8e8fe3b78f4a2d105859bd7032cf3be151f9bc728af5e785"\n    },\n    "src/modelos/comun.py": {\n      "sha256": "8ef64ab3f283550bc9a259624a4b934f140fb435dda3324eedc250ba84e1e1b8"\n    },\n    "src/modelos/construir_variables.py": {\n      "sha256": "4bed5c645da2bd540ed2738f63ea281dedf1c91c857eefbcaaf3356e2f6e72b8"\n    },\n    "src/modelos/deriva_nivel.py": {\n      "sha256": "c87de9852298d665c702ebf571d3ace599b90332d8854990893d75bc82a31be3"\n    },\n    "src/modelos/diagnostico_cuerpo_cola.py": {\n      "sha256": "1969c8b06f39f943f1e65235d6e20948b99d263e4a213bed1aa19fef41fc2ddc"\n    },\n    "src/modelos/diagnostico_forma_nivel.py": {\n      "sha256": "9bdba52658a287e069a66419f4659dfd6f816ac3c0def7eef9f860d3382145bf"\n    },\n    "src/modelos/escenarios_reserva.py": {\n      "sha256": "6589370402480b84d9bad902d5cd06dc17fad1e6f59dd6e38dd607292f1a8da7"\n    },\n    "src/modelos/evaluar.py": {\n      "sha256": "05ec0e71d74e50158bfbe77c37706cb339d30b196ac9a61135f019b076879288"\n    },\n    "src/modelos/experimentos.py": {\n      "sha256": "650e9120bc6556f23c632d4cf4824223a36d05e0cb62845005dbbe85d195a72e"\n    },\n    "src/modelos/exportar_predicciones.py": {\n      "sha256": "2383f82e6d1e60ed505939fa9c5495c491d6df0019a7d240c3fa7b161ab7fb2d"\n    },\n    "src/modelos/frecsev2.py": {\n      "sha256": "c4b93e38bd171f119283b23754e818c2b4cfb4bc9198d2ace84e7940f18cf431"\n    },\n    "src/modelos/influencia_reservas.py": {\n      "sha256": "959d19667f21fa3a54e2b835f5ea400c003802544f6bf69929eee063693baae7"\n    },\n    "src/modelos/intervalos_region.py": {\n      "sha256": "10646ab45a5e60d56a2230e5edf86b85e937eb89ab5dfc318134766fd21df317"\n    },\n    "src/modelos/mezcla3.py": {\n      "sha256": "a3448950d8c0544ddf1d4bb904385ae522344870ab89526f9397a65541d1f2ba"\n    },\n    "src/modelos/mlp_tweedie.py": {\n      "sha256": "d7d2f2470cd5c0ffec0634deb12fac3d01cef3f8c7434624a114ff93d075d885"\n    },\n    "src/modelos/produccion.py": {\n      "sha256": "11e15b69f6526e068862abbd6654a9348b7718656ce2a1a9dff1786e32763848"\n    },\n    "src/modelos/publico_simulado.py": {\n      "sha256": "135e9d63c7c75acfc10395abdfad6b53720b8e2967c0dd35afb84ef8d7f1fa99"\n    },\n    "src/modelos/rejilla_pesos.py": {\n      "sha256": "74298e214f614d4f24de5f1142db1d93eeef022276f5d4943ae05d6ad09c977e"\n    },\n    "src/modelos/rejilla_robusta.py": {\n      "sha256": "9a2a233c28e7ce067128258e9af34d9ed1242e415795108c05776a1c3c68264f"\n    },\n    "src/modelos/reproducir_mezcla.py": {\n      "sha256": "cfc3cc16e8915141c08b9b09a26ad8f53174a8b90de0e7e6acfdde1f01bed593"\n    },\n    "src/modelos/robustez.py": {\n      "sha256": "f5f3bcaed050d09ed95a67ad1d4176b0b3cc8aada593df0c7411180d83adcc72"\n    },\n    "src/modelos/semestres.py": {\n      "sha256": "8c1f52ee69fbe9977a5cd21790d96aeb4e6b86597765af00c259d6ae644cebd4"\n    },\n    "src/modelos/severidad_cfs.py": {\n      "sha256": "cf9e7ffd2518da52627e39bcc2e6890e51863d4abc782d26aff8a7e218a7d210"\n    },\n    "src/modelos/severidad_cfs3.py": {\n      "sha256": "a320db90b36c8fdde9edde3a7652f5793faa0ea4ba3e739836617e0df99f19c5"\n    },\n    "src/linea_b/mezcla3_b.py": {\n      "sha256": "2cb8c523927aaac4376b23ea6abb15e8da06fbb405cb9765a75648402fd53194"\n    },\n    "src/linea_b/nivel_b.py": {\n      "sha256": "f5d93edb8b84abe28101082633048336274d588305d1812090371c7477700f47"\n    },\n    "src/linea_b/reproducir_s30_b.py": {\n      "sha256": "ac82dc78d4b8ebfc749a91b8fd0818b5925ef9b3a0f8ac46e9d2d48ded99adbf"\n    },\n    "src/linea_b/severidad_tendencia.py": {\n      "sha256": "0c5870b8ffd59ffdf0d9dc3deca4c56f2ac0728798d6951a25418e55700a4831"\n    },\n    "src/datos/bases_actuariales.py": {\n      "sha256": "49e5bef169c6ede2fa4a98985b9f6e50edaa9f6e4182c33d8b9d1ff1323766df"\n    },\n    "src/datos/construir_f0.py": {\n      "sha256": "4cac9625164d59923661c3d5a1fde226f3bcf04fa3de4f7077f98dd9a76b8aed"\n    },\n    "src/datos/n62/rebuild_base.py": {\n      "sha256": "0881dbcd0df4dd54bfb181dfce1bbe20559207af340bc6f9ac72c8aedad03d29"\n    }\n  }\n}\n', 'receta/especificaciones/cfg_hogar_s2.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 981\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 334,\n  "add": [\n   "prior_claims_5y",\n   "credit_band",\n   "missed_payments_12m",\n   "household_size",\n   "years_at_address",\n   "marital_status",\n   "licence_points"\n  ]\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 322,\n  "add": [\n   "prior_claims_5y",\n   "credit_band",\n   "missed_payments_12m",\n   "household_size",\n   "years_at_address",\n   "marital_status",\n   "licence_points"\n  ]\n }\n}', 'receta/especificaciones/cfg_hogar_s3.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 981\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 1000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 395,\n  "add": [\n   "prior_claims_5y",\n   "credit_band",\n   "missed_payments_12m",\n   "household_size",\n   "years_at_address",\n   "marital_status",\n   "licence_points"\n  ]\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 322,\n  "add": [\n   "prior_claims_5y",\n   "credit_band",\n   "missed_payments_12m",\n   "household_size",\n   "years_at_address",\n   "marital_status",\n   "licence_points"\n  ]\n }\n}', 'receta/especificaciones/cfg_m595_f3.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 595\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 350\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 303\n }\n}', 'receta/especificaciones/cfg_motor_red.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 620,\n  "drop": [\n   "make_model",\n   "previous_insurer",\n   "overnight_parking",\n   "mes_inicio",\n   "wind_exposure_index_prev1",\n   "wildfire_index_mprev",\n   "years_at_address",\n   "energy_rating",\n   "rainfall_mm_mprev",\n   "household_size",\n   "distance_to_river_km",\n   "marital_status",\n   "basement",\n   "days_quote_to_inception",\n   "fuel_type",\n   "branch_id",\n   "periodo_corto",\n   "wind_exposure_index_mprev",\n   "body_type",\n   "email_domain",\n   "flood_zone",\n   "listed_building",\n   "marketing_consent",\n   "roof_type",\n   "soil_type",\n   "compulsory_excess",\n   "f1x_notif_own_damage",\n   "f1x_notif_third_party_prop",\n   "f1x_notif_theft",\n   "f1x_notif_bodily_injury",\n   "solar_panels",\n   "transmission",\n   "driver_assist",\n   "f1x_notif_glass_hail",\n   "freezing_days_mprev",\n   "mean_temperature_c_mprev"\n  ]\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 350\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 303\n }\n}', 'receta/especificaciones/cfg_s03.json': '{"motor": {"params": {"learning_rate": 0.02, "num_leaves": 7, "min_data_in_leaf": 2000, "cat_smooth": 200, "cat_l2": 50, "min_data_per_group": 1000, "max_cat_threshold": 16, "feature_fraction": 0.6, "cap": 15000.0}, "rounds": 981},\n "buildings": {"params": {"learning_rate": 0.02, "num_leaves": 5, "min_data_in_leaf": 2000, "cat_smooth": 200, "cat_l2": 50, "min_data_per_group": 1000, "max_cat_threshold": 16}, "rounds": 350},\n "contents": {"params": {"learning_rate": 0.02, "num_leaves": 7, "min_data_in_leaf": 4000, "cat_smooth": 200, "cat_l2": 50, "min_data_per_group": 1000, "max_cat_threshold": 16}, "rounds": 303}}\n', 'receta/especificaciones/cfg_total.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 981\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 350\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 303\n }\n}', 'receta/especificaciones/cfg_total_motor_red.json': '{\n "motor": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16,\n   "feature_fraction": 0.6,\n   "cap": 15000.0\n  },\n  "rounds": 981,\n  "drop": [\n   "make_model",\n   "previous_insurer",\n   "overnight_parking",\n   "mes_inicio",\n   "wind_exposure_index_prev1",\n   "wildfire_index_mprev",\n   "years_at_address",\n   "energy_rating",\n   "rainfall_mm_mprev",\n   "household_size",\n   "distance_to_river_km",\n   "marital_status",\n   "basement",\n   "days_quote_to_inception",\n   "fuel_type",\n   "branch_id",\n   "periodo_corto",\n   "wind_exposure_index_mprev",\n   "body_type",\n   "email_domain",\n   "flood_zone",\n   "listed_building",\n   "marketing_consent",\n   "roof_type",\n   "soil_type",\n   "compulsory_excess",\n   "f1x_notif_own_damage",\n   "f1x_notif_third_party_prop",\n   "f1x_notif_theft",\n   "f1x_notif_bodily_injury",\n   "solar_panels",\n   "transmission",\n   "driver_assist",\n   "f1x_notif_glass_hail",\n   "freezing_days_mprev",\n   "mean_temperature_c_mprev"\n  ]\n },\n "buildings": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 5,\n   "min_data_in_leaf": 2000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 350\n },\n "contents": {\n  "params": {\n   "learning_rate": 0.02,\n   "num_leaves": 7,\n   "min_data_in_leaf": 4000,\n   "cat_smooth": 200,\n   "cat_l2": 50,\n   "min_data_per_group": 1000,\n   "max_cat_threshold": 16\n  },\n  "rounds": 303\n }\n}', 'receta/especificaciones/spec_s22_propio.json': '{\n "motor": [\n  [\n   "CAT_motor_4s",\n   "cat1a",\n   0.2\n  ],\n  [\n   "CFSS_motor",\n   "CFSS_motor",\n   0.6\n  ],\n  [\n   "S04b_lgbm_F0124_F3autos",\n   "total_F01234",\n   0.1\n  ],\n  [\n   "S06c_lgbm_motor_red",\n   "total_motor_red",\n   0.1\n  ]\n ],\n "buildings": [\n  [\n   "S12h_edif_md1000",\n   "hS3",\n   0.4\n  ],\n  [\n   "CFSS4_buildings",\n   "CFSS4_buildings",\n   0.3\n  ],\n  [\n   "FSAS4_buildings",\n   "FSAS4_buildings",\n   0.3\n  ]\n ],\n "contents": [\n  [\n   "S08h_hogar_F3s2",\n   "hS2",\n   0.2\n  ],\n  [\n   "MLP3_contents",\n   "MLP3_contents",\n   0.1\n  ],\n  [\n   "FSA_contents",\n   "FSA_contents",\n   0.2\n  ],\n  [\n   "FSAXc4_contents",\n   "FSAXc4_contents",\n   0.5\n  ]\n ]\n}', 'receta/especificaciones/spec_s30.json': '{\n "motor": [\n  [\n   "CAT_motor_4s",\n   "cat1a",\n   0.2\n  ],\n  [\n   "B_CFST_motor_tendencia",\n   "B_CFST_motor_tendencia",\n   0.6\n  ],\n  [\n   "S04b_lgbm_F0124_F3autos",\n   "total_F01234",\n   0.1\n  ],\n  [\n   "S06c_lgbm_motor_red",\n   "total_motor_red",\n   0.1\n  ]\n ],\n "buildings": [\n  [\n   "S12h_edif_md1000",\n   "hS3",\n   0.4\n  ],\n  [\n   "B_CFST_buildings_tendencia",\n   "B_CFST_buildings_tendencia",\n   0.3\n  ],\n  [\n   "B_FSAST_buildings_tendencia",\n   "B_FSAST_buildings_tendencia",\n   0.3\n  ]\n ],\n "contents": [\n  [\n   "S08h_hogar_F3s2",\n   "hS2",\n   0.3\n  ],\n  [\n   "FSA_contents",\n   "FSA_contents",\n   0.3\n  ],\n  [\n   "FSAXc4_contents",\n   "FSAXc4_contents",\n   0.4\n  ]\n ]\n}', 'receta/huellas_esperadas.json': '{\n "creado_utc": "2026-10-19T02:14:08+00:00",\n "origenes": [\n  "/home/participant/work",\n  "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b"\n ],\n "semana1": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1",\n "entradas": {\n  "Datos/weather.parquet": {\n   "bytes": 41863008,\n   "sha256": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49",\n   "contenido": "f53977e7ef041173e620734d70592414e9a3e84cea453b291da9657289c2c214",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Datos/weather.parquet"\n  },\n  "Datos/sample_submission.parquet": {\n   "bytes": 3271617,\n   "sha256": "9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95",\n   "contenido": "f94c1d55d824842b510af78f8f431a512ef4d4c0671ce82a8100ee54ee0bd32e",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Datos/sample_submission.parquet"\n  },\n  "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_motor.parquet": {\n   "bytes": 98141422,\n   "sha256": "c503d0142e83818b2c202747ddbce53882ed9add6f8309ced714655caf25b392",\n   "contenido": "20f1a2ecfb2c012244ad8c4222ec8503e98f2efcc021c40cf1089fcc70afbbe6",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_motor.parquet"\n  },\n  "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_buildings.parquet": {\n   "bytes": 66919547,\n   "sha256": "2def2e2433062704e3762eae63f5b000e73d8284a62a6cc1a689aa63c2842454",\n   "contenido": "1506da562e28d3f28a9338b93c25424bfbdbe68c0efa1010867743977b0a4362",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_buildings.parquet"\n  },\n  "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_contents.parquet": {\n   "bytes": 65668739,\n   "sha256": "42328ddba499a77f9cc3c814506dc97fde7c3e2adf88cfc491058c7de92c76ff",\n   "contenido": "7b06b565833323090adcff185f2b5ab869b61936df75850acb69b80fe420c190",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_contents.parquet"\n  },\n  "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/particiones.parquet": {\n   "bytes": 15848536,\n   "sha256": "d365981d07feec50e487ff0d8fa4b9e5b8a00d5af8f8e521e03f2a158931035b",\n   "contenido": "0987960a3498983daa25bbd27bd4258e3e4dbfbf372bedb33ccaa02bba6c4105",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/particiones.parquet"\n  },\n  "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/contrato_disponibilidad.json": {\n   "bytes": 54543,\n   "sha256": "7e00fb2bb9efba56837ca3e6b04c016ae3ca1a8321ed6d7786eba11c6e2c541d",\n   "contenido": "388a625923b898238b01893d3b0b3ae83dda68927ebeec57b6eca839d1ec1c0a",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Matriz_F0/runs/f0_20261017T141732Z_65df7fca/contrato_disponibilidad.json"\n  },\n  "Bases_actuariales/runs/bases_20261017T120251Z_45443ff0/tables/asignacion_por_siniestro.parquet": {\n   "bytes": 4805740,\n   "sha256": "535327e1bbba95f171820d27e345988b3c2cbdeec6f86eefefaedefc52992ae5",\n   "contenido": "f760aed27500d172209434c89967949ded6e324b7ecd8a414c34dceb065f2ed3",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Bases_actuariales/runs/bases_20261017T120251Z_45443ff0/tables/asignacion_por_siniestro.parquet"\n  }\n },\n "salidas": {\n  "feat_motor.parquet": {\n   "bytes": 126378469,\n   "sha256": "1941042d83ae7ba5609627a8dd12201351871f79659b428f29f74d17410768c5",\n   "contenido": "c4a34c2725944199bb9c928d5a5bdd8d53b965c04e6b078e885065e66ccc8359",\n   "origen": "/home/participant/work/feat_motor.parquet",\n   "nodo": "variables"\n  },\n  "feat_buildings.parquet": {\n   "bytes": 85622382,\n   "sha256": "f0809330d8bd828d22eec686d0ceda30d8c855be2dbd43f173406d13e0d8efd4",\n   "contenido": "ca33fab819676987a2a486bed3c97e3182c677da041a51a5078dfce22e94d745",\n   "origen": "/home/participant/work/feat_buildings.parquet",\n   "nodo": "variables"\n  },\n  "feat_contents.parquet": {\n   "bytes": 84020097,\n   "sha256": "82334958a4e2d9bce76c9833d16df1a154403f8822e65dfea6f5a0f761b53a8f",\n   "contenido": "6dfaa65541341ebd054267896a39344befe7cda4b0a5b77ff22d583751f6023a",\n   "origen": "/home/participant/work/feat_contents.parquet",\n   "nodo": "variables"\n  },\n  "sel_total_F0124.parquet": {\n   "bytes": 4649229,\n   "sha256": "f5c512a4469ce73a368c4f86b1092f6c6f6888ce73bc7e25d9446177905690a3",\n   "contenido": "9d696c5bb25d63147c4df749e091fc0f94df5f459d57c7110b2c5227df66fd63",\n   "origen": "/home/participant/work/sel_total_F0124.parquet",\n   "nodo": "sel_total_F0124"\n  },\n  "sel_total_F01234.parquet": {\n   "bytes": 4649229,\n   "sha256": "1fd58048eeeed3dcb7f5719ac35f2c61552254806cb71e9046c53f3d84f2b1c4",\n   "contenido": "f04b5251e4034a1fe2587a338a00fa112f5599ac315ed36de083af936258da07",\n   "origen": "/home/participant/work/sel_total_F01234.parquet",\n   "nodo": "sel_total_F01234"\n  },\n  "sel_total_motor_red.parquet": {\n   "bytes": 4649229,\n   "sha256": "23c0cea11720a4a7f01e71ac0c78cf687bf110dc51033dd16444674a2c01f407",\n   "contenido": "02fe38788ae79e54b46573381aa1b5028c63e1aee329e104292429f3913cdefe",\n   "origen": "/home/participant/work/sel_total_motor_red.parquet",\n   "nodo": "sel_total_motor_red"\n  },\n  "sel_hS2.parquet": {\n   "bytes": 2893890,\n   "sha256": "e1cde41f5e1cbc88958a59d380c39206bb935865a80e24210e358057ef0f79f4",\n   "contenido": "9d3116fc3efbf1c9633c98d7ef2bdc53a4b61e47fd6b2be634d69091d7778008",\n   "origen": "/home/participant/work/sel_hS2.parquet",\n   "nodo": "sel_hS2"\n  },\n  "sel_hS3.parquet": {\n   "bytes": 1574274,\n   "sha256": "b707d8aa2336323c58cfe2b104d3694f38a5b9f48cd095cbc2ce9b2a60c199b1",\n   "contenido": "35aa73651e11b154a6aabc3392e74911af9a490cb7e3abf1e25fd1353f32638d",\n   "origen": "/home/participant/work/sel_hS3.parquet",\n   "nodo": "sel_hS3"\n  },\n  "sel_cat1a.parquet": {\n   "bytes": 2184468,\n   "sha256": "b93d1920cdb06ca2e7f3248f39bfed1fd832422a944ab22af5fc9641c8c10916",\n   "contenido": "2be5c9b5994e32f6bcede94413c65245c4a24d40a94013fe2ab273609ecdb71b",\n   "origen": "/home/participant/work/sel_cat1a.parquet",\n   "nodo": "sel_cat1a"\n  },\n  "S03_lgbm_F0124/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "1e7e184e95807462ab37443f397bbefba8334d737c3a7853107cce7b55fb2ac8",\n   "contenido": "6c4fff5cfcce7b7ba3253959764ca91de148e926f86ea36efbb9dd490fce8b47",\n   "origen": "/home/participant/work/S03_lgbm_F0124/v24.parquet",\n   "nodo": "S03_lgbm_F0124"\n  },\n  "S03_lgbm_F0124/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "53723ac0c3e49d9818b0451d3ed80b2e1ada3aecd40a3edfdd3f1b171d546556",\n   "contenido": "513660c46ad7c17fbfc47c9b10d768240a5260e09ec327aa53050bb8e3b6f3ac",\n   "origen": "/home/participant/work/S03_lgbm_F0124/prediccion_2025.parquet",\n   "nodo": "S03_lgbm_F0124"\n  },\n  "S03_lgbm_F0124/S03_lgbm_F0124.csv": {\n   "bytes": 12995807,\n   "sha256": "7877e9e968dfc094df6b9450f51ff1d5f97f12fa82b4d985e2fce18d6af93217",\n   "contenido": "7877e9e968dfc094df6b9450f51ff1d5f97f12fa82b4d985e2fce18d6af93217",\n   "origen": "/home/participant/work/S03_lgbm_F0124/S03_lgbm_F0124.csv",\n   "nodo": "S03_lgbm_F0124"\n  },\n  "S04b_lgbm_F0124_F3autos/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "acfb4eec8ac56d1f3a97d6a0a09e32f79681fd6379e0f3e86e495d9f5e30fb37",\n   "contenido": "b1882aa092b5e9d13b693c1e2424ae3e64ebf373e0662b387b5ada89303b82f8",\n   "origen": "/home/participant/work/S04b_lgbm_F0124_F3autos/v24.parquet",\n   "nodo": "S04b_lgbm_F0124_F3autos"\n  },\n  "S04b_lgbm_F0124_F3autos/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "264ce035ffdd1c07ebcb8c3227aa6f4f2563e6b759c1b7a7c45d2581eafd4564",\n   "contenido": "08b40bd4ef1a0cb624d5d35f4af1cd646ae3dfe6ecd38e3b887d75985c349eb5",\n   "origen": "/home/participant/work/S04b_lgbm_F0124_F3autos/prediccion_2025.parquet",\n   "nodo": "S04b_lgbm_F0124_F3autos"\n  },\n  "S04b_lgbm_F0124_F3autos/S04b_lgbm_F0124_F3autos.csv": {\n   "bytes": 12996206,\n   "sha256": "23e09d9f27326d7a20d508c32b4da039bac43085c1726498618f668d1eaba1fe",\n   "contenido": "23e09d9f27326d7a20d508c32b4da039bac43085c1726498618f668d1eaba1fe",\n   "origen": "/home/participant/work/S04b_lgbm_F0124_F3autos/S04b_lgbm_F0124_F3autos.csv",\n   "nodo": "S04b_lgbm_F0124_F3autos"\n  },\n  "S06c_lgbm_motor_red/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "eaba43ef4455670133e763c6dddb6648acd9f864ad7e93f4c57ce9b8432825bf",\n   "contenido": "9e4835764d7a16c4d7afb6a4ccff5a6355f2a97f55629a750504ca36e4019393",\n   "origen": "/home/participant/work/S06c_lgbm_motor_red/v24.parquet",\n   "nodo": "S06c_lgbm_motor_red"\n  },\n  "S06c_lgbm_motor_red/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "4a49db207b76127f3bf91c9bcd439e2bd90fab6a0eb79529d081604efb04b7ab",\n   "contenido": "ce20bc09337efb20e6b075f065a076be1ad1cef323e5382b3b66f05e45bbcb48",\n   "origen": "/home/participant/work/S06c_lgbm_motor_red/prediccion_2025.parquet",\n   "nodo": "S06c_lgbm_motor_red"\n  },\n  "S06c_lgbm_motor_red/S06c_lgbm_motor_red.csv": {\n   "bytes": 12997557,\n   "sha256": "8624c2331ab5d9eeed7ee6d6331528c1e7bf5c0bee2b1b02a9e421fb83d4a68e",\n   "contenido": "8624c2331ab5d9eeed7ee6d6331528c1e7bf5c0bee2b1b02a9e421fb83d4a68e",\n   "origen": "/home/participant/work/S06c_lgbm_motor_red/S06c_lgbm_motor_red.csv",\n   "nodo": "S06c_lgbm_motor_red"\n  },\n  "S08h_hogar_F3s2/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "cc604c6e645e27c525528b2de2c02138ccec56ddec186d7cd566edfb6020eedb",\n   "contenido": "40c627d97edc9c82650bd4c03d0a7bf1af9b9244790f118f8a7a121ee51e983e",\n   "origen": "/home/participant/work/S08h_hogar_F3s2/v24.parquet",\n   "nodo": "S08h_hogar_F3s2"\n  },\n  "S08h_hogar_F3s2/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "f91932f301b817308c50a48bd5bf6297dd62c711fe815f9915dd4fa3d93e2cb2",\n   "contenido": "578cad2b2004e9777a05bd7d78e83095063f1f2102effb32f0bf5e54fc6b21dd",\n   "origen": "/home/participant/work/S08h_hogar_F3s2/prediccion_2025.parquet",\n   "nodo": "S08h_hogar_F3s2"\n  },\n  "S08h_hogar_F3s2/S08h_hogar_F3s2.csv": {\n   "bytes": 12995502,\n   "sha256": "9623c7b7585248e92a717643464a87d5610153ed77b86c4ca344087df1848ac7",\n   "contenido": "9623c7b7585248e92a717643464a87d5610153ed77b86c4ca344087df1848ac7",\n   "origen": "/home/participant/work/S08h_hogar_F3s2/S08h_hogar_F3s2.csv",\n   "nodo": "S08h_hogar_F3s2"\n  },\n  "S12h_edif_md1000/v24.parquet": {\n   "bytes": 8878138,\n   "sha256": "d535388f050b6893cd35d89ccc3c966b7f43e186a13841af3de1b885f37c4c3e",\n   "contenido": "b777023647e18780004936d4970645208d51a3d2c85a6d8618814d537730f5cd",\n   "origen": "/home/participant/work/S12h_edif_md1000/v24.parquet",\n   "nodo": "S12h_edif_md1000"\n  },\n  "S12h_edif_md1000/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "70a3b8b61c40477c9f44725350aabe6adbe83ffc5498c2ea8ef564fd2cf6e663",\n   "contenido": "4716e9466d537e334279d5aaeac6ca28da4d6a81bc688d9fb6eab995f86fab60",\n   "origen": "/home/participant/work/S12h_edif_md1000/prediccion_2025.parquet",\n   "nodo": "S12h_edif_md1000"\n  },\n  "S12h_edif_md1000/S12h_edif_md1000.csv": {\n   "bytes": 12994639,\n   "sha256": "2ca725c0569daf79abf56516278afc8af8ade2ed9f9116662690767a380bdc19",\n   "contenido": "2ca725c0569daf79abf56516278afc8af8ade2ed9f9116662690767a380bdc19",\n   "origen": "/home/participant/work/S12h_edif_md1000/S12h_edif_md1000.csv",\n   "nodo": "S12h_edif_md1000"\n  },\n  "CAT_motor/v24.parquet": {\n   "bytes": 3958638,\n   "sha256": "ee475f23e30ac89f6410ce2f8a19424a505fca758da7a0c33271172abe9a902a",\n   "contenido": "e11fb2fd15be309b7c2d028e534dae6030aba40f5263ed6328712123b4bacb37",\n   "origen": "/home/participant/work/CAT_motor/v24.parquet",\n   "nodo": "CAT_motor"\n  },\n  "CAT_motor/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "4e197e05938aa70e23c934909796d9392c7df46d5e6c045be444f59245a21cc9",\n   "contenido": "99a8bee3434d4be9f95937fe656f7181c5726a13906ccef1134f42bd577181dc",\n   "origen": "/home/participant/work/CAT_motor/prediccion_2025.parquet",\n   "nodo": "CAT_motor"\n  },\n  "CAT_motor_s23/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "3895cf1b73442c532da5172401de53db6f93ec2c596e8a3a5f746e770850b6d7",\n   "contenido": "c2ac3b0329020c4c81fb91d04d79593c001bacb7ede724795682268d0f4a8597",\n   "origen": "/home/participant/work/CAT_motor_s23/v24.parquet",\n   "nodo": "CAT_motor_s23"\n  },\n  "CAT_motor_s23/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "ae5ee520b2f83d873bf9708629a536b3d1f44979ee6656a273a331d0a39470a4",\n   "contenido": "4d2fc235c404f224c71f33d86d285b5ae9631dfd57010d7feb4683fa3f816918",\n   "origen": "/home/participant/work/CAT_motor_s23/prediccion_2025.parquet",\n   "nodo": "CAT_motor_s23"\n  },\n  "CAT_motor_4s/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "62274dd4fab69a5ebd465da88315450414fadaf585013567b8ca44d8bc44ed9c",\n   "contenido": "ce3a64c582d95c8685c6210b1c1f27533a4f405be58964f3061d71db87c8e2da",\n   "origen": "/home/participant/work/CAT_motor_4s/v24.parquet",\n   "nodo": "CAT_motor_4s"\n  },\n  "CAT_motor_4s/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "85b97f8c006497cf46dca694dd24f4bc537b28154a6d3a6042568884ef923c98",\n   "contenido": "a53ce47dd9c1c5248d5ff573d3956a929b054a0027aefd2f068565584e903c3b",\n   "origen": "/home/participant/work/CAT_motor_4s/prediccion_2025.parquet",\n   "nodo": "CAT_motor_4s"\n  },\n  "CFS_motor_v23/v23_own_damage.npz": {\n   "bytes": 1019396,\n   "sha256": "a86234806dee320a21ce6b6a574b913af61dc5f108fbf85e21f5f1c0491aec46",\n   "contenido": "9425f95af7a7cea889c32314a26c20d58c18c35f51be0af11fda06a246eca639",\n   "origen": "/home/participant/work/CFS_motor_v23/v23_own_damage.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor_v23/v23_third_party_prop.npz": {\n   "bytes": 1019396,\n   "sha256": "cd33ea937a6053ab457c6a17a34e0e9d6cf8726f780bd7c083ca4bb9ec2cb12a",\n   "contenido": "8cc4f0034db4333f4a8c9a6e0000cd80b9a576e3016a7120588a4ba0b73bdbb6",\n   "origen": "/home/participant/work/CFS_motor_v23/v23_third_party_prop.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor_v23/v23_bodily_injury.npz": {\n   "bytes": 1019396,\n   "sha256": "b90099af6169d02c3bf02424f6dc9fda942f2f87fec89a6eaeee48ea7d20de7a",\n   "contenido": "ea21503a8f0dc5ff308f814e32603fb0222c8b61028f689638cf255e99f38cc3",\n   "origen": "/home/participant/work/CFS_motor_v23/v23_bodily_injury.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor_v23/v23_theft.npz": {\n   "bytes": 1019396,\n   "sha256": "f288a47025a9895e639129d2db271b29ecd2a88e888d6012a8cfb2f693098da5",\n   "contenido": "fa956280856087080425e43adff7089398ce2858c26e43b619fb6cd210f41add",\n   "origen": "/home/participant/work/CFS_motor_v23/v23_theft.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor_v23/v23_glass_hail.npz": {\n   "bytes": 1019396,\n   "sha256": "411b4dadf36d4b4056a17fda53156a0b9eafe497199edaaec15107502d4a726f",\n   "contenido": "68c2e89ba917c93bc72f3666c5f7893fb45a4c5df1147f36713dc7cc14a30fe6",\n   "origen": "/home/participant/work/CFS_motor_v23/v23_glass_hail.npz",\n   "nodo": "CFS_motor_v23"\n  },\n  "sel_CFS_motor_v23.parquet": {\n   "bytes": 2184468,\n   "sha256": "900ab1601fc2ac25ae6f983ee21faa8596acf95e8c23dd4bec7d69945119fb15",\n   "contenido": "19872c9181f376b9e5f06e41ade0aee9f321790a0fc0f219139bd52a2aab1b35",\n   "origen": "/home/participant/work/sel_CFS_motor_v23.parquet",\n   "nodo": "CFS_motor_v23"\n  },\n  "CFS_motor/prod_own_damage.npy": {\n   "bytes": 2040016,\n   "sha256": "a3b99f874b3d52f644c5b82da96d212bf636ee6069be651ac125004f836a2517",\n   "contenido": "d047b991d0bf126b600f2e4d987083571f61f3e500a81a51b8fc8cb5da684425",\n   "origen": "/home/participant/work/CFS_motor/prod_own_damage.npy",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prod_third_party_prop.npy": {\n   "bytes": 2040016,\n   "sha256": "690a4c3bcec92d55bd861b4da095c0cbd370e227494ecbf9b5e8f4a8a8707259",\n   "contenido": "dea39f8e7a118c3736b19412d5b24e6b103ccebc70f2aad86a9493d0a348065c",\n   "origen": "/home/participant/work/CFS_motor/prod_third_party_prop.npy",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prod_bodily_injury.npy": {\n   "bytes": 2040016,\n   "sha256": "67d5aa89fa68feefe1646db753380940225c7fb26fe7852d1f93f9fd16e0a1ac",\n   "contenido": "0f4bd15add1ff1be34938900abb19eef53771612ac968bbe72e50302a301cb1c",\n   "origen": "/home/participant/work/CFS_motor/prod_bodily_injury.npy",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prod_theft.npy": {\n   "bytes": 2040016,\n   "sha256": "07f82c8865581daf90e7084b7a5876856f6404b683c9ebcdc06e8528a6dec3a8",\n   "contenido": "0b529132474385d25c402a7a6f62ce1f3b8ca66a2d3e5525fd0dd864dd9eb9bc",\n   "origen": "/home/participant/work/CFS_motor/prod_theft.npy",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prod_glass_hail.npy": {\n   "bytes": 2040016,\n   "sha256": "9387d36f3bccd89f661bde91f10859bb56e702d5d17ba4a39c2eb3acc5cc877f",\n   "contenido": "2f1e1ea5522131b6506209a692913a4c25f0497fdc089e73f2dd71b23c1cc6df",\n   "origen": "/home/participant/work/CFS_motor/prod_glass_hail.npy",\n   "nodo": "CFS_motor"\n  },\n  "sel_CFS_motor.parquet": {\n   "bytes": 2184468,\n   "sha256": "900ab1601fc2ac25ae6f983ee21faa8596acf95e8c23dd4bec7d69945119fb15",\n   "contenido": "19872c9181f376b9e5f06e41ade0aee9f321790a0fc0f219139bd52a2aab1b35",\n   "origen": "/home/participant/work/sel_CFS_motor.parquet",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "cba09ca9072764aa37c89429db73a4639e227eb312b914136d7deae48bad3f3e",\n   "contenido": "74ac7d827138db3f62c99d14e9a23b133db6d9f8b465b7513f43598655ae2fc0",\n   "origen": "/home/participant/work/CFS_motor/v24.parquet",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "4d17abc843b341c27dfc4629d93bf07a874f6e02aa88016774476e5899791ee2",\n   "contenido": "996cc87eeedfe208b568cf75e6646b27790170ed80cf82878b5463a32f23ce79",\n   "origen": "/home/participant/work/CFS_motor/prediccion_2025.parquet",\n   "nodo": "CFS_motor"\n  },\n  "CFS_motor_s23/v24_own_damage.npy": {\n   "bytes": 2037712,\n   "sha256": "4d7afe91ca0b9870e42d8229a88116163ef8e97f210763df77fcad76ea93304b",\n   "contenido": "a5ef6ce5af02ad239f84441cf1ecc78ca03ece4fd7e013fd01b0d8abab2eb7fa",\n   "origen": "/home/participant/work/CFS_motor_s23/v24_own_damage.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24_third_party_prop.npy": {\n   "bytes": 2037712,\n   "sha256": "3b115c00c5c4793dffa7c6c0a25c9db979c0fecd22e419011faef6123f9b5856",\n   "contenido": "4e4ee5ee93b23cc2cbda594888f1f511c8937c6536f16eb8be88fefac7e2957c",\n   "origen": "/home/participant/work/CFS_motor_s23/v24_third_party_prop.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24_bodily_injury.npy": {\n   "bytes": 2037712,\n   "sha256": "a9317779bb6fed4f5989329b385a533facd41b9f5bd61e7d1e886a79e5af0fdb",\n   "contenido": "202fcb4d21542c7dc2b3b8832a57dba641de6ff84c18d54b79aad6d67571338b",\n   "origen": "/home/participant/work/CFS_motor_s23/v24_bodily_injury.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24_theft.npy": {\n   "bytes": 2037712,\n   "sha256": "fb20dd3bffd53ff99c3f51b1691072e4414d551e844b92afa146a2b2f28f73b1",\n   "contenido": "518fd4c60b0d9c684813dc65aa79095658c424e1afa4a1415edf7b752b14df7e",\n   "origen": "/home/participant/work/CFS_motor_s23/v24_theft.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24_glass_hail.npy": {\n   "bytes": 2037712,\n   "sha256": "ea815c321fdddc24049fd47e3bbc326b91a7585fb3baf6fc1f6464963bd80f9a",\n   "contenido": "d4d2a93ef4da9b2165b5ed937c1b1d862b6a57000d90405b707428008387b818",\n   "origen": "/home/participant/work/CFS_motor_s23/v24_glass_hail.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_own_damage.npy": {\n   "bytes": 2040016,\n   "sha256": "81d33d4c78c5d22450fb9d636ee51dd921e67b0524dee1a1a45567811a0aa568",\n   "contenido": "a8d6b95778f2f03d28d6fc5bfd4b5ebd71cfa69e1bb9ea10ba9fcfbf93ba2c89",\n   "origen": "/home/participant/work/CFS_motor_s23/prod_own_damage.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_third_party_prop.npy": {\n   "bytes": 2040016,\n   "sha256": "30cb12405c05c5789b0e63e2b2f425cbfc814004deb3f0b22c3a1fb9dc2d2e0f",\n   "contenido": "d5e41e24cfbeec1f725ce54721614d5bb0782aac03252eef666ace2745dd9d34",\n   "origen": "/home/participant/work/CFS_motor_s23/prod_third_party_prop.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_bodily_injury.npy": {\n   "bytes": 2040016,\n   "sha256": "15b2bea45299746e98e08ba93074913d43b8d18b5eda58a8cf4c255fb0bcdfec",\n   "contenido": "16febd373be937250f773cbc62bfe339665e64f53bcb4008dfe8130145aadc0a",\n   "origen": "/home/participant/work/CFS_motor_s23/prod_bodily_injury.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_theft.npy": {\n   "bytes": 2040016,\n   "sha256": "956aa7e8db64d6b56d8b3459b80ab9b92bbf67c8ffbff723bb95e8eb10a54078",\n   "contenido": "323094f6b8d6d62e1ac948d7367660d4ee94cc15f3af55f94ef1004a5926d0a5",\n   "origen": "/home/participant/work/CFS_motor_s23/prod_theft.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prod_glass_hail.npy": {\n   "bytes": 2040016,\n   "sha256": "df52d7b2356006719b146ba0eb7a2e236b7c8d9b5ceea1f06200f99d40c6e2f4",\n   "contenido": "bf5308da3621fda7b258602bdfec2f544f113eaba4717955921ce9d1d8bf922a",\n   "origen": "/home/participant/work/CFS_motor_s23/prod_glass_hail.npy",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "edd218f167d18c74c7fc42f5b5cc41dabae4940adb47cc57de1df6f3e97928c6",\n   "contenido": "b9dcd78b36d43e5700270e5db1c34d499964513a48e25c5797d27cf7cb2b64a2",\n   "origen": "/home/participant/work/CFS_motor_s23/v24.parquet",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_motor_s23/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "30a06f85fb051c40a79fc7de2b8da730715062bdc1176f6ab76b5c23df6da9f3",\n   "contenido": "0079edf0dcc8fd51536593b66efe082cb1462572b36326c2f42f9a2e3e6c534f",\n   "origen": "/home/participant/work/CFS_motor_s23/prediccion_2025.parquet",\n   "nodo": "CFS_motor_s23"\n  },\n  "CFS_buildings/v23_storm.npz": {\n   "bytes": 738220,\n   "sha256": "f3198f0bd2607ccc2385f67e725d10b7dfdede63ca83f50191b71159378169bf",\n   "contenido": "f9bea81ded4f4cb3b8720942d0aef01f290fcede646c9e369b685e46cb3bd880",\n   "origen": "/home/participant/work/CFS_buildings/v23_storm.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_flood.npz": {\n   "bytes": 738220,\n   "sha256": "be7902e29055327a534df8455e72458c6d2aa0ba256c110f511052fb5cf6730e",\n   "contenido": "47cec25dec3967016a7cdf802c587192224d41d347700faaacd5289789401d98",\n   "origen": "/home/participant/work/CFS_buildings/v23_flood.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_escape_of_water.npz": {\n   "bytes": 738220,\n   "sha256": "2122c2ce064d000f82ff5ec4bec3b0736fb84c69e148e0bfb399ef45b080719c",\n   "contenido": "d1efc155f0b3cd90b9ffd207c46ece1e6045579e42d682d0c1fc72586ff4d768",\n   "origen": "/home/participant/work/CFS_buildings/v23_escape_of_water.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_subsidence.npz": {\n   "bytes": 738220,\n   "sha256": "533faac0e88a2c6a1b4a777e4f4a764a0c0ba6e9092d4f658f6211962d3968be",\n   "contenido": "4469f4e57564db66c421f7049f8c4f28ce8cc9997fdce059a139dfd3c4f5a493",\n   "origen": "/home/participant/work/CFS_buildings/v23_subsidence.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_fire.npz": {\n   "bytes": 738220,\n   "sha256": "8663b1ffe7a7cd4b3cee0de2c57bd9e2b33bbb6cfd920d98cf44f25d15a97da2",\n   "contenido": "06c0e05e3945a3268187f82600938f13099ea29012eb165f3889c583bb48734b",\n   "origen": "/home/participant/work/CFS_buildings/v23_fire.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v23_accidental_damage.npz": {\n   "bytes": 738220,\n   "sha256": "768d54f01f0b736ee6d0744afda2fdfe20a7f0ee3f6859c9f1294aa11e67ac5c",\n   "contenido": "60ff9f1f8db519b306c3ebd4a2e2501c3f4974482522bb0149346035cd337c08",\n   "origen": "/home/participant/work/CFS_buildings/v23_accidental_damage.npz",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_storm.npy": {\n   "bytes": 1472136,\n   "sha256": "af281fe9aad5f43c395c65ca4be2e569b01a0397af2f8d55373642724b36aca5",\n   "contenido": "502fcc5befbd612e88c5661fe407ed0e9c5f170e6e4018011d39301d92fdde30",\n   "origen": "/home/participant/work/CFS_buildings/v24_storm.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_flood.npy": {\n   "bytes": 1472136,\n   "sha256": "3fbf1e53eb69949cac4c60cedd74bedce084ff76b76a8a068cc983335896593a",\n   "contenido": "cb23588bd3715eaa6373043d14116f4bc7eaa9278e87e05eebd41e7cde44b2cb",\n   "origen": "/home/participant/work/CFS_buildings/v24_flood.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_escape_of_water.npy": {\n   "bytes": 1472136,\n   "sha256": "1472f036e25ad90fb8aa19db0b7db55f1bc3b1bea0be0e9308870300ec7000bb",\n   "contenido": "911b77aa064803159d52833f7fcd2abc6b59f06f20225486541d0a45145bf4ea",\n   "origen": "/home/participant/work/CFS_buildings/v24_escape_of_water.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_subsidence.npy": {\n   "bytes": 1472136,\n   "sha256": "ed0068208cc4990b2b568b040f013510a695c36000323c86806ffb3fb3f96e1a",\n   "contenido": "7ed2c89b18fc4fd0f9f87b7ca64716aa0d2e7c6bed512818facb6216d23819c6",\n   "origen": "/home/participant/work/CFS_buildings/v24_subsidence.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_fire.npy": {\n   "bytes": 1472136,\n   "sha256": "f0512ea785bd8cec311736b4005ac3a4d55c8dcb9324f48b50094b8c536bd39c",\n   "contenido": "3e21a01447bba5c26199b7c86eca4454d52f5e479d515183698905ef41d549f6",\n   "origen": "/home/participant/work/CFS_buildings/v24_fire.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24_accidental_damage.npy": {\n   "bytes": 1472136,\n   "sha256": "f0ae17d7bd550235dab7a9999e218c704f44ce34c77fbf81f9ae3bdbc83a9c89",\n   "contenido": "4dbf28bd059e514f6c049a168ce8eeac0ff57b1d319f351750df9c7cd5915c28",\n   "origen": "/home/participant/work/CFS_buildings/v24_accidental_damage.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_storm.npy": {\n   "bytes": 1473368,\n   "sha256": "081473d7a42e4240bc11153b454cf3f7a16d1f2000dd9d74bbcd871b65f86c1b",\n   "contenido": "6461a529c8bc04e5c8f0c8e7feecb17592627dd68a6228c0ca22af9ad78e6ffc",\n   "origen": "/home/participant/work/CFS_buildings/prod_storm.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_flood.npy": {\n   "bytes": 1473368,\n   "sha256": "3dff91ac41b110db777a4ccc7f28cb2ac1994d46097f7222fbdfd5390fec2142",\n   "contenido": "1ec7cd4011d92c9c8c9af051af10c9a3a971d59b9af2e7e62b1ad5ccf320734c",\n   "origen": "/home/participant/work/CFS_buildings/prod_flood.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_escape_of_water.npy": {\n   "bytes": 1473368,\n   "sha256": "c7aabb1e756189060c9d66505d2c1187bca9f32951a6a39891cc2e015e15ee29",\n   "contenido": "c8744d97be8b25db88268c66db67b1d6affbc659f59bfeb44df19b23a43d39a0",\n   "origen": "/home/participant/work/CFS_buildings/prod_escape_of_water.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_subsidence.npy": {\n   "bytes": 1473368,\n   "sha256": "a09ce50a92a08d75b974a7643ab2a365d912d94648daa0670e494bc5f6921a1b",\n   "contenido": "85df5a94a9b785f981b3b7d3ffdc265b3c19c62fc3e0ff976b5648dd4f749e33",\n   "origen": "/home/participant/work/CFS_buildings/prod_subsidence.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_fire.npy": {\n   "bytes": 1473368,\n   "sha256": "1643fc97d5a0dece1ff9489462f024701a4264027085ec737abc7d032ce42291",\n   "contenido": "6c56c719e49b422109faee8f363eb27ed54ef88842ca85b0bb39f7717a14079d",\n   "origen": "/home/participant/work/CFS_buildings/prod_fire.npy",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prod_accidental_damage.npy": {\n   "bytes": 1473368,\n   "sha256": "00935979b230f2746fc698a5766d6b3b3b9cfeaebd61c75714a01a607dc62e95",\n   "contenido": "840f274befc462fa14a346cd3f83aa996b9393f62ee336f1de35400538db1e5d",\n   "origen": "/home/participant/work/CFS_buildings/prod_accidental_damage.npy",\n   "nodo": "CFS_buildings"\n  },\n  "sel_CFS_buildings.parquet": {\n   "bytes": 1574274,\n   "sha256": "65f62e51072a50cc359424a42b061fe13d0a9dca384e4b2956d3038f451fd22f",\n   "contenido": "61796de6874ad0523e7599425b4091e23b393c91a88116896d50dc8814fd6d6f",\n   "origen": "/home/participant/work/sel_CFS_buildings.parquet",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "27493e4077bcc877a6769b516f7c477ff39364d7dc0ee9fa725584ae54c52d22",\n   "contenido": "3c2a85e509beb56dd83779ac2e893c0febd544b7bb4d2cfddfba050e33ee8b59",\n   "origen": "/home/participant/work/CFS_buildings/v24.parquet",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "682d515636eba93971f37e987938721185eac8700b81b14e52ac3a2b8b4f86e9",\n   "contenido": "6dc46287a66b9a542ab085ab10e672bc2419fd23c80e1e8b5a5ffb16e8a8e3ce",\n   "origen": "/home/participant/work/CFS_buildings/prediccion_2025.parquet",\n   "nodo": "CFS_buildings"\n  },\n  "CFS_buildings_s23/v24_storm.npy": {\n   "bytes": 1472136,\n   "sha256": "fd520b07ea423aaac1659688416bbd82380f1a70a732bae3c5162168440efe77",\n   "contenido": "d32856085b4d5db65fd281401be536c37c2f0af03245fe5cf8996fb2d97a3673",\n   "origen": "/home/participant/work/CFS_buildings_s23/v24_storm.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_flood.npy": {\n   "bytes": 1472136,\n   "sha256": "b048655eab511d59fb15f20ff08e722f212e023ad15ff46e8ee076722fb232f7",\n   "contenido": "247e42c844db7dab10dc1524e50a6c67b863560f1c7d587a0b4bf75977c2d02e",\n   "origen": "/home/participant/work/CFS_buildings_s23/v24_flood.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_escape_of_water.npy": {\n   "bytes": 1472136,\n   "sha256": "336980ef8844d6d23607ff926829ec81b1929d1425ffa2711310f95e9d777184",\n   "contenido": "e2a8e9ff6402859ee77eb71a54850b5e82b90e0e19d29a7a60c239e5fe2af5ce",\n   "origen": "/home/participant/work/CFS_buildings_s23/v24_escape_of_water.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_subsidence.npy": {\n   "bytes": 1472136,\n   "sha256": "f63610d9318c9ae267edd4697695b753afbacb62b2f22d4539b95f6ade6ac9e1",\n   "contenido": "07cad8bed4b1049fc29bb2edead44fe851657a3f839ad65244b690a1245994b8",\n   "origen": "/home/participant/work/CFS_buildings_s23/v24_subsidence.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_fire.npy": {\n   "bytes": 1472136,\n   "sha256": "c5ddb12650fbecddf7d92d734bc87057317ba8119eab7507707a050e99639d7e",\n   "contenido": "c7ebe61d1ab90192ca8db661587706b2683f2b6f050b39b2b78d8ea0b9895ed0",\n   "origen": "/home/participant/work/CFS_buildings_s23/v24_fire.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24_accidental_damage.npy": {\n   "bytes": 1472136,\n   "sha256": "2f416841812aefc2df37b2fadc3b5af5f036d275c5c284b1e4628424450f75e6",\n   "contenido": "5c1fbd3e2ba9a953edc3315b26981cfb6ba6c57aeae6bc3cafd8c848dac7fc94",\n   "origen": "/home/participant/work/CFS_buildings_s23/v24_accidental_damage.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_storm.npy": {\n   "bytes": 1473368,\n   "sha256": "18fdd28dfb122345a3fc8364ac48e4c604f1829da21fc9fe98f9b61fe9586a8c",\n   "contenido": "7514a722f92d47ecfd65301247fb5389c88673d1aa35411ca1f3348fbdb1a59b",\n   "origen": "/home/participant/work/CFS_buildings_s23/prod_storm.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_flood.npy": {\n   "bytes": 1473368,\n   "sha256": "3141b7e8b374e792971a0bbe5dd9e9cb74f923fd563849c3d4eeba9909ef758b",\n   "contenido": "f020daead573789d5195be84cf1fc57216ba2a84c05ad9eb3c9c988e6ba25e02",\n   "origen": "/home/participant/work/CFS_buildings_s23/prod_flood.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_escape_of_water.npy": {\n   "bytes": 1473368,\n   "sha256": "d2d911f199ac71b35bfdda30794f0344d0b71be44a90039f329c92b90fdbff2e",\n   "contenido": "3f651d5f48b87cf3d7942dc855fc56345f834c74800ba1f91bf04b4c8a874205",\n   "origen": "/home/participant/work/CFS_buildings_s23/prod_escape_of_water.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_subsidence.npy": {\n   "bytes": 1473368,\n   "sha256": "dcd98e60b1f8a82cb7df1ac613a90f4650f70cc31fda15b805c63c0369b75975",\n   "contenido": "5c0697bafa5c329671be1720672f28cb3c7adbba32d090f6f754e44b6d5c1e13",\n   "origen": "/home/participant/work/CFS_buildings_s23/prod_subsidence.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_fire.npy": {\n   "bytes": 1473368,\n   "sha256": "18402ad417833b5e4480295ccd224a29b9112a6ceb6108bc3c4130a3eab0117b",\n   "contenido": "ee6ae48c6cbfd5c5e0ed2cbf802d6111026f7564730682e5f7b5f5e7c4c29962",\n   "origen": "/home/participant/work/CFS_buildings_s23/prod_fire.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prod_accidental_damage.npy": {\n   "bytes": 1473368,\n   "sha256": "f4f2085fa3d307d15e08b9821ba7ce835abccedb04b24c9fabf0727a9c91a255",\n   "contenido": "819f879990d64982318bb2f02dbcaf20142907bb0ac5e4d19065a1db9ca44a37",\n   "origen": "/home/participant/work/CFS_buildings_s23/prod_accidental_damage.npy",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "4478cd503b550cc95a4a8b9377557b2f5b3bdf769f76b832ff9ac6a2e466c2f7",\n   "contenido": "0b10942386dfff15aeffd1604c6332dddb02f44b5d179ec72cb1c7a8925cb4a8",\n   "origen": "/home/participant/work/CFS_buildings_s23/v24.parquet",\n   "nodo": "CFS_buildings_s23"\n  },\n  "CFS_buildings_s23/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "9bad842a7c6690be04ab1be43f8710b03446cbe2f2c528c5fa7d3ab12ea1d89f",\n   "contenido": "b034db17088a976eb360764448b397415d2b2b2e76fad92ebcc55437f9717b65",\n   "origen": "/home/participant/work/CFS_buildings_s23/prediccion_2025.parquet",\n   "nodo": "CFS_buildings_s23"\n  },\n  "sel_FSA_contents.parquet": {\n   "bytes": 1583200,\n   "sha256": "5eb0386a57ee6ef715989e06a3b5f3fc4f767069fa2e65582c7ff87f0d91060a",\n   "contenido": "3d908254711203ca9da04c4113973934a7587415b84f0bc99d6daf99dd2eadc9",\n   "origen": "/home/participant/work/sel_FSA_contents.parquet",\n   "nodo": "FSA_contents"\n  },\n  "FSA_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "92a05c3174853240db42344dee1c233a47a1f0bf2d4be59d177a4f802e260ff8",\n   "contenido": "418936f3e377ba69795b8df736804a4295115a8fde963066a424439258caed78",\n   "origen": "/home/participant/work/FSA_contents/v24.parquet",\n   "nodo": "FSA_contents"\n  },\n  "FSA_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "f3d02e2face2cf7ae18c44f615d3ad1969b2ce4bfc12e244569d0c35f720c1c2",\n   "contenido": "7f8da0b4396fccf8c6fef3131a5b5d489af18050319035266572265fde0d98a3",\n   "origen": "/home/participant/work/FSA_contents/prediccion_2025.parquet",\n   "nodo": "FSA_contents"\n  },\n  "FSAP_buildings/v23_storm.npz": {\n   "bytes": 738220,\n   "sha256": "850b3b4951cc143d9f37f8ccaad1c01d7256ab9dff8411777f811a2b97838134",\n   "contenido": "961171a27239f6d77f5212b21da9b7692da6a3a23270cb46ec7ca6d939c05e6c",\n   "origen": "/home/participant/work/FSAP_buildings/v23_storm.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_flood.npz": {\n   "bytes": 738220,\n   "sha256": "58e74b143eac29633154aae8af5ba88eef8a13b6832d281a3d9050a476e19156",\n   "contenido": "e552f897970f1e1252d7351133f5e4a9d1024c756005b359e3da8862356dab48",\n   "origen": "/home/participant/work/FSAP_buildings/v23_flood.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_escape_of_water.npz": {\n   "bytes": 738220,\n   "sha256": "425c10af118f637be9c71d4340150c13db4b2b4b42fce60bd81472cb925edaea",\n   "contenido": "c718763ca4de27b4255e4035f81fea32f004e1625615eba9483ab92d7e4fee4e",\n   "origen": "/home/participant/work/FSAP_buildings/v23_escape_of_water.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_subsidence.npz": {\n   "bytes": 738220,\n   "sha256": "239ac4084751f448eccc5c2aba2c6ac9e874611ebabdb6ca325e620d6c72dc25",\n   "contenido": "027814bf5f1aeb153890b764cd1201c6e2b5d508c9ac3cec84eeac037f645615",\n   "origen": "/home/participant/work/FSAP_buildings/v23_subsidence.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_fire.npz": {\n   "bytes": 738220,\n   "sha256": "8f23b38286754d7dea82b20d1508698e652f44f26debb17252f89bd77795510b",\n   "contenido": "29fde6af2f6860e07dfe851b1b32383da1a5a8d55d6a24978200e09dda3602f9",\n   "origen": "/home/participant/work/FSAP_buildings/v23_fire.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v23_accidental_damage.npz": {\n   "bytes": 738220,\n   "sha256": "35b0ab7e6a8df8d161a0819815e382559697f127c138119471b92c06f6d80d72",\n   "contenido": "ded99cff8b3dd536a2980c1b7c8d7c5cd723bbf6f608d93bb7da159376bf48ba",\n   "origen": "/home/participant/work/FSAP_buildings/v23_accidental_damage.npz",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_storm.npy": {\n   "bytes": 1472136,\n   "sha256": "f187a0b5b41839749d1d462ecfd69ba2f60c3d6decaa7cdfeea9dc5c00dc2458",\n   "contenido": "051d204f575ba9cd70bc12030068dd8f6c20816d1aadc5a69b615b94c9ec25a6",\n   "origen": "/home/participant/work/FSAP_buildings/v24_storm.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_flood.npy": {\n   "bytes": 1472136,\n   "sha256": "e946ded4c829ad8d9fc1b74868db0534d347e81578ebbcd2fa8c417700a70ec1",\n   "contenido": "aeb76d81aa4e26ecc2af8491e847c968c0b946cc55b63cef4875f4a7ba912ac8",\n   "origen": "/home/participant/work/FSAP_buildings/v24_flood.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_escape_of_water.npy": {\n   "bytes": 1472136,\n   "sha256": "66987e0dc2c8e716aeaac725a935407c69aa28817e19bc7c8ba6fb80b2b3b8d6",\n   "contenido": "292469938160a6a06c012d94c434b3db8802626401abafc30537f97d65e50b55",\n   "origen": "/home/participant/work/FSAP_buildings/v24_escape_of_water.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_subsidence.npy": {\n   "bytes": 1472136,\n   "sha256": "9a3fe6a967a80ce3247ca10d06f9d9f24435ecd9031653b30a76073f1f539dc4",\n   "contenido": "55da1950628a2b90cb499df99f9301800c18782a450d8d60d967724f23c767f5",\n   "origen": "/home/participant/work/FSAP_buildings/v24_subsidence.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_fire.npy": {\n   "bytes": 1472136,\n   "sha256": "8b076b6b3d6a1fbd4c79d2498ef0450124587552601e0020aaed32ca9398d98c",\n   "contenido": "088544e713c5082e74e76abb3e5c19d35581fd66c5e074bfab36aacc91729219",\n   "origen": "/home/participant/work/FSAP_buildings/v24_fire.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24_accidental_damage.npy": {\n   "bytes": 1472136,\n   "sha256": "37337978de4dfc4d10fd549eb3153e7a19e3240653d3c9a5c7e074058fe864d6",\n   "contenido": "9226fc6b8303f70d11db01bd240b4c3113121552acb2d910c74e91b6f3704613",\n   "origen": "/home/participant/work/FSAP_buildings/v24_accidental_damage.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_storm.npy": {\n   "bytes": 1473368,\n   "sha256": "388eba379c72789e19971a36e30b4d06c2e33b90e1adf7b3db466e0ea912cd7c",\n   "contenido": "7470367dfae54d656e6df0c63196545ecf4db75b7b60e58e1318a8e4e7645471",\n   "origen": "/home/participant/work/FSAP_buildings/prod_storm.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_flood.npy": {\n   "bytes": 1473368,\n   "sha256": "835b8d795e217a14a8e9b90e4c858aa84460e4025e07e506d44fa4ee840b6434",\n   "contenido": "7b8a2b4bc978c6b6d954f7c235270eb6b59ee4bd4b2a6d5c2388478636f0c53b",\n   "origen": "/home/participant/work/FSAP_buildings/prod_flood.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_escape_of_water.npy": {\n   "bytes": 1473368,\n   "sha256": "0a9bec1ecb3f5e7fba36e58a6b52aa63789b754892e307baf030961dd1d7637d",\n   "contenido": "bb0069ce0ebf05cece001ab3566953db3538fcea9c1dcf228dd0f3dec1588c65",\n   "origen": "/home/participant/work/FSAP_buildings/prod_escape_of_water.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_subsidence.npy": {\n   "bytes": 1473368,\n   "sha256": "62c9859786d37a6159cc44e91884429d259f8719d3cd485dc09cf3a51a7c8c5c",\n   "contenido": "16c2ad51a1e683b48b94654ff4f173bc4506f5eba3febbdac69f84ec8cce85b2",\n   "origen": "/home/participant/work/FSAP_buildings/prod_subsidence.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_fire.npy": {\n   "bytes": 1473368,\n   "sha256": "ffa156ed849afb427a6a4fd5f01dd9653f19611bbaa45c849219b5cc1704011d",\n   "contenido": "9fd207f858296b94926b73b64daf73d3f3f71feb6358591ef2c26224f163b7f2",\n   "origen": "/home/participant/work/FSAP_buildings/prod_fire.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prod_accidental_damage.npy": {\n   "bytes": 1473368,\n   "sha256": "08b9f79f33f867a1c240f15ecb6ae5943a8138fa632db09cf180378645faca1f",\n   "contenido": "af8df6b54f4ce6cd7bf2f4bf0530de9359291ed978a76f4d390c367d3becd2dd",\n   "origen": "/home/participant/work/FSAP_buildings/prod_accidental_damage.npy",\n   "nodo": "FSAP_buildings"\n  },\n  "sel_FSAP_buildings.parquet": {\n   "bytes": 1574274,\n   "sha256": "8651e0c38f36c7604ed7cf004c9f346b794ae600251cc89278baf0dbc08c564d",\n   "contenido": "4da2f65013274fe32e8ad5e550a436ab3b391c178fc9ccd7848cc245536627d3",\n   "origen": "/home/participant/work/sel_FSAP_buildings.parquet",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "a91de648d619bfdb1298988d25331d41bb45d296dd362c6e4faabe79af8c2348",\n   "contenido": "a1cab56bcf87ce5a6c80a03175b7467762a688df42d9c75f97dab912b85c39d5",\n   "origen": "/home/participant/work/FSAP_buildings/v24.parquet",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "e9fac7e4119262180180acb03a8a600137b65fd1920bb5aa4a363a0684da85e5",\n   "contenido": "8b500314fdbde9ffde0ac2abd14f7bc5d50a80b126c31c2c26387251032c9a41",\n   "origen": "/home/participant/work/FSAP_buildings/prediccion_2025.parquet",\n   "nodo": "FSAP_buildings"\n  },\n  "FSAP2_buildings/v24_storm.npy": {\n   "bytes": 1472136,\n   "sha256": "49034731247e67e0cebdb63a12280469f33e762e69c001c20723fab2221ff562",\n   "contenido": "5a3488b9190313e78922408356ddd9b279deb1a0c96118307eb911324f7907af",\n   "origen": "/home/participant/work/FSAP2_buildings/v24_storm.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_flood.npy": {\n   "bytes": 1472136,\n   "sha256": "6cf339c36691da4e7bf026bcad8c662a0b8d7e989aa03dc2aafd236f415b92f9",\n   "contenido": "0d5623bdde055f40f886313579343c4a449444c5c6f37e69acc4e717fd1222c8",\n   "origen": "/home/participant/work/FSAP2_buildings/v24_flood.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_escape_of_water.npy": {\n   "bytes": 1472136,\n   "sha256": "9177427a22ca85c120e6d8835a9bcc254c8d3d74f11041440eb324f6be1426a9",\n   "contenido": "a9b20654902aee33623c41575590caaa0eb90eb8355f31d45b49e16171512e4f",\n   "origen": "/home/participant/work/FSAP2_buildings/v24_escape_of_water.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_subsidence.npy": {\n   "bytes": 1472136,\n   "sha256": "658d42f0564f47bed553a64a11039fd93027ab8783bdaa82e1dd5fd08dbeea57",\n   "contenido": "1073cf493df2370ee4bf8c679198c26ef1af5372dc5ac321ad6aef5337cfa985",\n   "origen": "/home/participant/work/FSAP2_buildings/v24_subsidence.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_fire.npy": {\n   "bytes": 1472136,\n   "sha256": "4d07f0f0fb237a44c816a1a62af729ac25b49e2e6bf413476c123e3099da63f5",\n   "contenido": "2bc35b262ecb92d0773767a6d878f1b8d13dcb5bd7d6c997f18e990b59b22b4a",\n   "origen": "/home/participant/work/FSAP2_buildings/v24_fire.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24_accidental_damage.npy": {\n   "bytes": 1472136,\n   "sha256": "f04e2592832b7647c5e24e7e1a73f873f1f89bfbcdea710596e312fd382b77c2",\n   "contenido": "3236f232375e2213bc3706140a23b63d9c4cdf7e75f834e8085703416c774f04",\n   "origen": "/home/participant/work/FSAP2_buildings/v24_accidental_damage.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_storm.npy": {\n   "bytes": 1473368,\n   "sha256": "aa8c63e86c531dd6793ef0319e259a373293a5ff922ea5220c7b22b5cbd5ba0c",\n   "contenido": "9df133ce6b258e598f4a7e79adce3d8a2eeaff81025d5acfba853538fe3064c7",\n   "origen": "/home/participant/work/FSAP2_buildings/prod_storm.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_flood.npy": {\n   "bytes": 1473368,\n   "sha256": "da746c66ac3ac15c45ed769c96aeb15619f63a89090a278a0ccb97e19ae8afc6",\n   "contenido": "63f7430de45b45133ffab60ca6c43662edc63f5c305cce529490e316c17a73dd",\n   "origen": "/home/participant/work/FSAP2_buildings/prod_flood.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_escape_of_water.npy": {\n   "bytes": 1473368,\n   "sha256": "0aa258443517200e7b48ea41ec90bd37037b9790a162f27f076d53ec63f8ba73",\n   "contenido": "ad92d8ae2610ec85b2192edc9c01c877a85b2aef0613c03b90d698972e129b48",\n   "origen": "/home/participant/work/FSAP2_buildings/prod_escape_of_water.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_subsidence.npy": {\n   "bytes": 1473368,\n   "sha256": "52e3df3859fb429f940d6d59e97c7e660be29feaa4298723dbab030e54ca996d",\n   "contenido": "c171756be82f0ed42d375e4bce47aca1b1dca22072d13ee56d9cee609f0f792b",\n   "origen": "/home/participant/work/FSAP2_buildings/prod_subsidence.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_fire.npy": {\n   "bytes": 1473368,\n   "sha256": "0246bcf85c25b1ddf89a6ceed6d379150ccdd2dab9d544183d6d094a49f4eabe",\n   "contenido": "a3d56d033c08f71d2e5309805020f023857a68f13247fc890addf526eb8f17af",\n   "origen": "/home/participant/work/FSAP2_buildings/prod_fire.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prod_accidental_damage.npy": {\n   "bytes": 1473368,\n   "sha256": "1afbd944543d85ec1155f30d9d3343be142d37a01b2f0e2fd63b8f52fe7d4e82",\n   "contenido": "a8b6c5dd1c7aa4f411027e370558e07a40ac35855d6fb029e0782b1e05be115e",\n   "origen": "/home/participant/work/FSAP2_buildings/prod_accidental_damage.npy",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "ee8c8846a5d9abbdde283b8eebdf496dc089ac8c7983cb7cc42b6ffb60fbb011",\n   "contenido": "437cb4038ecff8caae42e517b32d3d85c93246d08a27f500d19d1d2ffada4bd3",\n   "origen": "/home/participant/work/FSAP2_buildings/v24.parquet",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP2_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "a72c60d348bb756577ee4cd9b5c899e554c9a756b0939ac1b0c7ce66cf2139db",\n   "contenido": "e7102e79075f911177eb670d5daf24b0ae8b19436ea27a9573d5c1cc0e6b4d3a",\n   "origen": "/home/participant/work/FSAP2_buildings/prediccion_2025.parquet",\n   "nodo": "FSAP2_buildings"\n  },\n  "FSAP_contents/v23_theft.npz": {\n   "bytes": 752028,\n   "sha256": "bbd6650303c3b940e075927cee9803413d87a9e9cffef5a2b37a2f79da0e3c4f",\n   "contenido": "5d3a97a2a8a843dd3227040e887ce13890b74705163658ce68f37d02762fa1de",\n   "origen": "/home/participant/work/FSAP_contents/v23_theft.npz",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v23_escape_of_water.npz": {\n   "bytes": 752028,\n   "sha256": "96f944cf147439c979d9195a4064c7645ed9ee0ecd700dc0507f76155ef7140c",\n   "contenido": "835011fbc9b139b7bb596313e81e268dfafa1ec55caf3be17d2ca48a10f2613b",\n   "origen": "/home/participant/work/FSAP_contents/v23_escape_of_water.npz",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v23_accidental_damage.npz": {\n   "bytes": 752028,\n   "sha256": "d9f5e0b63937d106d5dfc8c5cd9ca8c6e7811eed16ca5af8d86f983e8bfafb1d",\n   "contenido": "7196137a11c967c1b46013a7f9c8999025a9217e84e4a1ee019f3f9c3378421a",\n   "origen": "/home/participant/work/FSAP_contents/v23_accidental_damage.npz",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v23_flood.npz": {\n   "bytes": 752028,\n   "sha256": "25a2adf3ca65e4dabda00284e627724e269b73bf6e634a82010ac0ec76c8bb63",\n   "contenido": "4677f9dd8df9a0ba503e3f3dd5c26426051e838a14e3b89dc7f74d342eac6405",\n   "origen": "/home/participant/work/FSAP_contents/v23_flood.npz",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24_theft.npy": {\n   "bytes": 1500056,\n   "sha256": "cfe68a13afbbbda3335b46d7cfb01805c1bdaeec7f25938a8e194544a23cca29",\n   "contenido": "ea18aa845ea0529618fa4ff33927d3b5d1cbe8143bf5a9ca4d868599e7383105",\n   "origen": "/home/participant/work/FSAP_contents/v24_theft.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24_escape_of_water.npy": {\n   "bytes": 1500056,\n   "sha256": "8d9c3341a4c85ac780078b1ce295e0150c34348bc92f4e8eb43a05e0981db399",\n   "contenido": "a94ad4cc693a579dcf8e47c8d79f80796eeccf9cdc61f0f091ba67028ce96931",\n   "origen": "/home/participant/work/FSAP_contents/v24_escape_of_water.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24_accidental_damage.npy": {\n   "bytes": 1500056,\n   "sha256": "e57933947da62042f91b3d4156ffcba74131d13ea7abce90a0581fc04c4f9e54",\n   "contenido": "02e339ff1a89c3f29e9fbb3c1ce7745c26ad79304b17bf8305b78f211f872f6e",\n   "origen": "/home/participant/work/FSAP_contents/v24_accidental_damage.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24_flood.npy": {\n   "bytes": 1500056,\n   "sha256": "4aea53a01b546eab33b72268bd2c42c4f2eccf1037d1eb009fffde7c291afdda",\n   "contenido": "b62fcbbeae71f7114d61f92c467c75e12b481d35b0d9608bb4fa5e65c4e3514d",\n   "origen": "/home/participant/work/FSAP_contents/v24_flood.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prod_theft.npy": {\n   "bytes": 1500112,\n   "sha256": "22035d69d31c08bebfe539f8ee09296f50c36b40f9f8038de13d7963a80eb8c4",\n   "contenido": "fa471b8c9d0a3be4da31c3d01f621ffac76532ae95fb0fc45c6f6e2f1cf6bf26",\n   "origen": "/home/participant/work/FSAP_contents/prod_theft.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prod_escape_of_water.npy": {\n   "bytes": 1500112,\n   "sha256": "abafd8da5f6fac80251ff1a8baff4722847dacd2dfd1773007d0ef6d6a4cebec",\n   "contenido": "f6b43f005c642ed73589661ca5362a7c5c4701ac6a0dfcf42af5234c7089e7fe",\n   "origen": "/home/participant/work/FSAP_contents/prod_escape_of_water.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prod_accidental_damage.npy": {\n   "bytes": 1500112,\n   "sha256": "26641f6003ed76e7835d5145e3b231ffb560c05c58a838304552eea38b745ca7",\n   "contenido": "c5005e29e122cd98ad8e1445b4ff022e881bf4402efb4b176c04ef4553e6cb56",\n   "origen": "/home/participant/work/FSAP_contents/prod_accidental_damage.npy",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prod_flood.npy": {\n   "bytes": 1500112,\n   "sha256": "ccdbfe8c871e39f9267eab378e8d92c545dab44bee281ef2bc9d2374b4f0e61b",\n   "contenido": "87988ea67c8bc26a02b02e45fb634158adf1b9c4c7404409adcfdedd9c7eb0e3",\n   "origen": "/home/participant/work/FSAP_contents/prod_flood.npy",\n   "nodo": "FSAP_contents"\n  },\n  "sel_FSAP_contents.parquet": {\n   "bytes": 1583200,\n   "sha256": "5eb0386a57ee6ef715989e06a3b5f3fc4f767069fa2e65582c7ff87f0d91060a",\n   "contenido": "3d908254711203ca9da04c4113973934a7587415b84f0bc99d6daf99dd2eadc9",\n   "origen": "/home/participant/work/sel_FSAP_contents.parquet",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "b451b67b5a22020bd3ac2ba43fee68d1230a54087520433b6656e8dcfc08c161",\n   "contenido": "0293229b0a70a4d2d42df5e1f8bb0dc9b165d4be888db03694573bbdec786b29",\n   "origen": "/home/participant/work/FSAP_contents/v24.parquet",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "4e2721f75b11090cc3ebad37e97ba9060c0a5f6cbaf77346afcf27f56a90eb8a",\n   "contenido": "1ab6563930d58c91eff5e41a50032f1c2f12b22c7c44a740a5525eb099e5295b",\n   "origen": "/home/participant/work/FSAP_contents/prediccion_2025.parquet",\n   "nodo": "FSAP_contents"\n  },\n  "FSAP2_contents/v24_theft.npy": {\n   "bytes": 1500056,\n   "sha256": "d6b330d8f55508cd5e02ff459c851cfc523af68ec9f4c0e3dbf570299dffd1e0",\n   "contenido": "7107917ab6d0d5d51e547332db9fe78942be2e937cc3facbcff61c71687fa279",\n   "origen": "/home/participant/work/FSAP2_contents/v24_theft.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/v24_escape_of_water.npy": {\n   "bytes": 1500056,\n   "sha256": "6ee11df4a9fbb6f6747f7cb6c2a5579c9944285ac17b0dddb9689053df041ab3",\n   "contenido": "92fae148d1d97b8c1eb90c71fa2a1d298b49f13e04757a31467d094bf01791cc",\n   "origen": "/home/participant/work/FSAP2_contents/v24_escape_of_water.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/v24_accidental_damage.npy": {\n   "bytes": 1500056,\n   "sha256": "f276e677263db38f971491c24b246149671008da91c867f369abdd3983040ebf",\n   "contenido": "e960de5b9987702323fbf6d1e34b6ab219bcd85f44d10d5a188c934d893b76ab",\n   "origen": "/home/participant/work/FSAP2_contents/v24_accidental_damage.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/v24_flood.npy": {\n   "bytes": 1500056,\n   "sha256": "690e7b5b3c497c2df855d6f6923f96b64e5689a529d9485e46523e211b2e6205",\n   "contenido": "ead8ec1ee0c314469be019b1252e921a9298fa66db856bc1771b41cbb7624ac0",\n   "origen": "/home/participant/work/FSAP2_contents/v24_flood.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prod_theft.npy": {\n   "bytes": 1500112,\n   "sha256": "9d80e12df99f83f36c854a80dacc4c6740e072bb7b7d65dc329234bb13421af3",\n   "contenido": "1a927dc5ef777c95e949079d24f5fc4262cf2415096071644805896878b34906",\n   "origen": "/home/participant/work/FSAP2_contents/prod_theft.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prod_escape_of_water.npy": {\n   "bytes": 1500112,\n   "sha256": "7eadcd7a3e4a07457d4504c283513bbae835e79778005a97a9557d91fe8bc898",\n   "contenido": "48c21af0c89bd88414f418c9ce2d937ff2bb3c113118794381c343f602847cbb",\n   "origen": "/home/participant/work/FSAP2_contents/prod_escape_of_water.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prod_accidental_damage.npy": {\n   "bytes": 1500112,\n   "sha256": "46bd228c7352a8e8f2a157ebceb28c3c1ab00446094982d9d3b2fe7273e18188",\n   "contenido": "1665c7d122790a4acabbd09aeab95146ae5797da0a4a07bb155f30538a872ac8",\n   "origen": "/home/participant/work/FSAP2_contents/prod_accidental_damage.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prod_flood.npy": {\n   "bytes": 1500112,\n   "sha256": "88cf7525daf6c80c83d7f34c84f987be492c2566d05ded5be9a3b85cb904bc87",\n   "contenido": "79a9daa3ce6174dff505c36fb5f93badf37387bbdc93f7fc67ecb21ca6d02b9e",\n   "origen": "/home/participant/work/FSAP2_contents/prod_flood.npy",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "c170d5bb8d9f3fce5ad61a2b8d220b8a3589a423ab702f94f2a469b064538fa2",\n   "contenido": "73364651f8348112037de7604cdefdce01c8da21b5101ed62708efcb189e2ca9",\n   "origen": "/home/participant/work/FSAP2_contents/v24.parquet",\n   "nodo": "FSAP2_contents"\n  },\n  "FSAP2_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "1bfdd92e10f76ded32d5e3b1b20e1e482105a99cfbc1f6624330064289a2d4c8",\n   "contenido": "3893b4c9fa4772ec7222ae8e0a7a4e05dea79773beeba143a5d7df9e178fd560",\n   "origen": "/home/participant/work/FSAP2_contents/prediccion_2025.parquet",\n   "nodo": "FSAP2_contents"\n  },\n  "sel_FSAXc4_contents.parquet": {\n   "bytes": 1583200,\n   "sha256": "8d1bc6092d030392a8cccb718c6977a9fe5d961a66830699f3f68b6277bf6e91",\n   "contenido": "ead7277d3490bef603db7eca4ae65918116cb91115418c0dc55118a29a747857",\n   "origen": "/home/participant/work/sel_FSAXc4_contents.parquet",\n   "nodo": "FSAXc4_contents"\n  },\n  "FSAXc4_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "148c0fc478d5ecf6bfba4b7e49be6e092c305da96c74812e3007c1cff47e071b",\n   "contenido": "e55f3fa521d2d54e87fdf45777ca156709b8dd59eb04995ff327a8d287f44ec0",\n   "origen": "/home/participant/work/FSAXc4_contents/v24.parquet",\n   "nodo": "FSAXc4_contents"\n  },\n  "FSAXc4_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "7c1314c30b7dc66012acba3c31bdaf8614ba4003dfceb3c32c6d4947d5528693",\n   "contenido": "8e3d2185f0c53b77d7d0056fb503423cadd62ed2037b83cbd5b41ac47b373bcf",\n   "origen": "/home/participant/work/FSAXc4_contents/prediccion_2025.parquet",\n   "nodo": "FSAXc4_contents"\n  },\n  "CFS_motor_4s/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "1056f044b2b4ad4f44e00b707a5add3c8f270eb9f9d8eba50adc876b7f993d3b",\n   "contenido": "51da611d6fe392eacaa0a4cef21ec05d918b2b4aadb0d4e176a625d2bbd4d037",\n   "origen": "/home/participant/work/CFS_motor_4s/v24.parquet",\n   "nodo": "CFS_motor_4s"\n  },\n  "CFS_motor_4s/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "1a561d3974c5ada13d39cbb495f5d9fe8103899800d1c787369c83d7157ad270",\n   "contenido": "6a89e9b402ad16fc5558457556de8cd7baf3b826e20d0582d786e36d1e837c32",\n   "origen": "/home/participant/work/CFS_motor_4s/prediccion_2025.parquet",\n   "nodo": "CFS_motor_4s"\n  },\n  "sel_CFSS_motor.parquet": {\n   "bytes": 2184468,\n   "sha256": "3999126bf6a41a40e143a698c962734342576b243d86d14a100323638b27cd30",\n   "contenido": "451a411d6a0766749ae965a0820de4b424aaaf87f110bb83a2d2001d9a13681d",\n   "origen": "/home/participant/work/sel_CFSS_motor.parquet",\n   "nodo": "CFSS_motor"\n  },\n  "CFSS_motor/v24.parquet": {\n   "bytes": 3958638,\n   "sha256": "7e7dc1ae042af79eff417300700d29ffc7ebf1470285d0b46a1cade635aa5c64",\n   "contenido": "ddf65387e08b4ca4b810a8bb5990e03dc8a01ec1831191c07940407cbee0cd98",\n   "origen": "/home/participant/work/CFSS_motor/v24.parquet",\n   "nodo": "CFSS_motor"\n  },\n  "CFSS_motor/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "01879b6503846c2bcc787dffc568e2b4e54702e832a6188f8e28954a93636b0f",\n   "contenido": "f2c7f58463492168b95abf1288ce321dd5787f905d1d1bb78b26c6b6db03feda",\n   "origen": "/home/participant/work/CFSS_motor/prediccion_2025.parquet",\n   "nodo": "CFSS_motor"\n  },\n  "sel_CFSS4_buildings.parquet": {\n   "bytes": 1574274,\n   "sha256": "24720d1605dc96e864cc854e5d4b15140f810e45f824dd7b006181a7c77234dc",\n   "contenido": "f0fe47fda321f32a6886447d26bf0f7318d45f3bd2fb3b372dbb71a1f85609d6",\n   "origen": "/home/participant/work/sel_CFSS4_buildings.parquet",\n   "nodo": "CFSS4_buildings"\n  },\n  "CFSS4_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "5d62ca730fe58c37b2eba947cf3dd55bdbfe2f0c0d6664887aaf5e3aae985992",\n   "contenido": "a6a5c40bf934e4e0e77673c0e2d5199154b1c368676a5068a56d7d3a4431d48b",\n   "origen": "/home/participant/work/CFSS4_buildings/v24.parquet",\n   "nodo": "CFSS4_buildings"\n  },\n  "CFSS4_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "a0e9b7f325f203659ddc5d28b01b2a6d99025a4b3600a3188f6c2135f6e97287",\n   "contenido": "fe236611aa1b5f4bf67f521e2cf0d8361375f08d04f2dd47bcecdce0f34417e6",\n   "origen": "/home/participant/work/CFSS4_buildings/prediccion_2025.parquet",\n   "nodo": "CFSS4_buildings"\n  },\n  "sel_FSAS4_buildings.parquet": {\n   "bytes": 1574274,\n   "sha256": "30fd205adbccd000ad5778e586ba50e9bfe878c7c27c5537c6050662d7d7fa0b",\n   "contenido": "eea1f6aea6bebe86dd5b365c93be4ac6d61702b0b41d74a9c1529d0e520f0224",\n   "origen": "/home/participant/work/sel_FSAS4_buildings.parquet",\n   "nodo": "FSAS4_buildings"\n  },\n  "FSAS4_buildings/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "efbc789e12135c4cb664adc8a75f483f50b5b26c41382c7a3cd480a7044555d7",\n   "contenido": "9bbd5ac062a378429f68cdb3622482968d6c309afed2951a8df731e39a5ed707",\n   "origen": "/home/participant/work/FSAS4_buildings/v24.parquet",\n   "nodo": "FSAS4_buildings"\n  },\n  "FSAS4_buildings/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "e4c66502dafbd11706cede41cef0bf9157aa97d8fedc3186b90206b0fc51f513",\n   "contenido": "4f95dfb24f5f7168295054cbdc7168340c19ddea285dddf365ce25946fe599c4",\n   "origen": "/home/participant/work/FSAS4_buildings/prediccion_2025.parquet",\n   "nodo": "FSAS4_buildings"\n  },\n  "sel_MLP3_contents.parquet": {\n   "bytes": 1583200,\n   "sha256": "b0d6c5d6bd232a08fe68307a36882d2384f24100370a455d07a0e9c4999292bc",\n   "contenido": "0d43dc219dd678adf113bf7a8262d744d1a5bc74ed04109aa80b13e4496d099e",\n   "origen": "/home/participant/work/sel_MLP3_contents.parquet",\n   "nodo": "MLP3_contents"\n  },\n  "MLP3_contents/v24.parquet": {\n   "bytes": 2907654,\n   "sha256": "a32a06f39dc47a92bc4637fdf6cf7bba051f046d3395aab377d6f4b8abb60f09",\n   "contenido": "c5882d87bf4cd7d6f9a1986ee19f815b299d1a434e9980801b634c58ae6c49b1",\n   "origen": "/home/participant/work/MLP3_contents/v24.parquet",\n   "nodo": "MLP3_contents"\n  },\n  "MLP3_contents/prediccion_2025.parquet": {\n   "bytes": 2846950,\n   "sha256": "0dc7c784cbde615726f71c0f7fce473177a583de03dca0f06e69790227041d0d",\n   "contenido": "ec3a333653ace9fcc07e505a350ae2199272df0ec8f7e5273d87888b2c8248b8",\n   "origen": "/home/participant/work/MLP3_contents/prediccion_2025.parquet",\n   "nodo": "MLP3_contents"\n  },\n  "sel_B_CFST_motor_tendencia.parquet": {\n   "bytes": 2184468,\n   "sha256": "bab64358c15bc2937aebfbb52bf8614bb47030519bc895733796652ad6fa801e",\n   "contenido": "4c555e7e642ccb6716f2f2897b052cb6c08a41cef4ea0f134bd6098211a0fcf5",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_CFST_motor_tendencia/sel_B_CFST_motor_tendencia.parquet",\n   "nodo": "B_CFST_motor_tendencia"\n  },\n  "B_CFST_motor_tendencia/v24.parquet": {\n   "bytes": 3958636,\n   "sha256": "7c964a8afbff304b89badc17fd359f64bbd2937f3605573280cf794c719c883a",\n   "contenido": "b947fcda025e9854f2927db224abc5270cf716a38a7ae91c596f6272ba128dbc",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_CFST_motor_tendencia/v24.parquet",\n   "nodo": "B_CFST_motor_tendencia"\n  },\n  "B_CFST_motor_tendencia/prediccion_2025.parquet": {\n   "bytes": 3739277,\n   "sha256": "ce175b1f8bcf03869df82cbdc72bcb7576fbd0946b5ecd7a37eca3690c7cbd14",\n   "contenido": "ec1912f0241c5a0df24f89b1bf4de0c8ab641e4faba8c53f73a773b49518d1c2",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_CFST_motor_tendencia/prediccion_2025.parquet",\n   "nodo": "B_CFST_motor_tendencia"\n  },\n  "sel_B_CFST_buildings_tendencia.parquet": {\n   "bytes": 1574274,\n   "sha256": "b4ca50868d4f646fd99a2723e52cbc907f392e7d39985e9ded229a0aa48e6903",\n   "contenido": "c3d1abfbea05f10c764fd520865693f0b9c9f38966c69f6536fb7c5be8cffa40",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_CFST_buildings_tendencia/sel_B_CFST_buildings_tendencia.parquet",\n   "nodo": "B_CFST_buildings_tendencia"\n  },\n  "B_CFST_buildings_tendencia/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "548b3870f8e77e57d7134703845a59d53d83cd7c4142451e12f1494bb207f88d",\n   "contenido": "d84db3d67f3bdb02d19b07c215c6c96b1e99d9c4d81dc4d7cc9ec1af5f731845",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_CFST_buildings_tendencia/v24.parquet",\n   "nodo": "B_CFST_buildings_tendencia"\n  },\n  "B_CFST_buildings_tendencia/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "d07de2fdf5237cd391ba7304d36d3b1b343bd42a562e9ea048b3dc73194cc724",\n   "contenido": "e21facaf57951f15763253ca1dcd360779c330225a25764fcef5b98e33dbd626",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_CFST_buildings_tendencia/prediccion_2025.parquet",\n   "nodo": "B_CFST_buildings_tendencia"\n  },\n  "sel_B_FSAST_buildings_tendencia.parquet": {\n   "bytes": 1574274,\n   "sha256": "849c9710b1370ce397096291cc7816283dce9ab06cd2027b3336aa81ffa270b3",\n   "contenido": "e584bb6a5cb622f93c28576cbc90705cf9d1d128cacc726fd3bb6d416c0e4ded",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_FSAST_buildings_tendencia/sel_B_FSAST_buildings_tendencia.parquet",\n   "nodo": "B_FSAST_buildings_tendencia"\n  },\n  "B_FSAST_buildings_tendencia/v24.parquet": {\n   "bytes": 2867461,\n   "sha256": "b5eedbdfacf43bb4867863bc8bb2f69009dd9204d51abec087034ffd295013c4",\n   "contenido": "d5213e778557c64815a8df1f26905156a26ebdb7d9f003da9bf659a6319c5c20",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_FSAST_buildings_tendencia/v24.parquet",\n   "nodo": "B_FSAST_buildings_tendencia"\n  },\n  "B_FSAST_buildings_tendencia/prediccion_2025.parquet": {\n   "bytes": 2820627,\n   "sha256": "38e947a6be4ae6856a9d54608ec033e3bf036e472b5427c4930d94fb6430e9dc",\n   "contenido": "15a21294e404af324988cdd9b0b6548925c4eca8d2a354f91ff08e1a32a043c6",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/B_FSAST_buildings_tendencia/prediccion_2025.parquet",\n   "nodo": "B_FSAST_buildings_tendencia"\n  },\n  "S22_main/S22_main.csv": {\n   "bytes": 13012725,\n   "sha256": "c21a02e14de4a89f892604239b7cfd9ac4e95ea7d175460a25ba806a5a3541b0",\n   "contenido": "c21a02e14de4a89f892604239b7cfd9ac4e95ea7d175460a25ba806a5a3541b0",\n   "origen": "/home/participant/work/S22_main/S22_main.csv",\n   "nodo": "S22_main"\n  },\n  "S22_main/prediccion_2025.parquet": {\n   "bytes": 8557978,\n   "sha256": "35ffd557f43a32ad9a379a1df22386fc671e1f02f799c24f24e39c59da4719cf",\n   "contenido": "29ae79020644eebe6c5da25df5a99dfba0c48831cb5d52798c03d8ac98a1ee81",\n   "origen": "/home/participant/work/S22_main/prediccion_2025.parquet",\n   "nodo": "S22_main"\n  },\n  "S30_line_b/S30_line_b.csv": {\n   "bytes": 13012442,\n   "sha256": "fd9528e0b1ee4b674dea63834a97523fd4ddc743d1495a422ce31a2eb76e3bc5",\n   "contenido": "fd9528e0b1ee4b674dea63834a97523fd4ddc743d1495a422ce31a2eb76e3bc5",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/S30_line_b/S30_line_b.csv",\n   "nodo": "S30_line_b"\n  },\n  "S30_line_b/prediccion_2025.parquet": {\n   "bytes": 8557976,\n   "sha256": "231c6d174fa28fb73da432d4bae1dfb2287e7ec8bf9f07fa37f7bad772543e13",\n   "contenido": "d0c6cf5f0f3da2f01a519138648c74ccc91f64b2c2d9386a52aab199abb74e94",\n   "origen": "/mnt/user-data/uploads/Hakaton/Hackaton/Semana 1/Modelos_line_b/runs/S30_line_b/prediccion_2025.parquet",\n   "nodo": "S30_line_b"\n  }\n },\n "faltan": [\n  "CFS_motor/v23_own_damage.npz",\n  "CFS_motor/v23_third_party_prop.npz",\n  "CFS_motor/v23_bodily_injury.npz",\n  "CFS_motor/v23_theft.npz",\n  "CFS_motor/v23_glass_hail.npz",\n  "CFS_motor/v24_own_damage.npy",\n  "CFS_motor/v24_third_party_prop.npy",\n  "CFS_motor/v24_bodily_injury.npy",\n  "CFS_motor/v24_theft.npy",\n  "CFS_motor/v24_glass_hail.npy"\n ],\n "datos": {\n  "originales": {\n   "addresses.parquet": "2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1",\n   "claims.parquet": "323adf2e0f6c5975a026479d725d4cd04f51fce3aa069f897a99f664c6e392c0",\n   "customers.parquet": "065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb",\n   "policies_buildings.parquet": "eb84372c23c16ed0619d89c8eb1396ab6916e5ec46a5907ce42c3dfce4a91cbe",\n   "policies_contents.parquet": "8c067cc80baed583100df999e1c3d5e389295702f7bac21d5331bbd639b83fcd",\n   "policies_motor.parquet": "4d2d7ae03971d5d6df3a71bbaf0ebc168250a6807b27d1561d38b638434dc95e",\n   "sample_submission.parquet": "9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95",\n   "test_buildings.parquet": "7b2b96507de29241c70617c17df66146ed30b51f826ce5219bbda5da536d4957",\n   "test_contents.parquet": "c2212f62f3b7a21431c0073277549c3db5e63857565a8b88855e79a65fa1b8b6",\n   "test_motor.parquet": "8d28fb9fd091caf2a044913a013592226a4c0917d5c6fcc9c5e8cdcec2ab4fc5",\n   "weather.parquet": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49"\n  },\n  "capa": {\n   "policies_motor": "d6b5e28f2b291c6ce285bc75c2426f63d78b2556b40273197f4a42a0652c9ef8",\n   "policies_buildings": "0378b2801849ba257cc06fc671e7d3ae4f7eade41a20efc6d08fb1eb6cbaf0f9",\n   "policies_contents": "09cdc7e56daf6765cf93c338351f84bf7d474d4fe148e88b35caa5f74f890dd4",\n   "test_motor": "27ddb9a3e6f9b56af223c9d50d01586a81d732dcb571a23bb793810e0655f43b",\n   "test_buildings": "99e983e3203fee57ee32f6e47166cdd8d0bb033455c236e1967839dac88276ae",\n   "test_contents": "55de22d263c113646568a5fe778b71635ff51a475bc78f747e69baad5e3ff39a",\n   "customers": "065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb",\n   "addresses": "2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1",\n   "weather": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49",\n   "claims": "31b1d8007ca9ee3a6dd00e73f99e7a7fc0d1fb8b13087b687342c999bd7b8336",\n   "sample_submission": "9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95"\n  },\n  "decisiones": {\n   "Limpieza/runs/capendorse_20261017T010844Z_fbf8cc20/ledger/cumulative_retrospective_changes.parquet": {\n    "uso": "marca audit_capital_scale_confirmed (rama CAP02R a CAP06R)",\n    "sha256": "19682d0792858cf4b4df0172ef8658cfb1ff9a28f4431c5ba8e5bb5d25a704bd"\n   },\n   "Limpieza/runs/capfix_20261016T184904Z_099a22b1/ledger/all_capital_changes.parquet": {\n    "uso": "marca audit_capital_cap01",\n    "sha256": "4724971a9e788854cfa6e229b9acfd6b29cdb4012f3b2d87fd93b0cf662a921c"\n   },\n   "Decision_criterios_capital_primary/runs/dec_20261017T024905Z_c6e024cb/tables/autorizadas.csv": {\n    "uso": "marca audit_capital_scale_confirmed (lista D66)",\n    "sha256": "bf307355c2170b5d843751f0c543f6ffed25f318cb2e102a0fe07476b366d7c0"\n   }\n  },\n  "bases": {\n   "tables/t0_por_row_id.parquet": {\n    "bytes": 20018041,\n    "sha256": "caa8294e1d89dca5dfdc8b3de4db3f9a6fc2cba930506ec96a6abf5c31d3b340",\n    "contenido": "3bb22d23e7b5c10111c323733a808793acabd407918d0ab8ccb59ee6dfe0f70a"\n   },\n   "tables/asignacion_por_siniestro.parquet": {\n    "bytes": 4805740,\n    "sha256": "535327e1bbba95f171820d27e345988b3c2cbdeec6f86eefefaedefc52992ae5",\n    "contenido": "f760aed27500d172209434c89967949ded6e324b7ecd8a414c34dceb065f2ed3"\n   },\n   "tables/factores_desarrollo_conteos.csv": {\n    "bytes": 23362,\n    "sha256": "e02c7aa48b476ad7626cd7b2581745149830eabab19c4d2e95894a4e94d4e4b7",\n    "contenido": "e02c7aa48b476ad7626cd7b2581745149830eabab19c4d2e95894a4e94d4e4b7"\n   },\n   "tables/ibnr_por_peligro_anio.csv": {\n    "bytes": 4243,\n    "sha256": "4e1a723c0aa5691c64384fad32d6062b074a9794d72a4138a6f020f5ea572d71",\n    "contenido": "4e1a723c0aa5691c64384fad32d6062b074a9794d72a4138a6f020f5ea572d71"\n   },\n   "tables/completitud_2024_por_mes.csv": {\n    "bytes": 8572,\n    "sha256": "34ebf9ae430f8949f54d4e1d05cc1fd855e1bb40a86c0099a482ad44a70a9ff2",\n    "contenido": "34ebf9ae430f8949f54d4e1d05cc1fd855e1bb40a86c0099a482ad44a70a9ff2"\n   },\n   "tables/chain_ladder_conteos.csv": {\n    "bytes": 2366,\n    "sha256": "d643025262cbc9781febbb94217ffeb657ceadc0f750a17ea6c64d95ec0a88ef",\n    "contenido": "d643025262cbc9781febbb94217ffeb657ceadc0f750a17ea6c64d95ec0a88ef"\n   }\n  },\n  "f0": {\n   "tables/f0_motor.parquet": {\n    "bytes": 98141422,\n    "sha256": "c503d0142e83818b2c202747ddbce53882ed9add6f8309ced714655caf25b392",\n    "contenido": "20f1a2ecfb2c012244ad8c4222ec8503e98f2efcc021c40cf1089fcc70afbbe6"\n   },\n   "tables/f0_buildings.parquet": {\n    "bytes": 66919547,\n    "sha256": "2def2e2433062704e3762eae63f5b000e73d8284a62a6cc1a689aa63c2842454",\n    "contenido": "1506da562e28d3f28a9338b93c25424bfbdbe68c0efa1010867743977b0a4362"\n   },\n   "tables/f0_contents.parquet": {\n    "bytes": 65668739,\n    "sha256": "42328ddba499a77f9cc3c814506dc97fde7c3e2adf88cfc491058c7de92c76ff",\n    "contenido": "7b06b565833323090adcff185f2b5ab869b61936df75850acb69b80fe420c190"\n   },\n   "tables/particiones.parquet": {\n    "bytes": 15848536,\n    "sha256": "d365981d07feec50e487ff0d8fa4b9e5b8a00d5af8f8e521e03f2a158931035b",\n    "contenido": "0987960a3498983daa25bbd27bd4258e3e4dbfbf372bedb33ccaa02bba6c4105"\n   }\n  },\n  "f0_contrato_bloques": "5ddb45d8e2e07d2432f77cc7abe19226322b24391a813ae9e8e7efd071602972"\n },\n "mezcla": {\n  "insumos/exposicion_motor.parquet": {\n   "bytes": 8727967,\n   "sha256": "5555d11ca01e0afa16d6872295f24030f38be01e26e278a624d5fe53c47d54b1",\n   "contenido": "7a05a4165ca5029f508e188ff5dec1139a56fdd24f0b861655f61cefc99d70d5",\n   "origen": "Hackaton/Semana 1/Modelos_base/insumos/exposicion_motor.parquet",\n   "uso": "exposición que lee el modo mezcla cuando no hay feat_<ramo>.parquet; columnas row_id, e_ef_coste y exposure"\n  },\n  "insumos/exposicion_buildings.parquet": {\n   "bytes": 6057353,\n   "sha256": "5d6a099be0d72858ef749257ed374dde80f62fe4ef07bc657df48b710f8e2673",\n   "contenido": "940ccd79edf673e1cbd7209cd8437f0fa8947cdca3f72156383833740fb124dc",\n   "origen": "Hackaton/Semana 1/Modelos_base/insumos/exposicion_buildings.parquet",\n   "uso": "exposición que lee el modo mezcla cuando no hay feat_<ramo>.parquet; columnas row_id, e_ef_coste y exposure"\n  },\n  "insumos/exposicion_contents.parquet": {\n   "bytes": 6166024,\n   "sha256": "e115a4b0565cd134f47568b5cf0c35a9a931c72bd254967c17bf7bedd586ab5c",\n   "contenido": "d1b4ed9b35bc488254e1386b71ca3ec85699339bb1411d108a5186cdd9a0f881",\n   "origen": "Hackaton/Semana 1/Modelos_base/insumos/exposicion_contents.parquet",\n   "uso": "exposición que lee el modo mezcla cuando no hay feat_<ramo>.parquet; columnas row_id, e_ef_coste y exposure"\n  }\n }\n}', 'receta/receta.json': '{\n "nombre": "Receta de S30_line_b (selección final de la parte 1) y S22_main (respaldo)",\n "version": "2026-10-19",\n "entorno_de_referencia": "entorno/entorno_referencia.json",\n "entradas": {\n  "descripcion": "Rutas relativas a la carpeta «Hackaton/Semana 1» del proyecto (variable IAA_SEMANA1). Las etapas de capa base, bases actuariales y F0 que las producen desde los once Parquet originales están en la receta de datos (etapas 00 a 02).",\n  "archivos": [\n   "Datos/weather.parquet",\n   "Datos/sample_submission.parquet",\n   "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_motor.parquet",\n   "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_buildings.parquet",\n   "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/f0_contents.parquet",\n   "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/tables/particiones.parquet",\n   "Matriz_F0/runs/f0_20261017T141732Z_65df7fca/contrato_disponibilidad.json",\n   "Bases_actuariales/runs/bases_20261017T120251Z_45443ff0/tables/asignacion_por_siniestro.parquet"\n  ]\n },\n "objetivos": {\n  "S30": {\n   "nodo": "S30_line_b",\n   "csv": "S30_line_b/S30_line_b.csv",\n   "sha256": "fd9528e0b1ee4b674dea63834a97523fd4ddc743d1495a422ce31a2eb76e3bc5"\n  },\n  "S22": {\n   "nodo": "S22_main",\n   "csv": "S22_main/S22_main.csv",\n   "sha256": "c21a02e14de4a89f892604239b7cfd9ac4e95ea7d175460a25ba806a5a3541b0"\n  }\n },\n "nodos": [\n  {\n   "id": "variables",\n   "etapa": "variables",\n   "linea": "principal",\n   "script": "src/modelos/construir_variables.py",\n   "args": [],\n   "requiere": [],\n   "salidas": [\n    "feat_motor.parquet",\n    "feat_buildings.parquet",\n    "feat_contents.parquet"\n   ],\n   "verificar": [\n    "feat_motor.parquet",\n    "feat_buildings.parquet",\n    "feat_contents.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "construido una vez el 17-10 a las 13:14 (nube); comun.py con SHA 7b475c1b… igual al de los manifiestos de S03 y S04b",\n   "nota": "F0 (D68) + asignación de siniestros (D67) + clima original; variables F1, F2 y F4 con corte al 1 de enero del año del periodo",\n   "minutos_ref": 1\n  },\n  {\n   "id": "sel_total_F0124",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "total_F0124",\n    "F0124",\n    "receta/especificaciones/cfg_total.json",\n    "total"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_total_F0124.parquet"\n   ],\n   "verificar": [\n    "sel_total_F0124.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola1.sh, log_cola1.txt",\n   "nota": "sus etiquetas y son las de V23 que usa mezcla3.labels",\n   "minutos_ref": 2\n  },\n  {\n   "id": "sel_total_F01234",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "total_F01234",\n    "F01234",\n    "receta/especificaciones/cfg_total.json",\n    "total"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_total_F01234.parquet"\n   ],\n   "verificar": [\n    "sel_total_F01234.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola1.sh, log_cola1.txt",\n   "nota": "",\n   "minutos_ref": 2\n  },\n  {\n   "id": "sel_total_motor_red",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "total_motor_red",\n    "F01234",\n    "receta/especificaciones/cfg_total_motor_red.json",\n    "total"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_total_motor_red.parquet"\n   ],\n   "verificar": [\n    "sel_total_motor_red.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola9.sh, log_cola9.txt",\n   "nota": "",\n   "minutos_ref": 2\n  },\n  {\n   "id": "sel_hS2",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "hS2",\n    "F0124",\n    "receta/especificaciones/cfg_hogar_s2.json",\n    "total",\n    "buildings,contents"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_hS2.parquet"\n   ],\n   "verificar": [\n    "sel_hS2.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola19.sh, log_hS2.txt",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "sel_hS3",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/evaluar.py",\n   "args": [\n    "hS3",\n    "F0124",\n    "receta/especificaciones/cfg_hogar_s3.json",\n    "total",\n    "buildings"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_hS3.parquet"\n   ],\n   "verificar": [\n    "sel_hS3.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola29.sh, log_hS3.txt",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "sel_cat1a",\n   "etapa": "seleccion_v23",\n   "linea": "principal",\n   "script": "src/modelos/cat_experimento.py",\n   "args": [\n    "cat1a",\n    "motor"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_cat1a.parquet"\n   ],\n   "verificar": [\n    "sel_cat1a.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola7.sh, log_cola7.txt (mejor iteración 393)",\n   "nota": "",\n   "minutos_ref": 4\n  },\n  {\n   "id": "S03_lgbm_F0124",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S03_lgbm_F0124",\n    "F0124",\n    "receta/especificaciones/cfg_s03.json",\n    "--semillas=3"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "S03_lgbm_F0124/v24.parquet",\n    "S03_lgbm_F0124/prediccion_2025.parquet",\n    "S03_lgbm_F0124/S03_lgbm_F0124.csv"\n   ],\n   "verificar": [\n    "S03_lgbm_F0124/v24.parquet",\n    "S03_lgbm_F0124/prediccion_2025.parquet",\n    "S03_lgbm_F0124/S03_lgbm_F0124.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "README de Modelos_base, «Cómo reproducir»; manifest.json de la corrida",\n   "nota": "",\n   "minutos_ref": 12\n  },\n  {\n   "id": "S04b_lgbm_F0124_F3autos",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S04b_lgbm_F0124_F3autos",\n    "F01234",\n    "receta/especificaciones/cfg_m595_f3.json",\n    "--semillas=3",\n    "--solo=motor",\n    "--reusar=S03_lgbm_F0124"\n   ],\n   "requiere": [\n    "S03_lgbm_F0124"\n   ],\n   "salidas": [\n    "S04b_lgbm_F0124_F3autos/v24.parquet",\n    "S04b_lgbm_F0124_F3autos/prediccion_2025.parquet",\n    "S04b_lgbm_F0124_F3autos/S04b_lgbm_F0124_F3autos.csv"\n   ],\n   "verificar": [\n    "S04b_lgbm_F0124_F3autos/v24.parquet",\n    "S04b_lgbm_F0124_F3autos/prediccion_2025.parquet",\n    "S04b_lgbm_F0124_F3autos/S04b_lgbm_F0124_F3autos.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola2.sh; manifest.json de la corrida",\n   "nota": "",\n   "minutos_ref": 8\n  },\n  {\n   "id": "S06c_lgbm_motor_red",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S06c_lgbm_motor_red",\n    "F01234",\n    "receta/especificaciones/cfg_motor_red.json",\n    "--semillas=3",\n    "--solo=motor",\n    "--reusar=S04b_lgbm_F0124_F3autos"\n   ],\n   "requiere": [\n    "S04b_lgbm_F0124_F3autos"\n   ],\n   "salidas": [\n    "S06c_lgbm_motor_red/v24.parquet",\n    "S06c_lgbm_motor_red/prediccion_2025.parquet",\n    "S06c_lgbm_motor_red/S06c_lgbm_motor_red.csv"\n   ],\n   "verificar": [\n    "S06c_lgbm_motor_red/v24.parquet",\n    "S06c_lgbm_motor_red/prediccion_2025.parquet",\n    "S06c_lgbm_motor_red/S06c_lgbm_motor_red.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola9.sh, log_cola9.txt",\n   "nota": "",\n   "minutos_ref": 8\n  },\n  {\n   "id": "S08h_hogar_F3s2",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S08h_hogar_F3s2",\n    "F0124",\n    "receta/especificaciones/cfg_hogar_s2.json",\n    "--solo=buildings,contents",\n    "--reusar=S04b_lgbm_F0124_F3autos",\n    "--semillas=3"\n   ],\n   "requiere": [\n    "S04b_lgbm_F0124_F3autos"\n   ],\n   "salidas": [\n    "S08h_hogar_F3s2/v24.parquet",\n    "S08h_hogar_F3s2/prediccion_2025.parquet",\n    "S08h_hogar_F3s2/S08h_hogar_F3s2.csv"\n   ],\n   "verificar": [\n    "S08h_hogar_F3s2/v24.parquet",\n    "S08h_hogar_F3s2/prediccion_2025.parquet",\n    "S08h_hogar_F3s2/S08h_hogar_F3s2.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola19.sh, log_S08h.txt",\n   "nota": "",\n   "minutos_ref": 4\n  },\n  {\n   "id": "S12h_edif_md1000",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/produccion.py",\n   "args": [\n    "S12h_edif_md1000",\n    "F0124",\n    "receta/especificaciones/cfg_hogar_s3.json",\n    "--solo=buildings",\n    "--reusar=S08h_hogar_F3s2",\n    "--semillas=3"\n   ],\n   "requiere": [\n    "S08h_hogar_F3s2"\n   ],\n   "salidas": [\n    "S12h_edif_md1000/v24.parquet",\n    "S12h_edif_md1000/prediccion_2025.parquet",\n    "S12h_edif_md1000/S12h_edif_md1000.csv"\n   ],\n   "verificar": [\n    "S12h_edif_md1000/v24.parquet",\n    "S12h_edif_md1000/prediccion_2025.parquet",\n    "S12h_edif_md1000/S12h_edif_md1000.csv"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola29.sh, log_S12h.txt",\n   "nota": "",\n   "minutos_ref": 3\n  },\n  {\n   "id": "CAT_motor",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/cat_produccion.py",\n   "args": [\n    "CAT_motor",\n    "motor",\n    "393",\n    "--semillas=2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CAT_motor/v24.parquet",\n    "CAT_motor/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CAT_motor/v24.parquet",\n    "CAT_motor/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola10.sh",\n   "nota": "",\n   "minutos_ref": 40\n  },\n  {\n   "id": "CAT_motor_s23",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/cat_produccion.py",\n   "args": [\n    "CAT_motor_s23",\n    "motor",\n    "393",\n    "--semillas=2",\n    "--semilla_inicial=2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CAT_motor_s23/v24.parquet",\n    "CAT_motor_s23/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CAT_motor_s23/v24.parquet",\n    "CAT_motor_s23/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola30.sh y cola31.sh, log_cat_s23.txt",\n   "nota": "",\n   "minutos_ref": 40\n  },\n  {\n   "id": "CAT_motor_4s",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/combinar_semillas.py",\n   "args": [\n    "CAT_motor_4s",\n    "CAT_motor",\n    "CAT_motor_s23"\n   ],\n   "requiere": [\n    "CAT_motor",\n    "CAT_motor_s23"\n   ],\n   "salidas": [\n    "CAT_motor_4s/v24.parquet",\n    "CAT_motor_4s/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CAT_motor_4s/v24.parquet",\n    "CAT_motor_4s/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "resumen.json de CAT_motor_4s (corridas CAT_motor y CAT_motor_s23, semillas [2, 2])",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "CFS_motor_v23",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_motor_v23",\n    "motor"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CFS_motor_v23/v23_own_damage.npz",\n    "CFS_motor_v23/v23_third_party_prop.npz",\n    "CFS_motor_v23/v23_bodily_injury.npz",\n    "CFS_motor_v23/v23_theft.npz",\n    "CFS_motor_v23/v23_glass_hail.npz",\n    "sel_CFS_motor_v23.parquet"\n   ],\n   "verificar": [\n    "CFS_motor_v23/v23_own_damage.npz",\n    "CFS_motor_v23/v23_third_party_prop.npz",\n    "CFS_motor_v23/v23_bodily_injury.npz",\n    "CFS_motor_v23/v23_theft.npz",\n    "CFS_motor_v23/v23_glass_hail.npz",\n    "sel_CFS_motor_v23.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola37.sh, log_cfs_motor_v23.txt",\n   "nota": "",\n   "minutos_ref": 60\n  },\n  {\n   "id": "CFS_motor",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_motor",\n    "motor",\n    "--prod",\n    "--semillas=2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CFS_motor/v23_own_damage.npz",\n    "CFS_motor/v23_third_party_prop.npz",\n    "CFS_motor/v23_bodily_injury.npz",\n    "CFS_motor/v23_theft.npz",\n    "CFS_motor/v23_glass_hail.npz",\n    "CFS_motor/v24_own_damage.npy",\n    "CFS_motor/v24_third_party_prop.npy",\n    "CFS_motor/v24_bodily_injury.npy",\n    "CFS_motor/v24_theft.npy",\n    "CFS_motor/v24_glass_hail.npy",\n    "CFS_motor/prod_own_damage.npy",\n    "CFS_motor/prod_third_party_prop.npy",\n    "CFS_motor/prod_bodily_injury.npy",\n    "CFS_motor/prod_theft.npy",\n    "CFS_motor/prod_glass_hail.npy",\n    "sel_CFS_motor.parquet",\n    "CFS_motor/v24.parquet",\n    "CFS_motor/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_motor/prod_own_damage.npy",\n    "CFS_motor/prod_third_party_prop.npy",\n    "CFS_motor/prod_bodily_injury.npy",\n    "CFS_motor/prod_theft.npy",\n    "CFS_motor/prod_glass_hail.npy",\n    "CFS_motor/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola23.sh y cola25.sh (V23), cola29.sh (--prod), log_cfs_motor.txt y log_cfs_motor_prod.txt",\n   "nota": "la V23 y la V24 originales de esta corrida se calcularon en float64, antes del paso a float32 que documenta cat_frecsev.py; aguas abajo solo se usan sus partes de 2025 (hechas ya en float32) y sus mejores iteraciones, iguales a las de CFS_motor_v23",\n   "minutos_ref": 150\n  },\n  {\n   "id": "CFS_motor_s23",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_motor_s23",\n    "motor",\n    "--prod",\n    "--semillas=2",\n    "--semilla_inicial=2",\n    "--desde=CFS_motor"\n   ],\n   "requiere": [\n    "CFS_motor"\n   ],\n   "salidas": [\n    "CFS_motor_s23/v24_own_damage.npy",\n    "CFS_motor_s23/v24_third_party_prop.npy",\n    "CFS_motor_s23/v24_bodily_injury.npy",\n    "CFS_motor_s23/v24_theft.npy",\n    "CFS_motor_s23/v24_glass_hail.npy",\n    "CFS_motor_s23/prod_own_damage.npy",\n    "CFS_motor_s23/prod_third_party_prop.npy",\n    "CFS_motor_s23/prod_bodily_injury.npy",\n    "CFS_motor_s23/prod_theft.npy",\n    "CFS_motor_s23/prod_glass_hail.npy",\n    "CFS_motor_s23/v24.parquet",\n    "CFS_motor_s23/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_motor_s23/v24_own_damage.npy",\n    "CFS_motor_s23/v24_third_party_prop.npy",\n    "CFS_motor_s23/v24_bodily_injury.npy",\n    "CFS_motor_s23/v24_theft.npy",\n    "CFS_motor_s23/v24_glass_hail.npy",\n    "CFS_motor_s23/prod_own_damage.npy",\n    "CFS_motor_s23/prod_third_party_prop.npy",\n    "CFS_motor_s23/prod_bodily_injury.npy",\n    "CFS_motor_s23/prod_theft.npy",\n    "CFS_motor_s23/prod_glass_hail.npy",\n    "CFS_motor_s23/v24.parquet",\n    "CFS_motor_s23/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola32.sh, log_cfs_motor_s23.txt (5.736 s)",\n   "nota": "",\n   "minutos_ref": 96\n  },\n  {\n   "id": "CFS_buildings",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_buildings",\n    "buildings",\n    "--prod",\n    "--semillas=2",\n    "--f3s2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "CFS_buildings/v23_storm.npz",\n    "CFS_buildings/v23_flood.npz",\n    "CFS_buildings/v23_escape_of_water.npz",\n    "CFS_buildings/v23_subsidence.npz",\n    "CFS_buildings/v23_fire.npz",\n    "CFS_buildings/v23_accidental_damage.npz",\n    "CFS_buildings/v24_storm.npy",\n    "CFS_buildings/v24_flood.npy",\n    "CFS_buildings/v24_escape_of_water.npy",\n    "CFS_buildings/v24_subsidence.npy",\n    "CFS_buildings/v24_fire.npy",\n    "CFS_buildings/v24_accidental_damage.npy",\n    "CFS_buildings/prod_storm.npy",\n    "CFS_buildings/prod_flood.npy",\n    "CFS_buildings/prod_escape_of_water.npy",\n    "CFS_buildings/prod_subsidence.npy",\n    "CFS_buildings/prod_fire.npy",\n    "CFS_buildings/prod_accidental_damage.npy",\n    "sel_CFS_buildings.parquet",\n    "CFS_buildings/v24.parquet",\n    "CFS_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_buildings/v23_storm.npz",\n    "CFS_buildings/v23_flood.npz",\n    "CFS_buildings/v23_escape_of_water.npz",\n    "CFS_buildings/v23_subsidence.npz",\n    "CFS_buildings/v23_fire.npz",\n    "CFS_buildings/v23_accidental_damage.npz",\n    "CFS_buildings/v24_storm.npy",\n    "CFS_buildings/v24_flood.npy",\n    "CFS_buildings/v24_escape_of_water.npy",\n    "CFS_buildings/v24_subsidence.npy",\n    "CFS_buildings/v24_fire.npy",\n    "CFS_buildings/v24_accidental_damage.npy",\n    "CFS_buildings/prod_storm.npy",\n    "CFS_buildings/prod_flood.npy",\n    "CFS_buildings/prod_escape_of_water.npy",\n    "CFS_buildings/prod_subsidence.npy",\n    "CFS_buildings/prod_fire.npy",\n    "CFS_buildings/prod_accidental_damage.npy",\n    "sel_CFS_buildings.parquet",\n    "CFS_buildings/v24.parquet",\n    "CFS_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola31.sh, log_cfs_buildings.txt (2.606 s)",\n   "nota": "",\n   "minutos_ref": 44\n  },\n  {\n   "id": "CFS_buildings_s23",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/cat_frecsev.py",\n   "args": [\n    "CFS_buildings_s23",\n    "buildings",\n    "--prod",\n    "--semillas=2",\n    "--f3s2",\n    "--desde=CFS_buildings",\n    "--semilla_inicial=2"\n   ],\n   "requiere": [\n    "CFS_buildings"\n   ],\n   "salidas": [\n    "CFS_buildings_s23/v24_storm.npy",\n    "CFS_buildings_s23/v24_flood.npy",\n    "CFS_buildings_s23/v24_escape_of_water.npy",\n    "CFS_buildings_s23/v24_subsidence.npy",\n    "CFS_buildings_s23/v24_fire.npy",\n    "CFS_buildings_s23/v24_accidental_damage.npy",\n    "CFS_buildings_s23/prod_storm.npy",\n    "CFS_buildings_s23/prod_flood.npy",\n    "CFS_buildings_s23/prod_escape_of_water.npy",\n    "CFS_buildings_s23/prod_subsidence.npy",\n    "CFS_buildings_s23/prod_fire.npy",\n    "CFS_buildings_s23/prod_accidental_damage.npy",\n    "CFS_buildings_s23/v24.parquet",\n    "CFS_buildings_s23/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_buildings_s23/v24_storm.npy",\n    "CFS_buildings_s23/v24_flood.npy",\n    "CFS_buildings_s23/v24_escape_of_water.npy",\n    "CFS_buildings_s23/v24_subsidence.npy",\n    "CFS_buildings_s23/v24_fire.npy",\n    "CFS_buildings_s23/v24_accidental_damage.npy",\n    "CFS_buildings_s23/prod_storm.npy",\n    "CFS_buildings_s23/prod_flood.npy",\n    "CFS_buildings_s23/prod_escape_of_water.npy",\n    "CFS_buildings_s23/prod_subsidence.npy",\n    "CFS_buildings_s23/prod_fire.npy",\n    "CFS_buildings_s23/prod_accidental_damage.npy",\n    "CFS_buildings_s23/v24.parquet",\n    "CFS_buildings_s23/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola42.sh, log_cfs_buildings_s23.txt (2.205 s)",\n   "nota": "",\n   "minutos_ref": 37\n  },\n  {\n   "id": "FSA_contents",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSA_contents",\n    "contents",\n    "--prod",\n    "--semillas=2",\n    "--aditivo"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_FSA_contents.parquet",\n    "FSA_contents/v24.parquet",\n    "FSA_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_FSA_contents.parquet",\n    "FSA_contents/v24.parquet",\n    "FSA_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola33.sh, log_fsa_contents.txt",\n   "nota": "",\n   "minutos_ref": 20\n  },\n  {\n   "id": "FSAP_buildings",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSAP_buildings",\n    "buildings",\n    "--prod",\n    "--semillas=2",\n    "--aditivo",\n    "--partes"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "FSAP_buildings/v23_storm.npz",\n    "FSAP_buildings/v23_flood.npz",\n    "FSAP_buildings/v23_escape_of_water.npz",\n    "FSAP_buildings/v23_subsidence.npz",\n    "FSAP_buildings/v23_fire.npz",\n    "FSAP_buildings/v23_accidental_damage.npz",\n    "FSAP_buildings/v24_storm.npy",\n    "FSAP_buildings/v24_flood.npy",\n    "FSAP_buildings/v24_escape_of_water.npy",\n    "FSAP_buildings/v24_subsidence.npy",\n    "FSAP_buildings/v24_fire.npy",\n    "FSAP_buildings/v24_accidental_damage.npy",\n    "FSAP_buildings/prod_storm.npy",\n    "FSAP_buildings/prod_flood.npy",\n    "FSAP_buildings/prod_escape_of_water.npy",\n    "FSAP_buildings/prod_subsidence.npy",\n    "FSAP_buildings/prod_fire.npy",\n    "FSAP_buildings/prod_accidental_damage.npy",\n    "sel_FSAP_buildings.parquet",\n    "FSAP_buildings/v24.parquet",\n    "FSAP_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "FSAP_buildings/v23_storm.npz",\n    "FSAP_buildings/v23_flood.npz",\n    "FSAP_buildings/v23_escape_of_water.npz",\n    "FSAP_buildings/v23_subsidence.npz",\n    "FSAP_buildings/v23_fire.npz",\n    "FSAP_buildings/v23_accidental_damage.npz",\n    "FSAP_buildings/v24_storm.npy",\n    "FSAP_buildings/v24_flood.npy",\n    "FSAP_buildings/v24_escape_of_water.npy",\n    "FSAP_buildings/v24_subsidence.npy",\n    "FSAP_buildings/v24_fire.npy",\n    "FSAP_buildings/v24_accidental_damage.npy",\n    "FSAP_buildings/prod_storm.npy",\n    "FSAP_buildings/prod_flood.npy",\n    "FSAP_buildings/prod_escape_of_water.npy",\n    "FSAP_buildings/prod_subsidence.npy",\n    "FSAP_buildings/prod_fire.npy",\n    "FSAP_buildings/prod_accidental_damage.npy",\n    "sel_FSAP_buildings.parquet",\n    "FSAP_buildings/v24.parquet",\n    "FSAP_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola39.sh, log_fsap.txt",\n   "nota": "",\n   "minutos_ref": 25\n  },\n  {\n   "id": "FSAP2_buildings",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSAP2_buildings",\n    "buildings",\n    "--prod",\n    "--semillas=2",\n    "--aditivo",\n    "--partes",\n    "--desde=FSAP_buildings",\n    "--semilla_inicial=2"\n   ],\n   "requiere": [\n    "FSAP_buildings"\n   ],\n   "salidas": [\n    "FSAP2_buildings/v24_storm.npy",\n    "FSAP2_buildings/v24_flood.npy",\n    "FSAP2_buildings/v24_escape_of_water.npy",\n    "FSAP2_buildings/v24_subsidence.npy",\n    "FSAP2_buildings/v24_fire.npy",\n    "FSAP2_buildings/v24_accidental_damage.npy",\n    "FSAP2_buildings/prod_storm.npy",\n    "FSAP2_buildings/prod_flood.npy",\n    "FSAP2_buildings/prod_escape_of_water.npy",\n    "FSAP2_buildings/prod_subsidence.npy",\n    "FSAP2_buildings/prod_fire.npy",\n    "FSAP2_buildings/prod_accidental_damage.npy",\n    "FSAP2_buildings/v24.parquet",\n    "FSAP2_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "FSAP2_buildings/v24_storm.npy",\n    "FSAP2_buildings/v24_flood.npy",\n    "FSAP2_buildings/v24_escape_of_water.npy",\n    "FSAP2_buildings/v24_subsidence.npy",\n    "FSAP2_buildings/v24_fire.npy",\n    "FSAP2_buildings/v24_accidental_damage.npy",\n    "FSAP2_buildings/prod_storm.npy",\n    "FSAP2_buildings/prod_flood.npy",\n    "FSAP2_buildings/prod_escape_of_water.npy",\n    "FSAP2_buildings/prod_subsidence.npy",\n    "FSAP2_buildings/prod_fire.npy",\n    "FSAP2_buildings/prod_accidental_damage.npy",\n    "FSAP2_buildings/v24.parquet",\n    "FSAP2_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola40.sh y cola40b.sh, log_fsap2.txt",\n   "nota": "",\n   "minutos_ref": 15\n  },\n  {\n   "id": "FSAP_contents",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSAP_contents",\n    "contents",\n    "--prod",\n    "--semillas=2",\n    "--aditivo",\n    "--partes"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "FSAP_contents/v23_theft.npz",\n    "FSAP_contents/v23_escape_of_water.npz",\n    "FSAP_contents/v23_accidental_damage.npz",\n    "FSAP_contents/v23_flood.npz",\n    "FSAP_contents/v24_theft.npy",\n    "FSAP_contents/v24_escape_of_water.npy",\n    "FSAP_contents/v24_accidental_damage.npy",\n    "FSAP_contents/v24_flood.npy",\n    "FSAP_contents/prod_theft.npy",\n    "FSAP_contents/prod_escape_of_water.npy",\n    "FSAP_contents/prod_accidental_damage.npy",\n    "FSAP_contents/prod_flood.npy",\n    "sel_FSAP_contents.parquet",\n    "FSAP_contents/v24.parquet",\n    "FSAP_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "FSAP_contents/v23_theft.npz",\n    "FSAP_contents/v23_escape_of_water.npz",\n    "FSAP_contents/v23_accidental_damage.npz",\n    "FSAP_contents/v23_flood.npz",\n    "FSAP_contents/v24_theft.npy",\n    "FSAP_contents/v24_escape_of_water.npy",\n    "FSAP_contents/v24_accidental_damage.npy",\n    "FSAP_contents/v24_flood.npy",\n    "FSAP_contents/prod_theft.npy",\n    "FSAP_contents/prod_escape_of_water.npy",\n    "FSAP_contents/prod_accidental_damage.npy",\n    "FSAP_contents/prod_flood.npy",\n    "sel_FSAP_contents.parquet",\n    "FSAP_contents/v24.parquet",\n    "FSAP_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola39.sh, log_fsap.txt",\n   "nota": "",\n   "minutos_ref": 25\n  },\n  {\n   "id": "FSAP2_contents",\n   "etapa": "partes",\n   "linea": "principal",\n   "script": "src/modelos/frecsev2.py",\n   "args": [\n    "FSAP2_contents",\n    "contents",\n    "--prod",\n    "--semillas=2",\n    "--aditivo",\n    "--partes",\n    "--desde=FSAP_contents",\n    "--semilla_inicial=2"\n   ],\n   "requiere": [\n    "FSAP_contents"\n   ],\n   "salidas": [\n    "FSAP2_contents/v24_theft.npy",\n    "FSAP2_contents/v24_escape_of_water.npy",\n    "FSAP2_contents/v24_accidental_damage.npy",\n    "FSAP2_contents/v24_flood.npy",\n    "FSAP2_contents/prod_theft.npy",\n    "FSAP2_contents/prod_escape_of_water.npy",\n    "FSAP2_contents/prod_accidental_damage.npy",\n    "FSAP2_contents/prod_flood.npy",\n    "FSAP2_contents/v24.parquet",\n    "FSAP2_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "FSAP2_contents/v24_theft.npy",\n    "FSAP2_contents/v24_escape_of_water.npy",\n    "FSAP2_contents/v24_accidental_damage.npy",\n    "FSAP2_contents/v24_flood.npy",\n    "FSAP2_contents/prod_theft.npy",\n    "FSAP2_contents/prod_escape_of_water.npy",\n    "FSAP2_contents/prod_accidental_damage.npy",\n    "FSAP2_contents/prod_flood.npy",\n    "FSAP2_contents/v24.parquet",\n    "FSAP2_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "cola40.sh y cola40b.sh, log_fsap2.txt",\n   "nota": "",\n   "minutos_ref": 15\n  },\n  {\n   "id": "FSAXc4_contents",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/severidad_cfs3.py",\n   "args": [\n    "FSAP_contents",\n    "contents",\n    "FSAXc4_contents",\n    "--v24=FSAP_contents,FSAP2_contents",\n    "--prod=FSAP_contents,FSAP2_contents"\n   ],\n   "requiere": [\n    "FSAP_contents",\n    "FSAP2_contents"\n   ],\n   "salidas": [\n    "sel_FSAXc4_contents.parquet",\n    "FSAXc4_contents/v24.parquet",\n    "FSAXc4_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_FSAXc4_contents.parquet",\n    "FSAXc4_contents/v24.parquet",\n    "FSAXc4_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22",\n    "S30"\n   ],\n   "evidencia": "resumen.json de FSAXc4_contents (origen v23 FSAP_contents; v24 y prod FSAP_contents y FSAP2_contents; lambda 20)",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "CFS_motor_4s",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/combinar_semillas.py",\n   "args": [\n    "CFS_motor_4s",\n    "CFS_motor",\n    "CFS_motor_s23"\n   ],\n   "requiere": [\n    "CFS_motor",\n    "CFS_motor_s23"\n   ],\n   "salidas": [\n    "CFS_motor_4s/v24.parquet",\n    "CFS_motor_4s/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "CFS_motor_4s/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "finalizar_s15.sh",\n   "nota": "solo hace falta su resumen.json, que severidad_cfs.py lee como referencia; su V24 hereda la V24 en float64 de CFS_motor y no se compara",\n   "minutos_ref": 1\n  },\n  {\n   "id": "CFSS_motor",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/severidad_cfs.py",\n   "args": [\n    "CFS_motor_4s",\n    "motor",\n    "CFSS_motor",\n    "--v23=CFS_motor_v23",\n    "--v24=CFS_motor_s23",\n    "--prod=CFS_motor,CFS_motor_s23"\n   ],\n   "requiere": [\n    "CFS_motor_4s",\n    "CFS_motor_v23"\n   ],\n   "salidas": [\n    "sel_CFSS_motor.parquet",\n    "CFSS_motor/v24.parquet",\n    "CFSS_motor/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_CFSS_motor.parquet",\n    "CFSS_motor/v24.parquet",\n    "CFSS_motor/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "resumen.json de CFSS_motor (origen CFS_motor_4s); misma orden que CFSSr_motor de cola45.sh sin --tope",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "CFSS4_buildings",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/severidad_cfs.py",\n   "args": [\n    "CFS_buildings",\n    "buildings",\n    "CFSS4_buildings",\n    "--v24=CFS_buildings,CFS_buildings_s23",\n    "--prod=CFS_buildings,CFS_buildings_s23"\n   ],\n   "requiere": [\n    "CFS_buildings_s23"\n   ],\n   "salidas": [\n    "sel_CFSS4_buildings.parquet",\n    "CFSS4_buildings/v24.parquet",\n    "CFSS4_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_CFSS4_buildings.parquet",\n    "CFSS4_buildings/v24.parquet",\n    "CFSS4_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "cola42.sh",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "FSAS4_buildings",\n   "etapa": "severidad",\n   "linea": "principal",\n   "script": "src/modelos/severidad_cfs.py",\n   "args": [\n    "FSAP_buildings",\n    "buildings",\n    "FSAS4_buildings",\n    "--v24=FSAP_buildings,FSAP2_buildings",\n    "--prod=FSAP_buildings,FSAP2_buildings"\n   ],\n   "requiere": [\n    "FSAP_buildings",\n    "FSAP2_buildings"\n   ],\n   "salidas": [\n    "sel_FSAS4_buildings.parquet",\n    "FSAS4_buildings/v24.parquet",\n    "FSAS4_buildings/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_FSAS4_buildings.parquet",\n    "FSAS4_buildings/v24.parquet",\n    "FSAS4_buildings/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "cola40.sh",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "MLP3_contents",\n   "etapa": "produccion",\n   "linea": "principal",\n   "script": "src/modelos/mlp_tweedie.py",\n   "args": [\n    "MLP3_contents",\n    "contents",\n    "--semillas=3",\n    "--v24",\n    "--prod",\n    "--wd=0.0001",\n    "--f3s2"\n   ],\n   "requiere": [\n    "variables"\n   ],\n   "salidas": [\n    "sel_MLP3_contents.parquet",\n    "MLP3_contents/v24.parquet",\n    "MLP3_contents/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_MLP3_contents.parquet",\n    "MLP3_contents/v24.parquet",\n    "MLP3_contents/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "cola21.sh, log_mlp3_contents.txt",\n   "nota": "red neuronal con torch en CPU; solo entra en S22 (peso 0,1)",\n   "minutos_ref": 3\n  },\n  {\n   "id": "B_CFST_motor_tendencia",\n   "etapa": "linea_b",\n   "linea": "B",\n   "script": "src/linea_b/severidad_tendencia.py",\n   "args": [\n    "motor",\n    "B_CFST_motor_tendencia",\n    "--regla=tendencia",\n    "--v23=CFS_motor_v23",\n    "--v24=CFS_motor_s23",\n    "--prod=CFS_motor,CFS_motor_s23"\n   ],\n   "requiere": [\n    "CFS_motor",\n    "CFS_motor_s23",\n    "CFS_motor_v23"\n   ],\n   "salidas": [\n    "sel_B_CFST_motor_tendencia.parquet",\n    "B_CFST_motor_tendencia/v24.parquet",\n    "B_CFST_motor_tendencia/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_B_CFST_motor_tendencia.parquet",\n    "B_CFST_motor_tendencia/v24.parquet",\n    "B_CFST_motor_tendencia/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S30"\n   ],\n   "evidencia": "resumen.json de B_CFST_motor_tendencia (regla tendencia; v23 CFS_motor_v23; v24 CFS_motor_s23; prod CFS_motor,CFS_motor_s23); Modelos_line_b/README.md §3",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "B_CFST_buildings_tendencia",\n   "etapa": "linea_b",\n   "linea": "B",\n   "script": "src/linea_b/severidad_tendencia.py",\n   "args": [\n    "buildings",\n    "B_CFST_buildings_tendencia",\n    "--regla=tendencia",\n    "--v23=CFS_buildings",\n    "--v24=CFS_buildings,CFS_buildings_s23",\n    "--prod=CFS_buildings,CFS_buildings_s23"\n   ],\n   "requiere": [\n    "CFS_buildings",\n    "CFS_buildings_s23"\n   ],\n   "salidas": [\n    "sel_B_CFST_buildings_tendencia.parquet",\n    "B_CFST_buildings_tendencia/v24.parquet",\n    "B_CFST_buildings_tendencia/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_B_CFST_buildings_tendencia.parquet",\n    "B_CFST_buildings_tendencia/v24.parquet",\n    "B_CFST_buildings_tendencia/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S30"\n   ],\n   "evidencia": "resumen.json de B_CFST_buildings_tendencia (regla tendencia; v23 CFS_buildings; v24 CFS_buildings,CFS_buildings_s23; prod CFS_buildings,CFS_buildings_s23); Modelos_line_b/README.md §3",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "B_FSAST_buildings_tendencia",\n   "etapa": "linea_b",\n   "linea": "B",\n   "script": "src/linea_b/severidad_tendencia.py",\n   "args": [\n    "buildings",\n    "B_FSAST_buildings_tendencia",\n    "--regla=tendencia",\n    "--v23=FSAP_buildings",\n    "--v24=FSAP_buildings,FSAP2_buildings",\n    "--prod=FSAP_buildings,FSAP2_buildings"\n   ],\n   "requiere": [\n    "FSAP2_buildings",\n    "FSAP_buildings"\n   ],\n   "salidas": [\n    "sel_B_FSAST_buildings_tendencia.parquet",\n    "B_FSAST_buildings_tendencia/v24.parquet",\n    "B_FSAST_buildings_tendencia/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "sel_B_FSAST_buildings_tendencia.parquet",\n    "B_FSAST_buildings_tendencia/v24.parquet",\n    "B_FSAST_buildings_tendencia/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S30"\n   ],\n   "evidencia": "resumen.json de B_FSAST_buildings_tendencia (regla tendencia; v23 FSAP_buildings; v24 FSAP_buildings,FSAP2_buildings; prod FSAP_buildings,FSAP2_buildings); Modelos_line_b/README.md §3",\n   "nota": "",\n   "minutos_ref": 1\n  },\n  {\n   "id": "S22_main",\n   "etapa": "mezcla",\n   "linea": "principal",\n   "script": "src/modelos/mezcla3.py",\n   "args": [\n    "S22_main",\n    "receta/especificaciones/spec_s22_propio.json",\n    "--forma=motor:1.09:0.95",\n    "--nivel=s17",\n    "--escribir"\n   ],\n   "requiere": [\n    "sel_total_F0124",\n    "S04b_lgbm_F0124_F3autos",\n    "CAT_motor_4s",\n    "sel_cat1a",\n    "CFSS_motor",\n    "sel_total_F01234",\n    "S06c_lgbm_motor_red",\n    "sel_total_motor_red",\n    "S12h_edif_md1000",\n    "sel_hS3",\n    "CFSS4_buildings",\n    "FSAS4_buildings",\n    "S08h_hogar_F3s2",\n    "sel_hS2",\n    "MLP3_contents",\n    "FSA_contents",\n    "FSAXc4_contents"\n   ],\n   "salidas": [\n    "S22_main/S22_main.csv",\n    "S22_main/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "S22_main/S22_main.csv",\n    "S22_main/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S22"\n   ],\n   "evidencia": "finalizar_s22.sh; D86",\n   "nota": "SHA-256 entregado c21a02e14de4a89f892604239b7cfd9ac4e95ea7d175460a25ba806a5a3541b0 (Kaggle 57370117, público 0,12988)",\n   "minutos_ref": 1\n  },\n  {\n   "id": "S30_line_b",\n   "etapa": "mezcla",\n   "linea": "B",\n   "script": "src/linea_b/mezcla3_b.py",\n   "args": [\n    "S30_line_b",\n    "receta/especificaciones/spec_s30.json",\n    "--forma=motor:1.09:0.95,contents:1:0.95",\n    "--nivel=s17",\n    "--escribir"\n   ],\n   "requiere": [\n    "sel_total_F0124",\n    "S04b_lgbm_F0124_F3autos",\n    "CAT_motor_4s",\n    "sel_cat1a",\n    "B_CFST_motor_tendencia",\n    "sel_total_F01234",\n    "S06c_lgbm_motor_red",\n    "sel_total_motor_red",\n    "S12h_edif_md1000",\n    "sel_hS3",\n    "B_CFST_buildings_tendencia",\n    "B_FSAST_buildings_tendencia",\n    "S08h_hogar_F3s2",\n    "sel_hS2",\n    "FSA_contents",\n    "FSAXc4_contents"\n   ],\n   "salidas": [\n    "S30_line_b/S30_line_b.csv",\n    "S30_line_b/prediccion_2025.parquet"\n   ],\n   "verificar": [\n    "S30_line_b/S30_line_b.csv",\n    "S30_line_b/prediccion_2025.parquet"\n   ],\n   "objetivos": [\n    "S30"\n   ],\n   "evidencia": "Modelos_line_b/README.md («Candidato S30_line_b»); reproducir_s30_b.py; D88 y D89",\n   "nota": "SHA-256 entregado fd9528e0b1ee4b674dea63834a97523fd4ddc743d1495a422ce31a2eb76e3bc5 (Kaggle 57371361, público 0,12996); selección final de la parte 1",\n   "minutos_ref": 1\n  }\n ],\n "etapas_datos": {\n  "descripcion": "Etapas 00 a 03, de los once Parquet originales a las variables. reproducir.py datos las ejecuta en una raíz de proyecto mínima dentro de --trabajo, porque bases_actuariales.py y construir_f0.py localizan sus entradas subiendo hasta RUTAS_PROYECTO.json y escriben su corrida junto al script; los scripts se copian allí sin cambios y se comprueba su SHA-256. Las corridas nuevas se enlazan con los nombres que espera comun.py (f0_20261017T141732Z_65df7fca y bases_20261017T120251Z_45443ff0).",\n  "pasos": [\n   {\n    "id": "capa",\n    "script": "src/datos/n62/rebuild_base.py",\n    "args": [\n     "--raw",\n     "<originales>",\n     "--rules",\n     "src/datos/n62/rules.json",\n     "--out",\n     "<trabajo>/proyecto/Hackaton/Semana 1/Limpieza/runs/preaudit_20261016T211816Z_817285a5"\n    ],\n    "autor": "reference (N62, COD-20261018-104)",\n    "verifica": "SHA-256 de las once tablas igual al de la capa preaudit"\n   },\n   {\n    "id": "bases",\n    "script": "src/datos/bases_actuariales.py",\n    "args": [],\n    "autor": "primary (D67)",\n    "verifica": "huella de contenido de t0_por_row_id y asignacion_por_siniestro; bytes de las tablas de desarrollo"\n   },\n   {\n    "id": "f0",\n    "script": "src/datos/construir_f0.py",\n    "args": [\n     "--ramo motor",\n     "--run <id> --ramo buildings",\n     "--run <id> --ramo contents",\n     "--run <id> --finalizar"\n    ],\n    "autor": "primary (D68)",\n    "verifica": "huella de contenido de las tres matrices y de las particiones; bloques del contrato de disponibilidad",\n    "decisiones_congeladas": "tres listas de row_id del proyecto que solo alimentan marcas audit_* (excluidas de los predictores), leídas de --decisiones y comprobadas por SHA-256"\n   },\n   {\n    "id": "variables",\n    "script": "src/modelos/construir_variables.py",\n    "args": [],\n    "autor": "primary",\n    "verifica": "huella de contenido de feat_<ramo>.parquet"\n   }\n  ],\n  "prueba_19_oct": "En el entorno de referencia: once tablas con el SHA de preaudit; bases y F0 iguales en contenido; variables iguales en contenido y marcos de modelo (matriz X con tipos y categorías, etiquetas, offsets y particiones) idénticos a los de las variables originales. Los bytes de F0 y de las variables cambian porque pandas 3 guarda las fechas de periodo en segundos y la VM las guardó en nanosegundos."\n }\n}\n', 'src/datos/n62/rules.json': '{\n  "version": "N62_frozen_preaudit_v1",\n  "originals_sha256": {\n    "addresses.parquet": "2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1",\n    "claims.parquet": "323adf2e0f6c5975a026479d725d4cd04f51fce3aa069f897a99f664c6e392c0",\n    "customers.parquet": "065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb",\n    "policies_buildings.parquet": "eb84372c23c16ed0619d89c8eb1396ab6916e5ec46a5907ce42c3dfce4a91cbe",\n    "policies_contents.parquet": "8c067cc80baed583100df999e1c3d5e389295702f7bac21d5331bbd639b83fcd",\n    "policies_motor.parquet": "4d2d7ae03971d5d6df3a71bbaf0ebc168250a6807b27d1561d38b638434dc95e",\n    "sample_submission.parquet": "9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95",\n    "test_buildings.parquet": "7b2b96507de29241c70617c17df66146ed30b51f826ce5219bbda5da536d4957",\n    "test_contents.parquet": "c2212f62f3b7a21431c0073277549c3db5e63857565a8b88855e79a65fa1b8b6",\n    "test_motor.parquet": "8d28fb9fd091caf2a044913a013592226a4c0917d5c6fcc9c5e8cdcec2ab4fc5",\n    "weather.parquet": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49"\n  },\n  "ID01": {\n    "pattern": "[0-9]{1,8}",\n    "width": 8\n  },\n  "EXP01": {\n    "inclusive_days": true,\n    "denominator": 365,\n    "decimals": 4\n  },\n  "CAP01": {\n    "thresholds": {\n      "buildings": 10000000,\n      "contents": 1000000\n    },\n    "scale": 1000,\n    "context_fields": {\n      "buildings": [\n        "customer_id",\n        "address_id",\n        "construction_type",\n        "occupancy",\n        "accidental_damage_cover",\n        "voluntary_excess",\n        "compulsory_excess",\n        "smoke_alarms",\n        "mortgage"\n      ],\n      "contents": [\n        "customer_id",\n        "address_id",\n        "tenure",\n        "security_level",\n        "high_value_items",\n        "accidental_damage_cover",\n        "works_from_home",\n        "lodgers",\n        "bicycle_cover",\n        "voluntary_excess",\n        "compulsory_excess"\n      ]\n    },\n    "approved_sets": {\n      "policies_buildings": {\n        "count": 802,\n        "row_ids_sha256": "b262deab1c0cb371635c1b042032708b2ba42324e9c706e6c43e4fdc7cb95c20"\n      },\n      "policies_contents": {\n        "count": 731,\n        "row_ids_sha256": "af19838c5321bc453d20c733ca9c60a91076f2cef8def69bd54bf2039014e8f0"\n      },\n      "test_buildings": {\n        "count": 272,\n        "row_ids_sha256": "9f8490d0fb51aeb9b311e3357d11b00d5086ff3fe28a14b410198af545144ce6"\n      },\n      "test_contents": {\n        "count": 268,\n        "row_ids_sha256": "0bed421cb45f521b29b90be0d2850051e9b2b414416e3d97924c95ca71596b15"\n      }\n    },\n    "selection": "Most recent strictly earlier non-extreme original anchor; complete same context; start gap1..400; divided capital ratio0.9..1.1. Never reselect after correcting."\n  },\n  "CAT01": {\n    "map": {\n      "Brick": "brick",\n      "BRICK": "brick",\n      "NON_STANDARD": "non_standard",\n      "STONE": "stone",\n      "T": "t",\n      "W": "w"\n    }\n  },\n  "AGE01": {\n    "calendar_completed": true,\n    "feb29_on_feb28_nonleap": "abort",\n    "maximum_source_difference": 1\n  },\n  "DUP01": {\n    "count": 604,\n    "pairs_sha256": "ad61daaa2e217bbf3259932a72cfae5ec8c9931dc53c30294f414e5178396689"\n  },\n  "REN00": {\n    "cohort_source_file_sha256": "4e68ecadb8a00976a1601b5067ce73c87a43c74069cb542c1cafbdf8d9af764f",\n    "approved_sets": {\n      "buildings": {\n        "count": 1010,\n        "policy_ids_sha256": "1b68021fc8111739d1bece6eafe3f6acc8bc1dd54098c83bf5a9f5ec6f9efcdf"\n      },\n      "contents": {\n        "count": 219,\n        "policy_ids_sha256": "64da56e61f813b75b58eb3e08451dee91b0d7c9359d853bb90372eef7b604a94"\n      },\n      "motor": {\n        "count": 1513,\n        "policy_ids_sha256": "049e7b3480b1f35612007d3e580bfe3544991d164334fcebe6c5ab9bc12dfc84"\n      }\n    }\n  },\n  "PRE00": {\n    "cohort_source_file_sha256": "90d1cba5de82320aae81966aa1d248446cb2c758d6880378bbde855a4d1226bf",\n    "count": 413,\n    "claim_ids_sha256": "f31e2e94e7bb9d9bca6f50929be016275be72848eb6a2afa33976ef7f14d471c"\n  }\n}\n'}


### Inspect the dependency graph

Each row is a stage needed for S30. The `requires` column shows which earlier stages supply its inputs. This graph is frozen; inspecting it does not train a model. The data-building stages precede the first row.


In [ ]:
recipe = json.loads(DATAFILES["receta/receta.json"])
nodes = {node["id"]: node for node in recipe["nodos"]}
needed, stack = set(), [recipe["objetivos"]["S30"]["nodo"]]
while stack:
    current = stack.pop()
    if current not in needed:
        needed.add(current)
        stack.extend(nodes[current]["requiere"])
kinds = {"variables": "features", "seleccion_v23": "2023 selection", "produccion": "production", "partes": "peril components", "severidad": "severity", "linea_b": "trend-adjusted severity", "mezcla": "blend"}
graph = [{"stage": node["id"], "kind": kinds[node["etapa"]], "requires": ", ".join(node["requiere"]), "source": node["script"]} for node in recipe["nodos"] if node["id"] in needed]
print(f"Stages in the S30 graph: {len(graph)}")
display(pd.DataFrame(graph))


In [ ]:
for rel, text in {**SOURCES, **DATAFILES}.items():
    dest = PACKAGE / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if rel.endswith(".py"): compile(text, str(dest), "exec")
    dest.write_text(text, encoding="utf-8", newline="\n")
work = RUN / "run"
work_dir = work / "work"
work_dir.mkdir(parents=True, exist_ok=True)
config = PACKAGE / "receta/especificaciones/cfg_total.json"
work_config = work_dir / "cfg_total.json"
if work_config.exists() and work_config.read_bytes() != config.read_bytes():
    raise RuntimeError(f"Existing configuration differs: {work_config}")
if not work_config.exists():
    shutil.copyfile(config, work_config)
cmd = [sys.executable, "-X", "utf8", str(PACKAGE / "reproducir.py"), "todo", "--objetivo=S30", f"--semana1={developed.parent}", f"--originales={developed}", f"--trabajo={work}", "--continuar-si-difiere"]
print("Starting S30. Logs and receipts are in", work)
result = subprocess.run(cmd, cwd=RUN, env=dict(os.environ, PYTHONUTF8="1", PYTHONHASHSEED="0", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2"))
if result.returncode not in (0, 3): raise RuntimeError(f"S30 run failed with code {result.returncode}; inspect {work / 'registros'}")
print("Runner return code:", result.returncode)


## Apply the S41 level factors and check the output

The three exact factors below are frozen from the historical credibility rule. I apply them to the newly generated six-decimal S30 CSV, preserving the template order. A hash comparison checks the final file against the historical S41 receipt.


In [ ]:
factors = {'motor': 0.9962490564279887, 'buildings': 1.0259995860825672, 'contents': 1.0126005668944555}
s30 = work / "work/S30_line_b/S30_line_b.csv"
pred = pd.read_csv(s30, dtype={"row_id": str})
line_lookup = pd.concat([pd.read_parquet(developed / f"test_{line}.parquet", columns=["row_id"]).assign(line=line) for line in LINES]).set_index("row_id").line
labels = line_lookup.reindex(pred.row_id).to_numpy()
assert pd.notna(labels).all()
for line, factor in factors.items():
    pred.loc[labels == line, "prediction"] *= factor
expected_ids = pd.read_parquet(developed / "sample_submission.parquet", columns=["row_id"]).row_id.astype(str)
assert list(pred.columns) == ["row_id", "prediction"] and pred.row_id.is_unique and pred.row_id.equals(expected_ids)
assert np.isfinite(pred.prediction).all() and (pred.prediction > 0).all()
out = RUN / "submission_week1_S41_candidate.csv"
pred.to_csv(out, index=False, float_format="%.6f", lineterminator="\n")
h = hashlib.sha256(out.read_bytes()).hexdigest()
expected_sha = "d8661c2874e9112bd3dd62ea4470a465ee309832dee53fd9dcaa4144dac643a5"
print("S41 SHA-256:", h, "matches historical:", h == expected_sha)
if h != expected_sha:
    raise RuntimeError(f"S41 differs from the historical CSV. Preserve and investigate {out} before treating this as an exact reproduction.")
final_output = EXPORT / "submission_week1_S41.csv"
shutil.copyfile(out, final_output)
print("Output:", final_output)


## Outputs and troubleshooting

After the final checks pass, I find `submission_week1_S41.csv` in the `EXPORT` directory printed near the beginning. Without an override, Kaggle exports to a run-specific subdirectory of `/kaggle/working`; local execution exports beneath the new run directory. I can set `IAA_NOTEBOOK_EXPORT` before starting to choose a persistent private destination. I keep row-level outputs private.

| If execution stops | What I check |
| --- | --- |
| Missing or ambiguous input directory | The explicit market path, required filenames and attached competition data |
| Missing library or version mismatch | The optional installation cell, kernel restart and reference environment |
| Failed model stage | The stage's log and receipt under the printed run directory |
| Final fingerprint mismatch | Environment and generated artifacts; I retain the file and investigate the difference |

For Week 1, return code 3 means that the historical final CSV matched while intermediate artifacts differed. I preserve and inspect the difference report. Within the same session the runner can reuse completed nodes after its hash checks; a new notebook session creates a new run folder.

I do not bypass a failed fingerprint check or interpret a resumed run as an uninterrupted execution. A schema-valid CSV and an exact historical file match answer different questions.


## License and materials

My project code and documentation are covered by the MIT License below. Competition data and third-party packages retain their own terms and are not included in that license.

MIT License

Copyright (c) 2026 Manuel Rodríguez

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
